In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# 路径和网格配置
# ============================================================
base_path = "/Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo"

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# 数值设置
# ============================================================
EPS = 1e-16

# True：绘制 log-y
USE_LOG_Y = True

# 如果曲线过密，可以只选择部分参数，例如：
# PLOT_ONLY_N_SUB = [4]
# PLOT_ONLY_ALPHA = [1]
# PLOT_ONLY_ETA = [0.01]
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 辅助函数
# ============================================================
def config_paths(N_sub, alpha_IVP, eta_in):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return casenum, folder_path, npz_file, final_model


def local_euclidean_arclength_weights(phi_internal):
    """
    计算内部 image 的 Euclidean 弧长 quadrature weights：

        w_i^E = 1/2 (
            |phi_i - phi_{i-1}|_2
            +
            |phi_{i+1} - phi_i|_2
        )

    Parameters
    ----------
    phi_internal : ndarray, shape (N, 2)
        Reference path 的内部 images。

    Returns
    -------
    weights : ndarray, shape (N,)
    """
    full_path = np.vstack(
        [
            C1[None, :],
            phi_internal,
            C2[None, :],
        ]
    )

    seg_minus = (
        full_path[1:-1]
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - full_path[1:-1]
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def local_G_arclength_weights(phi_internal, G_all):
    """
    计算内部 image 的 G-弧长 quadrature weights：

        w_i^G = 1/2 (
            |phi_i - phi_{i-1}|_{G_i}
            +
            |phi_{i+1} - phi_i|_{G_i}
        )

    Parameters
    ----------
    phi_internal : ndarray, shape (N, 2)
    G_all : ndarray, shape (N, 2, 2)

    Returns
    -------
    weights : ndarray, shape (N,)
    """
    full_path = np.vstack(
        [
            C1[None, :],
            phi_internal,
            C2[None, :],
        ]
    )

    seg_minus = (
        full_path[1:-1]
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - full_path[1:-1]
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(length_minus_sq, 0.0)
    )

    length_plus = np.sqrt(
        np.maximum(length_plus_sq, 0.0)
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def compute_path_integrals(phi_ref, M_ref, perpG_Mg):
    """
    对一整条 reference path 计算两个离散积分。

    1. 论文中的 Riemannian I_perp：

       I_perp^G
       = int |P^G M gradF|_G^2 |phi'|_G ds

    2. Euclidean 大小和 Euclidean 弧长下的积分：

       I_perp^E
       = int ||P^G M gradF||_2^2 |phi'|_2 ds

    Parameters
    ----------
    phi_ref : ndarray, shape (N, 2)
    M_ref : ndarray, shape (N, 2, 2)
    perpG_Mg : ndarray, shape (N, 2)

    Returns
    -------
    I_perp_G : float
    I_perp_E : float
    """
    phi_ref = np.asarray(
        phi_ref,
        dtype=float,
    )

    M_ref = np.asarray(
        M_ref,
        dtype=float,
    )

    perpG_Mg = np.asarray(
        perpG_Mg,
        dtype=float,
    )

    if phi_ref.shape != (N, 2):
        raise ValueError(
            f"phi_ref shape should be {(N, 2)}, "
            f"got {phi_ref.shape}"
        )

    if M_ref.shape != (N, 2, 2):
        raise ValueError(
            f"M_ref shape should be {(N, 2, 2)}, "
            f"got {M_ref.shape}"
        )

    if perpG_Mg.shape != (N, 2):
        raise ValueError(
            f"perpG_Mg shape should be {(N, 2)}, "
            f"got {perpG_Mg.shape}"
        )

    # G_i = M_i^{-1}
    G_all = np.linalg.inv(M_ref)

    # --------------------------------------------------------
    # 论文中的 G-metric residual：
    #
    # |r_i|_{G_i}^2 = r_i^T G_i r_i
    # --------------------------------------------------------
    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    # G-弧长 quadrature weights
    weights_G = local_G_arclength_weights(
        phi_ref,
        G_all,
    )

    I_perp_G = np.sum(
        residual_G_sq * weights_G
    )

    # --------------------------------------------------------
    # Euclidean residual：
    #
    # ||r_i||_2^2
    # --------------------------------------------------------
    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    # Euclidean 弧长 quadrature weights
    weights_E = local_euclidean_arclength_weights(
        phi_ref
    )

    I_perp_E = np.sum(
        residual_E_sq * weights_E
    )

    return float(I_perp_G), float(I_perp_E)


def selected_config(N_sub, alpha_IVP, eta_in):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 扫描并读取已经跑完的配置
# ============================================================
results = []
skipped_incomplete = []
skipped_missing_data = []

all_configs = [
    (N_sub, alpha_IVP, eta_in)
    for N_sub in GRID_N_SUB
    for alpha_IVP in GRID_ALPHA
    for eta_in in GRID_ETA
]

for N_sub, alpha_IVP, eta_in in all_configs:
    if not selected_config(
        N_sub,
        alpha_IVP,
        eta_in,
    ):
        continue

    batches = TOTAL_INNER_STEPS // N_sub

    (
        casenum,
        folder_path,
        npz_file,
        final_model,
    ) = config_paths(
        N_sub,
        alpha_IVP,
        eta_in,
    )

    # --------------------------------------------------------
    # 只有 final model 和 npz 都存在才认为可能完成
    # --------------------------------------------------------
    if (
        not os.path.exists(final_model)
        or not os.path.exists(npz_file)
    ):
        skipped_incomplete.append(
            casenum
        )
        continue

    try:
        with np.load(
            npz_file,
            allow_pickle=True,
        ) as data:
            if "index_seq" not in data.files:
                skipped_incomplete.append(
                    casenum
                )
                continue

            index_seq_saved = np.asarray(
                data["index_seq"],
                dtype=int,
            )

            if (
                index_seq_saved.size == 0
                or int(np.max(index_seq_saved)) < batches
            ):
                skipped_incomplete.append(
                    casenum
                )
                continue

            count_seq = []
            I_perp_G_seq = []
            I_perp_E_seq = []
            valid_index_seq = []

            missing_required_data = False

            # ------------------------------------------------
            # 对每次 outer reference path 重新计算整条路径积分
            # ------------------------------------------------
            for outer_index in index_seq_saved:
                phi_key = (
                    f"phi_ref_iter_{outer_index}"
                )

                M_key = (
                    f"M_ref_iter_{outer_index}"
                )

                residual_key = (
                    f"perpG_Mg_iter_{outer_index}"
                )

                required_keys = [
                    phi_key,
                    M_key,
                    residual_key,
                ]

                if not all(
                    key in data.files
                    for key in required_keys
                ):
                    missing_required_data = True
                    break

                phi_ref = np.asarray(
                    data[phi_key],
                    dtype=float,
                )

                M_ref = np.asarray(
                    data[M_key],
                    dtype=float,
                )

                perpG_Mg = np.asarray(
                    data[residual_key],
                    dtype=float,
                )

                I_perp_G, I_perp_E = (
                    compute_path_integrals(
                        phi_ref,
                        M_ref,
                        perpG_Mg,
                    )
                )

                # 与原程序 fig_save(i * N_sub, ...) 的计数一致
                current_count = (
                    int(outer_index)
                    * N_sub
                )

                valid_index_seq.append(
                    int(outer_index)
                )

                count_seq.append(
                    current_count
                )

                I_perp_G_seq.append(
                    I_perp_G
                )

                I_perp_E_seq.append(
                    I_perp_E
                )

            if missing_required_data:
                skipped_missing_data.append(
                    casenum
                )
                continue

            count_seq = np.asarray(
                count_seq,
                dtype=int,
            )

            I_perp_G_seq = np.asarray(
                I_perp_G_seq,
                dtype=float,
            )

            I_perp_E_seq = np.asarray(
                I_perp_E_seq,
                dtype=float,
            )

            valid_index_seq = np.asarray(
                valid_index_seq,
                dtype=int,
            )

            # 防止 log 图遇到精确零值
            I_perp_G_plot = np.maximum(
                I_perp_G_seq,
                np.finfo(float).tiny,
            )

            I_perp_E_plot = np.maximum(
                I_perp_E_seq,
                np.finfo(float).tiny,
            )

            results.append(
                {
                    "N_sub": N_sub,
                    "alpha_IVP": alpha_IVP,
                    "eta_in": eta_in,
                    "casenum": casenum,
                    "npz_file": npz_file,
                    "index_seq": valid_index_seq,
                    "count_seq": count_seq,
                    "I_perp_G_seq": I_perp_G_seq,
                    "I_perp_E_seq": I_perp_E_seq,
                    "I_perp_G_plot": I_perp_G_plot,
                    "I_perp_E_plot": I_perp_E_plot,
                }
            )

    except Exception as exc:
        skipped_missing_data.append(
            f"{casenum}: {exc}"
        )


# ============================================================
# 输出读取状态
# ============================================================
print("=" * 90)
print(f"网格配置总数：{len(all_configs)}")
print(f"成功读取完成配置：{len(results)}")
print(f"未跑完或结果文件不存在：{len(skipped_incomplete)}")
print(f"跑完但缺少 M/path/residual 数据：{len(skipped_missing_data)}")
print("=" * 90)

if skipped_incomplete:
    print("\n[跳过：尚未跑完]")
    for name in skipped_incomplete:
        print("  ", name)

if skipped_missing_data:
    print("\n[跳过：缺少重新积分所需的数据]")
    for name in skipped_missing_data:
        print("  ", name)

if len(results) == 0:
    raise RuntimeError(
        "没有找到可以绘图的完整配置。请检查 base_path、"
        "_final.pth、npz，以及 M_ref_iter_i/perpG_Mg_iter_i 是否存在。"
    )


# ============================================================
# 配色和线型
#
# 颜色：alpha
# 线型：eta
# marker：N_sub
# ============================================================
alpha_colors = {
    0.1: "#1f77b4",
    1: "#d62728",
    10: "#2ca02c",
}

eta_linestyles = {
    0.03: "-",
    0.01: "--",
    0.003: "-.",
    0.001: ":",
}

nsub_markers = {
    4: "o",
    5: "s",
    10: "^",
}


# ============================================================
# 绘图
# ============================================================
fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
    sharex=True,
)

ax_G, ax_E = axes

for result in results:
    N_sub = result["N_sub"]
    alpha_IVP = result["alpha_IVP"]
    eta_in = result["eta_in"]

    count_seq = result["count_seq"]

    label = (
        rf"$N_{{sub}}={N_sub}$, "
        rf"$\alpha={alpha_IVP}$, "
        rf"$\eta={eta_in}$"
    )

    color = alpha_colors.get(
        alpha_IVP,
        None,
    )

    linestyle = eta_linestyles.get(
        eta_in,
        "-",
    )

    marker = nsub_markers.get(
        N_sub,
        None,
    )

    # 为避免 marker 太密，只显示有限数量 marker
    markevery = max(
        1,
        len(count_seq) // 10,
    )

    ax_G.plot(
        count_seq,
        result["I_perp_G_plot"],
        color=color,
        linestyle=linestyle,
        marker=marker,
        markevery=markevery,
        markersize=4,
        linewidth=1.6,
        alpha=0.9,
        label=label,
    )

    ax_E.plot(
        count_seq,
        result["I_perp_E_plot"],
        color=color,
        linestyle=linestyle,
        marker=marker,
        markevery=markevery,
        markersize=4,
        linewidth=1.6,
        alpha=0.9,
        label=label,
    )


# ------------------------------------------------------------
# 左图：论文中的 G-metric I_perp
# ------------------------------------------------------------
ax_G.set_title(
    r"Riemannian normal residual",
    fontsize=14,
)

ax_G.set_xlabel(
    "count_seq",
    fontsize=12,
)

ax_G.set_ylabel(
    r"$\widehat I_{\perp}^{G}"
    r"=\int_0^1"
    r"|P_\varphi^G M\nabla F|_G^2"
    r"|\varphi'(s)|_G\,ds$",
    fontsize=12,
)

ax_G.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.3,
)


# ------------------------------------------------------------
# 右图：Euclidean residual 的整条路径积分
# ------------------------------------------------------------
ax_E.set_title(
    r"Euclidean magnitude of the normal residual",
    fontsize=14,
)

ax_E.set_xlabel(
    "count_seq",
    fontsize=12,
)

ax_E.set_ylabel(
    r"$\widehat I_{\perp}^{E}"
    r"=\int_0^1"
    r"\|P_\varphi^G M\nabla F\|_2^2"
    r"|\varphi'(s)|_2\,ds$",
    fontsize=12,
)

ax_E.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.3,
)

if USE_LOG_Y:
    ax_G.set_yscale("log")
    ax_E.set_yscale("log")


# ============================================================
# 分离式图例，避免 36 条完整标签占满图
# ============================================================
alpha_handles = [
    Line2D(
        [0],
        [0],
        color=alpha_colors[alpha],
        linestyle="-",
        linewidth=2,
        label=rf"$\alpha={alpha}$",
    )
    for alpha in GRID_ALPHA
]

eta_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=eta_linestyles[eta],
        linewidth=2,
        label=rf"$\eta={eta}$",
    )
    for eta in GRID_ETA
]

nsub_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="None",
        marker=nsub_markers[N_sub],
        markersize=6,
        label=rf"$N_{{sub}}={N_sub}$",
    )
    for N_sub in GRID_N_SUB
]

legend_alpha = ax_E.legend(
    handles=alpha_handles,
    title=r"Color: $\alpha$",
    loc="upper right",
    bbox_to_anchor=(1.35, 1.0),
    frameon=True,
)

ax_E.add_artist(
    legend_alpha
)

legend_eta = ax_E.legend(
    handles=eta_handles,
    title=r"Line: $\eta$",
    loc="center right",
    bbox_to_anchor=(1.35, 0.55),
    frameon=True,
)

ax_E.add_artist(
    legend_eta
)

ax_E.legend(
    handles=nsub_handles,
    title=r"Marker: $N_{\mathrm{sub}}$",
    loc="lower right",
    bbox_to_anchor=(1.35, 0.0),
    frameon=True,
)


fig.tight_layout(
    rect=[0.0, 0.0, 0.84, 1.0]
)

output_pdf = os.path.join(
    base_path,
    "MFEP_two_integrated_residuals_vs_count.pdf",
)

output_png = os.path.join(
    base_path,
    "MFEP_two_integrated_residuals_vs_count.png",
)

fig.savefig(
    output_pdf,
    bbox_inches="tight",
)

fig.savefig(
    output_png,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("\n图已保存到：")
print(output_pdf)
print(output_png)


# ============================================================
# 同时把重新计算的曲线汇总保存，便于以后直接读取
# ============================================================
integrated_results_file = os.path.join(
    base_path,
    "MFEP_TWO_INTEGRATED_RESIDUALS.npz",
)

np.savez(
    integrated_results_file,
    results=np.array(
        results,
        dtype=object,
    ),
)

print("\n重新积分后的曲线数据已保存到：")
print(integrated_results_file)

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# 基本路径与配置
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# 选择要绘制的 IVM 配置
#
# None 表示绘制所有已经完成的配置。
#
# 例如只画：
#   N_sub=5, alpha=1, eta=0.01
#
# 可以设置：
#
# PLOT_ONLY_N_SUB = [5]
# PLOT_ONLY_ALPHA = [1]
# PLOT_ONLY_ETA = [0.01]
# ============================================================
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 绘图设置
# ============================================================
USE_LOG_Y_RESIDUALS = True

# Delta F 通常使用线性坐标。
# 如果希望使用 log 坐标，可设为 True。
USE_LOG_Y_DELTA_F = False

# NEB 曲线样式
NEB_COLOR = "black"
NEB_LINESTYLE = "-"
NEB_MARKER = "D"
NEB_LINEWIDTH = 3.0

# 保存输出
SAVE_OUTPUT = True


# ============================================================
# IVM 配置路径
# ============================================================
def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 中心差分 tangent
#
# full_path: (N+2, 2)
# return:    (N, 2)
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


# ============================================================
# 构造 G-正交投影
#
# G_i = M_i^{-1}
#
# P_parallel^G
# = tau_G (G tau_G)^T
#
# P_perp^G
# = I - P_parallel^G
# ============================================================
def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    if tangent.shape != (N, 2):
        raise ValueError(
            f"tangent should have shape {(N, 2)}, "
            f"got {tangent.shape}"
        )

    if M_all.shape != (N, 2, 2):
        raise ValueError(
            f"M_all should have shape {(N, 2, 2)}, "
            f"got {M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (N, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (N, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(N):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G-tangent norm at image {i}: "
                f"{norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = G_i @ tau_G

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


# ============================================================
# G-弧长 quadrature weights
#
# w_i^G = 1/2 [
#     |phi_i - phi_{i-1}|_{G_i}
#     +
#     |phi_{i+1} - phi_i|_{G_i}
# ]
# ============================================================
def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = full_path[1:-1]

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# Euclidean 弧长 quadrature weights
#
# w_i^E = 1/2 [
#     ||phi_i - phi_{i-1}||_2
#     +
#     ||phi_{i+1} - phi_i||_2
# ]
# ============================================================
def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = full_path[1:-1]

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# 路径积分重建自由能
#
# 已知内部 image 上的梯度：
#
#     F_internal[i] = nabla F(phi_i)
#
# 端点没有梯度数据，因此采用：
#
# 第一段：
#     grad_segment = F_internal[0]
#
# 内部段：
#     grad_segment
#     = 1/2 (F_internal[i-1] + F_internal[i])
#
# 最后一段：
#     grad_segment = F_internal[-1]
#
# 然后：
#
#     Delta F_j
#     = sum_m grad_segment_m dot
#       (phi_{m+1} - phi_m)
#
# 返回：
#     profile[0] = 0
#     Delta F barrier = max(profile) - profile[0]
# ============================================================
def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    if full_path.shape != (N + 2, 2):
        raise ValueError(
            f"full_path should have shape {(N + 2, 2)}, "
            f"got {full_path.shape}"
        )

    if F_internal.shape != (N, 2):
        raise ValueError(
            f"F_internal should have shape {(N, 2)}, "
            f"got {F_internal.shape}"
        )

    # N+1 个相邻路径段
    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    # N+1 个 segment gradient
    segment_gradients = np.zeros(
        (N + 1, 2),
        dtype=float,
    )

    # 起点到第一个内部 image
    segment_gradients[0] = (
        F_internal[0]
    )

    # 内部 image 之间使用梯形平均
    if N > 1:
        segment_gradients[1:N] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    # 最后一个内部 image 到终点
    segment_gradients[N] = (
        F_internal[-1]
    )

    # 每一段的自由能增量
    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        N + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    # 以初始端点自由能为零
    profile = (
        profile
        - profile[0]
    )

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


# ============================================================
# 对一整条路径重新计算三个指标
#
# 使用原始、未截断的 M 和 grad F。
# ============================================================
def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    if full_path.shape != (N + 2, 2):
        raise ValueError(
            f"full_path should have shape {(N + 2, 2)}, "
            f"got {full_path.shape}"
        )

    if M_all.shape != (N, 2, 2):
        raise ValueError(
            f"M_all should have shape {(N, 2, 2)}, "
            f"got {M_all.shape}"
        )

    if F_all.shape != (N, 2):
        raise ValueError(
            f"F_all should have shape {(N, 2)}, "
            f"got {F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    # 原始 Riemannian gradient：
    # M nabla F
    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    # G-normal component
    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    # --------------------------------------------------------
    # 论文中的 G-metric residual
    # --------------------------------------------------------
    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    # --------------------------------------------------------
    # Euclidean residual integral
    # --------------------------------------------------------
    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    # --------------------------------------------------------
    # 路径积分自由能
    # --------------------------------------------------------
    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,
        "Delta_F_barrier": (
            free_energy["barrier"]
        ),
        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),
        "free_energy_profile": (
            free_energy["profile"]
        ),
        "free_energy_max_index": (
            free_energy["max_index"]
        ),
        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,
    }


# ============================================================
# 去除重复 count
#
# 如果同一个 count 保存了多个状态，保留最后一个。
# ============================================================
def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(count_seq):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda idx: count_seq[idx],
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(array)
        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 扫描并读取已经完成的 IVM 配置
# ============================================================
ivm_results = []

skipped_ivm_incomplete = []
skipped_ivm_missing_data = []

all_configs = [
    (N_sub, alpha_IVP, eta_in)
    for N_sub in GRID_N_SUB
    for alpha_IVP in GRID_ALPHA
    for eta_in in GRID_ETA
]

for (
    N_sub,
    alpha_IVP,
    eta_in,
) in all_configs:
    if not selected_config(
        N_sub,
        alpha_IVP,
        eta_in,
    ):
        continue

    batches = (
        TOTAL_INNER_STEPS
        // N_sub
    )

    (
        casenum,
        folder_path,
        npz_file,
        final_model,
    ) = ivm_config_paths(
        N_sub,
        alpha_IVP,
        eta_in,
    )

    # 必须有 final model 和 npz
    if (
        not os.path.exists(final_model)
        or not os.path.exists(npz_file)
    ):
        skipped_ivm_incomplete.append(
            casenum
        )
        continue

    try:
        with np.load(
            npz_file,
            allow_pickle=True,
        ) as data:
            if "index_seq" not in data.files:
                skipped_ivm_incomplete.append(
                    casenum
                )
                continue

            index_seq_saved = np.asarray(
                data["index_seq"],
                dtype=int,
            )

            if (
                index_seq_saved.size == 0
                or int(
                    np.max(index_seq_saved)
                ) < batches
            ):
                skipped_ivm_incomplete.append(
                    casenum
                )
                continue

            count_seq = []
            I_perp_G_seq = []
            I_perp_E_seq = []
            Delta_F_seq = []
            Delta_F_final_seq = []
            free_energy_profiles = []
            valid_index_seq = []

            missing_required_data = False

            for outer_index in index_seq_saved:
                phi_key = (
                    f"phi_ref_iter_"
                    f"{outer_index}"
                )

                M_key = (
                    f"M_ref_iter_"
                    f"{outer_index}"
                )

                F_key = (
                    f"F_ref_iter_"
                    f"{outer_index}"
                )

                # 兼容只有 g_ref_iter_i 的文件
                g_key = (
                    f"g_ref_iter_"
                    f"{outer_index}"
                )

                if F_key in data.files:
                    selected_F_key = F_key
                elif g_key in data.files:
                    selected_F_key = g_key
                else:
                    selected_F_key = None

                required_present = (
                    phi_key in data.files
                    and M_key in data.files
                    and selected_F_key is not None
                )

                if not required_present:
                    missing_required_data = True
                    break

                phi_internal = np.asarray(
                    data[phi_key],
                    dtype=float,
                )

                M_all = np.asarray(
                    data[M_key],
                    dtype=float,
                )

                F_all = np.asarray(
                    data[selected_F_key],
                    dtype=float,
                )

                full_path = np.vstack(
                    [
                        C1[None, :],
                        phi_internal,
                        C2[None, :],
                    ]
                )

                metrics = compute_all_path_metrics(
                    full_path,
                    M_all,
                    F_all,
                )

                current_count = (
                    int(outer_index)
                    * N_sub
                )

                valid_index_seq.append(
                    int(outer_index)
                )

                count_seq.append(
                    current_count
                )

                I_perp_G_seq.append(
                    metrics["I_perp_G"]
                )

                I_perp_E_seq.append(
                    metrics["I_perp_E"]
                )

                Delta_F_seq.append(
                    metrics[
                        "Delta_F_barrier"
                    ]
                )

                Delta_F_final_seq.append(
                    metrics[
                        "Delta_F_final"
                    ]
                )

                free_energy_profiles.append(
                    metrics[
                        "free_energy_profile"
                    ]
                )

            if missing_required_data:
                skipped_ivm_missing_data.append(
                    casenum
                )
                continue

            (
                count_seq,
                I_perp_G_seq,
                I_perp_E_seq,
                Delta_F_seq,
                Delta_F_final_seq,
                valid_index_seq,
                free_energy_profiles,
            ) = keep_last_for_duplicate_counts(
                np.asarray(
                    count_seq,
                    dtype=int,
                ),
                np.asarray(
                    I_perp_G_seq,
                    dtype=float,
                ),
                np.asarray(
                    I_perp_E_seq,
                    dtype=float,
                ),
                np.asarray(
                    Delta_F_seq,
                    dtype=float,
                ),
                np.asarray(
                    Delta_F_final_seq,
                    dtype=float,
                ),
                np.asarray(
                    valid_index_seq,
                    dtype=int,
                ),
                np.asarray(
                    free_energy_profiles,
                    dtype=float,
                ),
            )

            ivm_results.append(
                {
                    "method": "IVM",
                    "N_sub": N_sub,
                    "alpha_IVP": alpha_IVP,
                    "eta_in": eta_in,
                    "casenum": casenum,
                    "npz_file": npz_file,
                    "index_seq": (
                        valid_index_seq
                    ),
                    "count_seq": (
                        count_seq
                    ),
                    "I_perp_G_seq": (
                        I_perp_G_seq
                    ),
                    "I_perp_E_seq": (
                        I_perp_E_seq
                    ),
                    "Delta_F_seq": (
                        Delta_F_seq
                    ),
                    "Delta_F_final_seq": (
                        Delta_F_final_seq
                    ),
                    "free_energy_profiles": (
                        free_energy_profiles
                    ),
                }
            )

    except Exception as exc:
        skipped_ivm_missing_data.append(
            f"{casenum}: {exc}"
        )


# ============================================================
# 查找并读取已经完成的 NEB-ODE 结果
#
# 优先匹配 forceCap500。
# ============================================================
preferred_neb_folder = os.path.join(
    base_path,
    (
        f"MFEP_ASE_NEB_ODE_N{N}"
        f"_kDefault"
        f"_forceCap500"
        f"_centralImprovedTangent"
    ),
)

preferred_neb_npz = os.path.join(
    preferred_neb_folder,
    (
        f"alanine_MFEP_ASE_NEB_ODE_N{N}"
        f"_kDefault"
        f"_forceCap500"
        f"_centralImprovedTangent.npz"
    ),
)

preferred_neb_done = os.path.join(
    preferred_neb_folder,
    "DONE",
)

neb_candidates = []

if (
    os.path.exists(preferred_neb_npz)
    and os.path.exists(preferred_neb_done)
):
    neb_candidates.append(
        preferred_neb_npz
    )
else:
    glob_pattern = os.path.join(
        base_path,
        (
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_kDefault"
            f"_forceCap*"
            f"_centralImprovedTangent"
        ),
        "alanine_*.npz",
    )

    for candidate in sorted(
        glob.glob(glob_pattern)
    ):
        candidate_folder = os.path.dirname(
            candidate
        )

        candidate_done = os.path.join(
            candidate_folder,
            "DONE",
        )

        if os.path.exists(candidate_done):
            neb_candidates.append(
                candidate
            )

neb_result = None
neb_read_error = None

if neb_candidates:
    # 如果有多个，优先使用最后修改的结果
    neb_npz_file = max(
        neb_candidates,
        key=os.path.getmtime,
    )

    try:
        with np.load(
            neb_npz_file,
            allow_pickle=True,
        ) as data:
            required_neb_keys = [
                "count_seq",
                "images_seq",
                "M_ref_seq",
                "F_ref_seq",
            ]

            missing_neb_keys = [
                key
                for key in required_neb_keys
                if key not in data.files
            ]

            if missing_neb_keys:
                raise KeyError(
                    f"NEB file missing keys: "
                    f"{missing_neb_keys}"
                )

            neb_count_seq_raw = np.asarray(
                data["count_seq"],
                dtype=float,
            )

            neb_images_seq = np.asarray(
                data["images_seq"],
                dtype=float,
            )

            neb_M_ref_seq = np.asarray(
                data["M_ref_seq"],
                dtype=float,
            )

            neb_F_ref_seq = np.asarray(
                data["F_ref_seq"],
                dtype=float,
            )

            n_neb_states = min(
                len(neb_count_seq_raw),
                len(neb_images_seq),
                len(neb_M_ref_seq),
                len(neb_F_ref_seq),
            )

            neb_count_seq_raw = (
                neb_count_seq_raw[
                    :n_neb_states
                ]
            )

            neb_images_seq = (
                neb_images_seq[
                    :n_neb_states
                ]
            )

            neb_M_ref_seq = (
                neb_M_ref_seq[
                    :n_neb_states
                ]
            )

            neb_F_ref_seq = (
                neb_F_ref_seq[
                    :n_neb_states
                ]
            )

            neb_I_perp_G_seq = []
            neb_I_perp_E_seq = []
            neb_Delta_F_seq = []
            neb_Delta_F_final_seq = []
            neb_free_energy_profiles = []

            for state_index in range(
                n_neb_states
            ):
                full_path = np.asarray(
                    neb_images_seq[
                        state_index
                    ],
                    dtype=float,
                )

                M_all = np.asarray(
                    neb_M_ref_seq[
                        state_index
                    ],
                    dtype=float,
                )

                F_all = np.asarray(
                    neb_F_ref_seq[
                        state_index
                    ],
                    dtype=float,
                )

                metrics = compute_all_path_metrics(
                    full_path,
                    M_all,
                    F_all,
                )

                neb_I_perp_G_seq.append(
                    metrics["I_perp_G"]
                )

                neb_I_perp_E_seq.append(
                    metrics["I_perp_E"]
                )

                neb_Delta_F_seq.append(
                    metrics[
                        "Delta_F_barrier"
                    ]
                )

                neb_Delta_F_final_seq.append(
                    metrics[
                        "Delta_F_final"
                    ]
                )

                neb_free_energy_profiles.append(
                    metrics[
                        "free_energy_profile"
                    ]
                )

            (
                neb_count_seq,
                neb_I_perp_G_seq,
                neb_I_perp_E_seq,
                neb_Delta_F_seq,
                neb_Delta_F_final_seq,
                neb_free_energy_profiles,
            ) = keep_last_for_duplicate_counts(
                neb_count_seq_raw,
                np.asarray(
                    neb_I_perp_G_seq,
                    dtype=float,
                ),
                np.asarray(
                    neb_I_perp_E_seq,
                    dtype=float,
                ),
                np.asarray(
                    neb_Delta_F_seq,
                    dtype=float,
                ),
                np.asarray(
                    neb_Delta_F_final_seq,
                    dtype=float,
                ),
                np.asarray(
                    neb_free_energy_profiles,
                    dtype=float,
                ),
            )

            if "force_cap" in data.files:
                neb_force_cap = float(
                    np.asarray(
                        data["force_cap"]
                    ).reshape(-1)[0]
                )
            else:
                neb_force_cap = np.nan

            neb_result = {
                "method": "NEB-ODE",
                "npz_file": neb_npz_file,
                "force_cap": neb_force_cap,
                "count_seq": neb_count_seq,
                "I_perp_G_seq": (
                    neb_I_perp_G_seq
                ),
                "I_perp_E_seq": (
                    neb_I_perp_E_seq
                ),
                "Delta_F_seq": (
                    neb_Delta_F_seq
                ),
                "Delta_F_final_seq": (
                    neb_Delta_F_final_seq
                ),
                "free_energy_profiles": (
                    neb_free_energy_profiles
                ),
            }

    except Exception as exc:
        neb_read_error = str(exc)


# ============================================================
# 输出读取状态
# ============================================================
print("=" * 95)
print(
    f"IVM 网格配置总数："
    f"{len(all_configs)}"
)
print(
    f"成功读取完成的 IVM 配置："
    f"{len(ivm_results)}"
)
print(
    f"未跑完的 IVM 配置："
    f"{len(skipped_ivm_incomplete)}"
)
print(
    f"缺少路径/M/F 数据的 IVM 配置："
    f"{len(skipped_ivm_missing_data)}"
)
print(
    f"成功读取 NEB-ODE："
    f"{neb_result is not None}"
)
print("=" * 95)

if skipped_ivm_incomplete:
    print("\n[跳过：IVM 尚未跑完]")
    for name in skipped_ivm_incomplete:
        print("  ", name)

if skipped_ivm_missing_data:
    print(
        "\n[跳过：IVM 缺少 path/M/F 数据]"
    )
    for name in skipped_ivm_missing_data:
        print("  ", name)

if neb_result is not None:
    print("\n[NEB-ODE result]")
    print(
        "  file      :",
        neb_result["npz_file"],
    )
    print(
        "  force cap :",
        neb_result["force_cap"],
    )
    print(
        "  states    :",
        len(neb_result["count_seq"]),
    )
    print(
        "  count range:",
        (
            float(
                neb_result[
                    "count_seq"
                ][0]
            ),
            float(
                neb_result[
                    "count_seq"
                ][-1]
            ),
        ),
    )
else:
    print("\n[WARNING] 未读取到 NEB-ODE 结果。")
    if neb_read_error is not None:
        print(
            "NEB 读取错误：",
            neb_read_error,
        )

if (
    len(ivm_results) == 0
    and neb_result is None
):
    raise RuntimeError(
        "没有找到任何可以绘图的 IVM 或 NEB 结果。"
    )


# ============================================================
# 为 log-y 准备数据
# ============================================================
tiny = np.finfo(float).tiny

for result in ivm_results:
    result["I_perp_G_plot"] = np.maximum(
        result["I_perp_G_seq"],
        tiny,
    )

    result["I_perp_E_plot"] = np.maximum(
        result["I_perp_E_seq"],
        tiny,
    )

    result["Delta_F_plot"] = np.maximum(
        result["Delta_F_seq"],
        tiny,
    )

if neb_result is not None:
    neb_result["I_perp_G_plot"] = (
        np.maximum(
            neb_result["I_perp_G_seq"],
            tiny,
        )
    )

    neb_result["I_perp_E_plot"] = (
        np.maximum(
            neb_result["I_perp_E_seq"],
            tiny,
        )
    )

    neb_result["Delta_F_plot"] = (
        np.maximum(
            neb_result["Delta_F_seq"],
            tiny,
        )
    )


# ============================================================
# IVM 配色和线型
#
# 颜色：alpha
# 线型：eta
# marker：N_sub
# ============================================================
alpha_colors = {
    0.1: "#1f77b4",
    1: "#d62728",
    10: "#2ca02c",
}

eta_linestyles = {
    0.03: "-",
    0.01: "--",
    0.003: "-.",
    0.001: ":",
}

nsub_markers = {
    4: "o",
    5: "s",
    10: "^",
}


# ============================================================
# 绘图：三个指标
# ============================================================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(22, 6.5),
    sharex=False,
)

ax_G, ax_E, ax_DF = axes


# ------------------------------------------------------------
# IVM 曲线
# ------------------------------------------------------------
for result in ivm_results:
    N_sub = result["N_sub"]
    alpha_IVP = result["alpha_IVP"]
    eta_in = result["eta_in"]

    count_seq = result["count_seq"]

    color = alpha_colors.get(
        alpha_IVP,
        None,
    )

    linestyle = eta_linestyles.get(
        eta_in,
        "-",
    )

    marker = nsub_markers.get(
        N_sub,
        "o",
    )

    markevery = max(
        1,
        len(count_seq) // 10,
    )

    label = (
        rf"IVM: $N_{{sub}}={N_sub}$, "
        rf"$\alpha={alpha_IVP}$, "
        rf"$\eta={eta_in}$"
    )

    common_kwargs = dict(
        color=color,
        linestyle=linestyle,
        marker=marker,
        markevery=markevery,
        markersize=4,
        linewidth=1.4,
        alpha=0.75,
        label=label,
    )

    ax_G.plot(
        count_seq,
        result["I_perp_G_plot"],
        **common_kwargs,
    )

    ax_E.plot(
        count_seq,
        result["I_perp_E_plot"],
        **common_kwargs,
    )

    ax_DF.plot(
        count_seq,
        result["Delta_F_plot"],
        **common_kwargs,
    )


# ------------------------------------------------------------
# NEB-ODE 曲线
# ------------------------------------------------------------
if neb_result is not None:
    neb_count_seq = (
        neb_result["count_seq"]
    )

    neb_markevery = max(
        1,
        len(neb_count_seq) // 12,
    )

    if np.isfinite(
        neb_result["force_cap"]
    ):
        neb_label = (
            rf"NEB-ODE "
            rf"(force cap="
            rf"{neb_result['force_cap']:g})"
        )
    else:
        neb_label = "NEB-ODE"

    neb_common_kwargs = dict(
        color=NEB_COLOR,
        linestyle=NEB_LINESTYLE,
        marker=NEB_MARKER,
        markevery=neb_markevery,
        markersize=5.5,
        markerfacecolor="white",
        markeredgecolor=NEB_COLOR,
        markeredgewidth=1.2,
        linewidth=NEB_LINEWIDTH,
        alpha=1.0,
        label=neb_label,
        zorder=20,
    )

    ax_G.plot(
        neb_count_seq,
        neb_result["I_perp_G_plot"],
        **neb_common_kwargs,
    )

    ax_E.plot(
        neb_count_seq,
        neb_result["I_perp_E_plot"],
        **neb_common_kwargs,
    )

    ax_DF.plot(
        neb_count_seq,
        neb_result["Delta_F_plot"],
        **neb_common_kwargs,
    )


# ============================================================
# Panel 1: G-metric I_perp
# ============================================================
ax_G.set_title(
    r"Riemannian normal residual",
    fontsize=14,
)

ax_G.set_xlabel(
    "count_seq",
    fontsize=12,
)

ax_G.set_ylabel(
    r"$\widehat I_{\perp}^{G}$"
    "\n"
    r"$=\int_0^1"
    r"|P_\varphi^G M\nabla F|_G^2"
    r"|\varphi'(s)|_G\,ds$",
    fontsize=11,
)

ax_G.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.3,
)


# ============================================================
# Panel 2: Euclidean residual integral
# ============================================================
ax_E.set_title(
    r"Euclidean residual integral",
    fontsize=14,
)

ax_E.set_xlabel(
    "count_seq",
    fontsize=12,
)

ax_E.set_ylabel(
    r"$\widehat I_{\perp}^{E}$"
    "\n"
    r"$=\int_0^1"
    r"\|P_\varphi^G M\nabla F\|_2^2"
    r"|\varphi'(s)|_2\,ds$",
    fontsize=11,
)

ax_E.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.3,
)


# ============================================================
# Panel 3: Delta F barrier
# ============================================================
ax_DF.set_title(
    r"Path-integrated free-energy barrier",
    fontsize=14,
)

ax_DF.set_xlabel(
    "count_seq",
    fontsize=12,
)

ax_DF.set_ylabel(
    r"$\Delta F^{\dagger}"
    r"=\max_s\left[F(\varphi(s))-F(\varphi(0))\right]$"
    "\n"
    r"(kcal/mol, line-integral estimate)",
    fontsize=11,
)

ax_DF.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.3,
)


# ============================================================
# 坐标尺度
# ============================================================
if USE_LOG_Y_RESIDUALS:
    ax_G.set_yscale("log")
    ax_E.set_yscale("log")

if USE_LOG_Y_DELTA_F:
    ax_DF.set_yscale("log")


# ============================================================
# 分离式图例
# ============================================================
alpha_handles = [
    Line2D(
        [0],
        [0],
        color=alpha_colors[alpha],
        linestyle="-",
        linewidth=2,
        label=rf"IVM $\alpha={alpha}$",
    )
    for alpha in GRID_ALPHA
]

eta_handles = [
    Line2D(
        [0],
        [0],
        color="gray",
        linestyle=eta_linestyles[eta],
        linewidth=2,
        label=rf"IVM $\eta={eta}$",
    )
    for eta in GRID_ETA
]

nsub_handles = [
    Line2D(
        [0],
        [0],
        color="gray",
        linestyle="None",
        marker=nsub_markers[N_sub],
        markersize=6,
        label=rf"IVM $N_{{sub}}={N_sub}$",
    )
    for N_sub in GRID_N_SUB
]

method_handles = [
    Line2D(
        [0],
        [0],
        color="gray",
        linestyle="-",
        linewidth=1.5,
        alpha=0.75,
        label="IVM",
    )
]

if neb_result is not None:
    method_handles.append(
        Line2D(
            [0],
            [0],
            color=NEB_COLOR,
            linestyle=NEB_LINESTYLE,
            marker=NEB_MARKER,
            markerfacecolor="white",
            linewidth=NEB_LINEWIDTH,
            markersize=6,
            label="NEB-ODE",
        )
    )

legend_method = ax_DF.legend(
    handles=method_handles,
    title="Method",
    loc="upper left",
    bbox_to_anchor=(1.03, 1.0),
    frameon=True,
)

ax_DF.add_artist(
    legend_method
)

legend_alpha = ax_DF.legend(
    handles=alpha_handles,
    title=r"IVM color: $\alpha$",
    loc="upper left",
    bbox_to_anchor=(1.03, 0.78),
    frameon=True,
)

ax_DF.add_artist(
    legend_alpha
)

legend_eta = ax_DF.legend(
    handles=eta_handles,
    title=r"IVM line: $\eta$",
    loc="upper left",
    bbox_to_anchor=(1.03, 0.47),
    frameon=True,
)

ax_DF.add_artist(
    legend_eta
)

ax_DF.legend(
    handles=nsub_handles,
    title=r"IVM marker: $N_{\rm sub}$",
    loc="upper left",
    bbox_to_anchor=(1.03, 0.12),
    frameon=True,
)


# ============================================================
# 保存和显示
# ============================================================
fig.tight_layout(
    rect=[0.0, 0.0, 0.84, 1.0]
)

output_pdf = os.path.join(
    base_path,
    "MFEP_IVM_vs_NEB_residuals_and_DeltaF.pdf",
)

output_png = os.path.join(
    base_path,
    "MFEP_IVM_vs_NEB_residuals_and_DeltaF.png",
)

if SAVE_OUTPUT:
    fig.savefig(
        output_pdf,
        bbox_inches="tight",
    )

    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

if SAVE_OUTPUT:
    print("\n对比图已保存到：")
    print(output_pdf)
    print(output_png)


# ============================================================
# 打印最后状态的数值汇总
# ============================================================
print("\n" + "=" * 120)
print("FINAL VALUE SUMMARY")
print("=" * 120)

summary_rows = []

for result in ivm_results:
    summary_rows.append(
        {
            "method": "IVM",
            "N_sub": result["N_sub"],
            "alpha": result["alpha_IVP"],
            "eta": result["eta_in"],
            "final_count": float(
                result["count_seq"][-1]
            ),
            "final_I_perp_G": float(
                result[
                    "I_perp_G_seq"
                ][-1]
            ),
            "final_I_perp_E": float(
                result[
                    "I_perp_E_seq"
                ][-1]
            ),
            "final_Delta_F": float(
                result[
                    "Delta_F_seq"
                ][-1]
            ),
            "final_endpoint_Delta_F": float(
                result[
                    "Delta_F_final_seq"
                ][-1]
            ),
        }
    )

if neb_result is not None:
    summary_rows.append(
        {
            "method": "NEB-ODE",
            "N_sub": None,
            "alpha": None,
            "eta": None,
            "final_count": float(
                neb_result[
                    "count_seq"
                ][-1]
            ),
            "final_I_perp_G": float(
                neb_result[
                    "I_perp_G_seq"
                ][-1]
            ),
            "final_I_perp_E": float(
                neb_result[
                    "I_perp_E_seq"
                ][-1]
            ),
            "final_Delta_F": float(
                neb_result[
                    "Delta_F_seq"
                ][-1]
            ),
            "final_endpoint_Delta_F": float(
                neb_result[
                    "Delta_F_final_seq"
                ][-1]
            ),
        }
    )

summary_rows_sorted = sorted(
    summary_rows,
    key=lambda row: (
        row["final_I_perp_G"]
    ),
)

for row in summary_rows_sorted:
    if row["method"] == "IVM":
        method_text = (
            f"IVM "
            f"Nsub={row['N_sub']:>2} "
            f"alpha={row['alpha']:<4} "
            f"eta={row['eta']:<6}"
        )
    else:
        method_text = (
            "NEB-ODE"
        )

    print(
        f"{method_text:<43}"
        f" count={row['final_count']:>7.1f}"
        f" | I_G={row['final_I_perp_G']:.6e}"
        f" | I_E={row['final_I_perp_E']:.6e}"
        f" | DeltaF_barrier="
        f"{row['final_Delta_F']:.6f} kcal/mol"
        f" | DeltaF_end="
        f"{row['final_endpoint_Delta_F']:.6f} kcal/mol"
    )


# ============================================================
# 保存重新计算后的全部曲线
# ============================================================
comparison_results_file = os.path.join(
    base_path,
    "MFEP_IVM_VS_NEB_THREE_METRICS.npz",
)

np.savez(
    comparison_results_file,
    ivm_results=np.array(
        ivm_results,
        dtype=object,
    ),
    neb_result=np.array(
        [neb_result],
        dtype=object,
    ),
    summary=np.array(
        summary_rows,
        dtype=object,
    ),
)

print("\n重新计算后的对比数据已保存到：")
print(comparison_results_file)

In [ ]:
import os
import glob
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# Optimal-IVM 路径
#
# 与 adaptive IVM 脚本中的 casenum 一致
# ============================================================
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)

OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# 选择要绘制的普通 grid IVM 配置
#
# None 表示全部完成的配置。
#
# 如果只想画部分：
#
# PLOT_ONLY_N_SUB = [4]
# PLOT_ONLY_ALPHA = [0.1]
# PLOT_ONLY_ETA = [0.03]
# ============================================================
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 自动刷新设置
#
# AUTO_REFRESH=True:
#   每 REFRESH_SECONDS 秒重新读取 Optimal-IVM 的当前 npz。
#
# 当 Optimal-IVM 的 count 到达 200 后自动停止。
#
# 若只想读取一次：
#   AUTO_REFRESH=False
# ============================================================
AUTO_REFRESH = True
REFRESH_SECONDS = 60

# 如果 Optimal-IVM 尚未生成 npz，是否继续等待
WAIT_FOR_OPTIMAL_FILE = True


# ============================================================
# 绘图设置
# ============================================================
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False

SAVE_OUTPUT = True

# 普通 IVM 曲线透明度
GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

# Optimal-IVM 曲线
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

# NEB 曲线
NEB_COLOR = "black"
NEB_LINESTYLE = "-"
NEB_MARKER = "D"
NEB_LINEWIDTH = 3.0


# ============================================================
# 安全读取正在写入的 npz
#
# np.savez 写文件期间，Notebook 恰好读取可能出现：
#   BadZipFile
#   EOFError
#
# 因此这里自动重试。
# ============================================================
def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                # 将数据复制到内存，关闭文件后仍可访问
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        f"[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# 普通 grid IVM 配置路径
# ============================================================
def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 中心差分 tangent
# ============================================================
def central_difference_tangent(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


# ============================================================
# 构造 G-正交投影
#
# G = M^{-1}
# ============================================================
def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


# ============================================================
# G-弧长 quadrature weights
# ============================================================
def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# Euclidean 弧长 quadrature weights
# ============================================================
def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# 路径积分重建自由能
#
# 端点没有梯度，第一段和最后一段分别使用最近 image 梯度。
# 内部段使用梯形平均。
# ============================================================
def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            f"is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    # 起点到第一个内部 image
    segment_gradients[0] = (
        F_internal[0]
    )

    # 内部 image 之间
    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    # 最后一个内部 image 到终点
    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


# ============================================================
# 对整条路径重新计算三个指标
#
# 1. G-metric I_perp
# 2. Euclidean integral I_perp
# 3. Delta F barrier
# ============================================================
def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            f"Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            f"Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    # M grad F
    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    # P_G M grad F
    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    # --------------------------------------------------------
    # 论文中的 G-metric I_perp
    # --------------------------------------------------------
    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    # --------------------------------------------------------
    # Euclidean residual integral
    # --------------------------------------------------------
    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    # --------------------------------------------------------
    # 路径积分自由能
    # --------------------------------------------------------
    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,
        "Delta_F_barrier": (
            free_energy["barrier"]
        ),
        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),
        "free_energy_profile": (
            free_energy["profile"]
        ),
        "free_energy_max_index": (
            free_energy["max_index"]
        ),
        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后一个状态
# ============================================================
def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取可用的逐 outer 数据
#
# require_complete=True:
#   普通 grid IVM 必须完成。
#
# require_complete=False:
#   Optimal-IVM 有多少读多少。
# ============================================================
def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            f"final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                f"incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                f"expected at least "
                f"{expected_batches}"
            )

    # --------------------------------------------------------
    # 优先使用保存的 count_seq
    # --------------------------------------------------------
    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        # 对 Optimal-IVM：
        # 如果正在写入最后一个 outer，而数据尚不完整，
        # 只跳过该 outer，不丢弃前面已经完成的数据。
        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

    if len(count_seq) == 0:
        return None, (
            f"no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": (
            valid_index_seq
        ),

        "count_seq": (
            count_seq
        ),

        "I_perp_G_seq": (
            I_perp_G_seq
        ),

        "I_perp_E_seq": (
            I_perp_E_seq
        ),

        "Delta_F_seq": (
            Delta_F_seq
        ),

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    # Optimal-IVM 的学习率 schedule
    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取所有已经完成的普通 grid IVM
# ============================================================
def load_completed_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (N_sub, alpha_IVP, eta_in)
        for N_sub in GRID_N_SUB
        for alpha_IVP in GRID_ALPHA
        for eta_in in GRID_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        if not selected_config(
            N_sub,
            alpha_IVP,
            eta_in,
        ):
            continue

        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=True,
            final_model=final_model,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
        all_configs,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
#
# 不要求 final model，不要求 count 到 200。
# ============================================================
def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive 0.04->0.001"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# 查找并读取 NEB-ODE
# ============================================================
def load_neb_result():
    preferred_neb_folder = os.path.join(
        base_path,
        (
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_kDefault"
            f"_forceCap500"
            f"_centralImprovedTangent"
        ),
    )

    preferred_neb_npz = os.path.join(
        preferred_neb_folder,
        (
            f"alanine_"
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_kDefault"
            f"_forceCap500"
            f"_centralImprovedTangent.npz"
        ),
    )

    neb_candidates = []

    if os.path.exists(
        preferred_neb_npz
    ):
        neb_candidates.append(
            preferred_neb_npz
        )
    else:
        pattern = os.path.join(
            base_path,
            (
                f"MFEP_ASE_NEB_ODE_N{N}"
                f"_kDefault"
                f"_forceCap*"
                f"_centralImprovedTangent"
            ),
            "alanine_*.npz",
        )

        neb_candidates.extend(
            glob.glob(pattern)
        )

    if not neb_candidates:
        return None, (
            "No NEB result found."
        )

    neb_npz_file = max(
        neb_candidates,
        key=os.path.getmtime,
    )

    data = safe_load_npz(
        neb_npz_file
    )

    if data is None:
        return None, (
            f"Could not read {neb_npz_file}"
        )

    required_keys = [
        "count_seq",
        "images_seq",
        "M_ref_seq",
        "F_ref_seq",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in data
    ]

    if missing_keys:
        return None, (
            f"NEB missing keys: "
            f"{missing_keys}"
        )

    count_raw = np.asarray(
        data["count_seq"],
        dtype=float,
    )

    images_seq = np.asarray(
        data["images_seq"],
        dtype=float,
    )

    M_ref_seq = np.asarray(
        data["M_ref_seq"],
        dtype=float,
    )

    F_ref_seq = np.asarray(
        data["F_ref_seq"],
        dtype=float,
    )

    n_states = min(
        len(count_raw),
        len(images_seq),
        len(M_ref_seq),
        len(F_ref_seq),
    )

    count_raw = count_raw[:n_states]
    images_seq = images_seq[:n_states]
    M_ref_seq = M_ref_seq[:n_states]
    F_ref_seq = F_ref_seq[:n_states]

    valid_count = []

    I_perp_G_seq = []
    I_perp_E_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []

    for state_index in range(
        n_states
    ):
        try:
            metrics = compute_all_path_metrics(
                images_seq[state_index],
                M_ref_seq[state_index],
                F_ref_seq[state_index],
            )
        except Exception as exc:
            print(
                f"[WARNING] 跳过 NEB state "
                f"{state_index}: {exc}"
            )
            continue

        valid_count.append(
            count_raw[state_index]
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

    if len(valid_count) == 0:
        return None, (
            "No valid NEB states."
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        free_energy_profiles,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            valid_count,
            dtype=float,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
    )

    if "force_cap" in data:
        force_cap = float(
            np.asarray(
                data["force_cap"]
            ).reshape(-1)[0]
        )
    else:
        force_cap = np.nan

    result = {
        "method": "NEB-ODE",
        "npz_file": neb_npz_file,
        "force_cap": force_cap,

        "count_seq": (
            count_seq
        ),

        "I_perp_G_seq": (
            I_perp_G_seq
        ),

        "I_perp_E_seq": (
            I_perp_E_seq
        ),

        "Delta_F_seq": (
            Delta_F_seq
        ),

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),
    }

    return result, None


# ============================================================
# Grid IVM 样式
# ============================================================
alpha_colors = {
    0.1: "#377eb8",
    1: "#4daf4a",
    10: "#984ea3",
}

eta_linestyles = {
    0.03: "-",
    0.01: "--",
    0.003: "-.",
    0.001: ":",
}

nsub_markers = {
    4: "o",
    5: "s",
    10: "^",
}


# ============================================================
# 绘制一次
# ============================================================
def load_and_plot_once():
    # --------------------------------------------------------
    # 读取结果
    # --------------------------------------------------------
    (
        grid_ivm_results,
        skipped_grid,
        all_configs,
    ) = load_completed_grid_ivm()

    optimal_result, optimal_error = (
        load_optimal_ivm()
    )

    neb_result, neb_error = (
        load_neb_result()
    )

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------
    print("=" * 100)
    print("MFEP IVM / Optimal-IVM / NEB comparison")
    print("=" * 100)

    print(
        f"完成的 grid IVM 配置："
        f"{len(grid_ivm_results)}"
    )

    print(
        f"跳过的 grid IVM 配置："
        f"{len(skipped_grid)}"
    )

    if optimal_result is not None:
        optimal_current_count = int(
            np.max(
                optimal_result["count_seq"]
            )
        )

        print(
            f"Optimal-IVM：已读取，"
            f"当前 count={optimal_current_count}/"
            f"{OPTIMAL_TOTAL_STEPS}"
        )

        print(
            f"Optimal-IVM states："
            f"{len(optimal_result['count_seq'])}"
        )

        if (
            len(
                optimal_result[
                    "skipped_outer_indices"
                ]
            )
            > 0
        ):
            print(
                "Optimal-IVM 尚未完整写入的 outer：",
                optimal_result[
                    "skipped_outer_indices"
                ],
            )

    else:
        optimal_current_count = -1

        print(
            "Optimal-IVM：尚未读取到有效数据"
        )

        print(
            "  ",
            optimal_error,
        )

    if neb_result is not None:
        print(
            f"NEB-ODE：已读取，"
            f"count range="
            f"({neb_result['count_seq'][0]}, "
            f"{neb_result['count_seq'][-1]})"
        )
    else:
        print(
            "NEB-ODE：读取失败"
        )
        print("  ", neb_error)

    print("=" * 100)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------
    tiny = np.finfo(float).tiny

    for result in grid_ivm_results:
        result["I_perp_G_plot"] = (
            np.maximum(
                result["I_perp_G_seq"],
                tiny,
            )
        )

        result["I_perp_E_plot"] = (
            np.maximum(
                result["I_perp_E_seq"],
                tiny,
            )
        )

        result["Delta_F_plot"] = (
            np.maximum(
                result["Delta_F_seq"],
                tiny,
            )
        )

    if optimal_result is not None:
        optimal_result["I_perp_G_plot"] = (
            np.maximum(
                optimal_result[
                    "I_perp_G_seq"
                ],
                tiny,
            )
        )

        optimal_result["I_perp_E_plot"] = (
            np.maximum(
                optimal_result[
                    "I_perp_E_seq"
                ],
                tiny,
            )
        )

        optimal_result["Delta_F_plot"] = (
            np.maximum(
                optimal_result[
                    "Delta_F_seq"
                ],
                tiny,
            )
        )

    if neb_result is not None:
        neb_result["I_perp_G_plot"] = (
            np.maximum(
                neb_result[
                    "I_perp_G_seq"
                ],
                tiny,
            )
        )

        neb_result["I_perp_E_plot"] = (
            np.maximum(
                neb_result[
                    "I_perp_E_seq"
                ],
                tiny,
            )
        )

        neb_result["Delta_F_plot"] = (
            np.maximum(
                neb_result[
                    "Delta_F_seq"
                ],
                tiny,
            )
        )

    # --------------------------------------------------------
    # 创建图
    # --------------------------------------------------------
    plt.close("all")

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(22, 6.5),
    )

    ax_G, ax_E, ax_DF = axes

    # --------------------------------------------------------
    # 普通 grid IVM
    # --------------------------------------------------------
    for result in grid_ivm_results:
        N_sub = result["N_sub"]
        alpha_IVP = result["alpha_IVP"]
        eta_in = result["eta_in"]

        count_seq = result["count_seq"]

        color = alpha_colors.get(
            alpha_IVP,
            "gray",
        )

        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )

        marker = nsub_markers.get(
            N_sub,
            "o",
        )

        markevery = max(
            1,
            len(count_seq) // 8,
        )

        common_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=3.5,
            linewidth=GRID_IVM_LINEWIDTH,
            alpha=GRID_IVM_ALPHA,
            zorder=2,
        )

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **common_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **common_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **common_kwargs,
        )

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------
    if optimal_result is not None:
        optimal_count = (
            optimal_result["count_seq"]
        )

        optimal_markevery = max(
            1,
            len(optimal_count) // 12,
        )

        optimal_kwargs = dict(
            color=OPTIMAL_COLOR,
            linestyle=OPTIMAL_LINESTYLE,
            marker=OPTIMAL_MARKER,
            markevery=optimal_markevery,
            markersize=8,
            markerfacecolor=OPTIMAL_COLOR,
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=OPTIMAL_LINEWIDTH,
            alpha=1.0,
            zorder=30,
            label=(
                "Optimal-IVM "
                r"($N_{\rm sub}=4$, "
                r"$\alpha=0.1$, adaptive Adam)"
            ),
        )

        ax_G.plot(
            optimal_count,
            optimal_result[
                "I_perp_G_plot"
            ],
            **optimal_kwargs,
        )

        ax_E.plot(
            optimal_count,
            optimal_result[
                "I_perp_E_plot"
            ],
            **optimal_kwargs,
        )

        ax_DF.plot(
            optimal_count,
            optimal_result[
                "Delta_F_plot"
            ],
            **optimal_kwargs,
        )

        # 标记当前最新点
        for axis, values in [
            (
                ax_G,
                optimal_result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                optimal_result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_DF,
                optimal_result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                optimal_count[-1],
                values[-1],
                s=130,
                marker="*",
                color=OPTIMAL_COLOR,
                edgecolor="black",
                linewidth=0.8,
                zorder=40,
            )

    # --------------------------------------------------------
    # NEB-ODE
    # --------------------------------------------------------
    if neb_result is not None:
        neb_count = (
            neb_result["count_seq"]
        )

        neb_markevery = max(
            1,
            len(neb_count) // 12,
        )

        neb_kwargs = dict(
            color=NEB_COLOR,
            linestyle=NEB_LINESTYLE,
            marker=NEB_MARKER,
            markevery=neb_markevery,
            markersize=5.5,
            markerfacecolor="white",
            markeredgecolor=NEB_COLOR,
            markeredgewidth=1.2,
            linewidth=NEB_LINEWIDTH,
            alpha=1.0,
            zorder=20,
            label="NEB-ODE",
        )

        ax_G.plot(
            neb_count,
            neb_result[
                "I_perp_G_plot"
            ],
            **neb_kwargs,
        )

        ax_E.plot(
            neb_count,
            neb_result[
                "I_perp_E_plot"
            ],
            **neb_kwargs,
        )

        ax_DF.plot(
            neb_count,
            neb_result[
                "Delta_F_plot"
            ],
            **neb_kwargs,
        )

    # ========================================================
    # Panel 1
    # ========================================================
    ax_G.set_title(
        "Riemannian normal residual",
        fontsize=14,
    )

    ax_G.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_G.set_ylabel(
        r"$\widehat I_{\perp}^{G}$"
        "\n"
        r"$=\int_0^1"
        r"|P_\varphi^G M\nabla F|_G^2"
        r"|\varphi'(s)|_G\,ds$",
        fontsize=10.5,
    )

    # ========================================================
    # Panel 2
    # ========================================================
    ax_E.set_title(
        "Euclidean residual integral",
        fontsize=14,
    )

    ax_E.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_E.set_ylabel(
        r"$\widehat I_{\perp}^{E}$"
        "\n"
        r"$=\int_0^1"
        r"\|P_\varphi^G M\nabla F\|_2^2"
        r"|\varphi'(s)|_2\,ds$",
        fontsize=10.5,
    )

    # ========================================================
    # Panel 3
    # ========================================================
    ax_DF.set_title(
        "Path-integrated free-energy barrier",
        fontsize=14,
    )

    ax_DF.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    # 使用简化标签，避免 MathText 版本兼容问题
    ax_DF.set_ylabel(
        r"$\Delta F^{\dagger}$ (kcal/mol)"
        "\n"
        "line-integral estimate",
        fontsize=10.5,
    )

    # ========================================================
    # 坐标和网格
    # ========================================================
    for axis in axes:
        axis.grid(
            True,
            which="both",
            linestyle="--",
            alpha=0.3,
        )

        axis.set_xlim(
            left=0,
        )

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log")
        ax_E.set_yscale("log")

    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # ========================================================
    # 图例
    # ========================================================
    method_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linewidth=1.5,
            alpha=GRID_IVM_ALPHA,
            label="Grid IVM",
        )
    ]

    if optimal_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=OPTIMAL_COLOR,
                linestyle=OPTIMAL_LINESTYLE,
                marker=OPTIMAL_MARKER,
                linewidth=OPTIMAL_LINEWIDTH,
                markersize=8,
                label="Optimal-IVM",
            )
        )

    if neb_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=NEB_COLOR,
                linestyle=NEB_LINESTYLE,
                marker=NEB_MARKER,
                markerfacecolor="white",
                linewidth=NEB_LINEWIDTH,
                markersize=6,
                label="NEB-ODE",
            )
        )

    legend_method = ax_DF.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(1.03, 1.0),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_method
    )

    alpha_handles = [
        Line2D(
            [0],
            [0],
            color=alpha_colors[alpha],
            linewidth=2,
            label=rf"$\alpha={alpha}$",
        )
        for alpha in GRID_ALPHA
    ]

    legend_alpha = ax_DF.legend(
        handles=alpha_handles,
        title="Grid IVM color",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.70),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_alpha
    )

    eta_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle=eta_linestyles[eta],
            linewidth=2,
            label=rf"$\eta={eta}$",
        )
        for eta in GRID_ETA
    ]

    legend_eta = ax_DF.legend(
        handles=eta_handles,
        title="Grid IVM line",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.37),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_eta
    )

    nsub_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle="None",
            marker=nsub_markers[N_sub],
            markersize=6,
            label=rf"$N_{{sub}}={N_sub}$",
        )
        for N_sub in GRID_N_SUB
    ]

    ax_DF.legend(
        handles=nsub_handles,
        title="Grid IVM marker",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.06),
        frameon=True,
    )

    # 不使用 tight_layout，避免 MathText/layout 错误
    fig.subplots_adjust(
        left=0.07,
        right=0.80,
        bottom=0.14,
        top=0.90,
        wspace=0.34,
    )

    # ========================================================
    # 保存
    # ========================================================
    output_pdf = os.path.join(
        base_path,
        "MFEP_OptimalIVM_vs_GridIVM_vs_NEB.pdf",
    )

    output_png = os.path.join(
        base_path,
        "MFEP_OptimalIVM_vs_GridIVM_vs_NEB.png",
    )

    if SAVE_OUTPUT:
        fig.savefig(
            output_pdf,
            bbox_inches="tight",
        )

        fig.savefig(
            output_png,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    # ========================================================
    # 最新结果汇总
    # ========================================================
    print("\n" + "=" * 115)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 115)

    summary_rows = []

    for result in grid_ivm_results:
        summary_rows.append(
            {
                "method": "IVM-grid",
                "description": (
                    f"Nsub={result['N_sub']} "
                    f"alpha={result['alpha_IVP']} "
                    f"eta={result['eta_in']}"
                ),
                "count": float(
                    result["count_seq"][-1]
                ),
                "I_G": float(
                    result["I_perp_G_seq"][-1]
                ),
                "I_E": float(
                    result["I_perp_E_seq"][-1]
                ),
                "Delta_F": float(
                    result["Delta_F_seq"][-1]
                ),
                "Delta_F_end": float(
                    result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    if optimal_result is not None:
        summary_rows.append(
            {
                "method": "Optimal-IVM",
                "description": (
                    "Nsub=4 alpha=0.1 "
                    "adaptive Adam"
                ),
                "count": float(
                    optimal_result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    optimal_result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    optimal_result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    optimal_result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "Delta_F_end": float(
                    optimal_result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    if neb_result is not None:
        summary_rows.append(
            {
                "method": "NEB-ODE",
                "description": (
                    f"force cap="
                    f"{neb_result['force_cap']:g}"
                    if np.isfinite(
                        neb_result["force_cap"]
                    )
                    else "NEB-ODE"
                ),
                "count": float(
                    neb_result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    neb_result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    neb_result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    neb_result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "Delta_F_end": float(
                    neb_result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    # Optimal 和 NEB 优先打印
    method_order = {
        "Optimal-IVM": 0,
        "NEB-ODE": 1,
        "IVM-grid": 2,
    }

    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (
            method_order.get(
                row["method"],
                99,
            ),
            row["I_G"],
        ),
    )

    for row in summary_rows_sorted:
        print(
            f"{row['method']:<13} "
            f"{row['description']:<37}"
            f" count={row['count']:>7.1f}"
            f" | I_G={row['I_G']:.6e}"
            f" | I_E={row['I_E']:.6e}"
            f" | DeltaF={row['Delta_F']:.6f}"
            f" | DeltaF_end="
            f"{row['Delta_F_end']:.6f}"
        )

    # ========================================================
    # 保存重新计算后的数据
    # ========================================================
    comparison_results_file = os.path.join(
        base_path,
        "MFEP_OPTIMAL_IVM_GRID_IVM_NEB_COMPARISON.npz",
    )

    np.savez(
        comparison_results_file,

        grid_ivm_results=np.array(
            grid_ivm_results,
            dtype=object,
        ),

        optimal_ivm_result=np.array(
            [optimal_result],
            dtype=object,
        ),

        neb_result=np.array(
            [neb_result],
            dtype=object,
        ),

        summary=np.array(
            summary_rows,
            dtype=object,
        ),
    )

    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)

    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": (
            grid_ivm_results
        ),
        "optimal_result": (
            optimal_result
        ),
        "neb_result": (
            neb_result
        ),
        "summary": (
            summary_rows
        ),
        "optimal_current_count": (
            optimal_current_count
        ),
    }


# ============================================================
# 执行一次或自动刷新
# ============================================================
if AUTO_REFRESH:
    print(
        "AUTO_REFRESH=True："
        f"每 {REFRESH_SECONDS} 秒刷新一次。"
    )

    print(
        "可点击 Jupyter 的停止按钮终止刷新；"
        "不会影响 Terminal 中运行的 Optimal-IVM。"
    )

    latest_output = None

    try:
        while True:
            if clear_output is not None:
                clear_output(
                    wait=True
                )

            latest_output = (
                load_and_plot_once()
            )

            current_count = (
                latest_output[
                    "optimal_current_count"
                ]
            )

            if (
                current_count
                >= OPTIMAL_TOTAL_STEPS
            ):
                print(
                    "\nOptimal-IVM 已到达 "
                    f"count={current_count}，"
                    "自动刷新结束。"
                )
                break

            if (
                current_count < 0
                and not WAIT_FOR_OPTIMAL_FILE
            ):
                print(
                    "\nOptimal-IVM 文件尚不存在，"
                    "且 WAIT_FOR_OPTIMAL_FILE=False，"
                    "停止刷新。"
                )
                break

            print(
                f"\n下次刷新将在 "
                f"{REFRESH_SECONDS} 秒后进行。"
            )

            time.sleep(
                REFRESH_SECONDS
            )

    except KeyboardInterrupt:
        print(
            "\n自动刷新已手动停止。"
        )

else:
    latest_output = (
        load_and_plot_once()
    )

In [ ]:
import os
import glob
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# Optimal-IVM 路径
#
# 必须与正在运行的 adaptive IVM 脚本中的 casenum 一致。
#
# 当前这里保留你给出的旧 schedule 名称：
#   lr0.04to0.001
#
# 如果训练脚本已经改成：
#   lr0.1to0.003_step30
#
# 则把下面对应字符串改成：
#   f"_lr0.1to0.003_step30"
# ============================================================
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)

OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# 选择要绘制的普通 grid IVM 配置
#
# None 表示全部完成的配置。
#
# 示例：
#
# PLOT_ONLY_N_SUB = [4]
# PLOT_ONLY_ALPHA = [0.1]
# PLOT_ONLY_ETA = [0.03]
# ============================================================
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 自动刷新设置
# ============================================================
AUTO_REFRESH = True
REFRESH_SECONDS = 60

# Optimal-IVM 文件尚未创建时是否继续等待
WAIT_FOR_OPTIMAL_FILE = True


# ============================================================
# 绘图设置
# ============================================================
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False

# arclength residual 通常跨多个数量级
USE_LOG_Y_ARC = True

SAVE_OUTPUT = True

# 普通 grid IVM
GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

# Optimal-IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

# NEB
NEB_COLOR = "black"
NEB_LINESTYLE = "-"
NEB_MARKER = "D"
NEB_LINEWIDTH = 3.0


# ============================================================
# 安全读取正在写入的 npz
# ============================================================
def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        f"[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# 普通 grid IVM 配置路径
# ============================================================
def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 中心差分 tangent
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


# ============================================================
# 构造 G-正交投影
#
# G = M^{-1}
# ============================================================
def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


# ============================================================
# G-弧长 quadrature weights
# ============================================================
def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# Euclidean 弧长 quadrature weights
# ============================================================
def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


# ============================================================
# 新增：arclength / equi-speed residual
#
# I_arc =
# int_0^1
#   (phi'(s) dot phi''(s))^2
#   |phi'(s)| ds
#
# 对均匀参数节点：
#
# ds = 1 / (N+1)
#
# phi'_i =
#   (phi_{i+1} - phi_{i-1}) / (2 ds)
#
# phi''_i =
#   (phi_{i+1} - 2 phi_i + phi_{i-1}) / ds^2
#
# 最后：
#
# I_arc ≈ sum_i
#   (phi'_i dot phi''_i)^2
#   |phi'_i| ds
#
# 当路径为等速参数化时：
#
# d/ds |phi'|^2 = 2 phi' dot phi'' = 0
#
# 因此该指标应趋近于零。
# ============================================================
def compute_arclength_residual(full_path):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            f"full_path must have shape (n,2), "
            f"got {full_path.shape}"
        )

    n_full = full_path.shape[0]
    n_internal = n_full - 2

    if n_internal < 1:
        raise ValueError(
            "The path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    tangent_second_inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        tangent_second_inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand) * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner_phi_prime_second": (
            tangent_second_inner
        ),
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


# ============================================================
# 路径积分重建自由能
# ============================================================
def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            f"is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    # 起点到第一个内部 image
    segment_gradients[0] = (
        F_internal[0]
    )

    # 内部 image 之间
    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    # 最后一个内部 image 到终点
    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


# ============================================================
# 对整条路径重新计算四个指标
#
# 1. G-metric I_perp
# 2. Euclidean integral I_perp
# 3. Delta F barrier
# 4. Arclength/equi-speed residual
# ============================================================
def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            f"Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            f"Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    # M grad F
    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    # P_G M grad F
    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    # --------------------------------------------------------
    # G-metric I_perp
    # --------------------------------------------------------
    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    # --------------------------------------------------------
    # Euclidean residual integral
    # --------------------------------------------------------
    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    # --------------------------------------------------------
    # 路径积分自由能
    # --------------------------------------------------------
    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    # --------------------------------------------------------
    # 新增：arclength/equi-speed residual
    # --------------------------------------------------------
    arc_result = compute_arclength_residual(
        full_path
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "free_energy_max_index": (
            free_energy["max_index"]
        ),

        # 新指标
        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result[
                "inner_phi_prime_second"
            ]
        ),

        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后状态
# ============================================================
def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取逐 outer 数据
# ============================================================
def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            f"final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                f"incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                f"expected at least "
                f"{expected_batches}"
            )

    # 优先使用保存的 count_seq
    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(count_seq) == 0:
        return None, (
            f"no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": valid_index_seq,
        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,

        # 新增指标
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取完成的普通 grid IVM
# ============================================================
def load_completed_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (N_sub, alpha_IVP, eta_in)
        for N_sub in GRID_N_SUB
        for alpha_IVP in GRID_ALPHA
        for eta_in in GRID_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        if not selected_config(
            N_sub,
            alpha_IVP,
            eta_in,
        ):
            continue

        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=True,
            final_model=final_model,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
        all_configs,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
# ============================================================
def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive 0.04->0.001"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# 读取 NEB-ODE
# ============================================================
def load_neb_result():
    preferred_neb_folder = os.path.join(
        base_path,
        (
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_kDefault"
            f"_forceCap500"
            f"_centralImprovedTangent"
        ),
    )

    preferred_neb_npz = os.path.join(
        preferred_neb_folder,
        (
            f"alanine_"
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_kDefault"
            f"_forceCap500"
            f"_centralImprovedTangent.npz"
        ),
    )

    neb_candidates = []

    if os.path.exists(
        preferred_neb_npz
    ):
        neb_candidates.append(
            preferred_neb_npz
        )
    else:
        pattern = os.path.join(
            base_path,
            (
                f"MFEP_ASE_NEB_ODE_N{N}"
                f"_kDefault"
                f"_forceCap*"
                f"_centralImprovedTangent"
            ),
            "alanine_*.npz",
        )

        neb_candidates.extend(
            glob.glob(pattern)
        )

    if not neb_candidates:
        return None, (
            "No NEB result found."
        )

    neb_npz_file = max(
        neb_candidates,
        key=os.path.getmtime,
    )

    data = safe_load_npz(
        neb_npz_file
    )

    if data is None:
        return None, (
            f"Could not read {neb_npz_file}"
        )

    required_keys = [
        "count_seq",
        "images_seq",
        "M_ref_seq",
        "F_ref_seq",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in data
    ]

    if missing_keys:
        return None, (
            f"NEB missing keys: "
            f"{missing_keys}"
        )

    count_raw = np.asarray(
        data["count_seq"],
        dtype=float,
    )

    images_seq = np.asarray(
        data["images_seq"],
        dtype=float,
    )

    M_ref_seq = np.asarray(
        data["M_ref_seq"],
        dtype=float,
    )

    F_ref_seq = np.asarray(
        data["F_ref_seq"],
        dtype=float,
    )

    n_states = min(
        len(count_raw),
        len(images_seq),
        len(M_ref_seq),
        len(F_ref_seq),
    )

    count_raw = count_raw[:n_states]
    images_seq = images_seq[:n_states]
    M_ref_seq = M_ref_seq[:n_states]
    F_ref_seq = F_ref_seq[:n_states]

    valid_count = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    for state_index in range(
        n_states
    ):
        try:
            metrics = compute_all_path_metrics(
                images_seq[state_index],
                M_ref_seq[state_index],
                F_ref_seq[state_index],
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 NEB state "
                f"{state_index}: {exc}"
            )
            continue

        valid_count.append(
            count_raw[state_index]
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(valid_count) == 0:
        return None, (
            "No valid NEB states."
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            valid_count,
            dtype=float,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    if "force_cap" in data:
        force_cap = float(
            np.asarray(
                data["force_cap"]
            ).reshape(-1)[0]
        )
    else:
        force_cap = np.nan

    result = {
        "method": "NEB-ODE",
        "npz_file": neb_npz_file,
        "force_cap": force_cap,

        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,

        # 新增
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),
    }

    return result, None


# ============================================================
# Grid IVM 样式
# ============================================================
alpha_colors = {
    0.1: "#377eb8",
    1: "#4daf4a",
    10: "#984ea3",
}

eta_linestyles = {
    0.03: "-",
    0.01: "--",
    0.003: "-.",
    0.001: ":",
}

nsub_markers = {
    4: "o",
    5: "s",
    10: "^",
}


# ============================================================
# 绘制一次
# ============================================================
def load_and_plot_once():
    (
        grid_ivm_results,
        skipped_grid,
        all_configs,
    ) = load_completed_grid_ivm()

    optimal_result, optimal_error = (
        load_optimal_ivm()
    )

    neb_result, neb_error = (
        load_neb_result()
    )

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------
    print("=" * 105)
    print(
        "MFEP IVM / Optimal-IVM / NEB comparison"
    )
    print("=" * 105)

    print(
        f"完成的 grid IVM 配置："
        f"{len(grid_ivm_results)}"
    )

    print(
        f"跳过的 grid IVM 配置："
        f"{len(skipped_grid)}"
    )

    if optimal_result is not None:
        optimal_current_count = int(
            np.max(
                optimal_result["count_seq"]
            )
        )

        print(
            f"Optimal-IVM：已读取，"
            f"当前 count="
            f"{optimal_current_count}/"
            f"{OPTIMAL_TOTAL_STEPS}"
        )

        print(
            f"Optimal-IVM states："
            f"{len(optimal_result['count_seq'])}"
        )

    else:
        optimal_current_count = -1

        print(
            "Optimal-IVM：尚未读取到有效数据"
        )

        print(
            "  ",
            optimal_error,
        )

    if neb_result is not None:
        print(
            f"NEB-ODE：已读取，"
            f"count range="
            f"({neb_result['count_seq'][0]}, "
            f"{neb_result['count_seq'][-1]})"
        )
    else:
        print(
            "NEB-ODE：读取失败"
        )
        print("  ", neb_error)

    print("=" * 105)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------
    tiny = np.finfo(float).tiny

    all_available_results = list(
        grid_ivm_results
    )

    if optimal_result is not None:
        all_available_results.append(
            optimal_result
        )

    if neb_result is not None:
        all_available_results.append(
            neb_result
        )

    for result in all_available_results:
        result["I_perp_G_plot"] = np.maximum(
            result["I_perp_G_seq"],
            tiny,
        )

        result["I_perp_E_plot"] = np.maximum(
            result["I_perp_E_seq"],
            tiny,
        )

        result["I_arc_plot"] = np.maximum(
            result["I_arc_seq"],
            tiny,
        )

        result["Delta_F_plot"] = np.maximum(
            result["Delta_F_seq"],
            tiny,
        )

    # --------------------------------------------------------
    # 创建 1 x 4 图
    # --------------------------------------------------------
    plt.close("all")

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(27, 6.5),
    )

    (
        ax_G,
        ax_E,
        ax_arc,
        ax_DF,
    ) = axes

    # --------------------------------------------------------
    # 普通 grid IVM
    # --------------------------------------------------------
    for result in grid_ivm_results:
        N_sub = result["N_sub"]
        alpha_IVP = result["alpha_IVP"]
        eta_in = result["eta_in"]

        count_seq = result["count_seq"]

        color = alpha_colors.get(
            alpha_IVP,
            "gray",
        )

        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )

        marker = nsub_markers.get(
            N_sub,
            "o",
        )

        markevery = max(
            1,
            len(count_seq) // 8,
        )

        common_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=3.5,
            linewidth=GRID_IVM_LINEWIDTH,
            alpha=GRID_IVM_ALPHA,
            zorder=2,
        )

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **common_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **common_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **common_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **common_kwargs,
        )

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------
    if optimal_result is not None:
        optimal_count = (
            optimal_result["count_seq"]
        )

        optimal_markevery = max(
            1,
            len(optimal_count) // 12,
        )

        optimal_kwargs = dict(
            color=OPTIMAL_COLOR,
            linestyle=OPTIMAL_LINESTYLE,
            marker=OPTIMAL_MARKER,
            markevery=optimal_markevery,
            markersize=8,
            markerfacecolor=OPTIMAL_COLOR,
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=OPTIMAL_LINEWIDTH,
            alpha=1.0,
            zorder=30,
        )

        ax_G.plot(
            optimal_count,
            optimal_result[
                "I_perp_G_plot"
            ],
            **optimal_kwargs,
        )

        ax_E.plot(
            optimal_count,
            optimal_result[
                "I_perp_E_plot"
            ],
            **optimal_kwargs,
        )

        ax_arc.plot(
            optimal_count,
            optimal_result[
                "I_arc_plot"
            ],
            **optimal_kwargs,
        )

        ax_DF.plot(
            optimal_count,
            optimal_result[
                "Delta_F_plot"
            ],
            **optimal_kwargs,
        )

        # 标记最新点
        for axis, values in [
            (
                ax_G,
                optimal_result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                optimal_result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                optimal_result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                optimal_result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                optimal_count[-1],
                values[-1],
                s=130,
                marker="*",
                color=OPTIMAL_COLOR,
                edgecolor="black",
                linewidth=0.8,
                zorder=40,
            )

    # --------------------------------------------------------
    # NEB-ODE
    # --------------------------------------------------------
    if neb_result is not None:
        neb_count = (
            neb_result["count_seq"]
        )

        neb_markevery = max(
            1,
            len(neb_count) // 12,
        )

        neb_kwargs = dict(
            color=NEB_COLOR,
            linestyle=NEB_LINESTYLE,
            marker=NEB_MARKER,
            markevery=neb_markevery,
            markersize=5.5,
            markerfacecolor="white",
            markeredgecolor=NEB_COLOR,
            markeredgewidth=1.2,
            linewidth=NEB_LINEWIDTH,
            alpha=1.0,
            zorder=20,
        )

        ax_G.plot(
            neb_count,
            neb_result[
                "I_perp_G_plot"
            ],
            **neb_kwargs,
        )

        ax_E.plot(
            neb_count,
            neb_result[
                "I_perp_E_plot"
            ],
            **neb_kwargs,
        )

        ax_arc.plot(
            neb_count,
            neb_result[
                "I_arc_plot"
            ],
            **neb_kwargs,
        )

        ax_DF.plot(
            neb_count,
            neb_result[
                "Delta_F_plot"
            ],
            **neb_kwargs,
        )

    # ========================================================
    # Panel 1: G residual
    # ========================================================
    ax_G.set_title(
        "Riemannian normal residual",
        fontsize=14,
    )

    ax_G.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_G.set_ylabel(
        r"$\widehat I_{\perp}^{G}$"
        "\n"
        r"$=\int |P_\varphi^G M\nabla F|_G^2"
        r"|\varphi'|_G\,ds$",
        fontsize=10,
    )

    # ========================================================
    # Panel 2: Euclidean residual
    # ========================================================
    ax_E.set_title(
        "Euclidean residual integral",
        fontsize=14,
    )

    ax_E.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_E.set_ylabel(
        r"$\widehat I_{\perp}^{E}$"
        "\n"
        r"$=\int \|P_\varphi^G M\nabla F\|_2^2"
        r"|\varphi'|_2\,ds$",
        fontsize=10,
    )

    # ========================================================
    # Panel 3: arclength residual
    # ========================================================
    ax_arc.set_title(
        "Arclength / equi-speed residual",
        fontsize=14,
    )

    ax_arc.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_arc.set_ylabel(
        r"$I_{\rm arc}$"
        "\n"
        r"$=\int_0^1"
        r"(\varphi'\cdot\varphi'')^2"
        r"|\varphi'|\,ds$",
        fontsize=10,
    )

    # ========================================================
    # Panel 4: Delta F
    # ========================================================
    ax_DF.set_title(
        "Path-integrated free-energy barrier",
        fontsize=14,
    )

    ax_DF.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_DF.set_ylabel(
        r"$\Delta F^{\dagger}$ (kcal/mol)"
        "\n"
        "line-integral estimate",
        fontsize=10,
    )

    # ========================================================
    # 坐标设置
    # ========================================================
    for axis in axes:
        axis.grid(
            True,
            which="both",
            linestyle="--",
            alpha=0.3,
        )

        axis.set_xlim(
            left=0,
        )

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log")
        ax_E.set_yscale("log")

    if USE_LOG_Y_ARC:
        ax_arc.set_yscale("log")

    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # ========================================================
    # 图例
    # ========================================================
    method_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linewidth=1.5,
            alpha=GRID_IVM_ALPHA,
            label="Grid IVM",
        )
    ]

    if optimal_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=OPTIMAL_COLOR,
                linestyle=OPTIMAL_LINESTYLE,
                marker=OPTIMAL_MARKER,
                linewidth=OPTIMAL_LINEWIDTH,
                markersize=8,
                label="Optimal-IVM",
            )
        )

    if neb_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=NEB_COLOR,
                linestyle=NEB_LINESTYLE,
                marker=NEB_MARKER,
                markerfacecolor="white",
                linewidth=NEB_LINEWIDTH,
                markersize=6,
                label="NEB-ODE",
            )
        )

    legend_method = ax_DF.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(1.03, 1.0),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_method
    )

    alpha_handles = [
        Line2D(
            [0],
            [0],
            color=alpha_colors[alpha],
            linewidth=2,
            label=rf"$\alpha={alpha}$",
        )
        for alpha in GRID_ALPHA
    ]

    legend_alpha = ax_DF.legend(
        handles=alpha_handles,
        title="Grid IVM color",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.70),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_alpha
    )

    eta_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle=eta_linestyles[eta],
            linewidth=2,
            label=rf"$\eta={eta}$",
        )
        for eta in GRID_ETA
    ]

    legend_eta = ax_DF.legend(
        handles=eta_handles,
        title="Grid IVM line",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.37),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_eta
    )

    nsub_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle="None",
            marker=nsub_markers[N_sub],
            markersize=6,
            label=rf"$N_{{sub}}={N_sub}$",
        )
        for N_sub in GRID_N_SUB
    ]

    ax_DF.legend(
        handles=nsub_handles,
        title="Grid IVM marker",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.06),
        frameon=True,
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.82,
        bottom=0.14,
        top=0.90,
        wspace=0.38,
    )

    # ========================================================
    # 保存图片
    # ========================================================
    output_pdf = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_NEB_"
            "four_metrics.pdf"
        ),
    )

    output_png = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_NEB_"
            "four_metrics.png"
        ),
    )

    if SAVE_OUTPUT:
        fig.savefig(
            output_pdf,
            bbox_inches="tight",
        )

        fig.savefig(
            output_png,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    # ========================================================
    # 汇总
    # ========================================================
    print("\n" + "=" * 140)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 140)

    summary_rows = []

    for result in grid_ivm_results:
        summary_rows.append(
            {
                "method": "IVM-grid",

                "description": (
                    f"Nsub={result['N_sub']} "
                    f"alpha={result['alpha_IVP']} "
                    f"eta={result['eta_in']}"
                ),

                "count": float(
                    result["count_seq"][-1]
                ),

                "I_G": float(
                    result["I_perp_G_seq"][-1]
                ),

                "I_E": float(
                    result["I_perp_E_seq"][-1]
                ),

                "I_arc": float(
                    result["I_arc_seq"][-1]
                ),

                "Delta_F": float(
                    result["Delta_F_seq"][-1]
                ),

                "Delta_F_end": float(
                    result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    if optimal_result is not None:
        summary_rows.append(
            {
                "method": "Optimal-IVM",

                "description": (
                    "Nsub=4 alpha=0.1 "
                    "adaptive Adam"
                ),

                "count": float(
                    optimal_result[
                        "count_seq"
                    ][-1]
                ),

                "I_G": float(
                    optimal_result[
                        "I_perp_G_seq"
                    ][-1]
                ),

                "I_E": float(
                    optimal_result[
                        "I_perp_E_seq"
                    ][-1]
                ),

                "I_arc": float(
                    optimal_result[
                        "I_arc_seq"
                    ][-1]
                ),

                "Delta_F": float(
                    optimal_result[
                        "Delta_F_seq"
                    ][-1]
                ),

                "Delta_F_end": float(
                    optimal_result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    if neb_result is not None:
        summary_rows.append(
            {
                "method": "NEB-ODE",

                "description": (
                    f"force cap="
                    f"{neb_result['force_cap']:g}"
                    if np.isfinite(
                        neb_result["force_cap"]
                    )
                    else "NEB-ODE"
                ),

                "count": float(
                    neb_result[
                        "count_seq"
                    ][-1]
                ),

                "I_G": float(
                    neb_result[
                        "I_perp_G_seq"
                    ][-1]
                ),

                "I_E": float(
                    neb_result[
                        "I_perp_E_seq"
                    ][-1]
                ),

                "I_arc": float(
                    neb_result[
                        "I_arc_seq"
                    ][-1]
                ),

                "Delta_F": float(
                    neb_result[
                        "Delta_F_seq"
                    ][-1]
                ),

                "Delta_F_end": float(
                    neb_result[
                        "Delta_F_final_seq"
                    ][-1]
                ),
            }
        )

    method_order = {
        "Optimal-IVM": 0,
        "NEB-ODE": 1,
        "IVM-grid": 2,
    }

    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (
            method_order.get(
                row["method"],
                99,
            ),
            row["I_G"],
        ),
    )

    for row in summary_rows_sorted:
        print(
            f"{row['method']:<13} "
            f"{row['description']:<37}"
            f" count={row['count']:>7.1f}"
            f" | I_G={row['I_G']:.6e}"
            f" | I_E={row['I_E']:.6e}"
            f" | I_arc={row['I_arc']:.6e}"
            f" | DeltaF={row['Delta_F']:.6f}"
            f" | DeltaF_end="
            f"{row['Delta_F_end']:.6f}"
        )

    # ========================================================
    # 保存重新计算后的全部数据
    # ========================================================
    comparison_results_file = os.path.join(
        base_path,
        (
            "MFEP_OPTIMAL_IVM_GRID_IVM_NEB_"
            "FOUR_METRICS.npz"
        ),
    )

    np.savez(
        comparison_results_file,

        grid_ivm_results=np.array(
            grid_ivm_results,
            dtype=object,
        ),

        optimal_ivm_result=np.array(
            [optimal_result],
            dtype=object,
        ),

        neb_result=np.array(
            [neb_result],
            dtype=object,
        ),

        summary=np.array(
            summary_rows,
            dtype=object,
        ),
    )

    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)

    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": (
            grid_ivm_results
        ),

        "optimal_result": (
            optimal_result
        ),

        "neb_result": (
            neb_result
        ),

        "summary": (
            summary_rows
        ),

        "optimal_current_count": (
            optimal_current_count
        ),
    }


# ============================================================
# 执行一次或自动刷新
# ============================================================
if AUTO_REFRESH:
    print(
        "AUTO_REFRESH=True："
        f"每 {REFRESH_SECONDS} 秒刷新一次。"
    )

    print(
        "点击 Jupyter 的停止按钮可终止刷新；"
        "不会影响 Terminal 中运行的 Optimal-IVM。"
    )

    latest_output = None

    try:
        while True:
            if clear_output is not None:
                clear_output(
                    wait=True
                )

            latest_output = (
                load_and_plot_once()
            )

            current_count = (
                latest_output[
                    "optimal_current_count"
                ]
            )

            if (
                current_count
                >= OPTIMAL_TOTAL_STEPS
            ):
                print(
                    "\nOptimal-IVM 已到达 "
                    f"count={current_count}，"
                    "自动刷新结束。"
                )
                break

            if (
                current_count < 0
                and not WAIT_FOR_OPTIMAL_FILE
            ):
                print(
                    "\nOptimal-IVM 文件尚不存在，"
                    "且 WAIT_FOR_OPTIMAL_FILE=False，"
                    "停止刷新。"
                )
                break

            print(
                f"\n下次刷新将在 "
                f"{REFRESH_SECONDS} 秒后进行。"
            )

            time.sleep(
                REFRESH_SECONDS
            )

    except KeyboardInterrupt:
        print(
            "\n自动刷新已手动停止。"
        )

else:
    latest_output = (
        load_and_plot_once()
    )

In [ ]:
import os
import glob
import re
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# Optimal-IVM 配置
# ============================================================
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)

OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# NEB k-sweep 配置
#
# 必须与当前运行的 NEB 脚本 GRID_K 对应。
#
# 你当前运行脚本中：
#
# GRID_K = [20.0]
#
# 所以这里使用：
#
# NEB_EXPECTED_K = [20.0]
#
# 如果以后运行：
#
# GRID_K = [1.0, 20.0, 200.0]
#
# 则改成：
#
# NEB_EXPECTED_K = [1.0, 20.0, 200.0]
#
# None：
#   不指定预期 k，只自动发现已经创建的结果目录。
#   此时无法判断尚未开始的 k，因此不建议用于自动停止。
# ============================================================
NEB_EXPECTED_K = [20.0, 200.0, 2000.0]

NEB_EXPECTED_FORCE_CAP = 500.0

# 是否读取旧的 kDefault NEB。
# 当前主要绘制 k-sweep，因此默认关闭。
INCLUDE_LEGACY_DEFAULT_NEB = False


# ============================================================
# 选择要绘制的普通 grid IVM 配置
# ============================================================
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 自动刷新设置
# ============================================================
AUTO_REFRESH = True
REFRESH_SECONDS = 60

# Optimal-IVM 文件尚未创建时是否继续等待
WAIT_FOR_OPTIMAL_FILE = True

# NEB 文件尚未创建时是否继续等待
WAIT_FOR_NEB_FILE = True

# 自动停止条件
#
# True:
#   对应方法必须完成，刷新循环才会结束。
#
# False:
#   不把该方法纳入自动停止判断。
# ============================================================
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_NEB_COMPLETE = True


# ============================================================
# 绘图设置
# ============================================================
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True

SAVE_OUTPUT = True

# 普通 grid IVM
GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

# Optimal-IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

# NEB k-sweep
NEB_LINEWIDTH = 2.8
NEB_MARKER = "D"

NEB_K_COLORS = {
    1.0: "#000000",
    20.0: "#ff7f00",
    200.0: "#00a6d6",
}

NEB_K_LINESTYLES = {
    1.0: "-",
    20.0: "--",
    200.0: "-.",
}


# ============================================================
# 安全读取正在写入的 npz
# ============================================================
def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        f"[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# 普通 grid IVM 配置路径
# ============================================================
def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 数值工具
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def compute_arclength_residual(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            f"full_path must have shape (n,2), "
            f"got {full_path.shape}"
        )

    n_full = full_path.shape[0]
    n_internal = n_full - 2

    if n_internal < 1:
        raise ValueError(
            "The path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    tangent_second_inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        tangent_second_inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand) * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner_phi_prime_second": (
            tangent_second_inner
        ),
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            f"is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    segment_gradients[0] = (
        F_internal[0]
    )

    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            f"Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            f"Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    arc_result = compute_arclength_residual(
        full_path
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "free_energy_max_index": (
            free_energy["max_index"]
        ),

        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result[
                "inner_phi_prime_second"
            ]
        ),

        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后状态
# ============================================================
def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取逐 outer 数据
# ============================================================
def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            f"final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                f"incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                f"expected at least "
                f"{expected_batches}"
            )

    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(count_seq) == 0:
        return None, (
            f"no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": valid_index_seq,
        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取完成的普通 grid IVM
# ============================================================
def load_completed_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (N_sub, alpha_IVP, eta_in)
        for N_sub in GRID_N_SUB
        for alpha_IVP in GRID_ALPHA
        for eta_in in GRID_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        if not selected_config(
            N_sub,
            alpha_IVP,
            eta_in,
        ):
            continue

        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=True,
            final_model=final_model,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
        all_configs,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
# ============================================================
def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive 0.04->0.001"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# NEB k 值工具
# ============================================================
def format_k_tag(k_spring):
    return f"{float(k_spring):g}"


def parse_k_from_path(path):
    folder_name = os.path.basename(
        os.path.dirname(path)
    )

    match = re.search(
        r"_k([^_]+)_forceCap",
        folder_name,
    )

    if match is None:
        return np.nan

    text = match.group(1)

    if text.lower() == "default":
        return np.nan

    try:
        return float(text)
    except ValueError:
        return np.nan


def k_values_equal(k1, k2):
    return bool(
        np.isclose(
            float(k1),
            float(k2),
            rtol=0.0,
            atol=1e-12,
        )
    )


def neb_expected_paths(k_spring):
    k_tag = format_k_tag(
        k_spring
    )

    casenum = (
        f"MFEP_ASE_NEB_ODE_N{N}"
        f"_k{k_tag}"
        f"_forceCap{NEB_EXPECTED_FORCE_CAP:g}"
        f"_centralImprovedTangent"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"alanine_{casenum}.npz",
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        done_file,
    )


# ============================================================
# 自动发现所有 NEB k-sweep npz
# ============================================================
def discover_neb_npz_files():
    candidates = []

    pattern = os.path.join(
        base_path,
        (
            f"MFEP_ASE_NEB_ODE_N{N}"
            f"_k*"
            f"_forceCap*"
            f"_centralImprovedTangent"
        ),
        "alanine_*.npz",
    )

    candidates.extend(
        glob.glob(pattern)
    )

    if INCLUDE_LEGACY_DEFAULT_NEB:
        legacy_pattern = os.path.join(
            base_path,
            (
                f"MFEP_ASE_NEB_ODE_N{N}"
                f"_kDefault"
                f"_forceCap*"
                f"_centralImprovedTangent"
            ),
            "alanine_*.npz",
        )

        candidates.extend(
            glob.glob(legacy_pattern)
        )

    # 显式加入预期路径，避免 glob 时序问题
    if NEB_EXPECTED_K is not None:
        for k_spring in NEB_EXPECTED_K:
            (
                casenum,
                folder_path,
                npz_file,
                done_file,
            ) = neb_expected_paths(
                k_spring
            )

            if os.path.exists(npz_file):
                candidates.append(
                    npz_file
                )

    unique_candidates = []

    seen = set()

    for candidate in candidates:
        absolute = os.path.abspath(
            candidate
        )

        if absolute in seen:
            continue

        seen.add(absolute)
        unique_candidates.append(
            candidate
        )

    return unique_candidates


# ============================================================
# 读取单个 NEB k 结果
# ============================================================
def load_single_neb_result(
    neb_npz_file,
):
    data = safe_load_npz(
        neb_npz_file
    )

    if data is None:
        return None, (
            f"Could not read {neb_npz_file}"
        )

    required_keys = [
        "count_seq",
        "images_seq",
        "M_ref_seq",
        "F_ref_seq",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in data
    ]

    if missing_keys:
        return None, (
            f"NEB missing keys: "
            f"{missing_keys}"
        )

    count_raw = np.asarray(
        data["count_seq"],
        dtype=float,
    )

    images_seq = np.asarray(
        data["images_seq"],
        dtype=float,
    )

    M_ref_seq = np.asarray(
        data["M_ref_seq"],
        dtype=float,
    )

    F_ref_seq = np.asarray(
        data["F_ref_seq"],
        dtype=float,
    )

    n_states = min(
        len(count_raw),
        len(images_seq),
        len(M_ref_seq),
        len(F_ref_seq),
    )

    if n_states == 0:
        return None, (
            f"No saved NEB state: "
            f"{neb_npz_file}"
        )

    count_raw = count_raw[:n_states]
    images_seq = images_seq[:n_states]
    M_ref_seq = M_ref_seq[:n_states]
    F_ref_seq = F_ref_seq[:n_states]

    valid_count = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    for state_index in range(
        n_states
    ):
        try:
            metrics = compute_all_path_metrics(
                images_seq[state_index],
                M_ref_seq[state_index],
                F_ref_seq[state_index],
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 NEB state "
                f"{state_index} in "
                f"{os.path.basename(neb_npz_file)}: "
                f"{exc}"
            )
            continue

        valid_count.append(
            count_raw[state_index]
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(valid_count) == 0:
        return None, (
            f"No valid NEB states: "
            f"{neb_npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            valid_count,
            dtype=float,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    if "k_spring" in data:
        k_spring = float(
            np.asarray(
                data["k_spring"]
            ).reshape(-1)[0]
        )
    else:
        k_spring = parse_k_from_path(
            neb_npz_file
        )

    if "force_cap" in data:
        force_cap = float(
            np.asarray(
                data["force_cap"]
            ).reshape(-1)[0]
        )
    else:
        force_cap = np.nan

    folder_path = os.path.dirname(
        neb_npz_file
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    done = os.path.exists(
        done_file
    )

    if "optimizer_step_seq" in data:
        optimizer_step_seq = np.asarray(
            data["optimizer_step_seq"],
            dtype=int,
        )
    else:
        optimizer_step_seq = np.array(
            [],
            dtype=int,
        )

    if optimizer_step_seq.size > 0:
        latest_optimizer_step = int(
            optimizer_step_seq[-1]
        )
    else:
        latest_optimizer_step = np.nan

    result = {
        "method": "NEB-ODE",
        "npz_file": neb_npz_file,
        "folder_path": folder_path,
        "done_file": done_file,
        "done": done,

        "k_spring": k_spring,
        "force_cap": force_cap,

        "latest_optimizer_step": (
            latest_optimizer_step
        ),

        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),
    }

    return result, None


# ============================================================
# 读取所有已经出现的 NEB k-sweep 结果
# ============================================================
def load_all_neb_results():
    candidates = discover_neb_npz_files()

    neb_results = []
    errors = []

    for neb_npz_file in candidates:
        result, error = load_single_neb_result(
            neb_npz_file
        )

        if result is None:
            errors.append(
                f"{neb_npz_file}: {error}"
            )
            continue

        # 默认不读取 kDefault
        if (
            not INCLUDE_LEGACY_DEFAULT_NEB
            and not np.isfinite(
                result["k_spring"]
            )
        ):
            continue

        neb_results.append(
            result
        )

    # 如果同一个 k 意外找到多个文件，
    # 保留修改时间最新的文件。
    newest_for_k = {}
    nonfinite_results = []

    for result in neb_results:
        k_spring = result["k_spring"]

        if not np.isfinite(k_spring):
            nonfinite_results.append(
                result
            )
            continue

        key = float(k_spring)
        mtime = os.path.getmtime(
            result["npz_file"]
        )

        if key not in newest_for_k:
            newest_for_k[key] = (
                mtime,
                result,
            )
        else:
            old_mtime, old_result = (
                newest_for_k[key]
            )

            if mtime >= old_mtime:
                newest_for_k[key] = (
                    mtime,
                    result,
                )

    neb_results = [
        item[1]
        for item in newest_for_k.values()
    ]

    neb_results.extend(
        nonfinite_results
    )

    neb_results.sort(
        key=lambda result: (
            1
            if not np.isfinite(
                result["k_spring"]
            )
            else 0,
            result["k_spring"]
            if np.isfinite(
                result["k_spring"]
            )
            else np.inf,
        )
    )

    return (
        neb_results,
        errors,
        candidates,
    )


# ============================================================
# 判断预期 NEB 是否全部完成
# ============================================================
def evaluate_neb_completion(
    neb_results,
):
    if NEB_EXPECTED_K is None:
        if len(neb_results) == 0:
            return False, []

        all_done = all(
            result["done"]
            for result in neb_results
        )

        return (
            all_done,
            [
                {
                    "k_spring": result["k_spring"],
                    "found": True,
                    "done": result["done"],
                }
                for result in neb_results
            ],
        )

    status_rows = []

    for expected_k in NEB_EXPECTED_K:
        matching = None

        for result in neb_results:
            if (
                np.isfinite(
                    result["k_spring"]
                )
                and k_values_equal(
                    result["k_spring"],
                    expected_k,
                )
            ):
                matching = result
                break

        status_rows.append(
            {
                "k_spring": float(
                    expected_k
                ),
                "found": (
                    matching is not None
                ),
                "done": (
                    bool(matching["done"])
                    if matching is not None
                    else False
                ),
            }
        )

    all_complete = all(
        row["found"] and row["done"]
        for row in status_rows
    )

    return (
        all_complete,
        status_rows,
    )


# ============================================================
# Grid IVM 样式
# ============================================================
alpha_colors = {
    0.1: "#377eb8",
    1: "#4daf4a",
    10: "#984ea3",
}

eta_linestyles = {
    0.03: "-",
    0.01: "--",
    0.003: "-.",
    0.001: ":",
}

nsub_markers = {
    4: "o",
    5: "s",
    10: "^",
}


def neb_style(
    k_spring,
    index,
):
    if np.isfinite(k_spring):
        for known_k, color in (
            NEB_K_COLORS.items()
        ):
            if k_values_equal(
                known_k,
                k_spring,
            ):
                selected_color = color
                break
        else:
            cmap = plt.get_cmap("tab10")
            selected_color = cmap(
                index % 10
            )

        for known_k, linestyle in (
            NEB_K_LINESTYLES.items()
        ):
            if k_values_equal(
                known_k,
                k_spring,
            ):
                selected_linestyle = (
                    linestyle
                )
                break
        else:
            styles = [
                "-",
                "--",
                "-.",
                ":",
            ]

            selected_linestyle = (
                styles[
                    index % len(styles)
                ]
            )

    else:
        selected_color = "black"
        selected_linestyle = "-"

    return (
        selected_color,
        selected_linestyle,
    )


def neb_label(result):
    k_spring = result["k_spring"]

    if np.isfinite(k_spring):
        label = (
            f"NEB-ODE k="
            f"{k_spring:g}"
        )
    else:
        label = "NEB-ODE k=Default"

    if result["done"]:
        label += " [DONE]"
    else:
        label += " [running]"

    return label


# ============================================================
# 绘制一次
# ============================================================
def load_and_plot_once():
    (
        grid_ivm_results,
        skipped_grid,
        all_configs,
    ) = load_completed_grid_ivm()

    optimal_result, optimal_error = (
        load_optimal_ivm()
    )

    (
        neb_results,
        neb_errors,
        neb_candidates,
    ) = load_all_neb_results()

    (
        neb_all_complete,
        neb_completion_rows,
    ) = evaluate_neb_completion(
        neb_results
    )

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------
    print("=" * 115)
    print(
        "MFEP IVM / Optimal-IVM / NEB k-sweep comparison"
    )
    print("=" * 115)

    print(
        f"完成的 grid IVM 配置："
        f"{len(grid_ivm_results)}"
    )

    print(
        f"跳过的 grid IVM 配置："
        f"{len(skipped_grid)}"
    )

    if optimal_result is not None:
        optimal_current_count = int(
            np.max(
                optimal_result["count_seq"]
            )
        )

        optimal_complete = (
            optimal_current_count
            >= OPTIMAL_TOTAL_STEPS
        )

        print(
            f"Optimal-IVM：已读取，"
            f"当前 count="
            f"{optimal_current_count}/"
            f"{OPTIMAL_TOTAL_STEPS}"
        )

        print(
            f"Optimal-IVM states："
            f"{len(optimal_result['count_seq'])}"
        )

        print(
            f"Optimal-IVM complete："
            f"{optimal_complete}"
        )

    else:
        optimal_current_count = -1
        optimal_complete = False

        print(
            "Optimal-IVM：尚未读取到有效数据"
        )

        print(
            "  ",
            optimal_error,
        )

    print("-" * 115)
    print(
        f"发现的 NEB npz 文件数："
        f"{len(neb_candidates)}"
    )

    print(
        f"成功读取的 NEB k 数："
        f"{len(neb_results)}"
    )

    if len(neb_results) > 0:
        for result in neb_results:
            k_spring = result[
                "k_spring"
            ]

            if np.isfinite(k_spring):
                k_text = f"{k_spring:g}"
            else:
                k_text = "Default"

            count_seq = result[
                "count_seq"
            ]

            print(
                f"NEB k={k_text:<8} "
                f"| states={len(count_seq):4d} "
                f"| count range="
                f"({count_seq[0]:g}, "
                f"{count_seq[-1]:g}) "
                f"| optimizer step="
                f"{result['latest_optimizer_step']} "
                f"| DONE={result['done']}"
            )
    else:
        print(
            "尚未读取到有效 NEB k-sweep 数据。"
        )

    if len(neb_errors) > 0:
        print(
            f"NEB 读取警告数："
            f"{len(neb_errors)}"
        )

        for error in neb_errors[:5]:
            print(
                "  ",
                error,
            )

    print("-" * 115)
    print(
        "预期 NEB 完成状态："
    )

    if len(neb_completion_rows) == 0:
        print(
            "  未指定或未发现预期 NEB。"
        )
    else:
        for row in neb_completion_rows:
            print(
                f"  k={row['k_spring']:g}: "
                f"found={row['found']}, "
                f"DONE={row['done']}"
            )

    print(
        f"NEB all complete："
        f"{neb_all_complete}"
    )

    print("=" * 115)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------
    tiny = np.finfo(float).tiny

    all_available_results = list(
        grid_ivm_results
    )

    if optimal_result is not None:
        all_available_results.append(
            optimal_result
        )

    all_available_results.extend(
        neb_results
    )

    for result in all_available_results:
        result["I_perp_G_plot"] = np.maximum(
            result["I_perp_G_seq"],
            tiny,
        )

        result["I_perp_E_plot"] = np.maximum(
            result["I_perp_E_seq"],
            tiny,
        )

        result["I_arc_plot"] = np.maximum(
            result["I_arc_seq"],
            tiny,
        )

        result["Delta_F_plot"] = np.maximum(
            result["Delta_F_seq"],
            tiny,
        )

    # --------------------------------------------------------
    # 创建 1 x 4 图
    # --------------------------------------------------------
    plt.close("all")

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(27, 6.5),
    )

    (
        ax_G,
        ax_E,
        ax_arc,
        ax_DF,
    ) = axes

    # --------------------------------------------------------
    # 普通 grid IVM
    # --------------------------------------------------------
    for result in grid_ivm_results:
        N_sub = result["N_sub"]
        alpha_IVP = result["alpha_IVP"]
        eta_in = result["eta_in"]

        count_seq = result["count_seq"]

        color = alpha_colors.get(
            alpha_IVP,
            "gray",
        )

        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )

        marker = nsub_markers.get(
            N_sub,
            "o",
        )

        markevery = max(
            1,
            len(count_seq) // 8,
        )

        common_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=3.5,
            linewidth=GRID_IVM_LINEWIDTH,
            alpha=GRID_IVM_ALPHA,
            zorder=2,
        )

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **common_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **common_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **common_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **common_kwargs,
        )

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------
    if optimal_result is not None:
        optimal_count = (
            optimal_result["count_seq"]
        )

        optimal_markevery = max(
            1,
            len(optimal_count) // 12,
        )

        optimal_kwargs = dict(
            color=OPTIMAL_COLOR,
            linestyle=OPTIMAL_LINESTYLE,
            marker=OPTIMAL_MARKER,
            markevery=optimal_markevery,
            markersize=8,
            markerfacecolor=OPTIMAL_COLOR,
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=OPTIMAL_LINEWIDTH,
            alpha=1.0,
            zorder=30,
        )

        ax_G.plot(
            optimal_count,
            optimal_result[
                "I_perp_G_plot"
            ],
            **optimal_kwargs,
        )

        ax_E.plot(
            optimal_count,
            optimal_result[
                "I_perp_E_plot"
            ],
            **optimal_kwargs,
        )

        ax_arc.plot(
            optimal_count,
            optimal_result[
                "I_arc_plot"
            ],
            **optimal_kwargs,
        )

        ax_DF.plot(
            optimal_count,
            optimal_result[
                "Delta_F_plot"
            ],
            **optimal_kwargs,
        )

        for axis, values in [
            (
                ax_G,
                optimal_result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                optimal_result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                optimal_result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                optimal_result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                optimal_count[-1],
                values[-1],
                s=130,
                marker="*",
                color=OPTIMAL_COLOR,
                edgecolor="black",
                linewidth=0.8,
                zorder=40,
            )

    # --------------------------------------------------------
    # NEB k-sweep：有多少画多少
    # --------------------------------------------------------
    for neb_index, result in enumerate(
        neb_results
    ):
        neb_count = result[
            "count_seq"
        ]

        (
            color,
            linestyle,
        ) = neb_style(
            result["k_spring"],
            neb_index,
        )

        neb_markevery = max(
            1,
            len(neb_count) // 12,
        )

        neb_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=NEB_MARKER,
            markevery=neb_markevery,
            markersize=5.5,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=1.2,
            linewidth=NEB_LINEWIDTH,
            alpha=1.0,
            zorder=20 + neb_index,
        )

        ax_G.plot(
            neb_count,
            result[
                "I_perp_G_plot"
            ],
            **neb_kwargs,
        )

        ax_E.plot(
            neb_count,
            result[
                "I_perp_E_plot"
            ],
            **neb_kwargs,
        )

        ax_arc.plot(
            neb_count,
            result[
                "I_arc_plot"
            ],
            **neb_kwargs,
        )

        ax_DF.plot(
            neb_count,
            result[
                "Delta_F_plot"
            ],
            **neb_kwargs,
        )

        # 标记每个 k 的最新状态
        latest_marker = (
            "D"
            if result["done"]
            else "o"
        )

        for axis, values in [
            (
                ax_G,
                result["I_perp_G_plot"],
            ),
            (
                ax_E,
                result["I_perp_E_plot"],
            ),
            (
                ax_arc,
                result["I_arc_plot"],
            ),
            (
                ax_DF,
                result["Delta_F_plot"],
            ),
        ]:
            axis.scatter(
                neb_count[-1],
                values[-1],
                s=75,
                marker=latest_marker,
                facecolor=color,
                edgecolor="black",
                linewidth=0.8,
                zorder=50 + neb_index,
            )

    # ========================================================
    # Panel 设置
    # ========================================================
    ax_G.set_title(
        "Riemannian normal residual",
        fontsize=14,
    )

    ax_G.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_G.set_ylabel(
        r"$\widehat I_{\perp}^{G}$"
        "\n"
        r"$=\int |P_\varphi^G M\nabla F|_G^2"
        r"|\varphi'|_G\,ds$",
        fontsize=10,
    )

    ax_E.set_title(
        "Euclidean residual integral",
        fontsize=14,
    )

    ax_E.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_E.set_ylabel(
        r"$\widehat I_{\perp}^{E}$"
        "\n"
        r"$=\int \|P_\varphi^G M\nabla F\|_2^2"
        r"|\varphi'|_2\,ds$",
        fontsize=10,
    )

    ax_arc.set_title(
        "Arclength / equi-speed residual",
        fontsize=14,
    )

    ax_arc.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_arc.set_ylabel(
        r"$I_{\rm arc}$"
        "\n"
        r"$=\int_0^1"
        r"(\varphi'\cdot\varphi'')^2"
        r"|\varphi'|\,ds$",
        fontsize=10,
    )

    ax_DF.set_title(
        "Path-integrated free-energy barrier",
        fontsize=14,
    )

    ax_DF.set_xlabel(
        "count_seq",
        fontsize=12,
    )

    ax_DF.set_ylabel(
        r"$\Delta F^{\dagger}$ (kcal/mol)"
        "\n"
        "line-integral estimate",
        fontsize=10,
    )

    for axis in axes:
        axis.grid(
            True,
            which="both",
            linestyle="--",
            alpha=0.3,
        )

        axis.set_xlim(
            left=0,
        )

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log")
        ax_E.set_yscale("log")

    if USE_LOG_Y_ARC:
        ax_arc.set_yscale("log")

    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # ========================================================
    # 图例
    # ========================================================
    method_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linewidth=1.5,
            alpha=GRID_IVM_ALPHA,
            label="Grid IVM",
        )
    ]

    if optimal_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=OPTIMAL_COLOR,
                linestyle=OPTIMAL_LINESTYLE,
                marker=OPTIMAL_MARKER,
                linewidth=OPTIMAL_LINEWIDTH,
                markersize=8,
                label="Optimal-IVM",
            )
        )

    for neb_index, result in enumerate(
        neb_results
    ):
        (
            color,
            linestyle,
        ) = neb_style(
            result["k_spring"],
            neb_index,
        )

        method_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                linestyle=linestyle,
                marker=NEB_MARKER,
                markerfacecolor="white",
                markeredgecolor=color,
                linewidth=NEB_LINEWIDTH,
                markersize=6,
                label=neb_label(result),
            )
        )

    legend_method = ax_DF.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(1.03, 1.0),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_method
    )

    alpha_handles = [
        Line2D(
            [0],
            [0],
            color=alpha_colors[alpha],
            linewidth=2,
            label=rf"$\alpha={alpha}$",
        )
        for alpha in GRID_ALPHA
    ]

    legend_alpha = ax_DF.legend(
        handles=alpha_handles,
        title="Grid IVM color",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.68),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_alpha
    )

    eta_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle=eta_linestyles[eta],
            linewidth=2,
            label=rf"$\eta={eta}$",
        )
        for eta in GRID_ETA
    ]

    legend_eta = ax_DF.legend(
        handles=eta_handles,
        title="Grid IVM line",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.36),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_eta
    )

    nsub_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle="None",
            marker=nsub_markers[N_sub],
            markersize=6,
            label=rf"$N_{{sub}}={N_sub}$",
        )
        for N_sub in GRID_N_SUB
    ]

    ax_DF.legend(
        handles=nsub_handles,
        title="Grid IVM marker",
        loc="upper left",
        bbox_to_anchor=(1.03, 0.05),
        frameon=True,
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.80,
        bottom=0.14,
        top=0.90,
        wspace=0.38,
    )

    # ========================================================
    # 保存图片
    # ========================================================
    output_pdf = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_"
            "NEB_KSWEEP_four_metrics.pdf"
        ),
    )

    output_png = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_"
            "NEB_KSWEEP_four_metrics.png"
        ),
    )

    if SAVE_OUTPUT:
        fig.savefig(
            output_pdf,
            bbox_inches="tight",
        )

        fig.savefig(
            output_png,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    # ========================================================
    # 当前值汇总
    # ========================================================
    print("\n" + "=" * 150)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 150)

    summary_rows = []

    for result in grid_ivm_results:
        summary_rows.append(
            {
                "method": "IVM-grid",

                "description": (
                    f"Nsub={result['N_sub']} "
                    f"alpha={result['alpha_IVP']} "
                    f"eta={result['eta_in']}"
                ),

                "count": float(
                    result["count_seq"][-1]
                ),

                "I_G": float(
                    result["I_perp_G_seq"][-1]
                ),

                "I_E": float(
                    result["I_perp_E_seq"][-1]
                ),

                "I_arc": float(
                    result["I_arc_seq"][-1]
                ),

                "Delta_F": float(
                    result["Delta_F_seq"][-1]
                ),

                "Delta_F_end": float(
                    result[
                        "Delta_F_final_seq"
                    ][-1]
                ),

                "done": True,
            }
        )

    if optimal_result is not None:
        summary_rows.append(
            {
                "method": "Optimal-IVM",

                "description": (
                    "Nsub=4 alpha=0.1 "
                    "adaptive Adam"
                ),

                "count": float(
                    optimal_result[
                        "count_seq"
                    ][-1]
                ),

                "I_G": float(
                    optimal_result[
                        "I_perp_G_seq"
                    ][-1]
                ),

                "I_E": float(
                    optimal_result[
                        "I_perp_E_seq"
                    ][-1]
                ),

                "I_arc": float(
                    optimal_result[
                        "I_arc_seq"
                    ][-1]
                ),

                "Delta_F": float(
                    optimal_result[
                        "Delta_F_seq"
                    ][-1]
                ),

                "Delta_F_end": float(
                    optimal_result[
                        "Delta_F_final_seq"
                    ][-1]
                ),

                "done": optimal_complete,
            }
        )

    for result in neb_results:
        k_spring = result[
            "k_spring"
        ]

        if np.isfinite(k_spring):
            k_description = (
                f"k={k_spring:g}"
            )
        else:
            k_description = (
                "k=Default"
            )

        if np.isfinite(
            result["force_cap"]
        ):
            k_description += (
                f" forceCap="
                f"{result['force_cap']:g}"
            )

        summary_rows.append(
            {
                "method": "NEB-ODE",

                "description": (
                    k_description
                ),

                "count": float(
                    result[
                        "count_seq"
                    ][-1]
                ),

                "I_G": float(
                    result[
                        "I_perp_G_seq"
                    ][-1]
                ),

                "I_E": float(
                    result[
                        "I_perp_E_seq"
                    ][-1]
                ),

                "I_arc": float(
                    result[
                        "I_arc_seq"
                    ][-1]
                ),

                "Delta_F": float(
                    result[
                        "Delta_F_seq"
                    ][-1]
                ),

                "Delta_F_end": float(
                    result[
                        "Delta_F_final_seq"
                    ][-1]
                ),

                "done": bool(
                    result["done"]
                ),

                "k_spring": (
                    result["k_spring"]
                ),
            }
        )

    method_order = {
        "Optimal-IVM": 0,
        "NEB-ODE": 1,
        "IVM-grid": 2,
    }

    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (
            method_order.get(
                row["method"],
                99,
            ),
            row.get(
                "k_spring",
                np.inf,
            ),
            row["I_G"],
        ),
    )

    for row in summary_rows_sorted:
        print(
            f"{row['method']:<13} "
            f"{row['description']:<39}"
            f" count={row['count']:>7.1f}"
            f" | I_G={row['I_G']:.6e}"
            f" | I_E={row['I_E']:.6e}"
            f" | I_arc={row['I_arc']:.6e}"
            f" | DeltaF={row['Delta_F']:.6f}"
            f" | DeltaF_end="
            f"{row['Delta_F_end']:.6f}"
            f" | done={row['done']}"
        )

    # ========================================================
    # 保存重新计算后的全部数据
    # ========================================================
    comparison_results_file = os.path.join(
        base_path,
        (
            "MFEP_OPTIMAL_IVM_GRID_IVM_"
            "NEB_KSWEEP_FOUR_METRICS.npz"
        ),
    )

    np.savez(
        comparison_results_file,

        grid_ivm_results=np.array(
            grid_ivm_results,
            dtype=object,
        ),

        optimal_ivm_result=np.array(
            [optimal_result],
            dtype=object,
        ),

        neb_results=np.array(
            neb_results,
            dtype=object,
        ),

        neb_completion_rows=np.array(
            neb_completion_rows,
            dtype=object,
        ),

        summary=np.array(
            summary_rows,
            dtype=object,
        ),
    )

    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)

    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": (
            grid_ivm_results
        ),

        "optimal_result": (
            optimal_result
        ),

        "neb_results": (
            neb_results
        ),

        "summary": (
            summary_rows
        ),

        "optimal_current_count": (
            optimal_current_count
        ),

        "optimal_complete": (
            optimal_complete
        ),

        "neb_all_complete": (
            neb_all_complete
        ),

        "neb_completion_rows": (
            neb_completion_rows
        ),
    }


# ============================================================
# 判断是否满足自动停止条件
# ============================================================
def should_stop_auto_refresh(
    latest_output,
):
    required_status = []

    if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
        required_status.append(
            bool(
                latest_output[
                    "optimal_complete"
                ]
            )
        )

    if AUTO_STOP_REQUIRE_NEB_COMPLETE:
        required_status.append(
            bool(
                latest_output[
                    "neb_all_complete"
                ]
            )
        )

    if len(required_status) == 0:
        return False

    return all(
        required_status
    )


# ============================================================
# 执行一次或自动刷新
# ============================================================
if AUTO_REFRESH:
    print(
        "AUTO_REFRESH=True："
        f"每 {REFRESH_SECONDS} 秒刷新一次。"
    )

    print(
        "当前会实时读取 Optimal-IVM 和 NEB k-sweep；"
        "NEB 已保存多少个 state 就绘制多少个 state。"
    )

    print(
        "点击 Jupyter 的停止按钮可终止刷新；"
        "不会影响 Terminal 中正在运行的计算。"
    )

    latest_output = None

    try:
        while True:
            if clear_output is not None:
                clear_output(
                    wait=True
                )

            latest_output = (
                load_and_plot_once()
            )

            optimal_current_count = (
                latest_output[
                    "optimal_current_count"
                ]
            )

            optimal_complete = (
                latest_output[
                    "optimal_complete"
                ]
            )

            neb_all_complete = (
                latest_output[
                    "neb_all_complete"
                ]
            )

            if should_stop_auto_refresh(
                latest_output
            ):
                print(
                    "\n所有要求监控的计算均已完成："
                )

                if (
                    AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
                ):
                    print(
                        f"  Optimal-IVM complete="
                        f"{optimal_complete}, "
                        f"count="
                        f"{optimal_current_count}"
                    )

                if (
                    AUTO_STOP_REQUIRE_NEB_COMPLETE
                ):
                    print(
                        f"  NEB k-sweep complete="
                        f"{neb_all_complete}"
                    )

                print(
                    "自动刷新结束。"
                )
                break

            if (
                optimal_current_count < 0
                and not WAIT_FOR_OPTIMAL_FILE
                and AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
            ):
                print(
                    "\nOptimal-IVM 文件尚不存在，"
                    "且 WAIT_FOR_OPTIMAL_FILE=False，"
                    "停止刷新。"
                )
                break

            if (
                len(
                    latest_output[
                        "neb_results"
                    ]
                ) == 0
                and not WAIT_FOR_NEB_FILE
                and AUTO_STOP_REQUIRE_NEB_COMPLETE
            ):
                print(
                    "\nNEB 文件尚不存在，"
                    "且 WAIT_FOR_NEB_FILE=False，"
                    "停止刷新。"
                )
                break

            print(
                f"\n下次刷新将在 "
                f"{REFRESH_SECONDS} 秒后进行。"
            )

            time.sleep(
                REFRESH_SECONDS
            )

    except KeyboardInterrupt:
        print(
            "\n自动刷新已手动停止。"
        )

else:
    latest_output = (
        load_and_plot_once()
    )

In [ ]:
import os
import glob
import re
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)

GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]


# ============================================================
# Optimal-IVM 配置
# ============================================================
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)

OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# FENEB 配置（与新跑的 FENEB 脚本完全一致）
# ============================================================
REPARAM_ALPHA = 1e-3
REPARAM_TOL   = 1e-3

GRID_K = [
    1.0,
]

GRID_FENEB_ALPHA = [
    # 1e-2,
    # 3e-3,
    # 1e-3,
    # 3e-4,
    1e-4,
    3e-5,
    1e-5,
]

FENEB_FMAX_TOL = 0.01
MAX_FENEB_STEPS = 200


# ============================================================
# 选择要绘制的普通 grid IVM 配置
# ============================================================
PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 自动刷新设置
# ============================================================
AUTO_REFRESH = True
REFRESH_SECONDS = 60

# Optimal-IVM 文件尚未创建时是否继续等待
WAIT_FOR_OPTIMAL_FILE = True

# FENEB 文件尚未创建时是否继续等待
WAIT_FOR_FENEB_FILE = True

# 自动停止条件
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True


# ============================================================
# 绘图设置
# ============================================================
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True

SAVE_OUTPUT = True

# 普通 grid IVM
GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

# Optimal-IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

# FENEB 默认样式
FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8


# ============================================================
# 安全读取正在写入的 npz
# ============================================================
def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        f"[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# 普通 grid IVM 配置路径
# ============================================================
def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 数值工具（与原脚本相同）
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def compute_arclength_residual(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            f"full_path must have shape (n,2), "
            f"got {full_path.shape}"
        )

    n_full = full_path.shape[0]
    n_internal = n_full - 2

    if n_internal < 1:
        raise ValueError(
            "The path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    tangent_second_inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        tangent_second_inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand) * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner_phi_prime_second": (
            tangent_second_inner
        ),
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            f"is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    segment_gradients[0] = (
        F_internal[0]
    )

    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            f"Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            f"Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    arc_result = compute_arclength_residual(
        full_path
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "free_energy_max_index": (
            free_energy["max_index"]
        ),

        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result[
                "inner_phi_prime_second"
            ]
        ),

        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后状态
# ============================================================
def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取逐 outer 数据
# ============================================================
def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            f"final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                f"incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                f"expected at least "
                f"{expected_batches}"
            )

    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                f"[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(count_seq) == 0:
        return None, (
            f"no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": valid_index_seq,
        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取完成的普通 grid IVM
# ============================================================
def load_completed_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (N_sub, alpha_IVP, eta_in)
        for N_sub in GRID_N_SUB
        for alpha_IVP in GRID_ALPHA
        for eta_in in GRID_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        if not selected_config(
            N_sub,
            alpha_IVP,
            eta_in,
        ):
            continue

        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=True,
            final_model=final_model,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
        all_configs,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
# ============================================================
def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive 0.04->0.001"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# FENEB 文件路径
# ============================================================
def float_tag(value):
    return f"{float(value):g}"

def feneb_result_paths(k_spring, alpha):
    k_tag = float_tag(k_spring)
    alpha_tag = float_tag(alpha)

    casenum = (
        f"MFEP_FENEB_N{N}"
        f"_k{k_tag}"
        f"_alpha{alpha_tag}"
        f"_reparamAlpha{REPARAM_ALPHA:g}"
        f"_reparamTol{REPARAM_TOL:g}"
        f"_fmax{FENEB_FMAX_TOL:g}"
        f"_steps{MAX_FENEB_STEPS}"
        f"_centralTangent"
        f"_straightInit"
    )

    folder_path = os.path.join(base_path, casenum)
    result_file = os.path.join(folder_path, f"alanine_{casenum}.npz")
    done_file = os.path.join(folder_path, "DONE")

    return (casenum, folder_path, result_file, done_file)


# ============================================================
# 读取单个 FENEB 结果
# ============================================================
def load_single_feneb_result(npz_file):
    data = safe_load_npz(npz_file)
    if data is None:
        return None, f"Could not read {npz_file}"

    required_keys = [
        "count_seq",
        "I_perp_seq",
        "I_perp_euclid_arc_seq",
        "I_arc_seq",
        "Delta_F_seq",
    ]
    missing = [k for k in required_keys if k not in data]
    if missing:
        return None, f"Missing keys: {missing}"

    # 读取原始序列
    count_raw = np.asarray(data["count_seq"], dtype=float)
    I_G_raw = np.asarray(data["I_perp_seq"], dtype=float)
    I_E_raw = np.asarray(data["I_perp_euclid_arc_seq"], dtype=float)
    I_arc_raw = np.asarray(data["I_arc_seq"], dtype=float)
    Delta_F_raw = np.asarray(data["Delta_F_seq"], dtype=float)

    n_states = min(
        len(count_raw),
        len(I_G_raw),
        len(I_E_raw),
        len(I_arc_raw),
        len(Delta_F_raw),
    )
    if n_states == 0:
        return None, "No saved FENEB states"

    count_raw = count_raw[:n_states]
    I_G_raw = I_G_raw[:n_states]
    I_E_raw = I_E_raw[:n_states]
    I_arc_raw = I_arc_raw[:n_states]
    Delta_F_raw = Delta_F_raw[:n_states]

    # 去重
    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
    ) = keep_last_for_duplicate_counts(
        count_raw,
        I_G_raw,
        I_E_raw,
        I_arc_raw,
        Delta_F_raw,
    )

    # 读取元数据
    k_spring = float(np.asarray(data.get("k_spring", [1.0])).reshape(-1)[0])
    alpha = float(np.asarray(data.get("alpha", [1e-2])).reshape(-1)[0])

    folder_path = os.path.dirname(npz_file)
    done_file = os.path.join(folder_path, "DONE")
    done = os.path.exists(done_file)

    result = {
        "method": "FENEB",
        "npz_file": npz_file,
        "folder_path": folder_path,
        "done_file": done_file,
        "done": done,
        "k_spring": k_spring,
        "alpha": alpha,
        "count_seq": count_seq,
        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,
        "Delta_F_seq": Delta_F_seq,
    }
    return result, None


# ============================================================
# 读取所有 FENEB 结果（按 GRID_K x GRID_FENEB_ALPHA）
# ============================================================
def load_all_feneb_results():
    feneb_results = []
    errors = []

    for k_spring in GRID_K:
        for alpha in GRID_FENEB_ALPHA:
            _, _, npz_file, _ = feneb_result_paths(k_spring, alpha)
            result, error = load_single_feneb_result(npz_file)
            if result is not None:
                feneb_results.append(result)
            else:
                errors.append(f"k={k_spring}, alpha={alpha}: {error}")

    return feneb_results, errors


# ============================================================
# 判断 FENEB 是否全部完成
# ============================================================
def evaluate_feneb_completion(feneb_results):
    expected = [(k, a) for k in GRID_K for a in GRID_FENEB_ALPHA]
    status = []
    for k, a in expected:
        matching = [r for r in feneb_results if
                    np.isclose(r["k_spring"], k) and np.isclose(r["alpha"], a)]
        found = len(matching) > 0
        done = bool(matching[0]["done"]) if found else False
        status.append({"k_spring": k, "alpha": a, "found": found, "done": done})
    all_complete = all(row["found"] and row["done"] for row in status)
    return all_complete, status


# ============================================================
# FENEB 绘图样式
# ============================================================
# 不同 k 用不同颜色，不同 alpha 用不同线型
FENEB_K_COLORS = {
    1.0: "#ff7f00",
    20.0: "#00a6d6",
    200.0: "#4daf4a",
}

FENEB_ALPHA_LINESTYLES = {
    1e-2: "-",
    3e-3: "--",
    1e-3: "-.",
    3e-4: ":",
    1e-4: (0, (3, 5, 1, 5)),
    3e-5: (0, (1, 1)),
    1e-5: (0, (5, 10)),
}


def feneb_style(k_spring, alpha, index):
    # 颜色：若 k 在预定义中就用预定义，否则循环 colormap
    if k_spring in FENEB_K_COLORS:
        color = FENEB_K_COLORS[k_spring]
    else:
        cmap = plt.get_cmap("tab10")
        color = cmap(index % 10)

    # 线型：若 alpha 在预定义中就用预定义，否则循环
    if alpha in FENEB_ALPHA_LINESTYLES:
        linestyle = FENEB_ALPHA_LINESTYLES[alpha]
    else:
        styles = ["-", "--", "-.", ":", (0, (3, 5, 1, 5)), (0, (1, 1)), (0, (5, 10))]
        linestyle = styles[index % len(styles)]

    return color, linestyle


def feneb_label(result):
    k = result["k_spring"]
    a = result["alpha"]
    label = f"FENEB k={k:g}, α={a:g}"
    if result["done"]:
        label += " [DONE]"
    else:
        label += " [running]"
    return label


# ============================================================
# 绘制一次
# ============================================================
def load_and_plot_once():
    (
        grid_ivm_results,
        skipped_grid,
        all_configs,
    ) = load_completed_grid_ivm()

    optimal_result, optimal_error = load_optimal_ivm()

    feneb_results, feneb_errors = load_all_feneb_results()
    feneb_all_complete, feneb_completion_rows = evaluate_feneb_completion(feneb_results)

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------
    print("=" * 115)
    print("MFEP IVM / Optimal-IVM / FENEB comparison")
    print("=" * 115)

    print(f"完成的 grid IVM 配置：{len(grid_ivm_results)}")
    print(f"跳过的 grid IVM 配置：{len(skipped_grid)}")

    if optimal_result is not None:
        optimal_current_count = int(np.max(optimal_result["count_seq"]))
        optimal_complete = optimal_current_count >= OPTIMAL_TOTAL_STEPS
        print(f"Optimal-IVM：已读取，当前 count={optimal_current_count}/{OPTIMAL_TOTAL_STEPS}")
        print(f"Optimal-IVM states：{len(optimal_result['count_seq'])}")
        print(f"Optimal-IVM complete：{optimal_complete}")
    else:
        optimal_current_count = -1
        optimal_complete = False
        print("Optimal-IVM：尚未读取到有效数据")
        print("  ", optimal_error)

    print("-" * 115)
    print(f"成功读取的 FENEB 配置数：{len(feneb_results)}")
    if feneb_results:
        for r in feneb_results:
            print(f"FENEB k={r['k_spring']:g}, α={r['alpha']:g} | "
                  f"states={len(r['count_seq']):4d} | "
                  f"count range=({r['count_seq'][0]:g}, {r['count_seq'][-1]:g}) | "
                  f"DONE={r['done']}")
    else:
        print("尚未读取到有效 FENEB 数据。")
    if feneb_errors:
        print(f"FENEB 读取警告数：{len(feneb_errors)}")
        for e in feneb_errors[:5]:
            print("  ", e)

    print("-" * 115)
    print("预期 FENEB 完成状态：")
    if not feneb_completion_rows:
        print("  未指定或未发现预期 FENEB。")
    else:
        for row in feneb_completion_rows:
            print(f"  k={row['k_spring']:g}, α={row['alpha']:g}: "
                  f"found={row['found']}, DONE={row['done']}")
    print(f"FENEB all complete：{feneb_all_complete}")
    print("=" * 115)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------
    tiny = np.finfo(float).tiny

    all_available = list(grid_ivm_results)
    if optimal_result is not None:
        all_available.append(optimal_result)
    all_available.extend(feneb_results)

    for res in all_available:
        res["I_perp_G_plot"] = np.maximum(res["I_perp_G_seq"], tiny)
        res["I_perp_E_plot"] = np.maximum(res["I_perp_E_seq"], tiny)
        res["I_arc_plot"] = np.maximum(res["I_arc_seq"], tiny)
        res["Delta_F_plot"] = np.maximum(res["Delta_F_seq"], tiny)

    # --------------------------------------------------------
    # 创建 1 x 4 图
    # --------------------------------------------------------
    plt.close("all")
    fig, axes = plt.subplots(1, 4, figsize=(27, 6.5))
    ax_G, ax_E, ax_arc, ax_DF = axes

    # --------------------------------------------------------
    # 普通 grid IVM
    # --------------------------------------------------------
    alpha_colors = {
        0.1: "#377eb8",
        1: "#4daf4a",
        10: "#984ea3",
    }
    eta_linestyles = {
        0.03: "-",
        0.01: "--",
        0.003: "-.",
        0.001: ":",
    }
    nsub_markers = {
        4: "o",
        5: "s",
        10: "^",
    }

    for result in grid_ivm_results:
        N_sub = result["N_sub"]
        alpha_IVP = result["alpha_IVP"]
        eta_in = result["eta_in"]
        count_seq = result["count_seq"]

        color = alpha_colors.get(alpha_IVP, "gray")
        linestyle = eta_linestyles.get(eta_in, "-")
        marker = nsub_markers.get(N_sub, "o")
        markevery = max(1, len(count_seq) // 8)

        common_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=3.5,
            linewidth=GRID_IVM_LINEWIDTH,
            alpha=GRID_IVM_ALPHA,
            zorder=2,
        )

        ax_G.plot(count_seq, result["I_perp_G_plot"], **common_kwargs)
        ax_E.plot(count_seq, result["I_perp_E_plot"], **common_kwargs)
        ax_arc.plot(count_seq, result["I_arc_plot"], **common_kwargs)
        ax_DF.plot(count_seq, result["Delta_F_plot"], **common_kwargs)

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------
    if optimal_result is not None:
        optimal_count = optimal_result["count_seq"]
        optimal_markevery = max(1, len(optimal_count) // 12)

        optimal_kwargs = dict(
            color=OPTIMAL_COLOR,
            linestyle=OPTIMAL_LINESTYLE,
            marker=OPTIMAL_MARKER,
            markevery=optimal_markevery,
            markersize=8,
            markerfacecolor=OPTIMAL_COLOR,
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=OPTIMAL_LINEWIDTH,
            alpha=1.0,
            zorder=30,
        )

        ax_G.plot(optimal_count, optimal_result["I_perp_G_plot"], **optimal_kwargs)
        ax_E.plot(optimal_count, optimal_result["I_perp_E_plot"], **optimal_kwargs)
        ax_arc.plot(optimal_count, optimal_result["I_arc_plot"], **optimal_kwargs)
        ax_DF.plot(optimal_count, optimal_result["Delta_F_plot"], **optimal_kwargs)

        for axis, values in [
            (ax_G, optimal_result["I_perp_G_plot"]),
            (ax_E, optimal_result["I_perp_E_plot"]),
            (ax_arc, optimal_result["I_arc_plot"]),
            (ax_DF, optimal_result["Delta_F_plot"]),
        ]:
            axis.scatter(
                optimal_count[-1], values[-1],
                s=130, marker="*", color=OPTIMAL_COLOR,
                edgecolor="black", linewidth=0.8, zorder=40,
            )

    # --------------------------------------------------------
    # FENEB 曲线
    # --------------------------------------------------------
    for idx, result in enumerate(feneb_results):
        count_seq = result["count_seq"]
        color, linestyle = feneb_style(result["k_spring"], result["alpha"], idx)
        markevery = max(1, len(count_seq) // 12)

        feneb_kwargs = dict(
            color=color,
            linestyle=linestyle,
            marker=FENEB_MARKER,
            markevery=markevery,
            markersize=5.5,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=1.2,
            linewidth=FENEB_LINEWIDTH,
            alpha=1.0,
            zorder=20 + idx,
        )

        ax_G.plot(count_seq, result["I_perp_G_plot"], **feneb_kwargs)
        ax_E.plot(count_seq, result["I_perp_E_plot"], **feneb_kwargs)
        ax_arc.plot(count_seq, result["I_arc_plot"], **feneb_kwargs)
        ax_DF.plot(count_seq, result["Delta_F_plot"], **feneb_kwargs)

        # 最新状态标记
        latest_marker = "D" if result["done"] else "o"
        for axis, values in [
            (ax_G, result["I_perp_G_plot"]),
            (ax_E, result["I_perp_E_plot"]),
            (ax_arc, result["I_arc_plot"]),
            (ax_DF, result["Delta_F_plot"]),
        ]:
            axis.scatter(
                count_seq[-1], values[-1],
                s=75, marker=latest_marker,
                facecolor=color, edgecolor="black",
                linewidth=0.8, zorder=50 + idx,
            )

    # ========================================================
    # Panel 设置
    # ========================================================
    ax_G.set_title("Riemannian normal residual", fontsize=14)
    ax_G.set_xlabel("count_seq", fontsize=12)
    ax_G.set_ylabel(r"$\widehat I_{\perp}^{G}$" "\n" r"$=\int |P_\varphi^G M\nabla F|_G^2 |\varphi'|_G\,ds$", fontsize=10)

    ax_E.set_title("Euclidean residual integral", fontsize=14)
    ax_E.set_xlabel("count_seq", fontsize=12)
    ax_E.set_ylabel(r"$\widehat I_{\perp}^{E}$" "\n" r"$=\int \|P_\varphi^G M\nabla F\|_2^2 |\varphi'|_2\,ds$", fontsize=10)

    ax_arc.set_title("Arclength / equi-speed residual", fontsize=14)
    ax_arc.set_xlabel("count_seq", fontsize=12)
    ax_arc.set_ylabel(r"$I_{\rm arc}$" "\n" r"$=\int_0^1 (\varphi'\cdot\varphi'')^2 |\varphi'|\,ds$", fontsize=10)

    ax_DF.set_title("Path-integrated free-energy barrier", fontsize=14)
    ax_DF.set_xlabel("count_seq", fontsize=12)
    ax_DF.set_ylabel(r"$\Delta F^{\dagger}$ (kcal/mol)" "\n" "line-integral estimate", fontsize=10)

    for axis in axes:
        axis.grid(True, which="both", linestyle="--", alpha=0.3)
        axis.set_xlim(left=0)

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log")
        ax_E.set_yscale("log")
    if USE_LOG_Y_ARC:
        ax_arc.set_yscale("log")
    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # ========================================================
    # 图例
    # ========================================================
    method_handles = [
        Line2D([0], [0], color="gray", linewidth=1.5,
               alpha=GRID_IVM_ALPHA, label="Grid IVM")
    ]
    if optimal_result is not None:
        method_handles.append(
            Line2D([0], [0], color=OPTIMAL_COLOR, linestyle=OPTIMAL_LINESTYLE,
                   marker=OPTIMAL_MARKER, linewidth=OPTIMAL_LINEWIDTH,
                   markersize=8, label="Optimal-IVM")
        )

    for idx, result in enumerate(feneb_results):
        color, linestyle = feneb_style(result["k_spring"], result["alpha"], idx)
        method_handles.append(
            Line2D([0], [0], color=color, linestyle=linestyle,
                   marker=FENEB_MARKER, markerfacecolor="white",
                   markeredgecolor=color, linewidth=FENEB_LINEWIDTH,
                   markersize=6, label=feneb_label(result))
        )

    legend_method = ax_DF.legend(
        handles=method_handles, title="Method",
        loc="upper left", bbox_to_anchor=(1.03, 1.0), frameon=True,
    )
    ax_DF.add_artist(legend_method)

    # 仍保留 IVM 样式的辅助图例
    alpha_handles = [
        Line2D([0], [0], color=alpha_colors[alpha], linewidth=2,
               label=rf"$\alpha={alpha}$")
        for alpha in GRID_ALPHA
    ]
    legend_alpha = ax_DF.legend(
        handles=alpha_handles, title="Grid IVM color",
        loc="upper left", bbox_to_anchor=(1.03, 0.68), frameon=True,
    )
    ax_DF.add_artist(legend_alpha)

    eta_handles = [
        Line2D([0], [0], color="gray", linestyle=eta_linestyles[eta],
               linewidth=2, label=rf"$\eta={eta}$")
        for eta in GRID_ETA
    ]
    legend_eta = ax_DF.legend(
        handles=eta_handles, title="Grid IVM line",
        loc="upper left", bbox_to_anchor=(1.03, 0.36), frameon=True,
    )
    ax_DF.add_artist(legend_eta)

    nsub_handles = [
        Line2D([0], [0], color="gray", linestyle="None",
               marker=nsub_markers[N_sub], markersize=6,
               label=rf"$N_{{sub}}={N_sub}$")
        for N_sub in GRID_N_SUB
    ]
    ax_DF.legend(
        handles=nsub_handles, title="Grid IVM marker",
        loc="upper left", bbox_to_anchor=(1.03, 0.05), frameon=True,
    )

    fig.subplots_adjust(
        left=0.055, right=0.80, bottom=0.14, top=0.90, wspace=0.38,
    )

    # ========================================================
    # 保存图片
    # ========================================================
    output_pdf = os.path.join(
        base_path,
        "MFEP_OptimalIVM_GridIVM_FENEB_four_metrics.pdf",
    )
    output_png = os.path.join(
        base_path,
        "MFEP_OptimalIVM_GridIVM_FENEB_four_metrics.png",
    )

    if SAVE_OUTPUT:
        fig.savefig(output_pdf, bbox_inches="tight")
        fig.savefig(output_png, dpi=300, bbox_inches="tight")

    plt.show()

    # ========================================================
    # 当前值汇总
    # ========================================================
    print("\n" + "=" * 150)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 150)

    summary_rows = []
    for result in grid_ivm_results:
        summary_rows.append({
            "method": "IVM-grid",
            "description": f"Nsub={result['N_sub']} alpha={result['alpha_IVP']} eta={result['eta_in']}",
            "count": float(result["count_seq"][-1]),
            "I_G": float(result["I_perp_G_seq"][-1]),
            "I_E": float(result["I_perp_E_seq"][-1]),
            "I_arc": float(result["I_arc_seq"][-1]),
            "Delta_F": float(result["Delta_F_seq"][-1]),
            "done": True,
        })

    if optimal_result is not None:
        summary_rows.append({
            "method": "Optimal-IVM",
            "description": "Nsub=4 alpha=0.1 adaptive Adam",
            "count": float(optimal_result["count_seq"][-1]),
            "I_G": float(optimal_result["I_perp_G_seq"][-1]),
            "I_E": float(optimal_result["I_perp_E_seq"][-1]),
            "I_arc": float(optimal_result["I_arc_seq"][-1]),
            "Delta_F": float(optimal_result["Delta_F_seq"][-1]),
            "done": optimal_complete,
        })

    for result in feneb_results:
        summary_rows.append({
            "method": "FENEB",
            "description": f"k={result['k_spring']:g} α={result['alpha']:g}",
            "count": float(result["count_seq"][-1]),
            "I_G": float(result["I_perp_G_seq"][-1]),
            "I_E": float(result["I_perp_E_seq"][-1]),
            "I_arc": float(result["I_arc_seq"][-1]),
            "Delta_F": float(result["Delta_F_seq"][-1]),
            "done": bool(result["done"]),
        })

    method_order = {"Optimal-IVM": 0, "FENEB": 1, "IVM-grid": 2}
    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (method_order.get(row["method"], 99), row["I_G"]),
    )

    for row in summary_rows_sorted:
        print(f"{row['method']:<13} {row['description']:<39}"
              f" count={row['count']:>7.1f}"
              f" | I_G={row['I_G']:.6e}"
              f" | I_E={row['I_E']:.6e}"
              f" | I_arc={row['I_arc']:.6e}"
              f" | DeltaF={row['Delta_F']:.6f}"
              f" | done={row['done']}")

    # 保存汇总数据
    comparison_results_file = os.path.join(
        base_path,
        "MFEP_OPTIMAL_IVM_GRID_IVM_FENEB_FOUR_METRICS.npz",
    )
    np.savez(
        comparison_results_file,
        grid_ivm_results=np.array(grid_ivm_results, dtype=object),
        optimal_ivm_result=np.array([optimal_result], dtype=object),
        feneb_results=np.array(feneb_results, dtype=object),
        feneb_completion_rows=np.array(feneb_completion_rows, dtype=object),
        summary=np.array(summary_rows, dtype=object),
    )
    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)
    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": grid_ivm_results,
        "optimal_result": optimal_result,
        "feneb_results": feneb_results,
        "summary": summary_rows,
        "optimal_current_count": optimal_current_count,
        "optimal_complete": optimal_complete,
        "feneb_all_complete": feneb_all_complete,
        "feneb_completion_rows": feneb_completion_rows,
    }


# ============================================================
# 判断是否满足自动停止条件
# ============================================================
def should_stop_auto_refresh(latest_output):
    required = []
    if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
        required.append(bool(latest_output["optimal_complete"]))
    if AUTO_STOP_REQUIRE_FENEB_COMPLETE:
        required.append(bool(latest_output["feneb_all_complete"]))
    if not required:
        return False
    return all(required)


# ============================================================
# 执行一次或自动刷新
# ============================================================
if AUTO_REFRESH:
    print("AUTO_REFRESH=True："
          f"每 {REFRESH_SECONDS} 秒刷新一次。")
    print("当前会实时读取 Optimal-IVM 和 FENEB；"
          "已保存多少个 state 就绘制多少个 state。")
    print("点击 Jupyter 的停止按钮可终止刷新。")
    latest_output = None
    try:
        while True:
            if clear_output is not None:
                clear_output(wait=True)
            latest_output = load_and_plot_once()

            optimal_complete = latest_output["optimal_complete"]
            feneb_all_complete = latest_output["feneb_all_complete"]

            if should_stop_auto_refresh(latest_output):
                print("\n所有要求监控的计算均已完成：")
                if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
                    print(f"  Optimal-IVM complete={optimal_complete}")
                if AUTO_STOP_REQUIRE_FENEB_COMPLETE:
                    print(f"  FENEB all complete={feneb_all_complete}")
                print("自动刷新结束。")
                break

            if (latest_output["optimal_current_count"] < 0
                    and not WAIT_FOR_OPTIMAL_FILE
                    and AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE):
                print("\nOptimal-IVM 文件尚不存在，"
                      "且 WAIT_FOR_OPTIMAL_FILE=False，停止刷新。")
                break

            if (len(latest_output["feneb_results"]) == 0
                    and not WAIT_FOR_FENEB_FILE
                    and AUTO_STOP_REQUIRE_FENEB_COMPLETE):
                print("\nFENEB 文件尚不存在，"
                      "且 WAIT_FOR_FENEB_FILE=False，停止刷新。")
                break

            print(f"\n下次刷新将在 {REFRESH_SECONDS} 秒后进行。")
            time.sleep(REFRESH_SECONDS)
    except KeyboardInterrupt:
        print("\n自动刷新已手动停止。")
else:
    latest_output = load_and_plot_once()

In [ ]:

import os
import glob
import re
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与端点
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array([-2.60866606, 2.77661136], dtype=float)
C2 = np.array([-1.3304716, 0.99118795], dtype=float)

# ------------------------------------------------------------
# Optimal‑IVM 固定配置
# ------------------------------------------------------------
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(base_path, OPTIMAL_CASENUM)
OPTIMAL_NPZ = os.path.join(OPTIMAL_FOLDER, f"alanine_{OPTIMAL_CASENUM}.npz")

# ------------------------------------------------------------
# FENEB 参数（必须与 FENEB 运行脚本完全一致）
# ------------------------------------------------------------
REPARAM_ALPHA = 1e-3
REPARAM_TOL   = 1e-3

GRID_K = [1.0]                     # 固定 k
GRID_FENEB_ALPHA = [1e-4, 3e-5, 1e-5]   # 当前正在/已经跑完的 α 列表
FENEB_FMAX_TOL = 0.01
MAX_FENEB_STEPS = 200

# ------------------------------------------------------------
# 自动刷新设置
# ------------------------------------------------------------
AUTO_REFRESH = True                # 是否定时刷新（Jupyter 中效果最好）
REFRESH_SECONDS = 60               # 刷新间隔秒数
WAIT_FOR_OPTIMAL_FILE = True
WAIT_FOR_FENEB_FILE = True
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True

# ------------------------------------------------------------
# 绘图样式
# ------------------------------------------------------------
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True
SAVE_OUTPUT = True

GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8

# ============================================================
# 安全读取正在写入的 npz
# ============================================================
def safe_load_npz(npz_file, retries=10, wait_seconds=2.0):
    if not os.path.exists(npz_file):
        return None
    last_error = None
    for attempt in range(retries):
        try:
            with np.load(npz_file, allow_pickle=True) as loaded:
                return {key: np.array(loaded[key], copy=True) for key in loaded.files}
        except (zipfile.BadZipFile, EOFError, OSError, ValueError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(wait_seconds)
    print(f"[WARNING] Cannot read npz: {npz_file}\n  error: {last_error}")
    return None


# ============================================================
# 数值工具（与运行脚本完全一致）
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(full_path, dtype=float)
    return full_path[2:] - full_path[:-2]

def compute_G_projectors(tangent, M_all):
    tangent = np.asarray(tangent, dtype=float)
    M_all = np.asarray(M_all, dtype=float)
    n_internal = tangent.shape[0]
    G_all = np.linalg.inv(M_all)
    identity = np.eye(2, dtype=float)
    P_parallel_all = np.zeros((n_internal, 2, 2))
    P_perp_all = np.zeros((n_internal, 2, 2))
    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]
        norm_G_sq = float(v_i @ G_i @ v_i)
        if norm_G_sq <= 1e-30:
            raise FloatingPointError(f"Near-zero G tangent norm at image {i}")
        tau_G = v_i / np.sqrt(norm_G_sq)
        G_tau_G = G_i @ tau_G
        denom = float(tau_G @ G_tau_G)
        P_parallel = np.outer(tau_G, G_tau_G) / max(denom, 1e-30)
        P_parallel_all[i] = P_parallel
        P_perp_all[i] = identity - P_parallel
    return G_all, P_parallel_all, P_perp_all

def local_G_arclength_weights(full_path, G_all):
    full_path = np.asarray(full_path, dtype=float)
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    length_minus = np.sqrt(np.maximum(np.einsum("ni,nij,nj->n", seg_minus, G_all, seg_minus), 0.0))
    length_plus  = np.sqrt(np.maximum(np.einsum("ni,nij,nj->n", seg_plus,  G_all, seg_plus),  0.0))
    return 0.5 * (length_minus + length_plus)

def local_euclidean_arclength_weights(full_path):
    full_path = np.asarray(full_path, dtype=float)
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus  = full_path[2:] - path_internal
    return 0.5 * (np.linalg.norm(seg_minus, axis=1) + np.linalg.norm(seg_plus, axis=1))

def compute_arclength_residual(full_path):
    full_path = np.asarray(full_path, dtype=float)
    n_full = full_path.shape[0]
    ds = 1.0 / (n_full - 1)
    phi_prime = (full_path[2:] - full_path[:-2]) / (2.0 * ds)
    phi_second = (full_path[2:] - 2.0 * full_path[1:-1] + full_path[:-2]) / (ds**2)
    inner = np.einsum("ni,ni->n", phi_prime, phi_second)
    speed = np.linalg.norm(phi_prime, axis=1)
    I_arc = float(np.sum(inner**2 * speed) * ds)
    return {"I_arc": I_arc, "integrand": inner**2 * speed, "speed": speed, "inner": inner}

def recover_free_energy_profile(full_path, F_internal):
    full_path = np.asarray(full_path, dtype=float)
    F_internal = np.asarray(F_internal, dtype=float)
    n_internal = full_path.shape[0] - 2
    segments = full_path[1:] - full_path[:-1]
    seg_grad = np.zeros((n_internal+1, 2))
    seg_grad[0] = F_internal[0]
    if n_internal > 1:
        seg_grad[1:n_internal] = 0.5 * (F_internal[:-1] + F_internal[1:])
    seg_grad[n_internal] = F_internal[-1]
    increments = np.einsum("ni,ni->n", seg_grad, segments)
    profile = np.zeros(n_internal+2)
    profile[1:] = np.cumsum(increments)
    profile -= profile[0]
    barrier = float(np.max(profile) - profile[0])
    return {"profile": profile, "barrier": barrier, "final_delta_F": float(profile[-1] - profile[0])}

def compute_all_path_metrics(full_path, M_all, F_all):
    tangent = central_difference_tangent(full_path)
    G_all, _, P_perp_G_all = compute_G_projectors(tangent, M_all)
    Mg_all = np.einsum("nij,nj->ni", M_all, F_all)
    perpG_Mg = np.einsum("nij,nj->ni", P_perp_G_all, Mg_all)
    res_G_sq = np.maximum(np.einsum("ni,nij,nj->n", perpG_Mg, G_all, perpG_Mg), 0.0)
    weights_G = local_G_arclength_weights(full_path, G_all)
    I_perp_G = float(np.sum(res_G_sq * weights_G))
    res_E_sq = np.sum(perpG_Mg**2, axis=1)
    weights_E = local_euclidean_arclength_weights(full_path)
    I_perp_E = float(np.sum(res_E_sq * weights_E))
    fe = recover_free_energy_profile(full_path, F_all)
    arc = compute_arclength_residual(full_path)
    return {
        "I_perp_G": I_perp_G, "I_perp_E": I_perp_E,
        "Delta_F_barrier": fe["barrier"], "Delta_F_final": fe["final_delta_F"],
        "I_arc": arc["I_arc"]
    }


# ============================================================
# 去重：同一 count 只保留最后一个状态
# ============================================================
def keep_last_for_duplicate_counts(count_seq, *arrays):
    count_seq = np.asarray(count_seq)
    if count_seq.size == 0:
        return (count_seq,) + arrays
    last_idx = {}
    for i, c in enumerate(count_seq):
        last_idx[float(c)] = i
    selected = np.array(sorted(last_idx.values(), key=lambda idx: count_seq[idx]), dtype=int)
    output = [count_seq[selected]]
    for arr in arrays:
        output.append(np.asarray(arr)[selected])
    return tuple(output)


# ============================================================
# 从 IVM npz 读取路径数据（支持未完成）
# ============================================================
def load_ivm_result(npz_file, method_name, N_sub, alpha_IVP, eta_description):
    if not os.path.exists(npz_file):
        return None, "npz does not exist"
    data = safe_load_npz(npz_file)
    if data is None:
        return None, "could not read"
    if "index_seq" not in data:
        return None, "index_seq missing"

    index_seq_saved = np.asarray(data["index_seq"], dtype=int)
    if index_seq_saved.size == 0:
        return None, "empty index_seq"

    saved_count_seq = np.asarray(data["count_seq"], dtype=int) if "count_seq" in data else index_seq_saved * N_sub
    n_avail = min(len(index_seq_saved), len(saved_count_seq))
    index_seq_saved = index_seq_saved[:n_avail]
    saved_count_seq = saved_count_seq[:n_avail]

    count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq = [], [], [], [], []
    for pos in range(n_avail):
        outer = int(index_seq_saved[pos])
        cnt = int(saved_count_seq[pos])
        phi_key = f"phi_ref_iter_{outer}"
        M_key   = f"M_ref_iter_{outer}"
        F_key   = f"F_ref_iter_{outer}"
        g_key   = f"g_ref_iter_{outer}"
        sel_F = F_key if F_key in data else (g_key if g_key in data else None)
        if phi_key not in data or M_key not in data or sel_F is None:
            continue
        try:
            phi_internal = np.asarray(data[phi_key], dtype=float)
            M_all = np.asarray(data[M_key], dtype=float)
            F_all = np.asarray(data[sel_F], dtype=float)
            full_path = np.vstack([C1[None,:], phi_internal, C2[None,:]])
            metrics = compute_all_path_metrics(full_path, M_all, F_all)
        except Exception as e:
            continue
        count_seq.append(cnt)
        I_G_seq.append(metrics["I_perp_G"])
        I_E_seq.append(metrics["I_perp_E"])
        I_arc_seq.append(metrics["I_arc"])
        Delta_F_seq.append(metrics["Delta_F_barrier"])

    if not count_seq:
        return None, "no valid states"

    (count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq) = keep_last_for_duplicate_counts(
        np.array(count_seq, dtype=int),
        np.array(I_G_seq, dtype=float),
        np.array(I_E_seq, dtype=float),
        np.array(I_arc_seq, dtype=float),
        np.array(Delta_F_seq, dtype=float)
    )

    result = {
        "method": method_name,
        "N_sub": N_sub, "alpha_IVP": alpha_IVP, "eta_description": eta_description,
        "npz_file": npz_file, "count_seq": count_seq,
        "I_perp_G_seq": I_G_seq, "I_perp_E_seq": I_E_seq,
        "I_arc_seq": I_arc_seq, "Delta_F_seq": Delta_F_seq,
    }
    return result, None


# ============================================================
# 自动扫描所有 IVM grid 配置，但只保留 alpha = 1e-4 的组合
# ============================================================
def load_all_grid_ivm(only_alpha=1e-4):
    pattern = os.path.join(base_path, "MFEP_image_Adam_N20_Nsub*_alpha*_lr*_NEBreparam")
    folders = sorted(glob.glob(pattern))
    results = []
    for folder in folders:
        casenum = os.path.basename(folder)
        m = re.search(r"Nsub(\d+)_alpha([\d.e+-]+)_lr([\d.e+-]+)_NEBreparam", casenum)
        if not m:
            continue
        N_sub = int(m.group(1))
        alpha = float(m.group(2))
        lr = float(m.group(3))
        # 只保留 alpha == 1e-4 的配置
        if not np.isclose(alpha, only_alpha, rtol=1e-12):
            continue
        npz_file = os.path.join(folder, f"alanine_{casenum}.npz")
        final_model = os.path.join(folder, f"model_MFEP_alanine_{casenum}_final.pth")
        result, err = load_ivm_result(npz_file, "IVM-grid", N_sub, alpha, str(lr))
        if result is not None:
            result["eta_in"] = lr
            result["casenum"] = casenum
            results.append(result)
    return results


# ============================================================
# Optimal‑IVM
# ============================================================
def load_optimal_ivm():
    result, err = load_ivm_result(OPTIMAL_NPZ, "Optimal-IVM", OPTIMAL_N_SUB, OPTIMAL_ALPHA, "adaptive 0.04→0.001")
    if result is not None:
        result["casenum"] = OPTIMAL_CASENUM
    return result, err


# ============================================================
# FENEB 路径（已包含 reparamAlpha / reparamTol）
# ============================================================
def float_tag(v):
    return f"{float(v):g}"

def feneb_result_paths(k_spring, alpha):
    k_tag = float_tag(k_spring)
    a_tag = float_tag(alpha)
    casenum = (
        f"MFEP_FENEB_N{N}"
        f"_k{k_tag}"
        f"_alpha{a_tag}"
        f"_reparamAlpha{REPARAM_ALPHA:g}"
        f"_reparamTol{REPARAM_TOL:g}"
        f"_fmax{FENEB_FMAX_TOL:g}"
        f"_steps{MAX_FENEB_STEPS}"
        f"_centralTangent"
        f"_straightInit"
    )
    folder_path = os.path.join(base_path, casenum)
    result_file = os.path.join(folder_path, f"alanine_{casenum}.npz")
    done_file = os.path.join(folder_path, "DONE")
    return casenum, folder_path, result_file, done_file

def load_single_feneb_result(npz_file):
    data = safe_load_npz(npz_file)
    if data is None:
        return None, f"Could not read {npz_file}"
    required = ["count_seq", "I_perp_seq", "I_perp_euclid_arc_seq", "I_arc_seq", "Delta_F_seq"]
    missing = [k for k in required if k not in data]
    if missing:
        return None, f"Missing keys: {missing}"

    count_raw = np.asarray(data["count_seq"], dtype=float)
    I_G_raw   = np.asarray(data["I_perp_seq"], dtype=float)
    I_E_raw   = np.asarray(data["I_perp_euclid_arc_seq"], dtype=float)
    I_arc_raw = np.asarray(data["I_arc_seq"], dtype=float)
    Delta_F_raw = np.asarray(data["Delta_F_seq"], dtype=float)

    n = min(len(count_raw), len(I_G_raw), len(I_E_raw), len(I_arc_raw), len(Delta_F_raw))
    if n == 0:
        return None, "No FENEB states"
    count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw = [arr[:n] for arr in [count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw]]
    (count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq) = keep_last_for_duplicate_counts(
        count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw
    )

    k = float(np.asarray(data.get("k_spring", [1.0])).ravel()[0])
    a = float(np.asarray(data.get("alpha", [1e-2])).ravel()[0])
    done = os.path.exists(os.path.join(os.path.dirname(npz_file), "DONE"))

    return {
        "method": "FENEB", "npz_file": npz_file, "k_spring": k, "alpha": a,
        "count_seq": count_seq, "I_perp_G_seq": I_G_seq, "I_perp_E_seq": I_E_seq,
        "I_arc_seq": I_arc_seq, "Delta_F_seq": Delta_F_seq, "done": done
    }, None

def load_all_feneb_results():
    results = []
    for k in GRID_K:
        for a in GRID_FENEB_ALPHA:
            _, _, npz_file, _ = feneb_result_paths(k, a)
            res, err = load_single_feneb_result(npz_file)
            if res is not None:
                results.append(res)
    return results


# ============================================================
# FENEB 样式
# ============================================================
FENEB_K_COLORS = {1.0: "#ff7f00"}
FENEB_ALPHA_LINESTYLES = {
    1e-2: "-", 3e-3: "--", 1e-3: "-.", 3e-4: ":",
    1e-4: (0, (3, 5, 1, 5)), 3e-5: (0, (1, 1)), 1e-5: (0, (5, 10)),
}

def feneb_style(k_spring, alpha, idx):
    color = FENEB_K_COLORS.get(k_spring, plt.cm.tab10(idx % 10))
    linestyle = FENEB_ALPHA_LINESTYLES.get(alpha, ["-","--","-.",":"][idx % 4])
    return color, linestyle

def feneb_label(res):
    done = " [DONE]" if res["done"] else " [running]"
    return f"FENEB k={res['k_spring']:g}, α={res['alpha']:g}{done}"


# ============================================================
# 绘图主函数（只绘制 Grid IVM 中 alpha=1e-4 的曲线）
# ============================================================
def load_and_plot_once():
    grid_ivm_results = load_all_grid_ivm(only_alpha=1e-4)
    optimal_result, optimal_err = load_optimal_ivm()
    feneb_results = load_all_feneb_results()

    # 汇总状态
    print("=" * 115)
    print("MFEP IVM (α=1e-4 only) / Optimal-IVM / FENEB comparison")
    print("=" * 115)
    print(f"扫描到的 Grid IVM (α=1e-4) 配置：{len(grid_ivm_results)}")
    if optimal_result is not None:
        opt_cnt = int(np.max(optimal_result["count_seq"]))
        opt_complete = opt_cnt >= OPTIMAL_TOTAL_STEPS
        print(f"Optimal-IVM count={opt_cnt}/{OPTIMAL_TOTAL_STEPS}, states={len(optimal_result['count_seq'])}, complete={opt_complete}")
    else:
        opt_complete = False
        print("Optimal-IVM 暂无数据")

    print(f"已加载 FENEB 配置：{len(feneb_results)}")
    for r in feneb_results:
        print(f"  FENEB k={r['k_spring']:g}, α={r['alpha']:g} | states={len(r['count_seq'])} | DONE={r['done']}")

    # 预处理对数安全值
    tiny = np.finfo(float).tiny
    all_data = list(grid_ivm_results)
    if optimal_result: all_data.append(optimal_result)
    all_data.extend(feneb_results)
    for d in all_data:
        d["I_perp_G_plot"] = np.maximum(d["I_perp_G_seq"], tiny)
        d["I_perp_E_plot"] = np.maximum(d["I_perp_E_seq"], tiny)
        d["I_arc_plot"]    = np.maximum(d["I_arc_seq"],    tiny)
        d["Delta_F_plot"]  = np.maximum(d["Delta_F_seq"],  tiny)

    # 建图
    plt.close("all")
    fig, axes = plt.subplots(1, 4, figsize=(27, 6.5))
    ax_G, ax_E, ax_arc, ax_DF = axes

    # 针对 alpha=0.001 的样式：用 N_sub 区分标记，用 eta 区分数线型
    nsub_markers = {4: "o", 5: "s", 10: "^", 3: "v", 2: "d", 6: "h", 8: "p"}
    eta_linestyles = {0.03: "-", 0.01: "--", 0.003: "-.", 0.001: ":"}
    # 颜色按 N_sub 分配不同颜色，以更好区分
    nsub_colors = {3: "#1f77b4", 4: "#ff7f0e", 5: "#2ca02c", 10: "#d62728"}

    for res in grid_ivm_results:
        ns, et = res["N_sub"], res["eta_in"]
        color = nsub_colors.get(ns, "gray")
        linestyle = eta_linestyles.get(et, "-")
        marker = nsub_markers.get(ns, "o")
        markevery = max(1, len(res["count_seq"]) // 8)
        kw = dict(color=color, linestyle=linestyle, marker=marker, markevery=markevery,
                  markersize=3.5, linewidth=GRID_IVM_LINEWIDTH, alpha=GRID_IVM_ALPHA, zorder=2)
        for ax, key in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                        (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(res["count_seq"], res[key], **kw)

    # --- Optimal-IVM ---
    if optimal_result is not None:
        oc = optimal_result["count_seq"]
        mev = max(1, len(oc)//12)
        okw = dict(color=OPTIMAL_COLOR, linestyle=OPTIMAL_LINESTYLE, marker=OPTIMAL_MARKER,
                   markevery=mev, markersize=8, markerfacecolor=OPTIMAL_COLOR,
                   markeredgecolor="white", markeredgewidth=0.8,
                   linewidth=OPTIMAL_LINEWIDTH, alpha=1.0, zorder=30)
        for ax, key in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                        (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(oc, optimal_result[key], **okw)
            ax.scatter(oc[-1], optimal_result[key][-1], s=130, marker="*",
                       color=OPTIMAL_COLOR, edgecolor="black", linewidth=0.8, zorder=40)

    # --- FENEB ---
    for idx, res in enumerate(feneb_results):
        cs = res["count_seq"]
        col, ls = feneb_style(res["k_spring"], res["alpha"], idx)
        mev = max(1, len(cs)//12)
        fkw = dict(color=col, linestyle=ls, marker=FENEB_MARKER,
                   markevery=mev, markersize=5.5, markerfacecolor="white",
                   markeredgecolor=col, markeredgewidth=1.2,
                   linewidth=FENEB_LINEWIDTH, alpha=1.0, zorder=20+idx)
        for ax, key in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                        (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(cs, res[key], **fkw)
            marker = "D" if res["done"] else "o"
            ax.scatter(cs[-1], res[key][-1], s=75, marker=marker,
                       facecolor=col, edgecolor="black", linewidth=0.8, zorder=50+idx)

    # 面板装饰
    titles = ["Riemannian normal residual", "Euclidean residual integral",
              "Arclength / equi-speed residual", "Path-integrated free-energy barrier"]
    ylabels = [r"$\widehat I_{\perp}^{G}$", r"$\widehat I_{\perp}^{E}$",
               r"$I_{\rm arc}$", r"$\Delta F^{\dagger}$ (kcal/mol)"]
    for ax, t, yl in zip(axes, titles, ylabels):
        ax.set_title(t, fontsize=14)
        ax.set_xlabel("count_seq", fontsize=12)
        ax.set_ylabel(yl, fontsize=10)
        ax.grid(True, which="both", linestyle="--", alpha=0.3)
        ax.set_xlim(left=0)
    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log"); ax_E.set_yscale("log")
    if USE_LOG_Y_ARC:
        ax_arc.set_yscale("log")
    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # 图例
    method_handles = []
    # Grid IVM (α=1e-4) 图例
    for ns in sorted(set(r["N_sub"] for r in grid_ivm_results)):
        method_handles.append(
            Line2D([0],[0], color=nsub_colors.get(ns, "gray"), linewidth=GRID_IVM_LINEWIDTH,
                   alpha=GRID_IVM_ALPHA, label=f"IVM α=1e-4, Nsub={ns}")
        )
    if optimal_result:
        method_handles.append(Line2D([0],[0], color=OPTIMAL_COLOR, linestyle=OPTIMAL_LINESTYLE,
                                     marker=OPTIMAL_MARKER, linewidth=OPTIMAL_LINEWIDTH,
                                     markersize=8, label="Optimal-IVM"))
    for idx, r in enumerate(feneb_results):
        c, ls = feneb_style(r["k_spring"], r["alpha"], idx)
        method_handles.append(Line2D([0],[0], color=c, linestyle=ls, marker=FENEB_MARKER,
                                     markerfacecolor="white", markeredgecolor=c,
                                     linewidth=FENEB_LINEWIDTH, markersize=6, label=feneb_label(r)))

    legend1 = ax_DF.legend(handles=method_handles, title="Method",
                           loc="upper left", bbox_to_anchor=(1.03, 1.0), frameon=True)
    ax_DF.add_artist(legend1)

    fig.subplots_adjust(left=0.055, right=0.80, bottom=0.14, top=0.90, wspace=0.38)

    if SAVE_OUTPUT:
        pdf = os.path.join(base_path, "MFEP_alpha00001_OptimalIVM_FENEB_four_metrics.pdf")
        png = os.path.join(base_path, "MFEP_alpha00001_OptimalIVM_FENEB_four_metrics.png")
        fig.savefig(pdf, bbox_inches="tight")
        fig.savefig(png, dpi=300, bbox_inches="tight")
        print(f"Saved {pdf} and {png}")

    plt.show()
    return opt_complete, all(r["done"] for r in feneb_results) if feneb_results else False


# ============================================================
# 自动刷新循环
# ============================================================
if AUTO_REFRESH:
    print(f"Auto-refresh every {REFRESH_SECONDS}s. Interrupt to stop.")
    while True:
        if clear_output: clear_output(wait=True)
        opt_done, feneb_all_done = load_and_plot_once()
        if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE and AUTO_STOP_REQUIRE_FENEB_COMPLETE:
            if opt_done and feneb_all_done:
                print("All required calculations completed. Auto-refresh stopped.")
                break
        print(f"Next refresh in {REFRESH_SECONDS}s ...")
        time.sleep(REFRESH_SECONDS)
else:
    load_and_plot_once()


In [ ]:
import os
import re
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与端点
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array([-2.60866606, 2.77661136], dtype=float)
C2 = np.array([-1.3304716, 0.99118795], dtype=float)

# ------------------------------------------------------------
# Optimal‑IVM 固定配置
# ------------------------------------------------------------
OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(base_path, OPTIMAL_CASENUM)
OPTIMAL_NPZ = os.path.join(OPTIMAL_FOLDER, f"alanine_{OPTIMAL_CASENUM}.npz")

# ------------------------------------------------------------
# FENEB 参数（必须与 FENEB 运行脚本完全一致）
# ------------------------------------------------------------
REPARAM_ALPHA = 1e-3
REPARAM_TOL   = 1e-3

GRID_K = [1.0]                     # 固定 k
GRID_FENEB_ALPHA = [1e-4, 3e-5, 1e-5]   # 当前正在/已经跑完的 α 列表
FENEB_FMAX_TOL = 0.01
MAX_FENEB_STEPS = 200

# ------------------------------------------------------------
# 自动刷新设置
# ------------------------------------------------------------
AUTO_REFRESH = True                # 是否定时刷新（Jupyter 中效果最好）
REFRESH_SECONDS = 60               # 刷新间隔秒数
WAIT_FOR_OPTIMAL_FILE = True
WAIT_FOR_FENEB_FILE = True
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True

# ------------------------------------------------------------
# 绘图样式
# ------------------------------------------------------------
USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True
SAVE_OUTPUT = True

GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1

OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8

# ============================================================
# 安全读取正在写入的 npz
# ============================================================
def safe_load_npz(npz_file, retries=10, wait_seconds=2.0):
    if not os.path.exists(npz_file):
        return None
    last_error = None
    for attempt in range(retries):
        try:
            with np.load(npz_file, allow_pickle=True) as loaded:
                return {key: np.array(loaded[key], copy=True) for key in loaded.files}
        except (zipfile.BadZipFile, EOFError, OSError, ValueError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(wait_seconds)
    print(f"[WARNING] Cannot read npz: {npz_file}\n  error: {last_error}")
    return None


# ============================================================
# 数值工具（与运行脚本完全一致）
# ============================================================
def central_difference_tangent(full_path):
    full_path = np.asarray(full_path, dtype=float)
    return full_path[2:] - full_path[:-2]

def compute_G_projectors(tangent, M_all):
    tangent = np.asarray(tangent, dtype=float)
    M_all = np.asarray(M_all, dtype=float)
    n_internal = tangent.shape[0]
    G_all = np.linalg.inv(M_all)
    identity = np.eye(2, dtype=float)
    P_parallel_all = np.zeros((n_internal, 2, 2))
    P_perp_all = np.zeros((n_internal, 2, 2))
    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]
        norm_G_sq = float(v_i @ G_i @ v_i)
        if norm_G_sq <= 1e-30:
            raise FloatingPointError(f"Near-zero G tangent norm at image {i}")
        tau_G = v_i / np.sqrt(norm_G_sq)
        G_tau_G = G_i @ tau_G
        denom = float(tau_G @ G_tau_G)
        P_parallel = np.outer(tau_G, G_tau_G) / max(denom, 1e-30)
        P_parallel_all[i] = P_parallel
        P_perp_all[i] = identity - P_parallel
    return G_all, P_parallel_all, P_perp_all

def local_G_arclength_weights(full_path, G_all):
    full_path = np.asarray(full_path, dtype=float)
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    length_minus = np.sqrt(np.maximum(np.einsum("ni,nij,nj->n", seg_minus, G_all, seg_minus), 0.0))
    length_plus  = np.sqrt(np.maximum(np.einsum("ni,nij,nj->n", seg_plus,  G_all, seg_plus),  0.0))
    return 0.5 * (length_minus + length_plus)

def local_euclidean_arclength_weights(full_path):
    full_path = np.asarray(full_path, dtype=float)
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus  = full_path[2:] - path_internal
    return 0.5 * (np.linalg.norm(seg_minus, axis=1) + np.linalg.norm(seg_plus, axis=1))

def compute_arclength_residual(full_path):
    full_path = np.asarray(full_path, dtype=float)
    n_full = full_path.shape[0]
    ds = 1.0 / (n_full - 1)
    phi_prime = (full_path[2:] - full_path[:-2]) / (2.0 * ds)
    phi_second = (full_path[2:] - 2.0 * full_path[1:-1] + full_path[:-2]) / (ds**2)
    inner = np.einsum("ni,ni->n", phi_prime, phi_second)
    speed = np.linalg.norm(phi_prime, axis=1)
    I_arc = float(np.sum(inner**2 * speed) * ds)
    return {"I_arc": I_arc, "integrand": inner**2 * speed, "speed": speed, "inner": inner}

def recover_free_energy_profile(full_path, F_internal):
    full_path = np.asarray(full_path, dtype=float)
    F_internal = np.asarray(F_internal, dtype=float)
    n_internal = full_path.shape[0] - 2
    segments = full_path[1:] - full_path[:-1]
    seg_grad = np.zeros((n_internal+1, 2))
    seg_grad[0] = F_internal[0]
    if n_internal > 1:
        seg_grad[1:n_internal] = 0.5 * (F_internal[:-1] + F_internal[1:])
    seg_grad[n_internal] = F_internal[-1]
    increments = np.einsum("ni,ni->n", seg_grad, segments)
    profile = np.zeros(n_internal+2)
    profile[1:] = np.cumsum(increments)
    profile -= profile[0]
    barrier = float(np.max(profile) - profile[0])
    return {"profile": profile, "barrier": barrier, "final_delta_F": float(profile[-1] - profile[0])}

def compute_all_path_metrics(full_path, M_all, F_all):
    tangent = central_difference_tangent(full_path)
    G_all, _, P_perp_G_all = compute_G_projectors(tangent, M_all)
    Mg_all = np.einsum("nij,nj->ni", M_all, F_all)
    perpG_Mg = np.einsum("nij,nj->ni", P_perp_G_all, Mg_all)
    res_G_sq = np.maximum(np.einsum("ni,nij,nj->n", perpG_Mg, G_all, perpG_Mg), 0.0)
    weights_G = local_G_arclength_weights(full_path, G_all)
    I_perp_G = float(np.sum(res_G_sq * weights_G))
    res_E_sq = np.sum(perpG_Mg**2, axis=1)
    weights_E = local_euclidean_arclength_weights(full_path)
    I_perp_E = float(np.sum(res_E_sq * weights_E))
    fe = recover_free_energy_profile(full_path, F_all)
    arc = compute_arclength_residual(full_path)
    return {
        "I_perp_G": I_perp_G, "I_perp_E": I_perp_E,
        "Delta_F_barrier": fe["barrier"], "Delta_F_final": fe["final_delta_F"],
        "I_arc": arc["I_arc"]
    }


# ============================================================
# 去重：同一 count 只保留最后一个状态
# ============================================================
def keep_last_for_duplicate_counts(count_seq, *arrays):
    count_seq = np.asarray(count_seq)
    if count_seq.size == 0:
        return (count_seq,) + arrays
    last_idx = {}
    for i, c in enumerate(count_seq):
        last_idx[float(c)] = i
    selected = np.array(sorted(last_idx.values(), key=lambda idx: count_seq[idx]), dtype=int)
    output = [count_seq[selected]]
    for arr in arrays:
        output.append(np.asarray(arr)[selected])
    return tuple(output)


# ============================================================
# 从 IVM npz 读取路径数据（支持未完成）
# ============================================================
def load_ivm_result(npz_file, method_name, N_sub, alpha_IVP, eta_description):
    if not os.path.exists(npz_file):
        return None, "npz does not exist"
    data = safe_load_npz(npz_file)
    if data is None:
        return None, "could not read"
    if "index_seq" not in data:
        return None, "index_seq missing"

    index_seq_saved = np.asarray(data["index_seq"], dtype=int)
    if index_seq_saved.size == 0:
        return None, "empty index_seq"

    saved_count_seq = np.asarray(data["count_seq"], dtype=int) if "count_seq" in data else index_seq_saved * N_sub
    n_avail = min(len(index_seq_saved), len(saved_count_seq))
    index_seq_saved = index_seq_saved[:n_avail]
    saved_count_seq = saved_count_seq[:n_avail]

    count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq = [], [], [], [], []
    for pos in range(n_avail):
        outer = int(index_seq_saved[pos])
        cnt = int(saved_count_seq[pos])
        phi_key = f"phi_ref_iter_{outer}"
        M_key   = f"M_ref_iter_{outer}"
        F_key   = f"F_ref_iter_{outer}"
        g_key   = f"g_ref_iter_{outer}"
        sel_F = F_key if F_key in data else (g_key if g_key in data else None)
        if phi_key not in data or M_key not in data or sel_F is None:
            continue
        try:
            phi_internal = np.asarray(data[phi_key], dtype=float)
            M_all = np.asarray(data[M_key], dtype=float)
            F_all = np.asarray(data[sel_F], dtype=float)
            full_path = np.vstack([C1[None,:], phi_internal, C2[None,:]])
            metrics = compute_all_path_metrics(full_path, M_all, F_all)
        except Exception as e:
            continue
        count_seq.append(cnt)
        I_G_seq.append(metrics["I_perp_G"])
        I_E_seq.append(metrics["I_perp_E"])
        I_arc_seq.append(metrics["I_arc"])
        Delta_F_seq.append(metrics["Delta_F_barrier"])

    if not count_seq:
        return None, "no valid states"

    (count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq) = keep_last_for_duplicate_counts(
        np.array(count_seq, dtype=int),
        np.array(I_G_seq, dtype=float),
        np.array(I_E_seq, dtype=float),
        np.array(I_arc_seq, dtype=float),
        np.array(Delta_F_seq, dtype=float)
    )

    result = {
        "method": method_name,
        "N_sub": N_sub, "alpha_IVP": alpha_IVP, "eta_description": eta_description,
        "npz_file": npz_file, "count_seq": count_seq,
        "I_perp_G_seq": I_G_seq, "I_perp_E_seq": I_E_seq,
        "I_arc_seq": I_arc_seq, "Delta_F_seq": Delta_F_seq,
    }
    return result, None


# ============================================================
# 自动扫描所有 IVM grid 配置（已存在 npz 的文件夹）
# ============================================================
def load_all_grid_ivm():
    pattern = os.path.join(base_path, "MFEP_image_Adam_N20_Nsub*_alpha*_lr*_NEBreparam")
    folders = sorted(glob.glob(pattern))
    results = []
    for folder in folders:
        casenum = os.path.basename(folder)
        # 解析 N_sub, alpha, lr 以便显示
        m = re.search(r"Nsub(\d+)_alpha([\d.e+-]+)_lr([\d.e+-]+)_NEBreparam", casenum)
        if not m:
            continue
        N_sub = int(m.group(1))
        alpha = float(m.group(2))
        lr = float(m.group(3))
        npz_file = os.path.join(folder, f"alanine_{casenum}.npz")
        final_model = os.path.join(folder, f"model_MFEP_alanine_{casenum}_final.pth")
        result, err = load_ivm_result(npz_file, "IVM-grid", N_sub, alpha, str(lr))
        if result is not None:
            result["eta_in"] = lr
            result["casenum"] = casenum
            results.append(result)
    return results


# ============================================================
# Optimal‑IVM
# ============================================================
def load_optimal_ivm():
    result, err = load_ivm_result(OPTIMAL_NPZ, "Optimal-IVM", OPTIMAL_N_SUB, OPTIMAL_ALPHA, "adaptive 0.04→0.001")
    if result is not None:
        result["casenum"] = OPTIMAL_CASENUM
    return result, err


# ============================================================
# FENEB 路径（已包含 reparamAlpha / reparamTol）
# ============================================================
def float_tag(v):
    return f"{float(v):g}"

def feneb_result_paths(k_spring, alpha):
    k_tag = float_tag(k_spring)
    a_tag = float_tag(alpha)
    casenum = (
        f"MFEP_FENEB_N{N}"
        f"_k{k_tag}"
        f"_alpha{a_tag}"
        f"_reparamAlpha{REPARAM_ALPHA:g}"
        f"_reparamTol{REPARAM_TOL:g}"
        f"_fmax{FENEB_FMAX_TOL:g}"
        f"_steps{MAX_FENEB_STEPS}"
        f"_centralTangent"
        f"_straightInit"
    )
    folder_path = os.path.join(base_path, casenum)
    result_file = os.path.join(folder_path, f"alanine_{casenum}.npz")
    done_file = os.path.join(folder_path, "DONE")
    return casenum, folder_path, result_file, done_file

def load_single_feneb_result(npz_file):
    data = safe_load_npz(npz_file)
    if data is None:
        return None, f"Could not read {npz_file}"
    required = ["count_seq", "I_perp_seq", "I_perp_euclid_arc_seq", "I_arc_seq", "Delta_F_seq"]
    missing = [k for k in required if k not in data]
    if missing:
        return None, f"Missing keys: {missing}"

    count_raw = np.asarray(data["count_seq"], dtype=float)
    I_G_raw   = np.asarray(data["I_perp_seq"], dtype=float)
    I_E_raw   = np.asarray(data["I_perp_euclid_arc_seq"], dtype=float)
    I_arc_raw = np.asarray(data["I_arc_seq"], dtype=float)
    Delta_F_raw = np.asarray(data["Delta_F_seq"], dtype=float)

    n = min(len(count_raw), len(I_G_raw), len(I_E_raw), len(I_arc_raw), len(Delta_F_raw))
    if n == 0:
        return None, "No FENEB states"
    count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw = [arr[:n] for arr in [count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw]]
    (count_seq, I_G_seq, I_E_seq, I_arc_seq, Delta_F_seq) = keep_last_for_duplicate_counts(
        count_raw, I_G_raw, I_E_raw, I_arc_raw, Delta_F_raw
    )

    k = float(np.asarray(data.get("k_spring", [1.0])).ravel()[0])
    a = float(np.asarray(data.get("alpha", [1e-2])).ravel()[0])
    done = os.path.exists(os.path.join(os.path.dirname(npz_file), "DONE"))

    return {
        "method": "FENEB", "npz_file": npz_file, "k_spring": k, "alpha": a,
        "count_seq": count_seq, "I_perp_G_seq": I_G_seq, "I_perp_E_seq": I_E_seq,
        "I_arc_seq": I_arc_seq, "Delta_F_seq": Delta_F_seq, "done": done
    }, None

def load_all_feneb_results():
    results = []
    for k in GRID_K:
        for a in GRID_FENEB_ALPHA:
            _, _, npz_file, _ = feneb_result_paths(k, a)
            res, err = load_single_feneb_result(npz_file)
            if res is not None:
                results.append(res)
    return results


# ============================================================
# FENEB 样式
# ============================================================
FENEB_K_COLORS = {1.0: "#ff7f00"}
FENEB_ALPHA_LINESTYLES = {
    1e-2: "-", 3e-3: "--", 1e-3: "-.", 3e-4: ":",
    1e-4: (0, (3, 5, 1, 5)), 3e-5: (0, (1, 1)), 1e-5: (0, (5, 10)),
}

def feneb_style(k_spring, alpha, idx):
    color = FENEB_K_COLORS.get(k_spring, plt.cm.tab10(idx % 10))
    linestyle = FENEB_ALPHA_LINESTYLES.get(alpha, ["-","--","-.",":"][idx % 4])
    return color, linestyle

def feneb_label(res):
    done = " [DONE]" if res["done"] else " [running]"
    return f"FENEB k={res['k_spring']:g}, α={res['alpha']:g}{done}"


# ============================================================
# 绘图主函数
# ============================================================
def load_and_plot_once():
    grid_ivm_results = load_all_grid_ivm()
    optimal_result, optimal_err = load_optimal_ivm()
    feneb_results = load_all_feneb_results()

    # 汇总状态
    print("=" * 115)
    print("MFEP IVM / Optimal-IVM / FENEB comparison")
    print("=" * 115)
    print(f"自动扫描到的 Grid IVM 配置：{len(grid_ivm_results)}")
    if optimal_result is not None:
        opt_cnt = int(np.max(optimal_result["count_seq"]))
        opt_complete = opt_cnt >= OPTIMAL_TOTAL_STEPS
        print(f"Optimal-IVM count={opt_cnt}/{OPTIMAL_TOTAL_STEPS}, states={len(optimal_result['count_seq'])}, complete={opt_complete}")
    else:
        opt_complete = False
        print("Optimal-IVM 暂无数据")

    print(f"已加载 FENEB 配置：{len(feneb_results)}")
    for r in feneb_results:
        print(f"  FENEB k={r['k_spring']:g}, α={r['alpha']:g} | states={len(r['count_seq'])} | DONE={r['done']}")

    # 预处理对数安全值
    tiny = np.finfo(float).tiny
    all_data = list(grid_ivm_results)
    if optimal_result: all_data.append(optimal_result)
    all_data.extend(feneb_results)
    for d in all_data:
        d["I_perp_G_plot"] = np.maximum(d["I_perp_G_seq"], tiny)
        d["I_perp_E_plot"] = np.maximum(d["I_perp_E_seq"], tiny)
        d["I_arc_plot"]    = np.maximum(d["I_arc_seq"],    tiny)
        d["Delta_F_plot"]  = np.maximum(d["Delta_F_seq"],  tiny)

    # 建图
    plt.close("all")
    fig, axes = plt.subplots(1, 4, figsize=(27, 6.5))
    ax_G, ax_E, ax_arc, ax_DF = axes

    # 预定义样式（对于扫描到的参数可能不在字典中，则自动循环配色）
    alpha_colors = {0.1: "#377eb8", 1: "#4daf4a", 10: "#984ea3"}
    eta_styles   = {0.03: "-", 0.01: "--", 0.003: "-.", 0.001: ":"}
    nsub_markers = {4: "o", 5: "s", 10: "^"}

    # --- 绘制 Grid IVM ---
    for res in grid_ivm_results:
        ns, al, et = res["N_sub"], res["alpha_IVP"], res["eta_in"]
        c = alpha_colors.get(al, plt.cm.tab10(hash(al) % 10))
        ls = eta_styles.get(et, "-")
        mk = nsub_markers.get(ns, "o")
        mev = max(1, len(res["count_seq"]) // 8)
        kw = dict(color=c, linestyle=ls, marker=mk, markevery=mev,
                  markersize=3.5, linewidth=GRID_IVM_LINEWIDTH, alpha=GRID_IVM_ALPHA, zorder=2)
        for ax, vals in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                         (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(res["count_seq"], res[vals], **kw)

    # --- Optimal-IVM ---
    if optimal_result is not None:
        oc = optimal_result["count_seq"]
        mev = max(1, len(oc)//12)
        okw = dict(color=OPTIMAL_COLOR, linestyle=OPTIMAL_LINESTYLE, marker=OPTIMAL_MARKER,
                   markevery=mev, markersize=8, markerfacecolor=OPTIMAL_COLOR,
                   markeredgecolor="white", markeredgewidth=0.8,
                   linewidth=OPTIMAL_LINEWIDTH, alpha=1.0, zorder=30)
        for ax, key in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                        (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(oc, optimal_result[key], **okw)
            ax.scatter(oc[-1], optimal_result[key][-1], s=130, marker="*",
                       color=OPTIMAL_COLOR, edgecolor="black", linewidth=0.8, zorder=40)

    # --- FENEB ---
    for idx, res in enumerate(feneb_results):
        cs = res["count_seq"]
        col, ls = feneb_style(res["k_spring"], res["alpha"], idx)
        mev = max(1, len(cs)//12)
        fkw = dict(color=col, linestyle=ls, marker=FENEB_MARKER,
                   markevery=mev, markersize=5.5, markerfacecolor="white",
                   markeredgecolor=col, markeredgewidth=1.2,
                   linewidth=FENEB_LINEWIDTH, alpha=1.0, zorder=20+idx)
        for ax, key in [(ax_G, "I_perp_G_plot"), (ax_E, "I_perp_E_plot"),
                        (ax_arc, "I_arc_plot"), (ax_DF, "Delta_F_plot")]:
            ax.plot(cs, res[key], **fkw)
            marker = "D" if res["done"] else "o"
            ax.scatter(cs[-1], res[key][-1], s=75, marker=marker,
                       facecolor=col, edgecolor="black", linewidth=0.8, zorder=50+idx)

    # 面板装饰
    titles = ["Riemannian normal residual", "Euclidean residual integral",
              "Arclength / equi-speed residual", "Path-integrated free-energy barrier"]
    ylabels = [r"$\widehat I_{\perp}^{G}$", r"$\widehat I_{\perp}^{E}$",
               r"$I_{\rm arc}$", r"$\Delta F^{\dagger}$ (kcal/mol)"]
    for ax, t, yl in zip(axes, titles, ylabels):
        ax.set_title(t, fontsize=14)
        ax.set_xlabel("count_seq", fontsize=12)
        ax.set_ylabel(yl, fontsize=10)
        ax.grid(True, which="both", linestyle="--", alpha=0.3)
        ax.set_xlim(left=0)
    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale("log"); ax_E.set_yscale("log")
    if USE_LOG_Y_ARC:
        ax_arc.set_yscale("log")
    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale("log")

    # 图例
    method_handles = [Line2D([0],[0], color="gray", linewidth=1.5, alpha=GRID_IVM_ALPHA, label="Grid IVM")]
    if optimal_result:
        method_handles.append(Line2D([0],[0], color=OPTIMAL_COLOR, linestyle=OPTIMAL_LINESTYLE,
                                     marker=OPTIMAL_MARKER, linewidth=OPTIMAL_LINEWIDTH,
                                     markersize=8, label="Optimal-IVM"))
    for idx, r in enumerate(feneb_results):
        c, ls = feneb_style(r["k_spring"], r["alpha"], idx)
        method_handles.append(Line2D([0],[0], color=c, linestyle=ls, marker=FENEB_MARKER,
                                     markerfacecolor="white", markeredgecolor=c,
                                     linewidth=FENEB_LINEWIDTH, markersize=6, label=feneb_label(r)))
    legend1 = ax_DF.legend(handles=method_handles, title="Method",
                           loc="upper left", bbox_to_anchor=(1.03, 1.0), frameon=True)
    ax_DF.add_artist(legend1)

    fig.subplots_adjust(left=0.055, right=0.80, bottom=0.14, top=0.90, wspace=0.38)

    if SAVE_OUTPUT:
        pdf = os.path.join(base_path, "MFEP_OptimalIVM_GridIVM_FENEB_four_metrics.pdf")
        png = os.path.join(base_path, "MFEP_OptimalIVM_GridIVM_FENEB_four_metrics.png")
        fig.savefig(pdf, bbox_inches="tight")
        fig.savefig(png, dpi=300, bbox_inches="tight")
        print(f"Saved {pdf} and {png}")

    plt.show()
    return opt_complete, all(r["done"] for r in feneb_results) if feneb_results else False


# ============================================================
# 自动刷新循环
# ============================================================
if AUTO_REFRESH:
    print(f"Auto-refresh every {REFRESH_SECONDS}s. Interrupt to stop.")
    while True:
        if clear_output: clear_output(wait=True)
        opt_done, feneb_all_done = load_and_plot_once()
        if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE and AUTO_STOP_REQUIRE_FENEB_COMPLETE:
            if opt_done and feneb_all_done:
                print("All required calculations completed. Auto-refresh stopped.")
                break
        print(f"Next refresh in {REFRESH_SECONDS}s ...")
        time.sleep(REFRESH_SECONDS)
else:
    load_and_plot_once()

In [ ]:
# -*- coding: utf-8 -*-

import os
import time
import zipfile

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)


N = 20
TOTAL_INNER_STEPS = 200


C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)


C2 = np.array(
    [-1.33047160, 0.99118795],
    dtype=float,
)


GRID_N_SUB = [
    4,
    5,
    10,
]


GRID_ALPHA = [
    0.1,
    1,
    10,
]


GRID_ETA = [
    0.03,
    0.01,
    0.003,
    0.001,
]


# ============================================================
# Optimal-IVM 配置
#
# 注意：
# 这里必须保持现有文件夹命名，即使文件夹名称中的学习率文字
# 和当前实际 schedule 不完全一致，也不能修改，否则找不到文件。
# ============================================================

OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200


OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)


OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)


OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# 新版 MFEP-FENEB 配置
#
# 必须与 FENEB_ala_compare.py 一致。
#
# 对应前面给出的新版 FENEB 脚本：
#
# case_number =
#     MFEP_FENEB_COMPARE
#     _N20
#     _lambda...
#     _Nupdates200
#     _Kspring1
#     _dtau0.001
#     _reparamTol0.001
#     _sameIVMsettings
# ============================================================

FENEB_REPARAM_KS = 1.0
FENEB_REPARAM_DTAU = 1.0e-3
FENEB_REPARAM_TOL = 1.0e-3
FENEB_REPARAM_MAXITER = 5000

FENEB_MAX_UPDATES = 200


# 必须和 FENEB_ala_compare.py 中的 GRID_PHYSICAL_STEP 一致
GRID_FENEB_PHYSICAL_STEP = [
    1.0e-2,
    5.0e-3,
    2.0e-3,
    1.0e-3,
]


# ============================================================
# 选择要绘制的普通 Grid-IVM 配置
#
# None 表示全部绘制。
#
# 例如：
#
# PLOT_ONLY_N_SUB = [4]
# PLOT_ONLY_ALPHA = [0.1]
# PLOT_ONLY_ETA = [0.003]
# ============================================================

PLOT_ONLY_N_SUB = None
PLOT_ONLY_ALPHA = None
PLOT_ONLY_ETA = None


# ============================================================
# 自动刷新设置
# ============================================================

AUTO_REFRESH = True
REFRESH_SECONDS = 60


# Optimal-IVM 文件尚未创建时是否继续等待
WAIT_FOR_OPTIMAL_FILE = True


# FENEB 文件尚未创建时是否继续等待
WAIT_FOR_FENEB_FILE = True


# 自动停止条件
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True


# ============================================================
# 绘图设置
# ============================================================

USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True


SAVE_OUTPUT = True


# 普通 Grid-IVM
GRID_IVM_ALPHA = 0.45
GRID_IVM_LINEWIDTH = 1.1


# Optimal-IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2


# MFEP-FENEB
FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8


# ============================================================
# 安全读取正在原子化写入的 npz
# ============================================================

def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        "[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# 普通 Grid-IVM 配置路径
#
# 这里的 _NEBreparam 是你已有 IVM 文件夹名称的一部分，
# 不表示本脚本绘制独立 NEB baseline。
# ============================================================

def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 数值工具
#
# 以下指标定义同时用于 IVM 和新版 MFEP-FENEB。
# ============================================================

def central_difference_tangent(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            "Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            "Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if (
            not np.isfinite(norm_G_sq)
            or norm_G_sq <= 1.0e-30
        ):
            raise FloatingPointError(
                "Near-zero or invalid G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        if (
            not np.isfinite(denom)
            or denom <= 1.0e-30
        ):
            raise FloatingPointError(
                "Invalid projector denominator "
                f"at image {i}: {denom}"
            )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / denom
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def compute_arclength_residual(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            "full_path must have shape (n,2), "
            f"got {full_path.shape}"
        )

    n_full = full_path.shape[0]
    n_internal = n_full - 2

    if n_internal < 1:
        raise ValueError(
            "The path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    tangent_second_inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        tangent_second_inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand)
        * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner_phi_prime_second": (
            tangent_second_inner
        ),
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            "is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    segment_gradients[0] = (
        F_internal[0]
    )

    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            "Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            "Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            "Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    arc_result = compute_arclength_residual(
        full_path
    )

    fmax_E = float(
        np.max(
            np.sqrt(
                np.maximum(
                    residual_E_sq,
                    0.0,
                )
            )
        )
    )

    fmax_G = float(
        np.max(
            np.sqrt(
                np.maximum(
                    residual_G_sq,
                    0.0,
                )
            )
        )
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "free_energy_max_index": (
            free_energy["max_index"]
        ),

        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result[
                "inner_phi_prime_second"
            ]
        ),

        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,

        "fmax_E": fmax_E,
        "fmax_G": fmax_G,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后状态
# ============================================================

def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取逐 outer 数据
# ============================================================

def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            "final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                "incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                "expected at least "
                f"{expected_batches}"
            )

    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                "[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(count_seq) == 0:
        return None, (
            "no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": valid_index_seq,
        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取完成的普通 Grid-IVM
# ============================================================

def load_completed_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (
            N_sub,
            alpha_IVP,
            eta_in,
        )
        for N_sub in GRID_N_SUB
        for alpha_IVP in GRID_ALPHA
        for eta_in in GRID_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        if not selected_config(
            N_sub,
            alpha_IVP,
            eta_in,
        ):
            continue

        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=True,
            final_model=final_model,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
        all_configs,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
# ============================================================

def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive Adam"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# 新版 MFEP-FENEB 文件路径
#
# 必须和 FENEB_ala_compare.py 中的 result_paths() 完全一致。
# ============================================================

def float_tag(value):
    return f"{float(value):g}"


def feneb_result_paths(
    physical_step,
):
    step_tag = float_tag(
        physical_step
    )

    casenum = (
        f"MFEP_FENEB_COMPARE"
        f"_N{N}"
        f"_lambda{step_tag}"
        f"_Nupdates{FENEB_MAX_UPDATES}"
        f"_Kspring{FENEB_REPARAM_KS:g}"
        f"_dtau{FENEB_REPARAM_DTAU:g}"
        f"_reparamTol{FENEB_REPARAM_TOL:g}"
        f"_sameIVMsettings"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    result_file = os.path.join(
        folder_path,
        f"alanine_{casenum}.npz",
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    return (
        casenum,
        folder_path,
        result_file,
        done_file,
    )


# ============================================================
# 读取新版 MFEP-FENEB 单个结果
#
# 关键：
#
# 新版 FENEB 保存：
#
#     mean_path_unwrapped_seq
#     mean_path_seq
#     M_ref_seq
#     F_ref_seq
#
# M 和 F 对应 restrained-MD mean path，因此指标必须在 mean_path
# 上重新计算，而不是使用 reference_path 或 reparameterized_path。
# ============================================================

def load_single_feneb_result(
    npz_file,
    expected_physical_step=None,
):
    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"Could not read {npz_file}"
        )

    required_basic_keys = [
        "count_seq",
        "M_ref_seq",
        "F_ref_seq",
    ]

    missing_basic = [
        key
        for key in required_basic_keys
        if key not in data
    ]

    if missing_basic:
        return None, (
            "Missing FENEB keys: "
            f"{missing_basic}"
        )

    # 优先使用 unwrapped mean path。
    if "mean_path_unwrapped_seq" in data:
        path_key = (
            "mean_path_unwrapped_seq"
        )
    elif "mean_path_seq" in data:
        path_key = "mean_path_seq"
    else:
        return None, (
            "Neither mean_path_unwrapped_seq "
            "nor mean_path_seq exists."
        )

    count_raw = np.asarray(
        data["count_seq"],
        dtype=float,
    )

    path_raw = np.asarray(
        data[path_key],
        dtype=float,
    )

    M_raw = np.asarray(
        data["M_ref_seq"],
        dtype=float,
    )

    F_raw = np.asarray(
        data["F_ref_seq"],
        dtype=float,
    )

    n_states = min(
        len(count_raw),
        len(path_raw),
        len(M_raw),
        len(F_raw),
    )

    if n_states == 0:
        return None, (
            "No saved FENEB states."
        )

    count_raw = count_raw[:n_states]
    path_raw = path_raw[:n_states]
    M_raw = M_raw[:n_states]
    F_raw = F_raw[:n_states]

    valid_count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    fmax_E_seq = []
    fmax_G_seq = []

    valid_state_indices = []
    skipped_state_indices = []

    for state_index in range(
        n_states
    ):
        try:
            current_path = np.asarray(
                path_raw[state_index],
                dtype=float,
            )

            current_M = np.asarray(
                M_raw[state_index],
                dtype=float,
            )

            current_F = np.asarray(
                F_raw[state_index],
                dtype=float,
            )

            # 新版 FENEB 应保存包含端点的完整路径，shape=(N+2,2)。
            #
            # 为了向后兼容，也允许只保存内部 images，shape=(N,2)。
            if current_path.shape == (
                N,
                2,
            ):
                full_path = np.vstack(
                    [
                        C1[None, :],
                        current_path,
                        C2[None, :],
                    ]
                )

            elif current_path.shape == (
                N + 2,
                2,
            ):
                full_path = (
                    current_path.copy()
                )

            else:
                raise ValueError(
                    "Unexpected FENEB path shape: "
                    f"{current_path.shape}; expected "
                    f"{(N, 2)} or {(N + 2, 2)}."
                )

            metrics = compute_all_path_metrics(
                full_path,
                current_M,
                current_F,
            )

        except Exception as exc:
            print(
                "[WARNING] 跳过 MFEP-FENEB "
                f"state={state_index}, "
                f"count={count_raw[state_index]}: "
                f"{exc}"
            )

            skipped_state_indices.append(
                state_index
            )

            continue

        valid_state_indices.append(
            state_index
        )

        valid_count_seq.append(
            count_raw[state_index]
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

        fmax_E_seq.append(
            metrics["fmax_E"]
        )

        fmax_G_seq.append(
            metrics["fmax_G"]
        )

    if len(valid_count_seq) == 0:
        return None, (
            "No valid FENEB state could be reconstructed."
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        free_energy_profiles,
        arc_integrands,
        fmax_E_seq,
        fmax_G_seq,
        valid_state_indices,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            valid_count_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
        np.asarray(
            fmax_E_seq,
            dtype=float,
        ),
        np.asarray(
            fmax_G_seq,
            dtype=float,
        ),
        np.asarray(
            valid_state_indices,
            dtype=int,
        ),
    )

    if "physical_step" in data:
        physical_step = float(
            np.asarray(
                data["physical_step"]
            ).reshape(-1)[0]
        )
    elif expected_physical_step is not None:
        physical_step = float(
            expected_physical_step
        )
    else:
        physical_step = np.nan

    if "spring_constant" in data:
        spring_constant = float(
            np.asarray(
                data["spring_constant"]
            ).reshape(-1)[0]
        )
    else:
        spring_constant = float(
            FENEB_REPARAM_KS
        )

    if "reparam_dtau" in data:
        reparam_dtau = float(
            np.asarray(
                data["reparam_dtau"]
            ).reshape(-1)[0]
        )
    else:
        reparam_dtau = float(
            FENEB_REPARAM_DTAU
        )

    if "reparam_tolerance" in data:
        reparam_tolerance = float(
            np.asarray(
                data["reparam_tolerance"]
            ).reshape(-1)[0]
        )
    else:
        reparam_tolerance = float(
            FENEB_REPARAM_TOL
        )

    folder_path = os.path.dirname(
        npz_file
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    done = os.path.exists(
        done_file
    )

    result = {
        "method": "MFEP-FENEB",

        "npz_file": npz_file,
        "folder_path": folder_path,
        "done_file": done_file,
        "done": done,

        "physical_step": physical_step,
        "spring_constant": spring_constant,
        "reparam_dtau": reparam_dtau,
        "reparam_tolerance": (
            reparam_tolerance
        ),

        "source_path_key": path_key,

        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,
        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "fmax_E_seq": fmax_E_seq,
        "fmax_G_seq": fmax_G_seq,

        "valid_state_indices": (
            valid_state_indices
        ),

        "skipped_state_indices": (
            np.asarray(
                skipped_state_indices,
                dtype=int,
            )
        ),
    }

    return result, None


# ============================================================
# 读取所有新版 MFEP-FENEB 结果
# ============================================================

def load_all_feneb_results():
    feneb_results = []
    errors = []

    for physical_step in (
        GRID_FENEB_PHYSICAL_STEP
    ):
        (
            casenum,
            folder_path,
            npz_file,
            done_file,
        ) = feneb_result_paths(
            physical_step
        )

        result, error = (
            load_single_feneb_result(
                npz_file,
                expected_physical_step=(
                    physical_step
                ),
            )
        )

        if result is not None:
            result["casenum"] = casenum
            result["expected_done_file"] = (
                done_file
            )

            feneb_results.append(
                result
            )

        else:
            errors.append(
                "lambda="
                f"{physical_step:g}: "
                f"{error}"
            )

    return (
        feneb_results,
        errors,
    )


# ============================================================
# 判断新版 MFEP-FENEB 是否全部完成
# ============================================================

def evaluate_feneb_completion(
    feneb_results,
):
    status = []

    for expected_step in (
        GRID_FENEB_PHYSICAL_STEP
    ):
        matching = [
            result
            for result in feneb_results
            if np.isclose(
                result["physical_step"],
                expected_step,
                rtol=1.0e-10,
                atol=1.0e-15,
            )
        ]

        found = len(matching) > 0

        if found:
            done = bool(
                matching[0]["done"]
            )

            latest_count = float(
                matching[0][
                    "count_seq"
                ][-1]
            )

        else:
            done = False
            latest_count = np.nan

        status.append(
            {
                "physical_step": float(
                    expected_step
                ),
                "found": found,
                "done": done,
                "latest_count": (
                    latest_count
                ),
            }
        )

    all_complete = all(
        row["found"] and row["done"]
        for row in status
    )

    return (
        all_complete,
        status,
    )


# ============================================================
# MFEP-FENEB 绘图样式
# ============================================================

FENEB_STEP_COLORS = {
    1.0e-2: "#ff7f00",
    5.0e-3: "#a65628",
    2.0e-3: "#f781bf",
    1.0e-3: "#00a6d6",
}


FENEB_STEP_LINESTYLES = {
    1.0e-2: "-",
    5.0e-3: "--",
    2.0e-3: "-.",
    1.0e-3: ":",
}


def dictionary_value_isclose(
    dictionary,
    value,
    default=None,
):
    for key, mapped_value in (
        dictionary.items()
    ):
        if np.isclose(
            float(key),
            float(value),
            rtol=1.0e-10,
            atol=1.0e-15,
        ):
            return mapped_value

    return default


def feneb_style(
    physical_step,
    index,
):
    color = dictionary_value_isclose(
        FENEB_STEP_COLORS,
        physical_step,
        default=None,
    )

    if color is None:
        cmap = plt.get_cmap(
            "tab10"
        )

        color = cmap(
            index % 10
        )

    linestyle = dictionary_value_isclose(
        FENEB_STEP_LINESTYLES,
        physical_step,
        default=None,
    )

    if linestyle is None:
        styles = [
            "-",
            "--",
            "-.",
            ":",
            (0, (3, 5, 1, 5)),
            (0, (1, 1)),
            (0, (5, 10)),
        ]

        linestyle = styles[
            index % len(styles)
        ]

    return (
        color,
        linestyle,
    )


def feneb_label(
    result,
):
    physical_step = (
        result["physical_step"]
    )

    label = (
        "MFEP-FENEB "
        rf"$\lambda={physical_step:g}$"
    )

    if result["done"]:
        label += " [DONE]"
    else:
        label += " [running]"

    return label


# ============================================================
# 绘制一次
# ============================================================

def load_and_plot_once():
    (
        grid_ivm_results,
        skipped_grid,
        all_configs,
    ) = load_completed_grid_ivm()

    del all_configs

    optimal_result, optimal_error = (
        load_optimal_ivm()
    )

    (
        feneb_results,
        feneb_errors,
    ) = load_all_feneb_results()

    (
        feneb_all_complete,
        feneb_completion_rows,
    ) = evaluate_feneb_completion(
        feneb_results
    )

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------

    print("=" * 120)
    print(
        "MFEP Grid-IVM / Optimal-IVM / "
        "metric-extended FENEB comparison"
    )
    print("=" * 120)

    print(
        "完成的 Grid-IVM 配置："
        f"{len(grid_ivm_results)}"
    )

    print(
        "跳过的 Grid-IVM 配置："
        f"{len(skipped_grid)}"
    )

    if optimal_result is not None:
        optimal_current_count = int(
            np.max(
                optimal_result[
                    "count_seq"
                ]
            )
        )

        optimal_complete = (
            optimal_current_count
            >= OPTIMAL_TOTAL_STEPS
        )

        print(
            "Optimal-IVM：已读取，"
            f"当前 count="
            f"{optimal_current_count}/"
            f"{OPTIMAL_TOTAL_STEPS}"
        )

        print(
            "Optimal-IVM states："
            f"{len(optimal_result['count_seq'])}"
        )

        print(
            "Optimal-IVM complete："
            f"{optimal_complete}"
        )

    else:
        optimal_current_count = -1
        optimal_complete = False

        print(
            "Optimal-IVM：尚未读取到有效数据"
        )

        print(
            "  ",
            optimal_error,
        )

    print("-" * 120)

    print(
        "成功读取的 MFEP-FENEB 配置数："
        f"{len(feneb_results)}"
    )

    if feneb_results:
        for result in feneb_results:
            print(
                "MFEP-FENEB "
                f"lambda="
                f"{result['physical_step']:g}"
                " | states="
                f"{len(result['count_seq']):4d}"
                " | count range=("
                f"{result['count_seq'][0]:g}, "
                f"{result['count_seq'][-1]:g})"
                " | path="
                f"{result['source_path_key']}"
                " | DONE="
                f"{result['done']}"
            )

    else:
        print(
            "尚未读取到有效 MFEP-FENEB 数据。"
        )

    if feneb_errors:
        print(
            "MFEP-FENEB 读取警告数："
            f"{len(feneb_errors)}"
        )

        for error in feneb_errors[:10]:
            print(
                "  ",
                error,
            )

    print("-" * 120)
    print("预期 MFEP-FENEB 完成状态：")

    for row in feneb_completion_rows:
        print(
            "  lambda="
            f"{row['physical_step']:g}: "
            f"found={row['found']}, "
            f"DONE={row['done']}, "
            f"latest_count="
            f"{row['latest_count']}"
        )

    print(
        "MFEP-FENEB all complete："
        f"{feneb_all_complete}"
    )

    print("=" * 120)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------

    tiny = np.finfo(
        float
    ).tiny

    all_available = list(
        grid_ivm_results
    )

    if optimal_result is not None:
        all_available.append(
            optimal_result
        )

    all_available.extend(
        feneb_results
    )

    for result in all_available:
        result["I_perp_G_plot"] = np.maximum(
            result["I_perp_G_seq"],
            tiny,
        )

        result["I_perp_E_plot"] = np.maximum(
            result["I_perp_E_seq"],
            tiny,
        )

        result["I_arc_plot"] = np.maximum(
            result["I_arc_seq"],
            tiny,
        )

        result["Delta_F_plot"] = np.maximum(
            result["Delta_F_seq"],
            tiny,
        )

    # --------------------------------------------------------
    # 创建 1 x 4 图
    # --------------------------------------------------------

    plt.close("all")

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(27, 6.5),
    )

    (
        ax_G,
        ax_E,
        ax_arc,
        ax_DF,
    ) = axes

    # --------------------------------------------------------
    # 普通 Grid-IVM
    # --------------------------------------------------------

    alpha_colors = {
        0.1: "#377eb8",
        1: "#4daf4a",
        10: "#984ea3",
    }

    eta_linestyles = {
        0.03: "-",
        0.01: "--",
        0.003: "-.",
        0.001: ":",
    }

    nsub_markers = {
        4: "o",
        5: "s",
        10: "^",
    }

    for result in grid_ivm_results:
        N_sub = result["N_sub"]
        alpha_IVP = result["alpha_IVP"]
        eta_in = result["eta_in"]
        count_seq = result["count_seq"]

        color = alpha_colors.get(
            alpha_IVP,
            "gray",
        )

        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )

        marker = nsub_markers.get(
            N_sub,
            "o",
        )

        markevery = max(
            1,
            len(count_seq) // 8,
        )

        common_kwargs = {
            "color": color,
            "linestyle": linestyle,
            "marker": marker,
            "markevery": markevery,
            "markersize": 3.5,
            "linewidth": (
                GRID_IVM_LINEWIDTH
            ),
            "alpha": GRID_IVM_ALPHA,
            "zorder": 2,
        }

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **common_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **common_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **common_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **common_kwargs,
        )

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------

    if optimal_result is not None:
        optimal_count = (
            optimal_result[
                "count_seq"
            ]
        )

        optimal_markevery = max(
            1,
            len(optimal_count) // 12,
        )

        optimal_kwargs = {
            "color": OPTIMAL_COLOR,
            "linestyle": (
                OPTIMAL_LINESTYLE
            ),
            "marker": OPTIMAL_MARKER,
            "markevery": (
                optimal_markevery
            ),
            "markersize": 8,
            "markerfacecolor": (
                OPTIMAL_COLOR
            ),
            "markeredgecolor": "white",
            "markeredgewidth": 0.8,
            "linewidth": (
                OPTIMAL_LINEWIDTH
            ),
            "alpha": 1.0,
            "zorder": 30,
        }

        ax_G.plot(
            optimal_count,
            optimal_result[
                "I_perp_G_plot"
            ],
            **optimal_kwargs,
        )

        ax_E.plot(
            optimal_count,
            optimal_result[
                "I_perp_E_plot"
            ],
            **optimal_kwargs,
        )

        ax_arc.plot(
            optimal_count,
            optimal_result[
                "I_arc_plot"
            ],
            **optimal_kwargs,
        )

        ax_DF.plot(
            optimal_count,
            optimal_result[
                "Delta_F_plot"
            ],
            **optimal_kwargs,
        )

        for axis, values in [
            (
                ax_G,
                optimal_result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                optimal_result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                optimal_result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                optimal_result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                optimal_count[-1],
                values[-1],
                s=130,
                marker="*",
                color=OPTIMAL_COLOR,
                edgecolor="black",
                linewidth=0.8,
                zorder=40,
            )

    # --------------------------------------------------------
    # 新版 MFEP-FENEB
    # --------------------------------------------------------

    for index, result in enumerate(
        feneb_results
    ):
        count_seq = result[
            "count_seq"
        ]

        (
            color,
            linestyle,
        ) = feneb_style(
            result["physical_step"],
            index,
        )

        markevery = max(
            1,
            len(count_seq) // 12,
        )

        feneb_kwargs = {
            "color": color,
            "linestyle": linestyle,
            "marker": FENEB_MARKER,
            "markevery": markevery,
            "markersize": 5.5,
            "markerfacecolor": "white",
            "markeredgecolor": color,
            "markeredgewidth": 1.2,
            "linewidth": FENEB_LINEWIDTH,
            "alpha": 1.0,
            "zorder": 20 + index,
        }

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **feneb_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **feneb_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **feneb_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **feneb_kwargs,
        )

        latest_marker = (
            "D"
            if result["done"]
            else "o"
        )

        for axis, values in [
            (
                ax_G,
                result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                count_seq[-1],
                values[-1],
                s=75,
                marker=latest_marker,
                facecolor=color,
                edgecolor="black",
                linewidth=0.8,
                zorder=50 + index,
            )

    # ========================================================
    # Panel 设置
    # ========================================================

    ax_G.set_title(
        "Riemannian normal residual",
        fontsize=14,
    )

    ax_G.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_G.set_ylabel(
        r"$\widehat I_{\perp}^{G}$"
        "\n"
        r"$=\int |P_\varphi^G M\nabla F|_G^2"
        r" |\varphi'|_G\,ds$",
        fontsize=10,
    )

    ax_E.set_title(
        "Euclidean residual integral",
        fontsize=14,
    )

    ax_E.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_E.set_ylabel(
        r"$\widehat I_{\perp}^{E}$"
        "\n"
        r"$=\int \|P_\varphi^G M\nabla F\|_2^2"
        r" |\varphi'|_2\,ds$",
        fontsize=10,
    )

    ax_arc.set_title(
        "Arclength / equi-speed residual",
        fontsize=14,
    )

    ax_arc.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_arc.set_ylabel(
        r"$I_{\rm arc}$"
        "\n"
        r"$=\int_0^1"
        r"(\varphi'\cdot\varphi'')^2"
        r"|\varphi'|\,ds$",
        fontsize=10,
    )

    ax_DF.set_title(
        "Path-integrated free-energy barrier",
        fontsize=14,
    )

    ax_DF.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_DF.set_ylabel(
        r"$\Delta F^{\dagger}$ (kcal/mol)"
        "\n"
        "line-integral estimate",
        fontsize=10,
    )

    for axis in axes:
        axis.grid(
            True,
            which="both",
            linestyle="--",
            alpha=0.3,
        )

        axis.set_xlim(
            left=0,
        )

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale(
            "log"
        )

        ax_E.set_yscale(
            "log"
        )

    if USE_LOG_Y_ARC:
        ax_arc.set_yscale(
            "log"
        )

    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale(
            "log"
        )

    # ========================================================
    # 图例
    # ========================================================

    method_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linewidth=1.5,
            alpha=GRID_IVM_ALPHA,
            label="Grid IVM",
        )
    ]

    if optimal_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=OPTIMAL_COLOR,
                linestyle=(
                    OPTIMAL_LINESTYLE
                ),
                marker=OPTIMAL_MARKER,
                linewidth=(
                    OPTIMAL_LINEWIDTH
                ),
                markersize=8,
                label="Optimal-IVM",
            )
        )

    for index, result in enumerate(
        feneb_results
    ):
        (
            color,
            linestyle,
        ) = feneb_style(
            result["physical_step"],
            index,
        )

        method_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                linestyle=linestyle,
                marker=FENEB_MARKER,
                markerfacecolor="white",
                markeredgecolor=color,
                linewidth=(
                    FENEB_LINEWIDTH
                ),
                markersize=6,
                label=feneb_label(
                    result
                ),
            )
        )

    legend_method = ax_DF.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(
            1.03,
            1.0,
        ),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_method
    )

    # Grid-IVM 颜色图例
    alpha_handles = [
        Line2D(
            [0],
            [0],
            color=alpha_colors[alpha],
            linewidth=2,
            label=rf"$\alpha={alpha}$",
        )
        for alpha in GRID_ALPHA
    ]

    legend_alpha = ax_DF.legend(
        handles=alpha_handles,
        title="Grid IVM color",
        loc="upper left",
        bbox_to_anchor=(
            1.03,
            0.68,
        ),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_alpha
    )

    # Grid-IVM 线型图例
    eta_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle=(
                eta_linestyles[eta]
            ),
            linewidth=2,
            label=rf"$\eta={eta}$",
        )
        for eta in GRID_ETA
    ]

    legend_eta = ax_DF.legend(
        handles=eta_handles,
        title="Grid IVM line",
        loc="upper left",
        bbox_to_anchor=(
            1.03,
            0.36,
        ),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_eta
    )

    # Grid-IVM marker 图例
    nsub_handles = [
        Line2D(
            [0],
            [0],
            color="gray",
            linestyle="None",
            marker=(
                nsub_markers[N_sub]
            ),
            markersize=6,
            label=(
                rf"$N_{{sub}}={N_sub}$"
            ),
        )
        for N_sub in GRID_N_SUB
    ]

    ax_DF.legend(
        handles=nsub_handles,
        title="Grid IVM marker",
        loc="upper left",
        bbox_to_anchor=(
            1.03,
            0.05,
        ),
        frameon=True,
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.80,
        bottom=0.14,
        top=0.90,
        wspace=0.38,
    )

    # ========================================================
    # 保存图片
    # ========================================================

    output_pdf = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_"
            "MetricFENEB_four_metrics.pdf"
        ),
    )

    output_png = os.path.join(
        base_path,
        (
            "MFEP_OptimalIVM_GridIVM_"
            "MetricFENEB_four_metrics.png"
        ),
    )

    if SAVE_OUTPUT:
        fig.savefig(
            output_pdf,
            bbox_inches="tight",
        )

        fig.savefig(
            output_png,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()
    plt.close(fig)

    # ========================================================
    # 当前值汇总
    # ========================================================

    print("\n" + "=" * 160)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 160)

    summary_rows = []

    for result in grid_ivm_results:
        summary_rows.append(
            {
                "method": "IVM-grid",
                "description": (
                    f"Nsub={result['N_sub']} "
                    f"alpha={result['alpha_IVP']} "
                    f"eta={result['eta_in']}"
                ),
                "count": float(
                    result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": True,
            }
        )

    if optimal_result is not None:
        summary_rows.append(
            {
                "method": "Optimal-IVM",
                "description": (
                    "Nsub=4 alpha=0.1 "
                    "adaptive Adam"
                ),
                "count": float(
                    optimal_result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    optimal_result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    optimal_result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    optimal_result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    optimal_result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": optimal_complete,
            }
        )

    for result in feneb_results:
        summary_rows.append(
            {
                "method": "MFEP-FENEB",
                "description": (
                    "lambda="
                    f"{result['physical_step']:g}"
                ),
                "count": float(
                    result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": bool(
                    result["done"]
                ),
            }
        )

    method_order = {
        "Optimal-IVM": 0,
        "MFEP-FENEB": 1,
        "IVM-grid": 2,
    }

    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (
            method_order.get(
                row["method"],
                99,
            ),
            row["I_G"],
        ),
    )

    for row in summary_rows_sorted:
        print(
            f"{row['method']:<14} "
            f"{row['description']:<42}"
            f" count={row['count']:>7.1f}"
            f" | I_G={row['I_G']:.6e}"
            f" | I_E={row['I_E']:.6e}"
            f" | I_arc={row['I_arc']:.6e}"
            f" | DeltaF={row['Delta_F']:.6f}"
            f" | done={row['done']}"
        )

    # ========================================================
    # 保存汇总数据
    # ========================================================

    comparison_results_file = os.path.join(
        base_path,
        (
            "MFEP_OPTIMAL_IVM_GRID_IVM_"
            "METRIC_FENEB_FOUR_METRICS.npz"
        ),
    )

    np.savez(
        comparison_results_file,

        grid_ivm_results=np.array(
            grid_ivm_results,
            dtype=object,
        ),

        optimal_ivm_result=np.array(
            [optimal_result],
            dtype=object,
        ),

        feneb_results=np.array(
            feneb_results,
            dtype=object,
        ),

        feneb_completion_rows=np.array(
            feneb_completion_rows,
            dtype=object,
        ),

        summary=np.array(
            summary_rows,
            dtype=object,
        ),
    )

    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)

    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": (
            grid_ivm_results
        ),

        "optimal_result": (
            optimal_result
        ),

        "feneb_results": (
            feneb_results
        ),

        "summary": summary_rows,

        "optimal_current_count": (
            optimal_current_count
        ),

        "optimal_complete": (
            optimal_complete
        ),

        "feneb_all_complete": (
            feneb_all_complete
        ),

        "feneb_completion_rows": (
            feneb_completion_rows
        ),
    }


# ============================================================
# 判断是否满足自动停止条件
# ============================================================

def should_stop_auto_refresh(
    latest_output,
):
    required = []

    if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
        required.append(
            bool(
                latest_output[
                    "optimal_complete"
                ]
            )
        )

    if AUTO_STOP_REQUIRE_FENEB_COMPLETE:
        required.append(
            bool(
                latest_output[
                    "feneb_all_complete"
                ]
            )
        )

    if not required:
        return False

    return all(
        required
    )


# ============================================================
# 执行一次或自动刷新
# ============================================================

if AUTO_REFRESH:
    print(
        "AUTO_REFRESH=True："
        f"每 {REFRESH_SECONDS} 秒刷新一次。"
    )

    print(
        "当前会实时读取 Optimal-IVM 和新版 MFEP-FENEB；"
        "已保存多少个 state 就绘制多少个 state。"
    )

    print(
        "点击 Jupyter 的停止按钮可终止刷新。"
    )

    latest_output = None

    try:
        while True:
            if clear_output is not None:
                clear_output(
                    wait=True
                )

            latest_output = (
                load_and_plot_once()
            )

            optimal_complete = (
                latest_output[
                    "optimal_complete"
                ]
            )

            feneb_all_complete = (
                latest_output[
                    "feneb_all_complete"
                ]
            )

            if should_stop_auto_refresh(
                latest_output
            ):
                print(
                    "\n所有要求监控的计算均已完成："
                )

                if (
                    AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
                ):
                    print(
                        "  Optimal-IVM complete="
                        f"{optimal_complete}"
                    )

                if (
                    AUTO_STOP_REQUIRE_FENEB_COMPLETE
                ):
                    print(
                        "  MFEP-FENEB all complete="
                        f"{feneb_all_complete}"
                    )

                print(
                    "自动刷新结束。"
                )

                break

            if (
                latest_output[
                    "optimal_current_count"
                ] < 0
                and not WAIT_FOR_OPTIMAL_FILE
                and AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
            ):
                print(
                    "\nOptimal-IVM 文件尚不存在，"
                    "且 WAIT_FOR_OPTIMAL_FILE=False，"
                    "停止刷新。"
                )

                break

            if (
                len(
                    latest_output[
                        "feneb_results"
                    ]
                ) == 0
                and not WAIT_FOR_FENEB_FILE
                and AUTO_STOP_REQUIRE_FENEB_COMPLETE
            ):
                print(
                    "\nMFEP-FENEB 文件尚不存在，"
                    "且 WAIT_FOR_FENEB_FILE=False，"
                    "停止刷新。"
                )

                break

            print(
                "\n下次刷新将在 "
                f"{REFRESH_SECONDS} 秒后进行。"
            )

            time.sleep(
                REFRESH_SECONDS
            )

    except KeyboardInterrupt:
        print(
            "\n自动刷新已手动停止。"
        )

else:
    latest_output = (
        load_and_plot_once()
    )

In [ ]:
# -*- coding: utf-8 -*-

import os
import time
import zipfile

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


# ============================================================
# 基本路径与配置
# ============================================================

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)


N = 20
TOTAL_INNER_STEPS = 200


C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)


C2 = np.array(
    [-1.33047160, 0.99118795],
    dtype=float,
)


# ============================================================
# 监控的 Grid-IVM 配置（与您当前正在运行的实验完全一致）
#
# GRID_N_SUB      = [3]
# GRID_ALPHA      = [0.0001]
# GRID_ETA        = [0.03, 0.01, 0.003, 0.001]
# ============================================================

MONITOR_N_SUB = [3]
MONITOR_ALPHA = [0.0001]
MONITOR_ETA = [0.03, 0.01, 0.003, 0.001]

# 选择要绘制的配置（None 表示全部绘制，此处已设置为仅监控网格）
PLOT_ONLY_N_SUB = MONITOR_N_SUB
PLOT_ONLY_ALPHA = MONITOR_ALPHA
PLOT_ONLY_ETA = MONITOR_ETA


# ============================================================
# Optimal-IVM 配置
#
# 注意：
# 这里必须保持现有文件夹命名，即使文件夹名称中的学习率文字
# 和当前实际 schedule 不完全一致，也不能修改，否则找不到文件。
# ============================================================

OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200


OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)


OPTIMAL_FOLDER = os.path.join(
    base_path,
    OPTIMAL_CASENUM,
)


OPTIMAL_NPZ = os.path.join(
    OPTIMAL_FOLDER,
    f"alanine_{OPTIMAL_CASENUM}.npz",
)


# ============================================================
# 新版 MFEP-FENEB 配置
#
# 必须与 FENEB_ala_compare.py 一致。
#
# 对应前面给出的新版 FENEB 脚本：
#
# case_number =
#     MFEP_FENEB_COMPARE
#     _N20
#     _lambda...
#     _Nupdates200
#     _Kspring1
#     _dtau0.001
#     _reparamTol0.001
#     _sameIVMsettings
# ============================================================

FENEB_REPARAM_KS = 1.0
FENEB_REPARAM_DTAU = 1.0e-3
FENEB_REPARAM_TOL = 1.0e-3
FENEB_REPARAM_MAXITER = 5000

FENEB_MAX_UPDATES = 200


# 必须和 FENEB_ala_compare.py 中的 GRID_PHYSICAL_STEP 一致
GRID_FENEB_PHYSICAL_STEP = [
    1.0e-2,
    5.0e-3,
    2.0e-3,
    1.0e-3,
]


# ============================================================
# 自动刷新设置
# ============================================================

AUTO_REFRESH = True
REFRESH_SECONDS = 60


# Optimal-IVM 文件尚未创建时是否继续等待
WAIT_FOR_OPTIMAL_FILE = True


# FENEB 文件尚未创建时是否继续等待
WAIT_FOR_FENEB_FILE = True


# 自动停止条件
AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True
AUTO_STOP_REQUIRE_MONITOR_GRID_COMPLETE = True   # 新增：监控的 Grid-IVM 全部完成时停止


# ============================================================
# 绘图设置
# ============================================================

USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True


SAVE_OUTPUT = True


# 普通 Grid-IVM (现在只有监控的那几条)
GRID_IVM_ALPHA = 0.8
GRID_IVM_LINEWIDTH = 2.0


# Optimal-IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2


# MFEP-FENEB
FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8


# ============================================================
# 为监控的 Grid-IVM 根据 η 分配颜色
# ============================================================
# 降序对应 η 列表，方便区分四条线
MONITOR_GRID_COLORS = {
    0.03: "#377eb8",
    0.01: "#4daf4a",
    0.003: "#984ea3",
    0.001: "#ff7f00",
}


# ============================================================
# 安全读取正在原子化写入的 npz
# ============================================================

def safe_load_npz(
    npz_file,
    retries=10,
    wait_seconds=2.0,
):
    if not os.path.exists(npz_file):
        return None

    last_error = None

    for attempt in range(retries):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as loaded:
                return {
                    key: np.array(
                        loaded[key],
                        copy=True,
                    )
                    for key in loaded.files
                }

        except (
            zipfile.BadZipFile,
            EOFError,
            OSError,
            ValueError,
        ) as exc:
            last_error = exc

            if attempt < retries - 1:
                time.sleep(wait_seconds)

    print(
        "[WARNING] 无法读取正在写入的 npz：\n"
        f"  {npz_file}\n"
        f"  error: {last_error}"
    )

    return None


# ============================================================
# Grid-IVM 配置路径
# ============================================================

def ivm_config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_{mycase}_{casenum}_final.pth",
    )

    return (
        casenum,
        folder_path,
        npz_file,
        final_model,
    )


def selected_config(
    N_sub,
    alpha_IVP,
    eta_in,
):
    if (
        PLOT_ONLY_N_SUB is not None
        and N_sub not in PLOT_ONLY_N_SUB
    ):
        return False

    if (
        PLOT_ONLY_ALPHA is not None
        and alpha_IVP not in PLOT_ONLY_ALPHA
    ):
        return False

    if (
        PLOT_ONLY_ETA is not None
        and eta_in not in PLOT_ONLY_ETA
    ):
        return False

    return True


# ============================================================
# 数值工具
# ============================================================

def central_difference_tangent(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            "Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            "Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v_i @ G_i @ v_i
        )

        if (
            not np.isfinite(norm_G_sq)
            or norm_G_sq <= 1.0e-30
        ):
            raise FloatingPointError(
                "Near-zero or invalid G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v_i
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = (
            G_i @ tau_G
        )

        denom = float(
            tau_G @ G_tau_G
        )

        if (
            not np.isfinite(denom)
            or denom <= 1.0e-30
        ):
            raise FloatingPointError(
                "Invalid projector denominator "
                f"at image {i}: {denom}"
            )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / denom
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


def local_G_arclength_weights(
    full_path,
    G_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    length_plus_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    length_minus = np.sqrt(
        np.maximum(
            length_minus_sq,
            0.0,
        )
    )

    length_plus = np.sqrt(
        np.maximum(
            length_plus_sq,
            0.0,
        )
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def local_euclidean_arclength_weights(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    length_minus = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    length_plus = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    return 0.5 * (
        length_minus
        + length_plus
    )


def compute_arclength_residual(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            "full_path must have shape (n,2), "
            f"got {full_path.shape}"
        )

    n_full = full_path.shape[0]
    n_internal = n_full - 2

    if n_internal < 1:
        raise ValueError(
            "The path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    tangent_second_inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        tangent_second_inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand)
        * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner_phi_prime_second": (
            tangent_second_inner
        ),
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            "is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    segment_gradients[0] = (
        F_internal[0]
    )

    if n_internal > 1:
        segment_gradients[1:n_internal] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    segment_gradients[n_internal] = (
        F_internal[-1]
    )

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    expected_path_shape = (
        n_internal + 2,
        2,
    )

    expected_M_shape = (
        n_internal,
        2,
        2,
    )

    expected_F_shape = (
        n_internal,
        2,
    )

    if full_path.shape != expected_path_shape:
        raise ValueError(
            "Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != expected_M_shape:
        raise ValueError(
            "Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != expected_F_shape:
        raise ValueError(
            "Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights_G = local_G_arclength_weights(
        full_path,
        G_all,
    )

    I_perp_G = float(
        np.sum(
            residual_G_sq
            * weights_G
        )
    )

    residual_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    weights_E = (
        local_euclidean_arclength_weights(
            full_path
        )
    )

    I_perp_E = float(
        np.sum(
            residual_E_sq
            * weights_E
        )
    )

    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    arc_result = compute_arclength_residual(
        full_path
    )

    fmax_E = float(
        np.max(
            np.sqrt(
                np.maximum(
                    residual_E_sq,
                    0.0,
                )
            )
        )
    )

    fmax_G = float(
        np.max(
            np.sqrt(
                np.maximum(
                    residual_G_sq,
                    0.0,
                )
            )
        )
    )

    return {
        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "free_energy_max_index": (
            free_energy["max_index"]
        ),

        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result[
                "inner_phi_prime_second"
            ]
        ),

        "G_all": G_all,
        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "weights_G": weights_G,
        "weights_E": weights_E,

        "fmax_E": fmax_E,
        "fmax_G": fmax_G,
    }


# ============================================================
# 去除重复 count，保留同一个 count 的最后状态
# ============================================================

def keep_last_for_duplicate_counts(
    count_seq,
    *arrays,
):
    count_seq = np.asarray(
        count_seq
    )

    if count_seq.size == 0:
        return (
            count_seq,
            *arrays,
        )

    last_index_for_count = {}

    for i, count in enumerate(
        count_seq
    ):
        last_index_for_count[
            float(count)
        ] = i

    selected_indices = np.array(
        sorted(
            last_index_for_count.values(),
            key=lambda index: (
                count_seq[index]
            ),
        ),
        dtype=int,
    )

    output = [
        count_seq[selected_indices]
    ]

    for array in arrays:
        array = np.asarray(
            array
        )

        output.append(
            array[selected_indices]
        )

    return tuple(output)


# ============================================================
# 从 IVM npz 中读取逐 outer 数据（通用）
# ============================================================

def load_ivm_result(
    npz_file,
    method_name,
    N_sub,
    alpha_IVP,
    eta_description,
    require_complete=False,
    final_model=None,
):
    if not os.path.exists(npz_file):
        return None, (
            f"npz does not exist: {npz_file}"
        )

    if (
        require_complete
        and final_model is not None
        and not os.path.exists(final_model)
    ):
        return None, (
            "final model does not exist: "
            f"{final_model}"
        )

    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"could not read: {npz_file}"
        )

    if "index_seq" not in data:
        return None, (
            f"index_seq missing: {npz_file}"
        )

    index_seq_saved = np.asarray(
        data["index_seq"],
        dtype=int,
    )

    if index_seq_saved.size == 0:
        return None, (
            f"empty index_seq: {npz_file}"
        )

    if require_complete:
        expected_batches = (
            TOTAL_INNER_STEPS
            // N_sub
        )

        if int(
            np.max(index_seq_saved)
        ) < expected_batches:
            return None, (
                "incomplete: max index="
                f"{np.max(index_seq_saved)}, "
                "expected at least "
                f"{expected_batches}"
            )

    if "count_seq" in data:
        saved_count_seq = np.asarray(
            data["count_seq"],
            dtype=int,
        )
    else:
        saved_count_seq = (
            index_seq_saved
            * N_sub
        )

    n_available = min(
        len(index_seq_saved),
        len(saved_count_seq),
    )

    index_seq_saved = (
        index_seq_saved[:n_available]
    )

    saved_count_seq = (
        saved_count_seq[:n_available]
    )

    valid_index_seq = []
    count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    skipped_outer_indices = []

    for position in range(
        n_available
    ):
        outer_index = int(
            index_seq_saved[position]
        )

        current_count = int(
            saved_count_seq[position]
        )

        phi_key = (
            f"phi_ref_iter_"
            f"{outer_index}"
        )

        M_key = (
            f"M_ref_iter_"
            f"{outer_index}"
        )

        F_key = (
            f"F_ref_iter_"
            f"{outer_index}"
        )

        g_key = (
            f"g_ref_iter_"
            f"{outer_index}"
        )

        if F_key in data:
            selected_F_key = F_key
        elif g_key in data:
            selected_F_key = g_key
        else:
            selected_F_key = None

        required_present = (
            phi_key in data
            and M_key in data
            and selected_F_key is not None
        )

        if not required_present:
            skipped_outer_indices.append(
                outer_index
            )
            continue

        try:
            phi_internal = np.asarray(
                data[phi_key],
                dtype=float,
            )

            M_all = np.asarray(
                data[M_key],
                dtype=float,
            )

            F_all = np.asarray(
                data[selected_F_key],
                dtype=float,
            )

            full_path = np.vstack(
                [
                    C1[None, :],
                    phi_internal,
                    C2[None, :],
                ]
            )

            metrics = compute_all_path_metrics(
                full_path,
                M_all,
                F_all,
            )

        except Exception as exc:
            print(
                "[WARNING] 跳过 "
                f"{method_name} outer={outer_index}: "
                f"{exc}"
            )

            skipped_outer_indices.append(
                outer_index
            )

            continue

        valid_index_seq.append(
            outer_index
        )

        count_seq.append(
            current_count
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

    if len(count_seq) == 0:
        return None, (
            "no complete outer state found: "
            f"{npz_file}"
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        valid_index_seq,
        free_energy_profiles,
        arc_integrands,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            count_seq,
            dtype=int,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            valid_index_seq,
            dtype=int,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
    )

    result = {
        "method": method_name,
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_description": eta_description,
        "npz_file": npz_file,

        "index_seq": valid_index_seq,
        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,

        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "skipped_outer_indices": (
            np.asarray(
                skipped_outer_indices,
                dtype=int,
            )
        ),
    }

    if "lr_count_seq" in data:
        result["lr_count_seq"] = np.asarray(
            data["lr_count_seq"],
            dtype=int,
        )

    if "lr_seq" in data:
        result["lr_seq"] = np.asarray(
            data["lr_seq"],
            dtype=float,
        )

    return result, None


# ============================================================
# 读取监控的 Grid-IVM（实时模式：不要求完成）
# ============================================================

def load_monitor_grid_ivm():
    ivm_results = []
    skipped = []

    all_configs = [
        (
            N_sub,
            alpha_IVP,
            eta_in,
        )
        for N_sub in MONITOR_N_SUB
        for alpha_IVP in MONITOR_ALPHA
        for eta_in in MONITOR_ETA
    ]

    for (
        N_sub,
        alpha_IVP,
        eta_in,
    ) in all_configs:
        (
            casenum,
            folder_path,
            npz_file,
            final_model,
        ) = ivm_config_paths(
            N_sub,
            alpha_IVP,
            eta_in,
        )

        result, error = load_ivm_result(
            npz_file=npz_file,
            method_name="IVM-grid",
            N_sub=N_sub,
            alpha_IVP=alpha_IVP,
            eta_description=str(eta_in),
            require_complete=False,          # 实时模式
            final_model=None,
        )

        if result is None:
            skipped.append(
                f"{casenum}: {error}"
            )
            continue

        result["eta_in"] = eta_in
        result["casenum"] = casenum
        result["final_model"] = final_model  # 保存路径用于判断完成

        ivm_results.append(
            result
        )

    return (
        ivm_results,
        skipped,
    )


# ============================================================
# 读取正在运行的 Optimal-IVM
# ============================================================

def load_optimal_ivm():
    result, error = load_ivm_result(
        npz_file=OPTIMAL_NPZ,
        method_name="Optimal-IVM",
        N_sub=OPTIMAL_N_SUB,
        alpha_IVP=OPTIMAL_ALPHA,
        eta_description=(
            "adaptive Adam"
        ),
        require_complete=False,
        final_model=None,
    )

    if result is not None:
        result["casenum"] = (
            OPTIMAL_CASENUM
        )

    return result, error


# ============================================================
# 新版 MFEP-FENEB 文件路径
# ============================================================

def float_tag(value):
    return f"{float(value):g}"


def feneb_result_paths(
    physical_step,
):
    step_tag = float_tag(
        physical_step
    )

    casenum = (
        f"MFEP_FENEB_COMPARE"
        f"_N{N}"
        f"_lambda{step_tag}"
        f"_Nupdates{FENEB_MAX_UPDATES}"
        f"_Kspring{FENEB_REPARAM_KS:g}"
        f"_dtau{FENEB_REPARAM_DTAU:g}"
        f"_reparamTol{FENEB_REPARAM_TOL:g}"
        f"_sameIVMsettings"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    result_file = os.path.join(
        folder_path,
        f"alanine_{casenum}.npz",
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    return (
        casenum,
        folder_path,
        result_file,
        done_file,
    )


# ============================================================
# 读取新版 MFEP-FENEB 单个结果
# ============================================================

def load_single_feneb_result(
    npz_file,
    expected_physical_step=None,
):
    data = safe_load_npz(
        npz_file
    )

    if data is None:
        return None, (
            f"Could not read {npz_file}"
        )

    required_basic_keys = [
        "count_seq",
        "M_ref_seq",
        "F_ref_seq",
    ]

    missing_basic = [
        key
        for key in required_basic_keys
        if key not in data
    ]

    if missing_basic:
        return None, (
            "Missing FENEB keys: "
            f"{missing_basic}"
        )

    if "mean_path_unwrapped_seq" in data:
        path_key = (
            "mean_path_unwrapped_seq"
        )
    elif "mean_path_seq" in data:
        path_key = "mean_path_seq"
    else:
        return None, (
            "Neither mean_path_unwrapped_seq "
            "nor mean_path_seq exists."
        )

    count_raw = np.asarray(
        data["count_seq"],
        dtype=float,
    )

    path_raw = np.asarray(
        data[path_key],
        dtype=float,
    )

    M_raw = np.asarray(
        data["M_ref_seq"],
        dtype=float,
    )

    F_raw = np.asarray(
        data["F_ref_seq"],
        dtype=float,
    )

    n_states = min(
        len(count_raw),
        len(path_raw),
        len(M_raw),
        len(F_raw),
    )

    if n_states == 0:
        return None, (
            "No saved FENEB states."
        )

    count_raw = count_raw[:n_states]
    path_raw = path_raw[:n_states]
    M_raw = M_raw[:n_states]
    F_raw = F_raw[:n_states]

    valid_count_seq = []

    I_perp_G_seq = []
    I_perp_E_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []

    free_energy_profiles = []
    arc_integrands = []

    fmax_E_seq = []
    fmax_G_seq = []

    valid_state_indices = []
    skipped_state_indices = []

    for state_index in range(
        n_states
    ):
        try:
            current_path = np.asarray(
                path_raw[state_index],
                dtype=float,
            )

            current_M = np.asarray(
                M_raw[state_index],
                dtype=float,
            )

            current_F = np.asarray(
                F_raw[state_index],
                dtype=float,
            )

            if current_path.shape == (
                N,
                2,
            ):
                full_path = np.vstack(
                    [
                        C1[None, :],
                        current_path,
                        C2[None, :],
                    ]
                )

            elif current_path.shape == (
                N + 2,
                2,
            ):
                full_path = (
                    current_path.copy()
                )

            else:
                raise ValueError(
                    "Unexpected FENEB path shape: "
                    f"{current_path.shape}; expected "
                    f"{(N, 2)} or {(N + 2, 2)}."
                )

            metrics = compute_all_path_metrics(
                full_path,
                current_M,
                current_F,
            )

        except Exception as exc:
            print(
                "[WARNING] 跳过 MFEP-FENEB "
                f"state={state_index}, "
                f"count={count_raw[state_index]}: "
                f"{exc}"
            )

            skipped_state_indices.append(
                state_index
            )

            continue

        valid_state_indices.append(
            state_index
        )

        valid_count_seq.append(
            count_raw[state_index]
        )

        I_perp_G_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_E_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profiles.append(
            metrics["free_energy_profile"]
        )

        arc_integrands.append(
            metrics["arc_integrand"]
        )

        fmax_E_seq.append(
            metrics["fmax_E"]
        )

        fmax_G_seq.append(
            metrics["fmax_G"]
        )

    if len(valid_count_seq) == 0:
        return None, (
            "No valid FENEB state could be reconstructed."
        )

    (
        count_seq,
        I_perp_G_seq,
        I_perp_E_seq,
        I_arc_seq,
        Delta_F_seq,
        Delta_F_final_seq,
        free_energy_profiles,
        arc_integrands,
        fmax_E_seq,
        fmax_G_seq,
        valid_state_indices,
    ) = keep_last_for_duplicate_counts(
        np.asarray(
            valid_count_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_G_seq,
            dtype=float,
        ),
        np.asarray(
            I_perp_E_seq,
            dtype=float,
        ),
        np.asarray(
            I_arc_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_seq,
            dtype=float,
        ),
        np.asarray(
            Delta_F_final_seq,
            dtype=float,
        ),
        np.asarray(
            free_energy_profiles,
            dtype=float,
        ),
        np.asarray(
            arc_integrands,
            dtype=float,
        ),
        np.asarray(
            fmax_E_seq,
            dtype=float,
        ),
        np.asarray(
            fmax_G_seq,
            dtype=float,
        ),
        np.asarray(
            valid_state_indices,
            dtype=int,
        ),
    )

    if "physical_step" in data:
        physical_step = float(
            np.asarray(
                data["physical_step"]
            ).reshape(-1)[0]
        )
    elif expected_physical_step is not None:
        physical_step = float(
            expected_physical_step
        )
    else:
        physical_step = np.nan

    if "spring_constant" in data:
        spring_constant = float(
            np.asarray(
                data["spring_constant"]
            ).reshape(-1)[0]
        )
    else:
        spring_constant = float(
            FENEB_REPARAM_KS
        )

    if "reparam_dtau" in data:
        reparam_dtau = float(
            np.asarray(
                data["reparam_dtau"]
            ).reshape(-1)[0]
        )
    else:
        reparam_dtau = float(
            FENEB_REPARAM_DTAU
        )

    if "reparam_tolerance" in data:
        reparam_tolerance = float(
            np.asarray(
                data["reparam_tolerance"]
            ).reshape(-1)[0]
        )
    else:
        reparam_tolerance = float(
            FENEB_REPARAM_TOL
        )

    folder_path = os.path.dirname(
        npz_file
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    done = os.path.exists(
        done_file
    )

    result = {
        "method": "MFEP-FENEB",

        "npz_file": npz_file,
        "folder_path": folder_path,
        "done_file": done_file,
        "done": done,

        "physical_step": physical_step,
        "spring_constant": spring_constant,
        "reparam_dtau": reparam_dtau,
        "reparam_tolerance": (
            reparam_tolerance
        ),

        "source_path_key": path_key,

        "count_seq": count_seq,

        "I_perp_G_seq": I_perp_G_seq,
        "I_perp_E_seq": I_perp_E_seq,
        "I_arc_seq": I_arc_seq,

        "Delta_F_seq": Delta_F_seq,
        "Delta_F_final_seq": (
            Delta_F_final_seq
        ),

        "free_energy_profiles": (
            free_energy_profiles
        ),

        "arc_integrands": (
            arc_integrands
        ),

        "fmax_E_seq": fmax_E_seq,
        "fmax_G_seq": fmax_G_seq,

        "valid_state_indices": (
            valid_state_indices
        ),

        "skipped_state_indices": (
            np.asarray(
                skipped_state_indices,
                dtype=int,
            )
        ),
    }

    return result, None


# ============================================================
# 读取所有新版 MFEP-FENEB 结果
# ============================================================

def load_all_feneb_results():
    feneb_results = []
    errors = []

    for physical_step in (
        GRID_FENEB_PHYSICAL_STEP
    ):
        (
            casenum,
            folder_path,
            npz_file,
            done_file,
        ) = feneb_result_paths(
            physical_step
        )

        result, error = (
            load_single_feneb_result(
                npz_file,
                expected_physical_step=(
                    physical_step
                ),
            )
        )

        if result is not None:
            result["casenum"] = casenum
            result["expected_done_file"] = (
                done_file
            )

            feneb_results.append(
                result
            )

        else:
            errors.append(
                "lambda="
                f"{physical_step:g}: "
                f"{error}"
            )

    return (
        feneb_results,
        errors,
    )


# ============================================================
# 判断新版 MFEP-FENEB 是否全部完成
# ============================================================

def evaluate_feneb_completion(
    feneb_results,
):
    status = []

    for expected_step in (
        GRID_FENEB_PHYSICAL_STEP
    ):
        matching = [
            result
            for result in feneb_results
            if np.isclose(
                result["physical_step"],
                expected_step,
                rtol=1.0e-10,
                atol=1.0e-15,
            )
        ]

        found = len(matching) > 0

        if found:
            done = bool(
                matching[0]["done"]
            )

            latest_count = float(
                matching[0][
                    "count_seq"
                ][-1]
            )

        else:
            done = False
            latest_count = np.nan

        status.append(
            {
                "physical_step": float(
                    expected_step
                ),
                "found": found,
                "done": done,
                "latest_count": (
                    latest_count
                ),
            }
        )

    all_complete = all(
        row["found"] and row["done"]
        for row in status
    )

    return (
        all_complete,
        status,
    )


# ============================================================
# MFEP-FENEB 绘图样式
# ============================================================

FENEB_STEP_COLORS = {
    1.0e-2: "#ff7f00",
    5.0e-3: "#a65628",
    2.0e-3: "#f781bf",
    1.0e-3: "#00a6d6",
}


FENEB_STEP_LINESTYLES = {
    1.0e-2: "-",
    5.0e-3: "--",
    2.0e-3: "-.",
    1.0e-3: ":",
}


def dictionary_value_isclose(
    dictionary,
    value,
    default=None,
):
    for key, mapped_value in (
        dictionary.items()
    ):
        if np.isclose(
            float(key),
            float(value),
            rtol=1.0e-10,
            atol=1.0e-15,
        ):
            return mapped_value

    return default


def feneb_style(
    physical_step,
    index,
):
    color = dictionary_value_isclose(
        FENEB_STEP_COLORS,
        physical_step,
        default=None,
    )

    if color is None:
        cmap = plt.get_cmap(
            "tab10"
        )

        color = cmap(
            index % 10
        )

    linestyle = dictionary_value_isclose(
        FENEB_STEP_LINESTYLES,
        physical_step,
        default=None,
    )

    if linestyle is None:
        styles = [
            "-",
            "--",
            "-.",
            ":",
            (0, (3, 5, 1, 5)),
            (0, (1, 1)),
            (0, (5, 10)),
        ]

        linestyle = styles[
            index % len(styles)
        ]

    return (
        color,
        linestyle,
    )


def feneb_label(
    result,
):
    physical_step = (
        result["physical_step"]
    )

    label = (
        "MFEP-FENEB "
        rf"$\lambda={physical_step:g}$"
    )

    if result["done"]:
        label += " [DONE]"
    else:
        label += " [running]"

    return label


# ============================================================
# 判断监控的 Grid-IVM 是否全部完成
# ============================================================

def monitor_grid_all_complete(grid_ivm_results):
    if len(grid_ivm_results) < len(MONITOR_ETA):
        return False
    for result in grid_ivm_results:
        # 完成的标志：final model 存在，或已经跑到 200
        if os.path.exists(result.get("final_model", "")):
            continue
        if result["count_seq"][-1] >= TOTAL_INNER_STEPS:
            continue
        return False
    return True


# ============================================================
# 绘制一次
# ============================================================

def load_and_plot_once():
    (
        grid_ivm_results,
        skipped_grid,
    ) = load_monitor_grid_ivm()

    optimal_result, optimal_error = (
        load_optimal_ivm()
    )

    (
        feneb_results,
        feneb_errors,
    ) = load_all_feneb_results()

    (
        feneb_all_complete,
        feneb_completion_rows,
    ) = evaluate_feneb_completion(
        feneb_results
    )

    grid_all_complete = monitor_grid_all_complete(
        grid_ivm_results
    )

    # --------------------------------------------------------
    # 状态输出
    # --------------------------------------------------------

    print("=" * 120)
    print(
        "MFEP Monitor Grid-IVM / Optimal-IVM / "
        "metric-extended FENEB comparison"
    )
    print("=" * 120)

    print(
        "已读取的 Monitor Grid-IVM 配置："
        f"{len(grid_ivm_results)} / {len(MONITOR_ETA)}"
    )

    print(
        "跳过的 Monitor Grid-IVM 配置："
        f"{len(skipped_grid)}"
    )

    if optimal_result is not None:
        optimal_current_count = int(
            np.max(
                optimal_result[
                    "count_seq"
                ]
            )
        )

        optimal_complete = (
            optimal_current_count
            >= OPTIMAL_TOTAL_STEPS
        )

        print(
            "Optimal-IVM：已读取，"
            f"当前 count="
            f"{optimal_current_count}/"
            f"{OPTIMAL_TOTAL_STEPS}"
        )

        print(
            "Optimal-IVM states："
            f"{len(optimal_result['count_seq'])}"
        )

        print(
            "Optimal-IVM complete："
            f"{optimal_complete}"
        )

    else:
        optimal_current_count = -1
        optimal_complete = False

        print(
            "Optimal-IVM：尚未读取到有效数据"
        )

        print(
            "  ",
            optimal_error,
        )

    print("-" * 120)

    print(
        "成功读取的 MFEP-FENEB 配置数："
        f"{len(feneb_results)}"
    )

    if feneb_results:
        for result in feneb_results:
            print(
                "MFEP-FENEB "
                f"lambda="
                f"{result['physical_step']:g}"
                " | states="
                f"{len(result['count_seq']):4d}"
                " | count range=("
                f"{result['count_seq'][0]:g}, "
                f"{result['count_seq'][-1]:g})"
                " | path="
                f"{result['source_path_key']}"
                " | DONE="
                f"{result['done']}"
            )

    else:
        print(
            "尚未读取到有效 MFEP-FENEB 数据。"
        )

    if feneb_errors:
        print(
            "MFEP-FENEB 读取警告数："
            f"{len(feneb_errors)}"
        )

        for error in feneb_errors[:10]:
            print(
                "  ",
                error,
            )

    print("-" * 120)
    print("预期 MFEP-FENEB 完成状态：")

    for row in feneb_completion_rows:
        print(
            "  lambda="
            f"{row['physical_step']:g}: "
            f"found={row['found']}, "
            f"DONE={row['done']}, "
            f"latest_count="
            f"{row['latest_count']}"
        )

    print(
        "MFEP-FENEB all complete："
        f"{feneb_all_complete}"
    )

    print(
        "Monitor Grid-IVM all complete："
        f"{grid_all_complete}"
    )

    print("=" * 120)

    # --------------------------------------------------------
    # Log-y 安全值
    # --------------------------------------------------------

    tiny = np.finfo(
        float
    ).tiny

    all_available = list(
        grid_ivm_results
    )

    if optimal_result is not None:
        all_available.append(
            optimal_result
        )

    all_available.extend(
        feneb_results
    )

    for result in all_available:
        result["I_perp_G_plot"] = np.maximum(
            result["I_perp_G_seq"],
            tiny,
        )

        result["I_perp_E_plot"] = np.maximum(
            result["I_perp_E_seq"],
            tiny,
        )

        result["I_arc_plot"] = np.maximum(
            result["I_arc_seq"],
            tiny,
        )

        result["Delta_F_plot"] = np.maximum(
            result["Delta_F_seq"],
            tiny,
        )

    # --------------------------------------------------------
    # 创建 1 x 4 图
    # --------------------------------------------------------

    plt.close("all")

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(27, 6.5),
    )

    (
        ax_G,
        ax_E,
        ax_arc,
        ax_DF,
    ) = axes

    # --------------------------------------------------------
    # Monitor Grid-IVM
    # --------------------------------------------------------
    # 固定 eta 线型，颜色根据 eta 分配
    eta_linestyles = {
        0.03: "-",
        0.01: "--",
        0.003: "-.",
        0.001: ":",
    }

    # N_sub 和 alpha 都固定，所以标记也固定
    fixed_marker = "o"

    for result in grid_ivm_results:
        eta_in = result["eta_in"]
        count_seq = result["count_seq"]

        color = MONITOR_GRID_COLORS.get(
            eta_in,
            "gray",
        )

        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )

        markevery = max(
            1,
            len(count_seq) // 8,
        )

        common_kwargs = {
            "color": color,
            "linestyle": linestyle,
            "marker": fixed_marker,
            "markevery": markevery,
            "markersize": 3.5,
            "linewidth": (
                GRID_IVM_LINEWIDTH
            ),
            "alpha": GRID_IVM_ALPHA,
            "zorder": 2,
        }

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **common_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **common_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **common_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **common_kwargs,
        )

    # --------------------------------------------------------
    # Optimal-IVM
    # --------------------------------------------------------

    if optimal_result is not None:
        optimal_count = (
            optimal_result[
                "count_seq"
            ]
        )

        optimal_markevery = max(
            1,
            len(optimal_count) // 12,
        )

        optimal_kwargs = {
            "color": OPTIMAL_COLOR,
            "linestyle": (
                OPTIMAL_LINESTYLE
            ),
            "marker": OPTIMAL_MARKER,
            "markevery": (
                optimal_markevery
            ),
            "markersize": 8,
            "markerfacecolor": (
                OPTIMAL_COLOR
            ),
            "markeredgecolor": "white",
            "markeredgewidth": 0.8,
            "linewidth": (
                OPTIMAL_LINEWIDTH
            ),
            "alpha": 1.0,
            "zorder": 30,
        }

        ax_G.plot(
            optimal_count,
            optimal_result[
                "I_perp_G_plot"
            ],
            **optimal_kwargs,
        )

        ax_E.plot(
            optimal_count,
            optimal_result[
                "I_perp_E_plot"
            ],
            **optimal_kwargs,
        )

        ax_arc.plot(
            optimal_count,
            optimal_result[
                "I_arc_plot"
            ],
            **optimal_kwargs,
        )

        ax_DF.plot(
            optimal_count,
            optimal_result[
                "Delta_F_plot"
            ],
            **optimal_kwargs,
        )

        for axis, values in [
            (
                ax_G,
                optimal_result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                optimal_result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                optimal_result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                optimal_result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                optimal_count[-1],
                values[-1],
                s=130,
                marker="*",
                color=OPTIMAL_COLOR,
                edgecolor="black",
                linewidth=0.8,
                zorder=40,
            )

    # --------------------------------------------------------
    # 新版 MFEP-FENEB
    # --------------------------------------------------------

    for index, result in enumerate(
        feneb_results
    ):
        count_seq = result[
            "count_seq"
        ]

        (
            color,
            linestyle,
        ) = feneb_style(
            result["physical_step"],
            index,
        )

        markevery = max(
            1,
            len(count_seq) // 12,
        )

        feneb_kwargs = {
            "color": color,
            "linestyle": linestyle,
            "marker": FENEB_MARKER,
            "markevery": markevery,
            "markersize": 5.5,
            "markerfacecolor": "white",
            "markeredgecolor": color,
            "markeredgewidth": 1.2,
            "linewidth": FENEB_LINEWIDTH,
            "alpha": 1.0,
            "zorder": 20 + index,
        }

        ax_G.plot(
            count_seq,
            result["I_perp_G_plot"],
            **feneb_kwargs,
        )

        ax_E.plot(
            count_seq,
            result["I_perp_E_plot"],
            **feneb_kwargs,
        )

        ax_arc.plot(
            count_seq,
            result["I_arc_plot"],
            **feneb_kwargs,
        )

        ax_DF.plot(
            count_seq,
            result["Delta_F_plot"],
            **feneb_kwargs,
        )

        latest_marker = (
            "D"
            if result["done"]
            else "o"
        )

        for axis, values in [
            (
                ax_G,
                result[
                    "I_perp_G_plot"
                ],
            ),
            (
                ax_E,
                result[
                    "I_perp_E_plot"
                ],
            ),
            (
                ax_arc,
                result[
                    "I_arc_plot"
                ],
            ),
            (
                ax_DF,
                result[
                    "Delta_F_plot"
                ],
            ),
        ]:
            axis.scatter(
                count_seq[-1],
                values[-1],
                s=75,
                marker=latest_marker,
                facecolor=color,
                edgecolor="black",
                linewidth=0.8,
                zorder=50 + index,
            )

    # ========================================================
    # Panel 设置
    # ========================================================

    ax_G.set_title(
        "Riemannian normal residual",
        fontsize=14,
    )

    ax_G.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_G.set_ylabel(
        r"$\widehat I_{\perp}^{G}$"
        "\n"
        r"$=\int |P_\varphi^G M\nabla F|_G^2"
        r" |\varphi'|_G\,ds$",
        fontsize=10,
    )

    ax_E.set_title(
        "Euclidean residual integral",
        fontsize=14,
    )

    ax_E.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_E.set_ylabel(
        r"$\widehat I_{\perp}^{E}$"
        "\n"
        r"$=\int \|P_\varphi^G M\nabla F\|_2^2"
        r" |\varphi'|_2\,ds$",
        fontsize=10,
    )

    ax_arc.set_title(
        "Arclength / equi-speed residual",
        fontsize=14,
    )

    ax_arc.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_arc.set_ylabel(
        r"$I_{\rm arc}$"
        "\n"
        r"$=\int_0^1"
        r"(\varphi'\cdot\varphi'')^2"
        r"|\varphi'|\,ds$",
        fontsize=10,
    )

    ax_DF.set_title(
        "Path-integrated free-energy barrier",
        fontsize=14,
    )

    ax_DF.set_xlabel(
        "restrained-MD / gradient count",
        fontsize=12,
    )

    ax_DF.set_ylabel(
        r"$\Delta F^{\dagger}$ (kcal/mol)"
        "\n"
        "line-integral estimate",
        fontsize=10,
    )

    for axis in axes:
        axis.grid(
            True,
            which="both",
            linestyle="--",
            alpha=0.3,
        )

        axis.set_xlim(
            left=0,
        )

    if USE_LOG_Y_RESIDUALS:
        ax_G.set_yscale(
            "log"
        )

        ax_E.set_yscale(
            "log"
        )

    if USE_LOG_Y_ARC:
        ax_arc.set_yscale(
            "log"
        )

    if USE_LOG_Y_DELTA_F:
        ax_DF.set_yscale(
            "log"
        )

    # ========================================================
    # 图例
    # ========================================================

    method_handles = []

    # Monitor Grid-IVM 图例（按 η 区分）
    for eta_in in MONITOR_ETA:
        color = MONITOR_GRID_COLORS.get(
            eta_in,
            "gray",
        )
        linestyle = eta_linestyles.get(
            eta_in,
            "-",
        )
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                linestyle=linestyle,
                marker=fixed_marker,
                linewidth=GRID_IVM_LINEWIDTH,
                markersize=6,
                label=rf"Grid-IVM $\eta={eta_in}$",
            )
        )

    if optimal_result is not None:
        method_handles.append(
            Line2D(
                [0],
                [0],
                color=OPTIMAL_COLOR,
                linestyle=(
                    OPTIMAL_LINESTYLE
                ),
                marker=OPTIMAL_MARKER,
                linewidth=(
                    OPTIMAL_LINEWIDTH
                ),
                markersize=8,
                label="Optimal-IVM",
            )
        )

    for index, result in enumerate(
        feneb_results
    ):
        (
            color,
            linestyle,
        ) = feneb_style(
            result["physical_step"],
            index,
        )

        method_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                linestyle=linestyle,
                marker=FENEB_MARKER,
                markerfacecolor="white",
                markeredgecolor=color,
                linewidth=(
                    FENEB_LINEWIDTH
                ),
                markersize=6,
                label=feneb_label(
                    result
                ),
            )
        )

    legend_method = ax_DF.legend(
        handles=method_handles,
        title="Method",
        loc="upper left",
        bbox_to_anchor=(
            1.03,
            1.0,
        ),
        frameon=True,
    )

    ax_DF.add_artist(
        legend_method
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.80,
        bottom=0.14,
        top=0.90,
        wspace=0.38,
    )

    # ========================================================
    # 保存图片
    # ========================================================

    output_pdf = os.path.join(
        base_path,
        (
            "MFEP_MonitorGridIVM_OptimalIVM_"
            "MetricFENEB_four_metrics.pdf"
        ),
    )

    output_png = os.path.join(
        base_path,
        (
            "MFEP_MonitorGridIVM_OptimalIVM_"
            "MetricFENEB_four_metrics.png"
        ),
    )

    if SAVE_OUTPUT:
        fig.savefig(
            output_pdf,
            bbox_inches="tight",
        )

        fig.savefig(
            output_png,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()
    plt.close(fig)

    # ========================================================
    # 当前值汇总
    # ========================================================

    print("\n" + "=" * 160)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 160)

    summary_rows = []

    for result in grid_ivm_results:
        summary_rows.append(
            {
                "method": "IVM-grid",
                "description": (
                    f"Nsub={result['N_sub']} "
                    f"alpha={result['alpha_IVP']} "
                    f"eta={result['eta_in']}"
                ),
                "count": float(
                    result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": bool(
                    os.path.exists(result.get("final_model", ""))
                    or result["count_seq"][-1] >= TOTAL_INNER_STEPS
                ),
            }
        )

    if optimal_result is not None:
        summary_rows.append(
            {
                "method": "Optimal-IVM",
                "description": (
                    "Nsub=4 alpha=0.1 "
                    "adaptive Adam"
                ),
                "count": float(
                    optimal_result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    optimal_result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    optimal_result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    optimal_result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    optimal_result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": optimal_complete,
            }
        )

    for result in feneb_results:
        summary_rows.append(
            {
                "method": "MFEP-FENEB",
                "description": (
                    "lambda="
                    f"{result['physical_step']:g}"
                ),
                "count": float(
                    result[
                        "count_seq"
                    ][-1]
                ),
                "I_G": float(
                    result[
                        "I_perp_G_seq"
                    ][-1]
                ),
                "I_E": float(
                    result[
                        "I_perp_E_seq"
                    ][-1]
                ),
                "I_arc": float(
                    result[
                        "I_arc_seq"
                    ][-1]
                ),
                "Delta_F": float(
                    result[
                        "Delta_F_seq"
                    ][-1]
                ),
                "done": bool(
                    result["done"]
                ),
            }
        )

    method_order = {
        "Optimal-IVM": 0,
        "MFEP-FENEB": 1,
        "IVM-grid": 2,
    }

    summary_rows_sorted = sorted(
        summary_rows,
        key=lambda row: (
            method_order.get(
                row["method"],
                99,
            ),
            row["I_G"],
        ),
    )

    for row in summary_rows_sorted:
        print(
            f"{row['method']:<14} "
            f"{row['description']:<42}"
            f" count={row['count']:>7.1f}"
            f" | I_G={row['I_G']:.6e}"
            f" | I_E={row['I_E']:.6e}"
            f" | I_arc={row['I_arc']:.6e}"
            f" | DeltaF={row['Delta_F']:.6f}"
            f" | done={row['done']}"
        )

    # ========================================================
    # 保存汇总数据
    # ========================================================

    comparison_results_file = os.path.join(
        base_path,
        (
            "MFEP_MONITOR_GRID_IVM_OPTIMAL_IVM_"
            "METRIC_FENEB_FOUR_METRICS.npz"
        ),
    )

    np.savez(
        comparison_results_file,

        monitor_grid_ivm_results=np.array(
            grid_ivm_results,
            dtype=object,
        ),

        optimal_ivm_result=np.array(
            [optimal_result],
            dtype=object,
        ),

        feneb_results=np.array(
            feneb_results,
            dtype=object,
        ),

        feneb_completion_rows=np.array(
            feneb_completion_rows,
            dtype=object,
        ),

        summary=np.array(
            summary_rows,
            dtype=object,
        ),
    )

    print("\n图已保存到：")
    print(output_pdf)
    print(output_png)

    print("\n重新计算后的数据已保存到：")
    print(comparison_results_file)

    return {
        "grid_ivm_results": (
            grid_ivm_results
        ),

        "optimal_result": (
            optimal_result
        ),

        "feneb_results": (
            feneb_results
        ),

        "summary": summary_rows,

        "optimal_current_count": (
            optimal_current_count
        ),

        "optimal_complete": (
            optimal_complete
        ),

        "feneb_all_complete": (
            feneb_all_complete
        ),

        "feneb_completion_rows": (
            feneb_completion_rows
        ),

        "grid_all_complete": (
            grid_all_complete
        ),
    }


# ============================================================
# 判断是否满足自动停止条件
# ============================================================

def should_stop_auto_refresh(
    latest_output,
):
    required = []

    if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
        required.append(
            bool(
                latest_output[
                    "optimal_complete"
                ]
            )
        )

    if AUTO_STOP_REQUIRE_FENEB_COMPLETE:
        required.append(
            bool(
                latest_output[
                    "feneb_all_complete"
                ]
            )
        )

    if AUTO_STOP_REQUIRE_MONITOR_GRID_COMPLETE:
        required.append(
            bool(
                latest_output[
                    "grid_all_complete"
                ]
            )
        )

    if not required:
        return False

    return all(
        required
    )


# ============================================================
# 执行一次或自动刷新
# ============================================================

if AUTO_REFRESH:
    print(
        "AUTO_REFRESH=True："
        f"每 {REFRESH_SECONDS} 秒刷新一次。"
    )

    print(
        "当前监控的 Grid-IVM 网格："
        f"N_sub={MONITOR_N_SUB}, "
        f"alpha={MONITOR_ALPHA}, "
        f"eta={MONITOR_ETA}"
    )

    print(
        "点击 Jupyter 的停止按钮可终止刷新。"
    )

    latest_output = None

    try:
        while True:
            if clear_output is not None:
                clear_output(
                    wait=True
                )

            latest_output = (
                load_and_plot_once()
            )

            if should_stop_auto_refresh(
                latest_output
            ):
                print(
                    "\n所有要求监控的计算均已完成："
                )

                if (
                    AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
                ):
                    print(
                        "  Optimal-IVM complete="
                        f"{latest_output['optimal_complete']}"
                    )

                if (
                    AUTO_STOP_REQUIRE_FENEB_COMPLETE
                ):
                    print(
                        "  MFEP-FENEB all complete="
                        f"{latest_output['feneb_all_complete']}"
                    )

                if (
                    AUTO_STOP_REQUIRE_MONITOR_GRID_COMPLETE
                ):
                    print(
                        "  Monitor Grid-IVM all complete="
                        f"{latest_output['grid_all_complete']}"
                    )

                print(
                    "自动刷新结束。"
                )

                break

            if (
                latest_output[
                    "optimal_current_count"
                ] < 0
                and not WAIT_FOR_OPTIMAL_FILE
                and AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE
            ):
                print(
                    "\nOptimal-IVM 文件尚不存在，"
                    "且 WAIT_FOR_OPTIMAL_FILE=False，"
                    "停止刷新。"
                )

                break

            if (
                len(
                    latest_output[
                        "feneb_results"
                    ]
                ) == 0
                and not WAIT_FOR_FENEB_FILE
                and AUTO_STOP_REQUIRE_FENEB_COMPLETE
            ):
                print(
                    "\nMFEP-FENEB 文件尚不存在，"
                    "且 WAIT_FOR_FENEB_FILE=False，"
                    "停止刷新。"
                )

                break

            print(
                "\n下次刷新将在 "
                f"{REFRESH_SECONDS} 秒后进行。"
            )

            time.sleep(
                REFRESH_SECONDS
            )

    except KeyboardInterrupt:
        print(
            "\n自动刷新已手动停止。"
        )

else:
    latest_output = (
        load_and_plot_once()
    )

In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import time
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None

# ============================================================
# 基本路径与配置
# ============================================================

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

N = 20
TOTAL_INNER_STEPS = 200

C1 = np.array([-2.60866606, 2.77661136], dtype=float)
C2 = np.array([-1.33047160, 0.99118795], dtype=float)

# ============================================================
# 自动发现所有 L‑BFGS 配置（原始 + Recycle）
# ============================================================
# 原始实验文件夹命名
_LBFGS_FOLDER_RE = re.compile(
    r"^MFEP_image_LBFGS_B_N20_Nsub(\d+)_alpha([\d.eE+-]+)_maxstep([\d.eE+-]+)_hist7_NEBreparam$"
)
# Recycle 实验文件夹命名
_LBFGS_RECYCLE_FOLDER_RE = re.compile(
    r"^MFEP_image_LBFGS_B_Recycle_N20_Nsub(\d+)_alpha([\d.eE+-]+)_maxstep([\d.eE+-]+)_hist7_NEBreparam$"
)

def discover_lbfgs_configs():
    """扫描 base_path 下所有 L‑BFGS 实验文件夹（原始和 Recycle），
       返回 list of dict，每个包含类型、参数和文件路径。
    """
    configs = []
    if not os.path.isdir(base_path):
        return configs
    for name in os.listdir(base_path):
        folder = os.path.join(base_path, name)
        if not os.path.isdir(folder):
            continue
        # 检查是否匹配原始模式
        m = _LBFGS_FOLDER_RE.match(name)
        if m:
            n_sub = int(m.group(1))
            alpha = float(m.group(2))
            maxstep = float(m.group(3))
            experiment_type = "L-BFGS"
        else:
            # 检查是否匹配 Recycle 模式
            m = _LBFGS_RECYCLE_FOLDER_RE.match(name)
            if m is None:
                continue
            n_sub = int(m.group(1))
            alpha = float(m.group(2))
            maxstep = float(m.group(3))
            experiment_type = "L-BFGS-Recycle"

        npz_file = os.path.join(folder, f"alanine_{name}.npz")
        final_model = os.path.join(folder, f"model_MFEP_alanine_{name}_final.pth")
        configs.append({
            "type": experiment_type,
            "N_sub": n_sub,
            "alpha": alpha,
            "maxstep": maxstep,
            "casenum": name,
            "folder": folder,
            "npz": npz_file,
            "final_model": final_model,
        })
    # 按类型和参数排序，保证顺序稳定
    return sorted(configs, key=lambda d: (d["type"], d["N_sub"], d["alpha"], d["maxstep"]))

# ============================================================
# Optimal‑IVM 配置（保持不变）
# ============================================================

OPTIMAL_N_SUB = 4
OPTIMAL_ALPHA = 0.1
OPTIMAL_TOTAL_STEPS = 200

OPTIMAL_CASENUM = (
    f"MFEP_image_AdamAdaptive"
    f"_N{N}"
    f"_Nsub{OPTIMAL_N_SUB}"
    f"_alpha{OPTIMAL_ALPHA}"
    f"_lr0.04to0.001"
    f"_steps{OPTIMAL_TOTAL_STEPS}"
    f"_sharedAdam"
    f"_NEBreparam"
)

OPTIMAL_FOLDER = os.path.join(base_path, OPTIMAL_CASENUM)
OPTIMAL_NPZ = os.path.join(OPTIMAL_FOLDER, f"alanine_{OPTIMAL_CASENUM}.npz")

# ============================================================
# 新版 MFEP‑FENEB 配置（保持不变）
# ============================================================

FENEB_REPARAM_KS = 1.0
FENEB_REPARAM_DTAU = 1.0e-3
FENEB_REPARAM_TOL = 1.0e-3
FENEB_REPARAM_MAXITER = 5000
FENEB_MAX_UPDATES = 200

GRID_FENEB_PHYSICAL_STEP = [5.0e-2, 2.0e-2, 1.0e-2, 5.0e-3, 2.0e-3, 1.0e-3]

# ============================================================
# 自动刷新设置
# ============================================================

AUTO_REFRESH = True
REFRESH_SECONDS = 60

WAIT_FOR_OPTIMAL_FILE = True
WAIT_FOR_FENEB_FILE = True

AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE = True
AUTO_STOP_REQUIRE_FENEB_COMPLETE = True
AUTO_STOP_REQUIRE_ALL_LBFGS_COMPLETE = True   # 所有发现的 L‑BFGS 完成后自动停止

# ============================================================
# 绘图设置
# ============================================================

USE_LOG_Y_RESIDUALS = True
USE_LOG_Y_DELTA_F = False
USE_LOG_Y_ARC = True

SAVE_OUTPUT = True

# L‑BFGS 绘图样式
LBFGS_LINEWIDTH = 2.0
LBFGS_ALPHA = 0.8
LBFGS_MARKER = "o"

# Optimal‑IVM
OPTIMAL_COLOR = "#e41a1c"
OPTIMAL_LINESTYLE = "-"
OPTIMAL_MARKER = "*"
OPTIMAL_LINEWIDTH = 3.2

# MFEP‑FENEB
FENEB_MARKER = "D"
FENEB_LINEWIDTH = 2.8

# ============================================================
# 安全读取正在写入的 npz
# ============================================================

def safe_load_npz(npz_file, retries=10, wait_seconds=2.0):
    if not os.path.exists(npz_file):
        return None
    last_error = None
    for attempt in range(retries):
        try:
            with np.load(npz_file, allow_pickle=True) as loaded:
                return {key: np.array(loaded[key], copy=True) for key in loaded.files}
        except (zipfile.BadZipFile, EOFError, OSError, ValueError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(wait_seconds)
    print(f"[WARNING] 无法读取正在写入的 npz：\n  {npz_file}\n  error: {last_error}")
    return None

# ============================================================
# L‑BFGS 文件夹路径（保留用于兼容，但实际不再调用）
# ============================================================
LBFGS_HISTORY_SIZE = 7

def ivm_config_paths(N_sub, alpha_IVP, maxstep):
    casenum = (
        f"MFEP_image_LBFGS_B_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_maxstep{maxstep}"
        f"_hist{LBFGS_HISTORY_SIZE}"
        f"_NEBreparam"
    )
    folder_path = os.path.join(base_path, casenum)
    npz_file = os.path.join(folder_path, f"alanine_{casenum}.npz")
    final_model = os.path.join(folder_path, f"model_MFEP_alanine_{casenum}_final.pth")
    return casenum, folder_path, npz_file, final_model

# ============================================================
# 数值工具（与之前完全一致，略去重复的 docstring）
# ============================================================

def central_difference_tangent(full_path):
    return full_path[2:] - full_path[:-2]

def compute_G_projectors(tangent, M_all):
    tangent = np.asarray(tangent, dtype=float)
    M_all = np.asarray(M_all, dtype=float)
    n_internal = tangent.shape[0]
    G_all = np.linalg.inv(M_all)
    P_parallel_all = np.zeros((n_internal, 2, 2))
    P_perp_all = np.zeros((n_internal, 2, 2))
    identity = np.eye(2)
    for i in range(n_internal):
        v_i = tangent[i]
        G_i = G_all[i]
        norm_G_sq = float(v_i @ G_i @ v_i)
        if not np.isfinite(norm_G_sq) or norm_G_sq <= 1e-30:
            raise FloatingPointError(f"Near-zero G tangent norm at image {i}")
        tau_G = v_i / np.sqrt(norm_G_sq)
        G_tau_G = G_i @ tau_G
        denom = float(tau_G @ G_tau_G)
        if not np.isfinite(denom) or denom <= 1e-30:
            raise FloatingPointError(f"Invalid projector denominator at image {i}")
        P_parallel = np.outer(tau_G, G_tau_G) / denom
        P_parallel_all[i] = P_parallel
        P_perp_all[i] = identity - P_parallel
    return G_all, P_parallel_all, P_perp_all

def local_G_arclength_weights(full_path, G_all):
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    length_minus_sq = np.einsum("ni,nij,nj->n", seg_minus, G_all, seg_minus)
    length_plus_sq = np.einsum("ni,nij,nj->n", seg_plus, G_all, seg_plus)
    return 0.5 * (np.sqrt(np.maximum(length_minus_sq, 0.0)) + np.sqrt(np.maximum(length_plus_sq, 0.0)))

def local_euclidean_arclength_weights(full_path):
    path_internal = full_path[1:-1]
    seg_minus = path_internal - full_path[:-2]
    seg_plus = full_path[2:] - path_internal
    return 0.5 * (np.linalg.norm(seg_minus, axis=1) + np.linalg.norm(seg_plus, axis=1))

def compute_arclength_residual(full_path):
    n_full = full_path.shape[0]
    ds = 1.0 / (n_full - 1)
    phi_prime = (full_path[2:] - full_path[:-2]) / (2.0 * ds)
    phi_second = (full_path[2:] - 2.0 * full_path[1:-1] + full_path[:-2]) / (ds ** 2)
    inner = np.einsum("ni,ni->n", phi_prime, phi_second)
    speed = np.linalg.norm(phi_prime, axis=1)
    integrand = inner ** 2 * speed
    I_arc = float(np.sum(integrand) * ds)
    return {"I_arc": I_arc, "phi_prime": phi_prime, "phi_second": phi_second,
            "inner_phi_prime_second": inner, "speed": speed, "integrand": integrand, "ds": ds}

def recover_free_energy_profile(full_path, F_internal):
    n_internal = full_path.shape[0] - 2
    segments = full_path[1:] - full_path[:-1]
    seg_grad = np.zeros((n_internal + 1, 2))
    seg_grad[0] = F_internal[0]
    if n_internal > 1:
        seg_grad[1:n_internal] = 0.5 * (F_internal[:-1] + F_internal[1:])
    seg_grad[n_internal] = F_internal[-1]
    increments = np.einsum("ni,ni->n", seg_grad, segments)
    profile = np.zeros(n_internal + 2)
    profile[1:] = np.cumsum(increments)
    profile -= profile[0]
    barrier = float(np.max(profile) - profile[0])
    max_idx = int(np.argmax(profile))
    return {"profile": profile, "barrier": barrier, "max_index": max_idx,
            "final_delta_F": float(profile[-1] - profile[0]), "increments": increments}

def compute_all_path_metrics(full_path, M_all, F_all):
    n_internal = full_path.shape[0] - 2
    tangent = central_difference_tangent(full_path)
    G_all, _, P_perp_G_all = compute_G_projectors(tangent, M_all)
    Mg_all = np.einsum("nij,nj->ni", M_all, F_all)
    perpG_Mg = np.einsum("nij,nj->ni", P_perp_G_all, Mg_all)
    residual_G_sq = np.maximum(np.einsum("ni,nij,nj->n", perpG_Mg, G_all, perpG_Mg), 0.0)
    weights_G = local_G_arclength_weights(full_path, G_all)
    I_perp_G = float(np.sum(residual_G_sq * weights_G))
    residual_E_sq = np.sum(perpG_Mg ** 2, axis=1)
    I_perp_E = float(np.sum(residual_E_sq * local_euclidean_arclength_weights(full_path)))
    free = recover_free_energy_profile(full_path, F_all)
    arc = compute_arclength_residual(full_path)
    fmax_E = float(np.max(np.sqrt(np.maximum(residual_E_sq, 0.0))))
    fmax_G = float(np.max(np.sqrt(np.maximum(residual_G_sq, 0.0))))
    return {"I_perp_G": I_perp_G, "I_perp_E": I_perp_E,
            "Delta_F_barrier": free["barrier"], "Delta_F_final": free["final_delta_F"],
            "free_energy_profile": free["profile"], "free_energy_max_index": free["max_index"],
            "I_arc": arc["I_arc"], "arc_integrand": arc["integrand"], "arc_speed": arc["speed"],
            "arc_inner": arc["inner_phi_prime_second"],
            "G_all": G_all, "Mg_all": Mg_all, "perpG_Mg": perpG_Mg,
            "weights_G": weights_G, "weights_E": local_euclidean_arclength_weights(full_path),
            "fmax_E": fmax_E, "fmax_G": fmax_G}

# ============================================================
# 去除重复 count
# ============================================================

def keep_last_for_duplicate_counts(count_seq, *arrays):
    count_seq = np.asarray(count_seq)
    if count_seq.size == 0:
        return (count_seq,) + arrays
    last_idx = {}
    for i, c in enumerate(count_seq):
        last_idx[float(c)] = i
    sel = np.array(sorted(last_idx.values(), key=lambda idx: count_seq[idx]))
    res = [count_seq[sel]]
    for a in arrays:
        res.append(np.asarray(a)[sel])
    return tuple(res)

# ============================================================
# 从 IVM npz 读取数据（兼容 L‑BFGS 的键）
# ============================================================

def load_ivm_result(npz_file, method_name, N_sub, alpha_IVP, eta_description,
                    require_complete=False, final_model=None):
    if not os.path.exists(npz_file):
        return None, f"npz does not exist: {npz_file}"
    if require_complete and final_model is not None and not os.path.exists(final_model):
        return None, f"final model missing: {final_model}"
    data = safe_load_npz(npz_file)
    if data is None:
        return None, f"could not read: {npz_file}"
    if "index_seq" not in data:
        return None, "index_seq missing"
    idx_saved = np.asarray(data["index_seq"], dtype=int)
    if idx_saved.size == 0:
        return None, "empty index_seq"
    if require_complete:
        expected = TOTAL_INNER_STEPS // N_sub
        if int(np.max(idx_saved)) < expected:
            return None, f"incomplete: max index={np.max(idx_saved)}, expected at least {expected}"

    count_saved = np.asarray(data.get("count_seq", idx_saved * N_sub), dtype=int)
    n = min(len(idx_saved), len(count_saved))
    idx_saved = idx_saved[:n]
    count_saved = count_saved[:n]

    valid_idx, counts, I_G, I_E, I_arc, DeltaF, DeltaF_final = [], [], [], [], [], [], []
    free_profs, arc_integ = [], []
    skipped = []
    for pos in range(n):
        outer = int(idx_saved[pos])
        cur_count = int(count_saved[pos])
        phi_key = f"phi_ref_iter_{outer}"
        M_key = f"M_ref_iter_{outer}"
        F_key = f"F_ref_iter_{outer}"
        g_key = f"g_ref_iter_{outer}"
        sel_F = None
        if F_key in data:
            sel_F = F_key
        elif g_key in data:
            sel_F = g_key
        else:
            skipped.append(outer)
            continue
        if phi_key not in data or M_key not in data or sel_F is None:
            skipped.append(outer)
            continue
        try:
            phi_int = np.asarray(data[phi_key], dtype=float)
            M_all = np.asarray(data[M_key], dtype=float)
            F_all = np.asarray(data[sel_F], dtype=float)
            full = np.vstack([C1[None, :], phi_int, C2[None, :]])
            m = compute_all_path_metrics(full, M_all, F_all)
        except Exception as e:
            print(f"[WARNING] 跳过 {method_name} outer={outer}: {e}")
            skipped.append(outer)
            continue
        valid_idx.append(outer)
        counts.append(cur_count)
        I_G.append(m["I_perp_G"]); I_E.append(m["I_perp_E"]); I_arc.append(m["I_arc"])
        DeltaF.append(m["Delta_F_barrier"]); DeltaF_final.append(m["Delta_F_final"])
        free_profs.append(m["free_energy_profile"]); arc_integ.append(m["arc_integrand"])
    if not counts:
        return None, "no complete outer state found"
    counts, I_G, I_E, I_arc, DeltaF, DeltaF_final, valid_idx, free_profs, arc_integ = \
        keep_last_for_duplicate_counts(np.asarray(counts, dtype=int),
                                       np.asarray(I_G), np.asarray(I_E), np.asarray(I_arc),
                                       np.asarray(DeltaF), np.asarray(DeltaF_final),
                                       np.asarray(valid_idx, dtype=int),
                                       np.asarray(free_profs), np.asarray(arc_integ))
    res = {
        "method": method_name, "N_sub": N_sub, "alpha_IVP": alpha_IVP,
        "eta_description": eta_description, "npz_file": npz_file,
        "index_seq": valid_idx, "count_seq": counts,
        "I_perp_G_seq": I_G, "I_perp_E_seq": I_E, "I_arc_seq": I_arc,
        "Delta_F_seq": DeltaF, "Delta_F_final_seq": DeltaF_final,
        "free_energy_profiles": free_profs, "arc_integrands": arc_integ,
        "skipped_outer_indices": np.asarray(skipped, dtype=int)
    }
    if "lr_count_seq" in data:
        res["lr_count_seq"] = np.asarray(data["lr_count_seq"], dtype=int)
    if "lr_seq" in data:
        res["lr_seq"] = np.asarray(data["lr_seq"], dtype=float)
    return res, None

# ============================================================
# 加载所有发现的 L‑BFGS 结果（原始 + Recycle）
# ============================================================

def load_all_lbfgs_results():
    results = []
    skipped = []
    configs = discover_lbfgs_configs()
    for cfg in configs:
        method_name = cfg["type"]      # "L-BFGS" 或 "L-BFGS-Recycle"
        res, err = load_ivm_result(cfg["npz"], method_name, cfg["N_sub"], cfg["alpha"],
                                   str(cfg["maxstep"]),
                                   require_complete=False, final_model=None)
        if res is None:
            skipped.append(f"{cfg['casenum']}: {err}")
            continue
        res["maxstep"] = cfg["maxstep"]
        res["casenum"] = cfg["casenum"]
        res["final_model"] = cfg["final_model"]
        # 保留 type 以用于样式区分
        res["experiment_type"] = cfg["type"]
        results.append(res)
    return results, skipped

# ============================================================
# Optimal‑IVM 加载
# ============================================================

def load_optimal_ivm():
    res, err = load_ivm_result(OPTIMAL_NPZ, "Optimal-IVM", OPTIMAL_N_SUB, OPTIMAL_ALPHA,
                               "adaptive Adam", require_complete=False, final_model=None)
    if res is not None:
        res["casenum"] = OPTIMAL_CASENUM
    return res, err

# ============================================================
# MFEP‑FENEB 路径与加载（保持不变）
# ============================================================

def float_tag(v):
    return f"{float(v):g}"

def feneb_result_paths(phys_step):
    st = float_tag(phys_step)
    casenum = (f"MFEP_FENEB_COMPARE_N{N}_lambda{st}_Nupdates{FENEB_MAX_UPDATES}"
               f"_Kspring{FENEB_REPARAM_KS:g}_dtau{FENEB_REPARAM_DTAU:g}"
               f"_reparamTol{FENEB_REPARAM_TOL:g}_sameIVMsettings")
    folder = os.path.join(base_path, casenum)
    npz_f = os.path.join(folder, f"alanine_{casenum}.npz")
    done_f = os.path.join(folder, "DONE")
    return casenum, folder, npz_f, done_f

def load_single_feneb_result(npz_file, expected_phys_step=None):
    data = safe_load_npz(npz_file)
    if data is None:
        return None, f"Could not read {npz_file}"
    for k in ["count_seq","M_ref_seq","F_ref_seq"]:
        if k not in data:
            return None, f"Missing FENEB key: {k}"
    path_key = "mean_path_unwrapped_seq" if "mean_path_unwrapped_seq" in data else \
               "mean_path_seq" if "mean_path_seq" in data else None
    if path_key is None:
        return None, "No mean_path key"
    cnt_raw = np.asarray(data["count_seq"], dtype=float)
    pth_raw = np.asarray(data[path_key])
    M_raw = np.asarray(data["M_ref_seq"])
    F_raw = np.asarray(data["F_ref_seq"])
    n_states = min(len(cnt_raw), len(pth_raw), len(M_raw), len(F_raw))
    if n_states == 0:
        return None, "No FENEB states"
    cnt_raw = cnt_raw[:n_states]; pth_raw = pth_raw[:n_states]; M_raw = M_raw[:n_states]; F_raw = F_raw[:n_states]
    valid_cnt, I_G, I_E, I_arc, DeltaF, DeltaF_fin = [], [], [], [], [], []
    free_profs, arc_ints = [], []
    fmaxE, fmaxG = [], []
    skip_idx = []
    for i in range(n_states):
        try:
            cur_path = np.asarray(pth_raw[i], dtype=float)
            cur_M = np.asarray(M_raw[i], dtype=float)
            cur_F = np.asarray(F_raw[i], dtype=float)
            if cur_path.shape == (N, 2):
                full = np.vstack([C1[None,:], cur_path, C2[None,:]])
            elif cur_path.shape == (N+2, 2):
                full = cur_path.copy()
            else:
                raise ValueError(f"Bad path shape {cur_path.shape}")
            m = compute_all_path_metrics(full, cur_M, cur_F)
        except Exception as e:
            print(f"[WARNING] 跳过 FENEB state={i}, count={cnt_raw[i]}: {e}")
            skip_idx.append(i)
            continue
        valid_cnt.append(cnt_raw[i]); I_G.append(m["I_perp_G"]); I_E.append(m["I_perp_E"])
        I_arc.append(m["I_arc"]); DeltaF.append(m["Delta_F_barrier"]); DeltaF_fin.append(m["Delta_F_final"])
        free_profs.append(m["free_energy_profile"]); arc_ints.append(m["arc_integrand"])
        fmaxE.append(m["fmax_E"]); fmaxG.append(m["fmax_G"])
    if not valid_cnt:
        return None, "No valid FENEB state"
    (valid_cnt, I_G, I_E, I_arc, DeltaF, DeltaF_fin,
     free_profs, arc_ints, fmaxE, fmaxG,
     ) = keep_last_for_duplicate_counts(
         np.asarray(valid_cnt), np.asarray(I_G), np.asarray(I_E), np.asarray(I_arc),
         np.asarray(DeltaF), np.asarray(DeltaF_fin), np.asarray(free_profs), np.asarray(arc_ints),
         np.asarray(fmaxE), np.asarray(fmaxG))
    phys_step = float(np.asarray(data.get("physical_step", expected_phys_step)).reshape(-1)[0]) if expected_phys_step is not None else np.nan
    folder = os.path.dirname(npz_file)
    done = os.path.exists(os.path.join(folder, "DONE"))
    return {"method":"MFEP-FENEB","npz_file":npz_file,"folder_path":folder,"done":done,
            "physical_step":phys_step,"count_seq":valid_cnt,
            "I_perp_G_seq":I_G,"I_perp_E_seq":I_E,"I_arc_seq":I_arc,
            "Delta_F_seq":DeltaF,"Delta_F_final_seq":DeltaF_fin,
            "free_energy_profiles":free_profs,"arc_integrands":arc_ints,
            "fmax_E_seq":fmaxE,"fmax_G_seq":fmaxG,
            "valid_state_indices":None,"skipped_state_indices":np.asarray(skip_idx),
            "source_path_key":path_key}, None

def load_all_feneb_results():
    res = []; errs = []
    for ps in GRID_FENEB_PHYSICAL_STEP:
        _, _, npz, _ = feneb_result_paths(ps)
        r, e = load_single_feneb_result(npz, expected_phys_step=ps)
        if r is not None:
            r["casenum"] = feneb_result_paths(ps)[0]
            r["expected_done_file"] = feneb_result_paths(ps)[3]
            res.append(r)
        else:
            errs.append(f"lambda={ps:g}: {e}")
    return res, errs

def evaluate_feneb_completion(feneb_results):
    status = []
    for ps in GRID_FENEB_PHYSICAL_STEP:
        matching = [r for r in feneb_results if np.isclose(r["physical_step"], ps, rtol=1e-10, atol=1e-15)]
        found = len(matching)>0
        done = bool(matching[0]["done"]) if found else False
        latest = float(matching[0]["count_seq"][-1]) if found else np.nan
        status.append({"physical_step":ps, "found":found, "done":done, "latest_count":latest})
    all_done = all(s["found"] and s["done"] for s in status)
    return all_done, status

# ============================================================
# MFEP‑FENEB 绘图样式（保持不变）
# ============================================================

FENEB_COLORS = {1e-2:"#ff7f00", 5e-3:"#a65628", 2e-3:"#f781bf", 1e-3:"#00a6d6"}
FENEB_LSTYLES = {1e-2:"-", 5e-3:"--", 2e-3:"-.", 1e-3:":"}

def feneb_style(phys_step, idx):
    c = None
    for k,v in FENEB_COLORS.items():
        if np.isclose(float(k), float(phys_step), rtol=1e-10, atol=1e-15):
            c = v; break
    if c is None:
        c = plt.get_cmap("tab10")(idx % 10)
    ls = None
    for k,v in FENEB_LSTYLES.items():
        if np.isclose(float(k), float(phys_step), rtol=1e-10, atol=1e-15):
            ls = v; break
    if ls is None:
        styles = ["-","--","-.",":",(0,(3,5,1,5)),(0,(1,1)),(0,(5,10))]
        ls = styles[idx % len(styles)]
    return c, ls

def feneb_label(r):
    label = f"MFEP-FENEB $\\lambda={r['physical_step']:g}$"
    label += " [DONE]" if r["done"] else " [running]"
    return label

# ============================================================
# L‑BFGS 全部完成判断
# ============================================================
def all_lbfgs_complete(lbfgs_results):
    configs = discover_lbfgs_configs()
    if len(lbfgs_results) < len(configs):
        return False
    for r in lbfgs_results:
        if os.path.exists(r.get("final_model","")):
            continue
        if r["count_seq"][-1] >= TOTAL_INNER_STEPS:
            continue
        return False
    return True

# ============================================================
# 分类型独立绘图辅助函数
# ============================================================

def record_label(result):
    """
    为不同方法生成图例标签。
    """
    method = result.get("method", "")

    if method == "L-BFGS":
        return (
            f"L-BFGS | "
            f"Nsub={result['N_sub']}, "
            f"alpha={result['alpha_IVP']:g}, "
            f"maxstep={result['maxstep']:g}"
        )

    if method == "L-BFGS-Recycle":
        return (
            f"L-BFGS-Recycle | "
            f"Nsub={result['N_sub']}, "
            f"alpha={result['alpha_IVP']:g}, "
            f"maxstep={result['maxstep']:g}"
        )

    if method == "Optimal-IVM":
        return (
            f"Optimal-IVM | "
            f"Nsub={result['N_sub']}, "
            f"alpha={result['alpha_IVP']:g}, "
            f"adaptive Adam"
        )

    if method == "MFEP-FENEB":
        state = "DONE" if result.get("done", False) else "running"
        return f"MFEP-FENEB | lambda={result['physical_step']:g} [{state}]"

    return method


def build_group_styles(results, group_name):
    """
    对同一方法类型内部的不同参数组合分配颜色、线型和 marker。

    这里不再让原始 L-BFGS 和 Recycle 共用颜色逻辑；
    因为它们已经分图展示，所以每一个参数配置都尽量有独立颜色，
    方便在同类型内部仔细比较。
    """
    if len(results) == 0:
        return {}

    styles = {}
    markers = ["o", "s", "^", "v", "P", "X", "<", ">", "h", "*", "D"]
    linestyles = ["-", "--", "-.", ":", (0, (5, 2)), (0, (2, 1))]

    # --------------------------------------------------------
    # FENEB：继续按 lambda 使用已有的颜色和线型
    # --------------------------------------------------------
    if group_name == "MFEP-FENEB":
        sorted_results = sorted(
            results,
            key=lambda r: float(r.get("physical_step", np.inf))
        )

        for i, r in enumerate(sorted_results):
            color, linestyle = feneb_style(r["physical_step"], i)

            styles[id(r)] = {
                "color": color,
                "linestyle": linestyle,
                "marker": FENEB_MARKER,
                "markersize": 6.5,
                "linewidth": FENEB_LINEWIDTH,
                "markerfacecolor": "white",
                "markeredgecolor": color,
                "markeredgewidth": 1.2,
                "alpha": 1.0,
            }

        return styles

    # --------------------------------------------------------
    # Optimal-IVM：只有一条，使用固定样式
    # --------------------------------------------------------
    if group_name == "Optimal-IVM":
        for r in results:
            styles[id(r)] = {
                "color": OPTIMAL_COLOR,
                "linestyle": OPTIMAL_LINESTYLE,
                "marker": OPTIMAL_MARKER,
                "markersize": 10,
                "linewidth": OPTIMAL_LINEWIDTH,
                "markerfacecolor": OPTIMAL_COLOR,
                "markeredgecolor": "white",
                "markeredgewidth": 0.8,
                "alpha": 1.0,
            }

        return styles

    # --------------------------------------------------------
    # L-BFGS / L-BFGS-Recycle：
    # 每个参数组合单独分配颜色和 marker
    # --------------------------------------------------------
    sorted_results = sorted(
        results,
        key=lambda r: (
            r.get("N_sub", 0),
            r.get("alpha_IVP", 0.0),
            r.get("maxstep", 0.0),
            r.get("casenum", ""),
        )
    )

    cmap = plt.get_cmap("tab20")

    for i, r in enumerate(sorted_results):
        color = cmap(i % 20)
        marker = markers[i % len(markers)]
        linestyle = linestyles[(i // len(markers)) % len(linestyles)]

        styles[id(r)] = {
            "color": color,
            "linestyle": linestyle,
            "marker": marker,
            "markersize": 4.5,
            "linewidth": LBFGS_LINEWIDTH,
            "markerfacecolor": color,
            "markeredgecolor": "white",
            "markeredgewidth": 0.5,
            "alpha": LBFGS_ALPHA,
        }

    return styles


def plot_one_method_group(results, group_name, output_stem):
    """
    为一种方法类型单独画一张大图。

    图的布局为：
        4 行 x 1 列

    从上到下依次是：
        1. Riemannian normal residual
        2. Euclidean residual integral
        3. Arclength / equi-speed residual
        4. Free-energy barrier
    """
    if len(results) == 0:
        print(f"[INFO] {group_name}: 没有可绘制的数据，跳过绘图。")
        return None

    metric_specs = [
        {
            "key": "I_perp_G_plot",
            "title": "Riemannian normal residual",
            "ylabel": r"$\widehat I_{\perp}^{G}$",
            "log_scale": USE_LOG_Y_RESIDUALS,
        },
        {
            "key": "I_perp_E_plot",
            "title": "Euclidean residual integral",
            "ylabel": r"$\widehat I_{\perp}^{E}$",
            "log_scale": USE_LOG_Y_RESIDUALS,
        },
        {
            "key": "I_arc_plot",
            "title": "Arclength / equi-speed residual",
            "ylabel": r"$I_{\rm arc}$",
            "log_scale": USE_LOG_Y_ARC,
        },
        {
            "key": "Delta_F_plot",
            "title": "Path-integrated free-energy barrier",
            "ylabel": r"$\Delta F^{\dagger}$ (kcal/mol)",
            "log_scale": USE_LOG_Y_DELTA_F,
        },
    ]

    # 一张大图，四个指标纵向排列
    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(14, 23),
        sharex=True,
    )

    fig.suptitle(
        f"{group_name}: internal parameter comparison",
        fontsize=20,
        fontweight="bold",
        y=0.995,
    )

    styles = build_group_styles(results, group_name)

    legend_handles = []

    for result in results:
        style = styles[id(result)]
        count_seq = np.asarray(result["count_seq"], dtype=float)

        if len(count_seq) == 0:
            continue

        markevery = max(1, len(count_seq) // 12)

        label = record_label(result)

        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=style["color"],
                linestyle=style["linestyle"],
                marker=style["marker"],
                linewidth=style["linewidth"],
                markersize=max(style["markersize"], 6),
                markerfacecolor=style["markerfacecolor"],
                markeredgecolor=style["markeredgecolor"],
                markeredgewidth=style["markeredgewidth"],
                label=label,
            )
        )

        for ax, spec in zip(axes, metric_specs):
            values = np.asarray(result[spec["key"]], dtype=float)

            ax.plot(
                count_seq,
                values,
                color=style["color"],
                linestyle=style["linestyle"],
                marker=style["marker"],
                markevery=markevery,
                markersize=style["markersize"],
                linewidth=style["linewidth"],
                markerfacecolor=style["markerfacecolor"],
                markeredgecolor=style["markeredgecolor"],
                markeredgewidth=style["markeredgewidth"],
                alpha=style["alpha"],
                zorder=3,
            )

            # 用更明显的点标记最终状态
            if group_name == "Optimal-IVM":
                end_marker = "*"
                end_size = 180
            elif group_name == "MFEP-FENEB":
                end_marker = "D" if result.get("done", False) else "o"
                end_size = 95
            else:
                end_marker = style["marker"]
                end_size = 75

            ax.scatter(
                count_seq[-1],
                values[-1],
                s=end_size,
                marker=end_marker,
                facecolor=style["color"],
                edgecolor="black",
                linewidth=0.9,
                zorder=20,
            )

    # --------------------------------------------------------
    # 统一设置坐标轴
    # --------------------------------------------------------
    for ax, spec in zip(axes, metric_specs):
        ax.set_title(spec["title"], fontsize=16, pad=10)
        ax.set_ylabel(spec["ylabel"], fontsize=14)

        ax.grid(
            True,
            which="both",
            linestyle="--",
            linewidth=0.8,
            alpha=0.35,
        )

        ax.set_xlim(left=0)

        if spec["log_scale"]:
            ax.set_yscale("log")

        ax.tick_params(axis="both", labelsize=12)

    axes[-1].set_xlabel(
        "restrained-MD / gradient count",
        fontsize=14,
    )

    # 每个子图也保留 x 轴标签，方便单独截取看图
    for ax in axes[:-1]:
        ax.set_xlabel(
            "restrained-MD / gradient count",
            fontsize=12,
        )

    # --------------------------------------------------------
    # 统一大图图例：放在右侧，不遮挡曲线
    # --------------------------------------------------------
    ncol_legend = 1

    fig.legend(
        handles=legend_handles,
        title=f"{group_name} configurations",
        loc="center left",
        bbox_to_anchor=(0.78, 0.50),
        frameon=True,
        fontsize=10.5,
        title_fontsize=12,
        ncol=ncol_legend,
    )

    fig.subplots_adjust(
        left=0.11,
        right=0.74,
        top=0.96,
        bottom=0.07,
        hspace=0.34,
    )

    pdf_out = os.path.join(base_path, f"{output_stem}.pdf")
    png_out = os.path.join(base_path, f"{output_stem}.png")

    if SAVE_OUTPUT:
        fig.savefig(pdf_out, bbox_inches="tight")
        fig.savefig(png_out, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"[SAVED] {group_name}")
    print(f"  PDF: {pdf_out}")
    print(f"  PNG: {png_out}")

    return {
        "group_name": group_name,
        "pdf": pdf_out,
        "png": png_out,
        "n_results": len(results),
    }


# ============================================================
# 绘图主函数
# ============================================================
def load_and_plot_once():
    lbfgs_results, lbfgs_skipped = load_all_lbfgs_results()
    opt_res, opt_err = load_optimal_ivm()
    feneb_res, feneb_errs = load_all_feneb_results()

    feneb_all_done, feneb_rows = evaluate_feneb_completion(feneb_res)
    lbfgs_done = all_lbfgs_complete(lbfgs_results)

    # --------------------------------------------------------
    # 控制台输出
    # --------------------------------------------------------
    print("=" * 120)
    print("MFEP: separated internal comparison by method type")
    print("=" * 120)

    total_configs = len(discover_lbfgs_configs())

    print(f"已发现的 L-BFGS 配置总数（原始 + Recycle）：{total_configs}")
    print(f"已成功读取的 L-BFGS 配置数：{len(lbfgs_results)}")

    if lbfgs_skipped:
        print("\n跳过的 L-BFGS 配置：")
        for s in lbfgs_skipped[:10]:
            print(f"  {s}")

    if opt_res is not None:
        opt_cur = int(np.max(opt_res["count_seq"]))
        opt_done = opt_cur >= OPTIMAL_TOTAL_STEPS

        print(
            f"\nOptimal-IVM: "
            f"count={opt_cur}/{OPTIMAL_TOTAL_STEPS}, "
            f"states={len(opt_res['count_seq'])}, "
            f"done={opt_done}"
        )
    else:
        opt_done = False
        opt_cur = -1
        print("\nOptimal-IVM: 无数据")

    print("-" * 120)
    print(f"FENEB 成功读取数量：{len(feneb_res)}")

    for fr in feneb_res:
        print(
            f"  lambda={fr['physical_step']:g}, "
            f"states={len(fr['count_seq']):4d}, "
            f"range=({fr['count_seq'][0]:g}, {fr['count_seq'][-1]:g}), "
            f"done={fr['done']}"
        )

    if feneb_errs:
        print("\n未读取到的 FENEB 配置：")
        for s in feneb_errs:
            print(f"  {s}")

    print("-" * 120)
    print(f"FENEB 全部完成：{feneb_all_done}")
    print(f"L-BFGS 全部完成：{lbfgs_done}")
    print("=" * 120)

    # --------------------------------------------------------
    # 准备绘图数值：避免 log scale 下出现 0
    # --------------------------------------------------------
    tiny = np.finfo(float).tiny

    all_data = list(lbfgs_results)

    if opt_res is not None:
        all_data.append(opt_res)

    all_data.extend(feneb_res)

    for d in all_data:
        d["I_perp_G_plot"] = np.maximum(
            np.asarray(d["I_perp_G_seq"], dtype=float),
            tiny,
        )

        d["I_perp_E_plot"] = np.maximum(
            np.asarray(d["I_perp_E_seq"], dtype=float),
            tiny,
        )

        d["I_arc_plot"] = np.maximum(
            np.asarray(d["I_arc_seq"], dtype=float),
            tiny,
        )

        d["Delta_F_plot"] = np.maximum(
            np.asarray(d["Delta_F_seq"], dtype=float),
            tiny,
        )

    # --------------------------------------------------------
    # 按方法类型拆分数据
    # --------------------------------------------------------
    lbfgs_original_results = [
        r for r in lbfgs_results
        if r.get("experiment_type", r.get("method", "")) == "L-BFGS"
    ]

    lbfgs_recycle_results = [
        r for r in lbfgs_results
        if r.get("experiment_type", r.get("method", "")) == "L-BFGS-Recycle"
    ]

    # --------------------------------------------------------
    # 分类型绘图：
    #
    # 1. L-BFGS 原始方法内部比较
    # 2. L-BFGS-Recycle 内部比较
    # 3. Optimal-IVM 单独一张大图
    # 4. FENEB 不同 lambda 内部比较
    # --------------------------------------------------------
    plt.close("all")

    saved_plot_info = []

    out = plot_one_method_group(
        results=lbfgs_original_results,
        group_name="L-BFGS",
        output_stem="MFEP_LBFGS_internal_comparison_four_metrics",
    )
    if out is not None:
        saved_plot_info.append(out)

    out = plot_one_method_group(
        results=lbfgs_recycle_results,
        group_name="L-BFGS-Recycle",
        output_stem="MFEP_LBFGS_Recycle_internal_comparison_four_metrics",
    )
    if out is not None:
        saved_plot_info.append(out)

    if opt_res is not None:
        out = plot_one_method_group(
            results=[opt_res],
            group_name="Optimal-IVM",
            output_stem="MFEP_OptimalIVM_four_metrics",
        )
        if out is not None:
            saved_plot_info.append(out)

    out = plot_one_method_group(
        results=feneb_res,
        group_name="MFEP-FENEB",
        output_stem="MFEP_FENEB_internal_comparison_four_metrics",
    )
    if out is not None:
        saved_plot_info.append(out)

    # --------------------------------------------------------
    # 数值汇总
    # --------------------------------------------------------
    print("\n" + "=" * 165)
    print("CURRENT / FINAL VALUE SUMMARY")
    print("=" * 165)

    summary = []

    for r in lbfgs_results:
        done = (
            os.path.exists(r.get("final_model", ""))
            or r["count_seq"][-1] >= TOTAL_INNER_STEPS
        )

        summary.append({
            "method": r["method"],
            "description": (
                f"Nsub={r['N_sub']} "
                f"alpha={r['alpha_IVP']} "
                f"maxstep={r['maxstep']}"
            ),
            "count": float(r["count_seq"][-1]),
            "I_G": float(r["I_perp_G_seq"][-1]),
            "I_E": float(r["I_perp_E_seq"][-1]),
            "I_arc": float(r["I_arc_seq"][-1]),
            "Delta_F": float(r["Delta_F_seq"][-1]),
            "done": done,
        })

    if opt_res is not None:
        summary.append({
            "method": "Optimal-IVM",
            "description": "Nsub=4 alpha=0.1 adaptive Adam",
            "count": float(opt_res["count_seq"][-1]),
            "I_G": float(opt_res["I_perp_G_seq"][-1]),
            "I_E": float(opt_res["I_perp_E_seq"][-1]),
            "I_arc": float(opt_res["I_arc_seq"][-1]),
            "Delta_F": float(opt_res["Delta_F_seq"][-1]),
            "done": opt_done,
        })

    for r in feneb_res:
        summary.append({
            "method": "MFEP-FENEB",
            "description": f"lambda={r['physical_step']:g}",
            "count": float(r["count_seq"][-1]),
            "I_G": float(r["I_perp_G_seq"][-1]),
            "I_E": float(r["I_perp_E_seq"][-1]),
            "I_arc": float(r["I_arc_seq"][-1]),
            "Delta_F": float(r["Delta_F_seq"][-1]),
            "done": bool(r["done"]),
        })

    method_order = {
        "Optimal-IVM": 0,
        "MFEP-FENEB": 1,
        "L-BFGS": 2,
        "L-BFGS-Recycle": 3,
    }

    for row in sorted(
        summary,
        key=lambda x: (
            method_order.get(x["method"], 999),
            x["I_G"],
        ),
    ):
        print(
            f"{row['method']:<16} "
            f"{row['description']:<46} "
            f"count={row['count']:>7.1f} | "
            f"I_G={row['I_G']:.6e} | "
            f"I_E={row['I_E']:.6e} | "
            f"I_arc={row['I_arc']:.6e} | "
            f"DeltaF={row['Delta_F']:.6f} | "
            f"done={row['done']}"
        )

    # --------------------------------------------------------
    # 保存全部读取结果与汇总
    # --------------------------------------------------------
    summary_npz_out = os.path.join(
        base_path,
        "MFEP_SeparatedMethodComparison_four_metrics.npz",
    )

    np.savez(
        summary_npz_out,
        lbfgs_results=np.array(lbfgs_results, dtype=object),
        lbfgs_original_results=np.array(lbfgs_original_results, dtype=object),
        lbfgs_recycle_results=np.array(lbfgs_recycle_results, dtype=object),
        optimal_ivm_result=np.array([opt_res], dtype=object),
        feneb_results=np.array(feneb_res, dtype=object),
        summary=np.array(summary, dtype=object),
        saved_plot_info=np.array(saved_plot_info, dtype=object),
    )

    print("\n" + "=" * 120)
    print("已分别保存的方法内部对比图：")

    for item in saved_plot_info:
        print(f"\n[{item['group_name']}]")
        print(f"  PDF: {item['pdf']}")
        print(f"  PNG: {item['png']}")

    print(f"\n汇总数据文件：\n  {summary_npz_out}")
    print("=" * 120)

    return {
        "lbfgs_results": lbfgs_results,
        "lbfgs_original_results": lbfgs_original_results,
        "lbfgs_recycle_results": lbfgs_recycle_results,
        "optimal_result": opt_res,
        "feneb_results": feneb_res,
        "summary": summary,
        "saved_plot_info": saved_plot_info,
        "optimal_current_count": opt_cur,
        "optimal_complete": opt_done,
        "feneb_all_complete": feneb_all_done,
        "lbfgs_all_complete": lbfgs_done,
    }
# ============================================================
# 自动停止逻辑
# ============================================================
def should_stop(latest):
    req = []
    if AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE:
        req.append(bool(latest.get("optimal_complete", False)))
    if AUTO_STOP_REQUIRE_FENEB_COMPLETE:
        req.append(bool(latest.get("feneb_all_complete", False)))
    if AUTO_STOP_REQUIRE_ALL_LBFGS_COMPLETE:
        req.append(bool(latest.get("lbfgs_all_complete", False)))
    return all(req) if req else False

# ============================================================
# 执行
# ============================================================
if AUTO_REFRESH:
    print(f"AUTO_REFRESH=True：每 {REFRESH_SECONDS} 秒刷新一次。")
    print("自动检测所有 L‑BFGS 实验（原始及 Recycle）并实时绘图。")
    print("点击停止按钮可终止。")
    latest = None
    try:
        while True:
            if clear_output is not None:
                clear_output(wait=True)
            latest = load_and_plot_once()
            if should_stop(latest):
                print("\n所有要求监控的计算均已完成，自动刷新结束。")
                break
            if (latest["optimal_current_count"] < 0 and not WAIT_FOR_OPTIMAL_FILE and AUTO_STOP_REQUIRE_OPTIMAL_COMPLETE):
                print("\nOptimal-IVM 文件不存在且 WAIT_FOR_OPTIMAL_FILE=False，停止刷新。")
                break
            if (len(latest["feneb_results"]) == 0 and not WAIT_FOR_FENEB_FILE and AUTO_STOP_REQUIRE_FENEB_COMPLETE):
                print("\nMFEP-FENEB 文件不存在且 WAIT_FOR_FENEB_FILE=False，停止刷新。")
                break
            print(f"\n下次刷新将在 {REFRESH_SECONDS} 秒后进行。")
            time.sleep(REFRESH_SECONDS)
    except KeyboardInterrupt:
        print("\n自动刷新已手动停止。")
else:
    latest = load_and_plot_once()

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
#  一行三张：仅对比 IVM(best) vs FENEB(lambda=0.02)
#  去掉 I_perp^E，Nature/Science 风格，IVM 用蓝色
#  依赖上一 cell 已定义的函数与变量
# ============================================================

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

# ------------------------------------------------------------
# 1. 选定的两条线
# ------------------------------------------------------------
BEST_IVM = dict(N_sub=1, alpha=1e-4, maxstep=0.1, exp_type="L-BFGS-Recycle")
BEST_FENEB_LAMBDA = 0.02

# 配色（色盲友好 Wong 调色板）
COLOR_IVM   = "#0072B2"   # 蓝色 —— 我们的方法
COLOR_FENEB = "#D55E00"   # 朱红 —— FENEB


# ------------------------------------------------------------
# 2. 定位并读取 best IVM (L-BFGS-Recycle)
# ------------------------------------------------------------
def find_best_ivm_config():
    for cfg in discover_lbfgs_configs():
        if (cfg["type"] == BEST_IVM["exp_type"]
                and cfg["N_sub"] == BEST_IVM["N_sub"]
                and np.isclose(cfg["alpha"], BEST_IVM["alpha"], rtol=1e-6, atol=1e-12)
                and np.isclose(cfg["maxstep"], BEST_IVM["maxstep"], rtol=1e-6, atol=1e-12)):
            return cfg
    return None

ivm_cfg = find_best_ivm_config()
if ivm_cfg is None:
    raise RuntimeError(
        "未找到指定的 L-BFGS-Recycle 配置 "
        f"(N_sub={BEST_IVM['N_sub']}, alpha={BEST_IVM['alpha']}, "
        f"maxstep={BEST_IVM['maxstep']})。请检查文件夹是否存在。"
    )

ivm_res, ivm_err = load_ivm_result(
    ivm_cfg["npz"], "IVM", ivm_cfg["N_sub"], ivm_cfg["alpha"],
    str(ivm_cfg["maxstep"]), require_complete=False, final_model=None,
)
if ivm_res is None:
    raise RuntimeError(f"读取 IVM 失败：{ivm_err}")


# ------------------------------------------------------------
# 3. 读取 FENEB (lambda = 0.02)
# ------------------------------------------------------------
_, _, feneb_npz, _ = feneb_result_paths(BEST_FENEB_LAMBDA)
feneb_res, feneb_err = load_single_feneb_result(
    feneb_npz, expected_phys_step=BEST_FENEB_LAMBDA
)
if feneb_res is None:
    raise RuntimeError(f"读取 FENEB(lambda={BEST_FENEB_LAMBDA}) 失败：{feneb_err}")


# ------------------------------------------------------------
# 4. Nature / Science 风格 rcParams
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset":   "dejavusans",
    "axes.linewidth":     1.1,
    "axes.edgecolor":     "#222222",
    "axes.labelcolor":    "#111111",
    "xtick.color":        "#222222",
    "ytick.color":        "#222222",
    "text.color":         "#111111",
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4.5,
    "ytick.major.size":   4.5,
    "xtick.major.width":  1.0,
    "ytick.major.width":  1.0,
    "xtick.minor.size":   2.6,
    "ytick.minor.size":   2.6,
    "xtick.minor.width":  0.8,
    "ytick.minor.width":  0.8,
    "font.size":          12,
    "axes.labelsize":     14,
    "legend.frameon":     False,
    "savefig.dpi":        400,
    "figure.dpi":         120,
})


# ------------------------------------------------------------
# 5. 三个指标（去掉 I_perp^E）
# ------------------------------------------------------------
tiny = np.finfo(float).tiny

metric_specs = [
    dict(key="I_perp_G_seq",
         ylabel=r"$\widehat{I}_{\perp}^{\,G}$",
         title="Riemannian normal residual",
         log=True),
    dict(key="I_arc_seq",
         ylabel=r"$I_{\mathrm{arc}}$",
         title="Equi-arclength residual",
         log=True),
    dict(key="Delta_F_seq",
         ylabel=r"$\Delta F^{\dagger}\ \mathrm{(kcal\,mol^{-1})}$",
         title="Free-energy barrier",
         log=False),
]

series = [
    dict(res=ivm_res,   color=COLOR_IVM,
         label=r"IVM (ours)",
         marker="o", z=5),
    dict(res=feneb_res, color=COLOR_FENEB,
         label=r"FENEB ($\lambda=0.02$)",
         marker="s", z=4),
]


# ------------------------------------------------------------
# 6. 绘图：一行三张
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

panel_tags = ["a", "b", "c"]

for ax, spec, tag in zip(axes, metric_specs, panel_tags):

    for s in series:
        r = s["res"]
        x = np.asarray(r["count_seq"], dtype=float)
        y = np.asarray(r[spec["key"]], dtype=float)

        if spec["log"]:
            y = np.maximum(y, tiny)

        order = np.argsort(x)
        x, y = x[order], y[order]

        markevery = max(1, len(x) // 10)

        ax.plot(
            x, y,
            color=s["color"],
            linestyle="-",
            linewidth=2.4,
            marker=s["marker"],
            markevery=markevery,
            markersize=6.5,
            markerfacecolor="white",
            markeredgecolor=s["color"],
            markeredgewidth=1.6,
            solid_capstyle="round",
            label=s["label"],
            zorder=s["z"],
            clip_on=False,
        )
        # 终点高亮实心点
        ax.scatter(
            x[-1], y[-1],
            s=70, marker=s["marker"],
            facecolor=s["color"], edgecolor="white",
            linewidth=1.2, zorder=s["z"] + 5, clip_on=False,
        )

    # 轴样式
    ax.set_xlabel("Restrained-MD / gradient count", labelpad=6)
    ax.set_ylabel(spec["ylabel"], labelpad=6)
    ax.set_title(spec["title"], fontsize=13, pad=12, fontweight="medium")
    ax.set_xlim(left=0)

    if spec["log"]:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
        ax.yaxis.set_minor_locator(
            LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=12)
        )
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.minorticks_on()

    # 只保留左、下边框（Nature 风格）
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(which="both", top=False, right=False)

    # 轻网格
    ax.grid(True, which="major", axis="y",
            linestyle="-", linewidth=0.6, alpha=0.18, color="#666666")

    # 面板字母
    ax.text(-0.16, 1.06, tag, transform=ax.transAxes,
            fontsize=17, fontweight="bold", va="top", ha="left")

# 统一图例（放在图顶部中央）
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center", bbox_to_anchor=(0.5, 1.06),
    ncol=2, fontsize=13, handlelength=2.2,
    columnspacing=2.4, borderaxespad=0.0,
)

fig.subplots_adjust(left=0.07, right=0.985, top=0.83,
                    bottom=0.15, wspace=0.30)

# ------------------------------------------------------------
# 7. 保存
# ------------------------------------------------------------
out_stem = os.path.join(base_path, "MFEP_IVM_vs_FENEB_bestpair_threepanel")
fig.savefig(out_stem + ".pdf", bbox_inches="tight")
fig.savefig(out_stem + ".png", dpi=400, bbox_inches="tight")
fig.savefig(out_stem + ".svg", bbox_inches="tight")

plt.show()

# ------------------------------------------------------------
# 8. 终值汇总
# ------------------------------------------------------------
print("=" * 70)
print("Final values (best pair)")
print("=" * 70)
for s in series:
    r = s["res"]
    name = s["label"]
    print(f"{name:>22} | count={r['count_seq'][-1]:>7.1f} | "
          f"I_G={r['I_perp_G_seq'][-1]:.4e} | "
          f"I_arc={r['I_arc_seq'][-1]:.4e} | "
          f"DeltaF={r['Delta_F_seq'][-1]:.4f}")
print("=" * 70)
print(f"[SAVED] {out_stem}.pdf / .png / .svg")

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
#  一行三张：仅对比 IVM(best) vs FENEB(lambda=0.02)
#  - 以公共初始数据 (MFEP_INITIAL_PATH_common.npz) 为共同起点
#  - 横坐标画到 100 即停止
#  去掉 I_perp^E，Nature/Science 风格，IVM 用蓝色
#  依赖上一 cell 已定义的函数与变量
# ============================================================

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

# ------------------------------------------------------------
# 0. 全局设置：横坐标上限 & 公共初始数据文件
# ------------------------------------------------------------
X_MAX = 100.0
INITIAL_NPZ = os.path.join(base_path, "MFEP_INITIAL_PATH_common.npz")

# 指标序列 key -> 公共初始 npz 中对应的标量 key
INIT_KEY_MAP = {
    "I_perp_G_seq": "I_perp_G",
    "I_arc_seq":    "I_arc",
    "Delta_F_seq":  "Delta_F_barrier",
}


# ------------------------------------------------------------
# 1. 选定的两条线
# ------------------------------------------------------------
BEST_IVM = dict(N_sub=1, alpha=1e-4, maxstep=0.1, exp_type="L-BFGS-Recycle")
BEST_FENEB_LAMBDA = 0.02

# 配色（色盲友好 Wong 调色板）
COLOR_IVM   = "#0072B2"   # 蓝色 —— 我们的方法
COLOR_FENEB = "#D55E00"   # 朱红 —— FENEB


# ------------------------------------------------------------
# 2. 定位并读取 best IVM (L-BFGS-Recycle)
# ------------------------------------------------------------
def find_best_ivm_config():
    for cfg in discover_lbfgs_configs():
        if (cfg["type"] == BEST_IVM["exp_type"]
                and cfg["N_sub"] == BEST_IVM["N_sub"]
                and np.isclose(cfg["alpha"], BEST_IVM["alpha"], rtol=1e-6, atol=1e-12)
                and np.isclose(cfg["maxstep"], BEST_IVM["maxstep"], rtol=1e-6, atol=1e-12)):
            return cfg
    return None

ivm_cfg = find_best_ivm_config()
if ivm_cfg is None:
    raise RuntimeError(
        "未找到指定的 L-BFGS-Recycle 配置 "
        f"(N_sub={BEST_IVM['N_sub']}, alpha={BEST_IVM['alpha']}, "
        f"maxstep={BEST_IVM['maxstep']})。请检查文件夹是否存在。"
    )

ivm_res, ivm_err = load_ivm_result(
    ivm_cfg["npz"], "IVM", ivm_cfg["N_sub"], ivm_cfg["alpha"],
    str(ivm_cfg["maxstep"]), require_complete=False, final_model=None,
)
if ivm_res is None:
    raise RuntimeError(f"读取 IVM 失败：{ivm_err}")


# ------------------------------------------------------------
# 3. 读取 FENEB (lambda = 0.02)
# ------------------------------------------------------------
_, _, feneb_npz, _ = feneb_result_paths(BEST_FENEB_LAMBDA)
feneb_res, feneb_err = load_single_feneb_result(
    feneb_npz, expected_phys_step=BEST_FENEB_LAMBDA
)
if feneb_res is None:
    raise RuntimeError(f"读取 FENEB(lambda={BEST_FENEB_LAMBDA}) 失败：{feneb_err}")


# ------------------------------------------------------------
# 3b. 读取公共初始数据
# ------------------------------------------------------------
if not os.path.exists(INITIAL_NPZ):
    raise FileNotFoundError(
        f"找不到公共初始数据文件：{INITIAL_NPZ}\n"
        f"请先运行生成 MFEP_INITIAL_PATH_common.npz 的脚本。"
    )

with np.load(INITIAL_NPZ, allow_pickle=True) as _init_loaded:
    INIT_COUNT = float(np.asarray(_init_loaded["count"]).item())
    INIT_VALUES = {
        seq_key: float(np.asarray(_init_loaded[init_key]).item())
        for seq_key, init_key in INIT_KEY_MAP.items()
    }

print("公共初始数据 (共同起点):")
print(f"  count = {INIT_COUNT:.1f}")
for seq_key, val in INIT_VALUES.items():
    print(f"  {seq_key:>14s} = {val:.6e}")


# ------------------------------------------------------------
# 3c. 以公共初始点为起点，并裁剪到 count <= X_MAX
# ------------------------------------------------------------
def augment_and_clip(res, metric_keys):
    """
    - 丢弃该方法自身 count<=INIT_COUNT 的点；
    - 在最前面插入公共初始点 (INIT_COUNT, INIT_VALUES)；
    - 只保留 count <= X_MAX 的点。
    返回 dict: {"count_seq": ..., <metric_key>: ...}
    """
    x = np.asarray(res["count_seq"], dtype=float)

    # 保留严格大于初始 count 的点，避免与公共起点重复
    keep = x > INIT_COUNT + 1e-9

    out = {}
    x_kept = x[keep]

    # 拼接公共起点
    out["count_seq"] = np.concatenate([[INIT_COUNT], x_kept])

    for mk in metric_keys:
        y = np.asarray(res[mk], dtype=float)[keep]
        out[mk] = np.concatenate([[INIT_VALUES[mk]], y])

    # 裁剪到 X_MAX
    mask = out["count_seq"] <= X_MAX + 1e-9
    for k in out:
        out[k] = out[k][mask]

    return out


metric_keys = list(INIT_KEY_MAP.keys())
ivm_plot   = augment_and_clip(ivm_res,   metric_keys)
feneb_plot = augment_and_clip(feneb_res, metric_keys)


# ------------------------------------------------------------
# 4. Nature / Science 风格 rcParams
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset":   "dejavusans",
    "axes.linewidth":     1.1,
    "axes.edgecolor":     "#222222",
    "axes.labelcolor":    "#111111",
    "xtick.color":        "#222222",
    "ytick.color":        "#222222",
    "text.color":         "#111111",
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   4.5,
    "ytick.major.size":   4.5,
    "xtick.major.width":  1.0,
    "ytick.major.width":  1.0,
    "xtick.minor.size":   2.6,
    "ytick.minor.size":   2.6,
    "xtick.minor.width":  0.8,
    "ytick.minor.width":  0.8,
    "font.size":          12,
    "axes.labelsize":     14,
    "legend.frameon":     False,
    "savefig.dpi":        400,
    "figure.dpi":         120,
})


# ------------------------------------------------------------
# 5. 三个指标（去掉 I_perp^E）
# ------------------------------------------------------------
tiny = np.finfo(float).tiny

metric_specs = [
    dict(key="I_perp_G_seq",
         ylabel=r"$\widehat{I}_{\perp}^{\,G}$",
         title="Riemannian normal residual",
         log=True),
    dict(key="I_arc_seq",
         ylabel=r"$I_{\mathrm{arc}}$",
         title="Equi-arclength residual",
         log=True),
    dict(key="Delta_F_seq",
         ylabel=r"$\Delta F^{\dagger}\ \mathrm{(kcal\,mol^{-1})}$",
         title="Free-energy barrier",
         log=False),
]

series = [
    dict(data=ivm_plot,   color=COLOR_IVM,
         label=r"IVM (ours)",
         marker="o", z=5),
    dict(data=feneb_plot, color=COLOR_FENEB,
         label=r"FENEB ($\lambda=0.02$)",
         marker="s", z=4),
]


# ------------------------------------------------------------
# 6. 绘图：一行三张
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

panel_tags = ["a", "b", "c"]

for ax, spec, tag in zip(axes, metric_specs, panel_tags):

    for s in series:
        d = s["data"]
        x = np.asarray(d["count_seq"], dtype=float)
        y = np.asarray(d[spec["key"]], dtype=float)

        if spec["log"]:
            y = np.maximum(y, tiny)

        order = np.argsort(x)
        x, y = x[order], y[order]

        markevery = max(1, len(x) // 10)

        ax.plot(
            x, y,
            color=s["color"],
            linestyle="-",
            linewidth=2.4,
            marker=s["marker"],
            markevery=markevery,
            markersize=6.5,
            markerfacecolor="white",
            markeredgecolor=s["color"],
            markeredgewidth=1.6,
            solid_capstyle="round",
            label=s["label"],
            zorder=s["z"],
            clip_on=False,
        )
        # 终点高亮实心点
        ax.scatter(
            x[-1], y[-1],
            s=70, marker=s["marker"],
            facecolor=s["color"], edgecolor="white",
            linewidth=1.2, zorder=s["z"] + 5, clip_on=False,
        )

    # 轴样式
    ax.set_xlabel("Restrained-MD / gradient count", labelpad=6)
    ax.set_ylabel(spec["ylabel"], labelpad=6)
    ax.set_title(spec["title"], fontsize=13, pad=12, fontweight="medium")
    ax.set_xlim(0, X_MAX)

    if spec["log"]:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
        ax.yaxis.set_minor_locator(
            LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=12)
        )
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.minorticks_on()

    # 只保留左、下边框（Nature 风格）
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(which="both", top=False, right=False)

    # 轻网格
    ax.grid(True, which="major", axis="y",
            linestyle="-", linewidth=0.6, alpha=0.18, color="#666666")

    # 面板字母
    ax.text(-0.16, 1.06, tag, transform=ax.transAxes,
            fontsize=17, fontweight="bold", va="top", ha="left")

# 统一图例（放在图顶部中央）
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center", bbox_to_anchor=(0.5, 1.06),
    ncol=2, fontsize=13, handlelength=2.2,
    columnspacing=2.4, borderaxespad=0.0,
)

fig.subplots_adjust(left=0.07, right=0.985, top=0.83,
                    bottom=0.15, wspace=0.30)

# ------------------------------------------------------------
# 7. 保存
# ------------------------------------------------------------
out_stem = os.path.join(base_path, "MFEP_IVM_vs_FENEB_bestpair_threepanel")
fig.savefig(out_stem + ".pdf", bbox_inches="tight")
fig.savefig(out_stem + ".png", dpi=400, bbox_inches="tight")
fig.savefig(out_stem + ".svg", bbox_inches="tight")

plt.show()

# ------------------------------------------------------------
# 8. 终值汇总（裁剪到 count<=100 后）
# ------------------------------------------------------------
print("=" * 70)
print(f"Final values (best pair, clipped to count <= {X_MAX:.0f})")
print("=" * 70)
for s in series:
    d = s["data"]
    name = s["label"]
    print(f"{name:>22} | count={d['count_seq'][-1]:>7.1f} | "
          f"I_G={d['I_perp_G_seq'][-1]:.4e} | "
          f"I_arc={d['I_arc_seq'][-1]:.4e} | "
          f"DeltaF={d['Delta_F_seq'][-1]:.4f}")
print("=" * 70)
print(f"[SAVED] {out_stem}.pdf / .png / .svg")

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
#  一行四张 contour:Initialization / count=10 / 20 / 100
#  背景 FES:0–55 黑->白, >55 全白;前景 IVM vs FENEB 路径
#  小标题:Force calls = per-image count * 20 (20 个可移动 image)
#  依赖前面 cell:ivm_res, feneb_res, base_path
# ============================================================

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# 0. 配置
# ------------------------------------------------------------
COLOR_IVM   = "#0072B2"          # 蓝 —— 我们的方法
COLOR_FENEB = "#D55E00"          # 朱红 —— FENEB

N_IMAGES_MOVABLE = 20            # Force calls = count * 20
TARGET_COUNTS = [0, 10, 20, 100]
PANEL_TITLES  = ["Initialization", None, None, None]  # None -> 用 Force calls

XLIM = (-np.pi, 0.0)
YLIM = ( 0.0,  np.pi)

# 手动能量色标:0 -> 黑, ENERGY_WHITE_CAP -> 白, 以上全白
ENERGY_WHITE_CAP = 60.0
BG_LO = 0.05                      # 谷底灰度 (0=黑; 想浅一点调大, 如 0.2)

# 固定端点(与实验一致),IVM 路径只存 20 个内部 image,需补端点
C1 = np.array([-2.60866606, 2.77661136])
C2 = np.array([-1.3304716,  0.99118795])

SPECIAL_POINTS = [
    (-2.62593029, 2.77365591, r"$C5$",           -0.05,  0.16),
    (-1.33433281, 0.99268968, r"$C7_{\mathrm{eq}}$", 0.10, -0.40),
]

FES_NPZ = os.path.join(base_path, "wtmeta_phi_psi", "fes_XYZ.npz")

IVM_RAW_NPZ   = str(np.asarray(ivm_res["npz_file"]).item())
FENEB_RAW_NPZ = str(np.asarray(feneb_res["npz_file"]).item())


# ------------------------------------------------------------
# 1. 读取并裁剪 FES
# ------------------------------------------------------------
with np.load(FES_NPZ) as L:
    gx = np.asarray(L["X"], dtype=float)
    gy = np.asarray(L["Y"], dtype=float)
    gz = np.asarray(L["Z"], dtype=float)

if gx.ndim == 1 and gy.ndim == 1:
    gx, gy = np.meshgrid(gx, gy, indexing="xy")

gx = np.deg2rad(gx)
gy = np.deg2rad(gy)
gz = np.ma.masked_invalid(gz)

x_line = gx[0, :]
y_line = gy[:, 0]
cols = np.where((x_line >= XLIM[0]) & (x_line <= XLIM[1]))[0]
rows = np.where((y_line >= YLIM[0]) & (y_line <= YLIM[1]))[0]
r0, r1 = rows[0], rows[-1] + 1
c0, c1 = cols[0], cols[-1] + 1

Xc = gx[r0:r1, c0:c1]
Yc = gy[r0:r1, c0:c1]
Zc = gz[r0:r1, c0:c1]

# 以区域最小值为 0 基准
Zc = Zc - np.ma.min(Zc)



from matplotlib.colors import LinearSegmentedColormap

# ---- 能量->灰度: 二阶曲线, CURV 控制曲率 ----
# CURV = 0.0  线性
# CURV > 0    低能量(黑->灰)变化快, 接近上限变化慢  <- 你要的
# 建议范围 0.0 ~ 0.9; 太大(接近1)顶端几乎不变化会丢细节
CURV = 2

_u    = np.linspace(0.0, 1.0, 256)
_t    = _u + CURV * _u * (1.0 - _u)      # 二阶映射
_t    = np.clip(_t, 0.0, 1.0)
_gray = BG_LO + (1.0 - BG_LO) * _t       # 谷底 BG_LO -> 顶端 1.0(白)

BG_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad", list(zip(_gray, _gray, _gray))
)
BG_CMAP.set_over("white")
levels = np.linspace(0.0, ENERGY_WHITE_CAP, 33)   # 41 段, 过渡更平滑


# ------------------------------------------------------------
# 2. 路径提取(键名已确认)
# ------------------------------------------------------------
# IVM: phi_ref_iter_{count} 只有 20 个内部 image,补端点 -> 22
_ivm_raw = np.load(IVM_RAW_NPZ, allow_pickle=True)

def get_ivm_path(count):
    P = np.asarray(_ivm_raw[f"phi_ref_iter_{count}"], dtype=float)  # (20,2)
    return np.vstack([C1, P, C2])                                  # (22,2)

# FENEB: mean_path_unwrapped_seq[count] 已含端点 -> (22,2)
with np.load(FENEB_RAW_NPZ, allow_pickle=True) as _fen_raw:
    _feneb_paths = np.asarray(_fen_raw["mean_path_unwrapped_seq"], dtype=float)

def get_feneb_path(count):
    return _feneb_paths[count]                                     # (22,2)


# ------------------------------------------------------------
# 3. Nature / Science 风格 rcParams
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset":  "dejavusans",
    "axes.linewidth":    1.1,
    "axes.edgecolor":    "#222222",
    "axes.labelcolor":   "#111111",
    "xtick.color":       "#222222",
    "ytick.color":       "#222222",
    "text.color":        "#111111",
    "xtick.direction":   "out",
    "ytick.direction":   "out",
    "font.size":         12,
    "axes.labelsize":    14,
    "legend.frameon":    False,
    "savefig.dpi":       400,
    "figure.dpi":        120,
})


# ------------------------------------------------------------
# 4. 绘图:一行四张
# ------------------------------------------------------------
series = [
    dict(getter=get_ivm_path,   color=COLOR_IVM,   marker="o",
         label=r"IVM (ours)",            z=6),
    dict(getter=get_feneb_path, color=COLOR_FENEB, marker="s",
         label=r"FENEB ($\lambda=0.02$)", z=5),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
panel_tags = ["a", "b", "c", "d"]
cf = None

for ax, tc, ptitle, tag in zip(axes, TARGET_COUNTS, PANEL_TITLES, panel_tags):

    # --- 背景灰度 FES (0-55 黑->白, >55 白) ---
    cf = ax.contourf(Xc, Yc, Zc, levels=levels, cmap=BG_CMAP, extend="max")

    # --- 亚稳态标注 ---
    for xx, yy, lab, dx, dy in SPECIAL_POINTS:
        ax.plot(xx, yy, marker="o", markersize=6,
                markerfacecolor="white", markeredgecolor="#111111",
                markeredgewidth=1.4, linestyle="None", zorder=9)
        ax.text(xx + dx, yy + dy, lab, color="#111111",
                fontsize=12, fontweight="bold", zorder=9,
                ha="right", va="bottom",
                bbox=dict(boxstyle="round,pad=0.12", fc="white",
                          ec="none", alpha=0.75))

    # --- 两条路径 (无 halo, 画全部 22 个点) ---
    for s in series:
        P = s["getter"](tc)
        ax.plot(P[:, 0], P[:, 1], color=s["color"],
                linewidth=2.2, marker=s["marker"], markevery=1,
                markersize=5.0, markerfacecolor="white",
                markeredgecolor=s["color"], markeredgewidth=1.3,
                solid_capstyle="round", zorder=s["z"])

    # --- 标题:优先自定义,否则 Force calls ---
    force_calls = tc * N_IMAGES_MOVABLE
    if ptitle is not None:
        title = ptitle
    else:
        title = f"Force calls: {force_calls / 1000:g}k"
    ax.set_title(title, fontsize=13, pad=10, fontweight="medium")

    ax.set_xlabel(r"$\phi$ (rad)", labelpad=5)
    if ax is axes[0]:
        ax.set_ylabel(r"$\psi$ (rad)", labelpad=5)
    else:
        ax.set_yticklabels([])

    ax.set_xlim(XLIM)
    ax.set_ylim(YLIM)
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(which="both", top=False, right=False)

    ax.text(-0.02, 1.11, tag, transform=ax.transAxes,
            fontsize=17, fontweight="bold", va="top", ha="left")

# --- 布局:先留出右侧空间给 colorbar ---
fig.subplots_adjust(left=0.05, right=0.88, top=0.82,
                    bottom=0.14, wspace=0.08)

# --- 独立 colorbar 轴, 与子图等高, 不再和第四张重合 ---
pos_last = axes[-1].get_position()
cax = fig.add_axes([pos_last.x1 + 0.015, pos_last.y0,
                    0.014, pos_last.height])
cbar = fig.colorbar(cf, cax=cax, extend="max")
cbar.set_label(r"Free energy (kcal mol$^{-1}$)", labelpad=8)

# --- 顶部统一图例 ---
handles = [
    Line2D([0], [0], color=s["color"], linewidth=2.2,
           marker=s["marker"], markerfacecolor="white",
           markeredgecolor=s["color"], markeredgewidth=1.3,
           label=s["label"])
    for s in series
]
fig.legend(handles=handles, loc="upper center",
           bbox_to_anchor=(0.5, 1.05), ncol=2, fontsize=13,
           handlelength=2.2, columnspacing=2.4, borderaxespad=0.0)

# ------------------------------------------------------------
# 5. 保存
# ------------------------------------------------------------
out_stem = os.path.join(base_path, "MFEP_IVM_vs_FENEB_contour_fourpanel")
fig.savefig(out_stem + ".pdf", bbox_inches="tight")
fig.savefig(out_stem + ".png", dpi=400, bbox_inches="tight")
fig.savefig(out_stem + ".svg", bbox_inches="tight")

_ivm_raw.close()
plt.show()
print(f"[SAVED] {out_stem}.pdf / .png / .svg")

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Euclidean MEP (G_opt = I)
#
# 一行三张：
#   (a) Euclidean MEP residual I_perp
#   (b) MFEP diagnostic I_perp^G evaluated on the MEP
#   (c) Initial path and final Euclidean MEP on the FES
#
# 只读取 Euclidean MEP 结果，不读取之前的 MFEP/FENEB 结果。
# ============================================================

import os
import glob
import re
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import LogLocator, NullFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable


# ============================================================
# 1. User settings
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# Euclidean MEP configuration to plot.
N_IMAGES = 20
N_SUB = 1
ALPHA = 0.05
MAXSTEP = 0.1
HISTORY_SIZE = 7

# The number of restrained-MD/gradient evaluations per outer iteration:
#
#   N images × N_sub gradient batches.
#
# The reference-path evaluation is reused as inner substep 0.
GRADIENT_CALLS_PER_OUTER = N_IMAGES * N_SUB

# Path plotting window.
XLIM = (-np.pi, 0.0)
YLIM = (0.0, np.pi)

# Output file.
OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        f"Euclidean_MEP_alpha{ALPHA}"
        f"_Nsub{N_SUB}"
        f"_maxstep{MAXSTEP}"
        f"_MEP_and_MFEP_metrics"
    ),
)

# Colors: color-blind-friendly Wong palette.
COLOR_MEP = "#0072B2"
COLOR_MFEP_TEST = "#D55E00"
COLOR_INITIAL = "#7F7F7F"
COLOR_FINAL_PATH = "#0072B2"


# ============================================================
# 2. Find the Euclidean MEP result file
# ============================================================

def parse_float_from_name(text, key):
    """
    Extract a floating-point value following a key in a directory name.

    Example
    -------
    text = "..._alpha0.05_maxstep0.1_..."
    key  = "alpha"
    """
    pattern = (
        rf"{re.escape(key)}"
        rf"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    )

    match = re.search(pattern, text)

    if match is None:
        return None

    try:
        return float(match.group(1))
    except ValueError:
        return None


def parse_int_from_name(text, key):
    """
    Extract an integer following a key in a directory name.
    """
    pattern = rf"{re.escape(key)}(\d+)"
    match = re.search(pattern, text)

    if match is None:
        return None

    try:
        return int(match.group(1))
    except ValueError:
        return None


def discover_euclidean_mep_results():
    """
    Discover result folders generated by the Euclidean-MEP code.

    Expected directory pattern
    --------------------------
    MEP_image_LBFGS_B_Recycle_N...
    """
    pattern = os.path.join(
        BASE_PATH,
        "MEP_image_LBFGS_B_Recycle_N*",
    )

    configs = []

    for folder in sorted(glob.glob(pattern)):
        if not os.path.isdir(folder):
            continue

        name = os.path.basename(folder)

        n_images = parse_int_from_name(
            name,
            "_N",
        )

        n_sub = parse_int_from_name(
            name,
            "_Nsub",
        )

        history_size = parse_int_from_name(
            name,
            "_hist",
        )

        alpha = parse_float_from_name(
            name,
            "_alpha",
        )

        maxstep = parse_float_from_name(
            name,
            "_maxstep",
        )

        npz_candidates = sorted(
            glob.glob(
                os.path.join(
                    folder,
                    "alanine_*.npz",
                )
            )
        )

        if not npz_candidates:
            continue

        # There should normally be one NPZ file per experiment folder.
        npz_path = npz_candidates[-1]

        configs.append(
            {
                "folder": folder,
                "name": name,
                "npz": npz_path,
                "N": n_images,
                "N_sub": n_sub,
                "alpha": alpha,
                "maxstep": maxstep,
                "history_size": history_size,
            }
        )

    return configs


def find_target_result():
    """
    Find the result matching the selected configuration.
    """
    configs = discover_euclidean_mep_results()

    matches = []

    for cfg in configs:
        if cfg["N"] != N_IMAGES:
            continue

        if cfg["N_sub"] != N_SUB:
            continue

        if cfg["history_size"] != HISTORY_SIZE:
            continue

        if cfg["alpha"] is None or not np.isclose(
            cfg["alpha"],
            ALPHA,
            rtol=1e-8,
            atol=1e-14,
        ):
            continue

        if cfg["maxstep"] is None or not np.isclose(
            cfg["maxstep"],
            MAXSTEP,
            rtol=1e-8,
            atol=1e-14,
        ):
            continue

        matches.append(cfg)

    if not matches:
        available = "\n".join(
            f"  - {cfg['name']}"
            for cfg in configs
        )

        raise RuntimeError(
            "未找到指定的 Euclidean MEP 结果：\n"
            f"  N={N_IMAGES}\n"
            f"  N_sub={N_SUB}\n"
            f"  alpha={ALPHA}\n"
            f"  maxstep={MAXSTEP}\n"
            f"  history={HISTORY_SIZE}\n\n"
            "当前发现的 MEP 结果目录：\n"
            f"{available if available else '  (none)'}"
        )

    if len(matches) > 1:
        print(
            f"[WARNING] 找到 {len(matches)} 个匹配结果，"
            f"将使用：{matches[-1]['npz']}"
        )

    return matches[-1]


TARGET_CONFIG = find_target_result()

print("=" * 80)
print("Selected Euclidean MEP result")
print("=" * 80)
print(f"Folder   : {TARGET_CONFIG['folder']}")
print(f"NPZ file : {TARGET_CONFIG['npz']}")
print(f"N        : {TARGET_CONFIG['N']}")
print(f"N_sub    : {TARGET_CONFIG['N_sub']}")
print(f"alpha    : {TARGET_CONFIG['alpha']}")
print(f"maxstep  : {TARGET_CONFIG['maxstep']}")
print(f"history  : {TARGET_CONFIG['history_size']}")
print("=" * 80)


# ============================================================
# 3. NPZ helpers
# ============================================================

def load_first_existing(loaded, keys, required=True):
    """
    Load the first available key from an NPZ file.
    """
    for key in keys:
        if key in loaded.files:
            return np.asarray(
                loaded[key],
                dtype=float,
            )

    if required:
        raise KeyError(
            "None of the requested keys exists: "
            + ", ".join(keys)
        )

    return None


def extract_iteration_numbers(loaded, prefix):
    """
    Extract iteration numbers from keys such as:

        path_iter_0
        path_iter_1
        ...
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    iteration_numbers = []

    for key in loaded.files:
        match = pattern.match(key)

        if match is not None:
            iteration_numbers.append(
                int(match.group(1))
            )

    return sorted(iteration_numbers)


def safe_positive(y):
    """
    Replace invalid or non-positive values for log plotting.
    """
    y = np.asarray(
        y,
        dtype=float,
    )

    tiny = np.finfo(float).tiny

    y = np.where(
        np.isfinite(y),
        y,
        np.nan,
    )

    positive = y[
        np.isfinite(y)
        & (y > 0.0)
    ]

    if positive.size > 0:
        floor = max(
            tiny,
            positive.min() * 1e-6,
        )
    else:
        floor = tiny

    return np.where(
        np.isfinite(y) & (y > 0.0),
        y,
        floor,
    )


def build_gradient_count(index_seq, loaded):
    """
    Construct cumulative restrained-MD/gradient count.

    Default:
        count = (outer iteration + 1) * N * N_sub.

    If the NPZ already contains count_seq, use it.
    """
    if "count_seq" in loaded.files:
        count = np.asarray(
            loaded["count_seq"],
            dtype=float,
        )

        if count.shape == index_seq.shape:
            return count

    return (
        index_seq + 1.0
    ) * float(
        GRADIENT_CALLS_PER_OUTER
    )


# ============================================================
# 4. Load Euclidean MEP result
# ============================================================

with np.load(
    TARGET_CONFIG["npz"],
    allow_pickle=True,
) as loaded:

    index_seq = load_first_existing(
        loaded,
        ["index_seq"],
    )

    # --------------------------------------------------------
    # Euclidean MEP residual:
    #
    # I_perp
    # =
    # sum_i |P_E grad F|^2 w_i^E.
    # --------------------------------------------------------
    I_MEP_seq = load_first_existing(
        loaded,
        [
            "I_perp_MEP_seq",
            "I_perp_seq",
        ],
    )

    # --------------------------------------------------------
    # MFEP diagnostic evaluated on the same Euclidean MEP:
    #
    # I_perp^G
    # =
    # sum_i
    # |P_{M^{-1}} M grad F|_{M^{-1}}^2 w_i^G.
    # --------------------------------------------------------
    I_MFEP_seq = load_first_existing(
        loaded,
        [
            "I_G_perp_seq",
            "I_perp_G_seq",
        ],
    )

    # Optional unweighted versions.
    I_MEP_mean_seq = load_first_existing(
        loaded,
        [
            "I_perp_euclid_mean_seq",
        ],
        required=False,
    )

    I_MFEP_mean_seq = load_first_existing(
        loaded,
        [
            "I_G_perp_euclid_mean_seq",
        ],
        required=False,
    )

    # Optional maximum residual norms.
    fmax_MEP_seq = load_first_existing(
        loaded,
        [
            "fmax_MEP_seq",
            "fmax_seq",
        ],
        required=False,
    )

    fmax_MFEP_seq = load_first_existing(
        loaded,
        [
            "fmax_G_seq",
        ],
        required=False,
    )

    count_seq = build_gradient_count(
        index_seq,
        loaded,
    )

    # Path data.
    path_iterations = extract_iteration_numbers(
        loaded,
        "path_iter_",
    )

    if not path_iterations:
        raise RuntimeError(
            "NPZ 中没有找到 path_iter_* 数据。"
        )

    final_iteration = path_iterations[-1]

    final_path = np.asarray(
        loaded[f"path_iter_{final_iteration}"],
        dtype=float,
    )

    # The initial frozen reference path is preferable for plotting
    # the initialization.
    if "phi_ref_iter_0" in loaded.files:
        initial_path = np.asarray(
            loaded["phi_ref_iter_0"],
            dtype=float,
        )
    elif "path_iter_0" in loaded.files:
        initial_path = np.asarray(
            loaded["path_iter_0"],
            dtype=float,
        )
    else:
        initial_path = None

    # Endpoints are not stored in path_iter_* because those arrays
    # contain only the internal images. Add known endpoints.
    C1 = np.asarray(
        [-2.60866606, 2.77661136],
        dtype=float,
    )

    C2 = np.asarray(
        [-1.3304716, 0.99118795],
        dtype=float,
    )

    final_path_full = np.vstack(
        [
            C1,
            final_path,
            C2,
        ]
    )

    if initial_path is not None:
        initial_path_full = np.vstack(
            [
                C1,
                initial_path,
                C2,
            ]
        )
    else:
        initial_path_full = None


# Ensure all convergence arrays have matching lengths.
n_common = min(
    len(index_seq),
    len(count_seq),
    len(I_MEP_seq),
    len(I_MFEP_seq),
)

index_seq = index_seq[:n_common]
count_seq = count_seq[:n_common]
I_MEP_seq = I_MEP_seq[:n_common]
I_MFEP_seq = I_MFEP_seq[:n_common]

if I_MEP_mean_seq is not None:
    I_MEP_mean_seq = I_MEP_mean_seq[:n_common]

if I_MFEP_mean_seq is not None:
    I_MFEP_mean_seq = I_MFEP_mean_seq[:n_common]

if fmax_MEP_seq is not None:
    fmax_MEP_seq = fmax_MEP_seq[:n_common]

if fmax_MFEP_seq is not None:
    fmax_MFEP_seq = fmax_MFEP_seq[:n_common]


# Sort by gradient count.
order = np.argsort(
    count_seq
)

index_seq = index_seq[order]
count_seq = count_seq[order]
I_MEP_seq = I_MEP_seq[order]
I_MFEP_seq = I_MFEP_seq[order]

if I_MEP_mean_seq is not None:
    I_MEP_mean_seq = I_MEP_mean_seq[order]

if I_MFEP_mean_seq is not None:
    I_MFEP_mean_seq = I_MFEP_mean_seq[order]

if fmax_MEP_seq is not None:
    fmax_MEP_seq = fmax_MEP_seq[order]

if fmax_MFEP_seq is not None:
    fmax_MFEP_seq = fmax_MFEP_seq[order]


# ============================================================
# 5. Load free-energy surface
# ============================================================

if not os.path.exists(FES_NPZ):
    raise FileNotFoundError(
        f"Cannot find FES file: {FES_NPZ}"
    )

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as fes_loaded:

    grid_x = np.asarray(
        fes_loaded["X"],
        dtype=float,
    )

    grid_y = np.asarray(
        fes_loaded["Y"],
        dtype=float,
    )

    grid_z = np.asarray(
        fes_loaded["Z"],
        dtype=float,
    )

if (
    grid_x.ndim == 1
    and grid_y.ndim == 1
):
    grid_x, grid_y = np.meshgrid(
        grid_x,
        grid_y,
        indexing="xy",
    )

if (
    grid_x.shape != grid_y.shape
    or grid_x.shape != grid_z.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{grid_x.shape}, "
        f"Y{grid_y.shape}, "
        f"Z{grid_z.shape}"
    )

# The stored FES coordinates are assumed to be in degrees.
grid_x = np.deg2rad(
    grid_x
)

grid_y = np.deg2rad(
    grid_y
)

grid_z = np.ma.masked_invalid(
    grid_z
)

# Crop FES.
inside = (
    (grid_x >= XLIM[0])
    & (grid_x <= XLIM[1])
    & (grid_y >= YLIM[0])
    & (grid_y <= YLIM[1])
)

if not np.any(inside):
    raise RuntimeError(
        f"No FES grid points inside XLIM={XLIM}, YLIM={YLIM}."
    )

rows = np.where(
    np.any(inside, axis=1)
)[0]

cols = np.where(
    np.any(inside, axis=0)
)[0]

r0 = rows[0]
r1 = rows[-1] + 1
c0 = cols[0]
c1 = cols[-1] + 1

X_plot = grid_x[
    r0:r1,
    c0:c1,
]

Y_plot = grid_y[
    r0:r1,
    c0:c1,
]

Z_plot = grid_z[
    r0:r1,
    c0:c1,
]


# ============================================================
# 6. Nature / Science style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 7. Plotting helpers
# ============================================================

def style_log_axis(ax):
    ax.set_yscale("log")

    ax.yaxis.set_major_locator(
        LogLocator(
            base=10.0,
            numticks=12,
        )
    )

    ax.yaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(2, 10) * 0.1,
            numticks=100,
        )
    )

    ax.yaxis.set_minor_formatter(
        NullFormatter()
    )


def style_nature_axis(ax):
    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.tick_params(
        which="both",
        top=False,
        right=False,
    )

    ax.grid(
        True,
        which="major",
        axis="y",
        linestyle="-",
        linewidth=0.6,
        alpha=0.18,
        color="#666666",
    )


def plot_convergence_curve(
    ax,
    x,
    y,
    color,
    label,
    marker,
):
    y_plot = safe_positive(
        y
    )

    markevery = max(
        1,
        len(x) // 10,
    )

    ax.plot(
        x,
        y_plot,
        color=color,
        linestyle="-",
        linewidth=2.5,
        marker=marker,
        markevery=markevery,
        markersize=6.5,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.6,
        solid_capstyle="round",
        label=label,
        zorder=5,
        clip_on=False,
    )

    # Highlight the final point.
    ax.scatter(
        x[-1],
        y_plot[-1],
        s=75,
        marker=marker,
        facecolor=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=10,
        clip_on=False,
    )


# ============================================================
# 8. Create the three-panel figure
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15.5, 4.8),
)

ax_mep = axes[0]
ax_mfep = axes[1]
ax_path = axes[2]


# ------------------------------------------------------------
# Panel (a): Euclidean MEP residual
# ------------------------------------------------------------

plot_convergence_curve(
    ax=ax_mep,
    x=count_seq,
    y=I_MEP_seq,
    color=COLOR_MEP,
    label=r"Euclidean MEP",
    marker="o",
)

ax_mep.set_xlabel(
    "Restrained-MD / gradient count",
    labelpad=6,
)

ax_mep.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,\mathrm{E}}$",
    labelpad=7,
)

ax_mep.set_title(
    "Euclidean MEP residual",
    pad=12,
    fontweight="medium",
)

ax_mep.set_xlim(
    left=0,
)

style_log_axis(
    ax_mep
)

style_nature_axis(
    ax_mep
)

ax_mep.text(
    -0.16,
    1.06,
    "a",
    transform=ax_mep.transAxes,
    fontsize=17,
    fontweight="bold",
    va="top",
    ha="left",
)


# ------------------------------------------------------------
# Panel (b): MFEP criterion evaluated on the Euclidean MEP
# ------------------------------------------------------------

plot_convergence_curve(
    ax=ax_mfep,
    x=count_seq,
    y=I_MFEP_seq,
    color=COLOR_MFEP_TEST,
    label=r"MFEP test on MEP",
    marker="s",
)

ax_mfep.set_xlabel(
    "Restrained-MD / gradient count",
    labelpad=6,
)

ax_mfep.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,G}$",
    labelpad=7,
)

ax_mfep.set_title(
    r"MFEP residual on the MEP",
    pad=12,
    fontweight="medium",
)

ax_mfep.set_xlim(
    left=0,
)

style_log_axis(
    ax_mfep
)

style_nature_axis(
    ax_mfep
)

ax_mfep.text(
    -0.16,
    1.06,
    "b",
    transform=ax_mfep.transAxes,
    fontsize=17,
    fontweight="bold",
    va="top",
    ha="left",
)


# ------------------------------------------------------------
# Panel (c): Initial path and final Euclidean MEP
# ------------------------------------------------------------

z_min = float(
    np.ma.min(
        Z_plot
    )
)

z_max = float(
    np.ma.max(
        Z_plot
    )
)

if z_max > 40.0:
    norm = mpl.colors.TwoSlopeNorm(
        vmin=z_min,
        vcenter=40.0,
        vmax=z_max,
    )

    contour = ax_path.contourf(
        X_plot,
        Y_plot,
        Z_plot,
        levels=50,
        cmap=mpl.cm.jet,
        norm=norm,
    )
else:
    contour = ax_path.contourf(
        X_plot,
        Y_plot,
        Z_plot,
        levels=50,
        cmap=mpl.cm.jet,
    )

if initial_path_full is not None:
    ax_path.plot(
        initial_path_full[:, 0],
        initial_path_full[:, 1],
        color=COLOR_INITIAL,
        linestyle="--",
        linewidth=1.8,
        marker="s",
        markersize=4.8,
        markerfacecolor="white",
        markeredgecolor=COLOR_INITIAL,
        markeredgewidth=1.1,
        alpha=0.9,
        label="Initialization",
        zorder=5,
    )

ax_path.plot(
    final_path_full[:, 0],
    final_path_full[:, 1],
    color=COLOR_FINAL_PATH,
    linestyle="-",
    linewidth=2.5,
    marker="o",
    markersize=5.7,
    markerfacecolor="white",
    markeredgecolor=COLOR_FINAL_PATH,
    markeredgewidth=1.5,
    label=r"Euclidean MEP ($G=I$)",
    zorder=8,
)

# Endpoint markers.
ax_path.scatter(
    final_path_full[0, 0],
    final_path_full[0, 1],
    s=120,
    marker="*",
    facecolor="#009E73",
    edgecolor="white",
    linewidth=1.0,
    zorder=12,
)

ax_path.scatter(
    final_path_full[-1, 0],
    final_path_full[-1, 1],
    s=120,
    marker="*",
    facecolor="#CC79A7",
    edgecolor="white",
    linewidth=1.0,
    zorder=12,
)

# Reference conformers.
special_points = [
    (
        -2.62593029,
        2.77365591,
        r"$C5$",
    ),
    (
        -1.33433281,
        0.99268968,
        r"$C7_{\mathrm{eq}}$",
    ),
]

for x_value, y_value, label in special_points:
    ax_path.plot(
        x_value,
        y_value,
        marker="x",
        color="white",
        markersize=9,
        markeredgewidth=2.2,
        linestyle="None",
        zorder=15,
    )

    ax_path.text(
        x_value - 0.43,
        y_value - 0.32,
        label,
        color="white",
        fontsize=15,
        fontweight="bold",
        zorder=15,
    )

ax_path.set_xlabel(
    r"$\phi$ (rad)",
    labelpad=6,
)

ax_path.set_ylabel(
    r"$\psi$ (rad)",
    labelpad=6,
)

ax_path.set_title(
    "Euclidean minimum-energy path",
    pad=12,
    fontweight="medium",
)

ax_path.set_xlim(
    XLIM
)

ax_path.set_ylim(
    YLIM
)

ax_path.set_aspect(
    "equal",
    adjustable="box",
)

ax_path.tick_params(
    which="both",
    top=False,
    right=False,
)

ax_path.legend(
    loc="best",
    fontsize=10.5,
    handlelength=2.0,
)

ax_path.text(
    -0.16,
    1.06,
    "c",
    transform=ax_path.transAxes,
    fontsize=17,
    fontweight="bold",
    va="top",
    ha="left",
)

# Colorbar.
divider = make_axes_locatable(
    ax_path
)

cax = divider.append_axes(
    "right",
    size="4.5%",
    pad=0.10,
)

colorbar = fig.colorbar(
    contour,
    cax=cax,
)

colorbar.set_label(
    r"$F$ (kcal mol$^{-1}$)",
    fontsize=12,
)


# ============================================================
# 9. Layout and save
# ============================================================

fig.subplots_adjust(
    left=0.072,
    right=0.965,
    top=0.86,
    bottom=0.16,
    wspace=0.32,
)

fig.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 10. Final-value summary
# ============================================================

print()
print("=" * 82)
print("Euclidean MEP result summary")
print("=" * 82)
print(f"Result file                    : {TARGET_CONFIG['npz']}")
print(f"Final stored outer iteration   : {final_iteration}")
print(f"Final gradient count           : {count_seq[-1]:.1f}")
print(f"Final Euclidean MEP I_perp     : {I_MEP_seq[-1]:.8e}")
print(f"Minimum Euclidean MEP I_perp   : {np.nanmin(I_MEP_seq):.8e}")
print(f"Final MFEP-test I_perp^G       : {I_MFEP_seq[-1]:.8e}")
print(f"Minimum MFEP-test I_perp^G     : {np.nanmin(I_MFEP_seq):.8e}")

if fmax_MEP_seq is not None:
    print(
        f"Final Euclidean MEP fmax       : "
        f"{fmax_MEP_seq[-1]:.8e}"
    )

if fmax_MFEP_seq is not None:
    print(
        f"Final MFEP-test fmax           : "
        f"{fmax_MFEP_seq[-1]:.8e}"
    )

if I_MEP_mean_seq is not None:
    print(
        f"Final mean Euclidean residual  : "
        f"{I_MEP_mean_seq[-1]:.8e}"
    )

if I_MFEP_mean_seq is not None:
    print(
        f"Final mean MFEP vector residual: "
        f"{I_MFEP_mean_seq[-1]:.8e}"
    )

# Ratio provides a direct indication of whether the Euclidean MEP
# also satisfies the physical MFEP condition.
final_ratio = (
    I_MFEP_seq[-1]
    / max(
        I_MEP_seq[-1],
        np.finfo(float).tiny,
    )
)

print(f"Final I_perp^G / I_perp ratio  : {final_ratio:.8e}")
print("=" * 82)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# MFEP-IVM vs Euclidean-MEP-IVM
#
# 一行四张：
#   (a) MFEP residual I_perp^G
#   (b) Euclidean MEP residual I_perp^E
#   (c) Free-energy barrier Delta F^dagger
#   (d) Final MFEP and Euclidean MEP on one grayscale contour
#
# 两种方法使用统一公式重新计算全部指标，避免不同 NPZ 键名
# 或旧版本诊断定义造成不一致。
# ============================================================

import os
import glob
import re
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, NullFormatter
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# 0. User configuration
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# ------------------------------------------------------------
# Experiment parameters
# ------------------------------------------------------------

N_IMAGES = 20
HISTORY_SIZE = 7

# MFEP-IVM configuration: optimization metric G=M^{-1}
MFEP_N_SUB = 1
MFEP_ALPHA = 0.0001
MFEP_MAXSTEP = 0.1

# Euclidean MEP-IVM configuration: optimization metric G=I
MEP_N_SUB = 1
MEP_ALPHA = 0.03
MEP_MAXSTEP = 0.1

# If True, the horizontal axis is total restrained-MD calls:
#
#     count = (outer iteration + 1) * N_IMAGES * N_SUB
#
# If False, use outer iteration.
USE_MD_CALL_COUNT = True


# ------------------------------------------------------------
# Plotting window and endpoints
# ------------------------------------------------------------

XLIM = (-np.pi, 0.0)
YLIM = (0.0, np.pi)

C1 = np.asarray(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.asarray(
    [-1.33047160, 0.99118795],
    dtype=float,
)

SPECIAL_POINTS = [
    (
        -2.62593029,
        2.77365591,
        r"$C5$",
        -0.05,
        0.16,
    ),
    (
        -1.33433281,
        0.99268968,
        r"$C7_{\mathrm{eq}}$",
        0.10,
        -0.40,
    ),
]


# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------

# Color-blind-friendly Wong palette.
COLOR_MFEP = "#0072B2"       # Blue
COLOR_MEP = "#D50087"        # Vermilion

MARKER_MFEP = "o"
MARKER_MEP = "s"


# ------------------------------------------------------------
# Black-to-white contour settings
# ------------------------------------------------------------

# Free energy at and above this value is white.
ENERGY_WHITE_CAP = 60.0

# Gray value at the minimum:
#
# 0.0 = pure black
# 0.05 = nearly black
BG_LO = 0.05

# Quadratic gray mapping:
#
# CURV = 0: linear
# CURV > 0: low-energy region changes faster from black to gray
CURV = 2.0

CONTOUR_LEVELS = 33


# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        f"MFEP_vs_Euclidean_MEP"
        f"_alpha{MEP_ALPHA}"
        f"_Nsub{MEP_N_SUB}"
        f"_maxstep{MEP_MAXSTEP}"
        f"_metrics_and_contour"
    ),
)


# ============================================================
# 1. Directory discovery helpers
# ============================================================

def parse_float_after_key(text, key):
    """
    Parse a floating-point value occurring immediately after key.

    Example
    -------
    text = "..._alpha0.05_maxstep0.1_..."
    key  = "_alpha"
    """
    pattern = (
        rf"{re.escape(key)}"
        rf"([-+]?(?:\d+(?:\.\d*)?|\.\d+)"
        rf"(?:[eE][-+]?\d+)?)"
    )

    match = re.search(
        pattern,
        text,
    )

    if match is None:
        return None

    try:
        return float(
            match.group(1)
        )
    except ValueError:
        return None


def parse_int_after_key(text, key):
    """
    Parse an integer occurring immediately after key.
    """
    pattern = (
        rf"{re.escape(key)}"
        rf"(\d+)"
    )

    match = re.search(
        pattern,
        text,
    )

    if match is None:
        return None

    try:
        return int(
            match.group(1)
        )
    except ValueError:
        return None


def discover_ivm_configs(prefix):
    """
    Discover IVM result folders.

    Parameters
    ----------
    prefix : str
        For example:
            "MFEP_image_LBFGS_B_Recycle_N"
            "MEP_image_LBFGS_B_Recycle_N"
    """
    pattern = os.path.join(
        BASE_PATH,
        prefix + "*",
    )

    configs = []

    for folder in sorted(
        glob.glob(pattern)
    ):
        if not os.path.isdir(folder):
            continue

        name = os.path.basename(
            folder
        )

        n_images = parse_int_after_key(
            name,
            "_N",
        )

        n_sub = parse_int_after_key(
            name,
            "_Nsub",
        )

        alpha = parse_float_after_key(
            name,
            "_alpha",
        )

        maxstep = parse_float_after_key(
            name,
            "_maxstep",
        )

        history = parse_int_after_key(
            name,
            "_hist",
        )

        npz_candidates = sorted(
            glob.glob(
                os.path.join(
                    folder,
                    "alanine_*.npz",
                )
            )
        )

        if not npz_candidates:
            continue

        configs.append(
            {
                "folder": folder,
                "name": name,
                "npz": npz_candidates[-1],
                "N": n_images,
                "N_sub": n_sub,
                "alpha": alpha,
                "maxstep": maxstep,
                "history": history,
            }
        )

    return configs


def find_target_config(
    prefix,
    n_images,
    n_sub,
    alpha,
    maxstep,
    history,
    label,
):
    """
    Find the matching result folder.
    """
    configs = discover_ivm_configs(
        prefix
    )

    matches = []

    for cfg in configs:
        if cfg["N"] != n_images:
            continue

        if cfg["N_sub"] != n_sub:
            continue

        if cfg["history"] != history:
            continue

        if (
            cfg["alpha"] is None
            or not np.isclose(
                cfg["alpha"],
                alpha,
                rtol=1e-8,
                atol=1e-14,
            )
        ):
            continue

        if (
            cfg["maxstep"] is None
            or not np.isclose(
                cfg["maxstep"],
                maxstep,
                rtol=1e-8,
                atol=1e-14,
            )
        ):
            continue

        matches.append(
            cfg
        )

    if not matches:
        available = "\n".join(
            f"  - {cfg['name']}"
            for cfg in configs
        )

        raise RuntimeError(
            f"未找到指定的 {label} 结果：\n"
            f"  prefix={prefix}\n"
            f"  N={n_images}\n"
            f"  N_sub={n_sub}\n"
            f"  alpha={alpha}\n"
            f"  maxstep={maxstep}\n"
            f"  history={history}\n\n"
            f"发现的 {label} 目录：\n"
            f"{available if available else '  (none)'}"
        )

    if len(matches) > 1:
        print(
            f"[WARNING] {label} 找到 {len(matches)} 个匹配目录，"
            f"使用最后一个：\n"
            f"  {matches[-1]['folder']}"
        )

    return matches[-1]


MFEP_CONFIG = find_target_config(
    prefix="MFEP_image_LBFGS_B_Recycle_N",
    n_images=N_IMAGES,
    n_sub=MFEP_N_SUB,
    alpha=MFEP_ALPHA,
    maxstep=MFEP_MAXSTEP,
    history=HISTORY_SIZE,
    label="MFEP-IVM",
)

MEP_CONFIG = find_target_config(
    prefix="MEP_image_LBFGS_B_Recycle_N",
    n_images=N_IMAGES,
    n_sub=MEP_N_SUB,
    alpha=MEP_ALPHA,
    maxstep=MEP_MAXSTEP,
    history=HISTORY_SIZE,
    label="Euclidean MEP-IVM",
)


print("=" * 88)
print("Selected configurations")
print("=" * 88)
print("MFEP-IVM")
print(f"  folder : {MFEP_CONFIG['folder']}")
print(f"  npz    : {MFEP_CONFIG['npz']}")
print(f"  alpha  : {MFEP_CONFIG['alpha']}")
print(f"  N_sub  : {MFEP_CONFIG['N_sub']}")
print(f"  maxstep: {MFEP_CONFIG['maxstep']}")
print()
print("Euclidean MEP-IVM")
print(f"  folder : {MEP_CONFIG['folder']}")
print(f"  npz    : {MEP_CONFIG['npz']}")
print(f"  alpha  : {MEP_CONFIG['alpha']}")
print(f"  N_sub  : {MEP_CONFIG['N_sub']}")
print(f"  maxstep: {MEP_CONFIG['maxstep']}")
print("=" * 88)


# ============================================================
# 2. Numerical geometry helpers
# ============================================================

def euclidean_normal_projector(tangent, eps=1e-16):
    """
    Euclidean normal projector:

        P_E = I - tau_hat tau_hat^T.
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    tangent_norm = np.linalg.norm(
        tangent
    )

    if tangent_norm <= eps:
        return np.eye(
            tangent.shape[0],
            dtype=float,
        )

    tau = (
        tangent
        / tangent_norm
    )

    return (
        np.eye(
            tangent.shape[0],
            dtype=float,
        )
        - np.outer(
            tau,
            tau,
        )
    )


def G_normal_projector(
    tangent,
    G,
    eps=1e-16,
):
    """
    G-orthogonal normal projector:

        P_G v
        =
        v
        -
        tau
        (tau^T G v) / (tau^T G tau).

    Matrix form:

        P_G
        =
        I
        -
        tau (tau^T G) / (tau^T G tau).
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    G = np.asarray(
        G,
        dtype=float,
    )

    denom = float(
        tangent @ G @ tangent
    )

    if denom <= eps:
        return np.eye(
            tangent.shape[0],
            dtype=float,
        )

    return (
        np.eye(
            tangent.shape[0],
            dtype=float,
        )
        - np.outer(
            tangent,
            tangent @ G,
        )
        / denom
    )


def central_tangents_from_full_path(
    path_full,
):
    """
    Central-difference tangents at the internal images.
    """
    path_full = np.asarray(
        path_full,
        dtype=float,
    )

    return (
        path_full[2:]
        - path_full[:-2]
    )


def path_with_endpoints(
    path_internal,
):
    """
    Add fixed endpoints to an internal-image path.
    """
    path_internal = np.asarray(
        path_internal,
        dtype=float,
    )

    return np.vstack(
        [
            C1,
            path_internal,
            C2,
        ]
    )


def compute_euclidean_node_weights(
    path_full,
):
    """
    Euclidean arclength quadrature weights:

        w_i^E
        =
        1/2 (
            |x_i-x_{i-1}|
            +
            |x_{i+1}-x_i|
        ).
    """
    internal = path_full[1:-1]

    seg_minus = (
        internal
        - path_full[:-2]
    )

    seg_plus = (
        path_full[2:]
        - internal
    )

    return 0.5 * (
        np.linalg.norm(
            seg_minus,
            axis=1,
        )
        +
        np.linalg.norm(
            seg_plus,
            axis=1,
        )
    )


def compute_G_node_weights(
    path_full,
    G_all,
):
    """
    G-arclength quadrature weights.
    """
    internal = path_full[1:-1]

    seg_minus = (
        internal
        - path_full[:-2]
    )

    seg_plus = (
        path_full[2:]
        - internal
    )

    seg_minus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    seg_plus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    seg_minus_G = np.sqrt(
        np.maximum(
            seg_minus_G_sq,
            0.0,
        )
    )

    seg_plus_G = np.sqrt(
        np.maximum(
            seg_plus_G_sq,
            0.0,
        )
    )

    return 0.5 * (
        seg_minus_G
        + seg_plus_G
    )


def compute_MEP_residual(
    path_internal,
    grad_F,
):
    """
    Euclidean MEP residual:

        I_perp^E
        =
        sum_i
        |P_E grad F|^2
        w_i^E.
    """
    path_full = path_with_endpoints(
        path_internal
    )

    tangents = central_tangents_from_full_path(
        path_full
    )

    residual_vectors = np.zeros_like(
        grad_F
    )

    for j in range(
        len(path_internal)
    ):
        P_E = euclidean_normal_projector(
            tangents[j]
        )

        residual_vectors[j] = (
            P_E
            @ grad_F[j]
        )

    residual_sq = np.sum(
        residual_vectors ** 2,
        axis=1,
    )

    weights = compute_euclidean_node_weights(
        path_full
    )

    I_E = float(
        np.sum(
            residual_sq
            * weights
        )
    )

    return {
        "value": I_E,
        "vectors": residual_vectors,
        "pointwise_sq": residual_sq,
        "weights": weights,
    }


def compute_MFEP_residual(
    path_internal,
    grad_F,
    M_all,
):
    """
    MFEP residual:

        G = M^{-1},

        R_G
        =
        P_G M grad F,

        I_perp^G
        =
        sum_i
        R_G^T G R_G
        w_i^G.
    """
    path_full = path_with_endpoints(
        path_internal
    )

    tangents = central_tangents_from_full_path(
        path_full
    )

    G_all = np.linalg.inv(
        M_all
    )

    Mg = np.einsum(
        "nij,nj->ni",
        M_all,
        grad_F,
    )

    residual_vectors = np.zeros_like(
        Mg
    )

    for j in range(
        len(path_internal)
    ):
        P_G = G_normal_projector(
            tangents[j],
            G_all[j],
        )

        residual_vectors[j] = (
            P_G
            @ Mg[j]
        )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        residual_vectors,
        G_all,
        residual_vectors,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights = compute_G_node_weights(
        path_full,
        G_all,
    )

    I_G = float(
        np.sum(
            residual_G_sq
            * weights
        )
    )

    return {
        "value": I_G,
        "vectors": residual_vectors,
        "pointwise_G_sq": residual_G_sq,
        "weights": weights,
        "G": G_all,
    }


def recover_free_energy_barrier(
    path_internal,
    grad_F,
):
    """
    Recover a relative free-energy profile along the internal path
    by integrating

        dF/ds = grad F · phi'(s).

    The same discretization is used for MFEP and MEP, so the
    barriers are directly comparable.

    Returns
    -------
    barrier : float
        max(F)-min(F), in kcal/mol.

    profile : ndarray
        Relative integrated free-energy profile at the internal images.
    """
    path_internal = np.asarray(
        path_internal,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    n = path_internal.shape[0]

    if n < 2:
        return np.nan, np.full(
            n,
            np.nan,
        )

    s = np.linspace(
        0.0,
        1.0,
        n,
    )

    ds = (
        1.0
        / (n - 1)
    )

    phi_prime = np.zeros_like(
        path_internal
    )

    if n > 2:
        phi_prime[1:-1] = (
            path_internal[2:]
            - path_internal[:-2]
        ) / (
            2.0 * ds
        )

    phi_prime[0] = (
        path_internal[1]
        - path_internal[0]
    ) / ds

    phi_prime[-1] = (
        path_internal[-1]
        - path_internal[-2]
    ) / ds

    dFds = np.sum(
        grad_F
        * phi_prime,
        axis=1,
    )

    profile = np.zeros(
        n,
        dtype=float,
    )

    for j in range(1, n):
        delta_s = (
            s[j]
            - s[j - 1]
        )

        profile[j] = (
            profile[j - 1]
            + 0.5
            * (
                dFds[j - 1]
                + dFds[j]
            )
            * delta_s
        )

    profile = (
        profile
        - np.nanmin(profile)
    )

    barrier = float(
        np.nanmax(profile)
        - np.nanmin(profile)
    )

    return (
        barrier,
        profile,
    )


# ============================================================
# 3. NPZ loading and uniform metric recomputation
# ============================================================

def extract_iteration_numbers(
    loaded,
    prefix,
):
    """
    Extract iteration numbers from keys such as:

        phi_ref_iter_0
        phi_ref_iter_1
        ...
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    values = []

    for key in loaded.files:
        match = pattern.match(
            key
        )

        if match is not None:
            values.append(
                int(
                    match.group(1)
                )
            )

    return sorted(
        values
    )


def load_and_recompute_metrics(
    npz_path,
    n_sub,
    method_name,
):
    """
    Load one IVM run and recompute:

        I_perp^G,
        I_perp^E,
        Delta F^dagger

    using exactly the same formulas for both methods.
    """
    with np.load(
        npz_path,
        allow_pickle=True,
    ) as loaded:

        iterations = extract_iteration_numbers(
            loaded,
            "phi_ref_iter_",
        )

        if not iterations:
            raise RuntimeError(
                f"{method_name}: NPZ 中没有 phi_ref_iter_* 数据：\n"
                f"  {npz_path}"
            )

        usable_iterations = []

        I_G_values = []
        I_E_values = []
        barrier_values = []

        paths = []
        profiles = []

        for iteration in iterations:
            path_key = (
                f"phi_ref_iter_{iteration}"
            )

            grad_key_candidates = [
                f"g_ref_iter_{iteration}",
                f"F_ref_iter_{iteration}",
            ]

            metric_key = (
                f"M_ref_iter_{iteration}"
            )

            grad_key = None

            for candidate in grad_key_candidates:
                if candidate in loaded.files:
                    grad_key = candidate
                    break

            if grad_key is None:
                continue

            if metric_key not in loaded.files:
                continue

            path_internal = np.asarray(
                loaded[path_key],
                dtype=float,
            )

            grad_F = np.asarray(
                loaded[grad_key],
                dtype=float,
            )

            M_all = np.asarray(
                loaded[metric_key],
                dtype=float,
            )

            if (
                path_internal.ndim != 2
                or path_internal.shape[1] != 2
            ):
                continue

            if grad_F.shape != path_internal.shape:
                continue

            if M_all.shape != (
                path_internal.shape[0],
                2,
                2,
            ):
                continue

            try:
                mep_result = compute_MEP_residual(
                    path_internal,
                    grad_F,
                )

                mfep_result = compute_MFEP_residual(
                    path_internal,
                    grad_F,
                    M_all,
                )

                barrier, profile = recover_free_energy_barrier(
                    path_internal,
                    grad_F,
                )

            except np.linalg.LinAlgError:
                print(
                    f"[WARNING] {method_name}: "
                    f"iteration {iteration} 的 M 不可逆，跳过。"
                )
                continue

            usable_iterations.append(
                iteration
            )

            I_E_values.append(
                mep_result["value"]
            )

            I_G_values.append(
                mfep_result["value"]
            )

            barrier_values.append(
                barrier
            )

            paths.append(
                path_internal.copy()
            )

            profiles.append(
                profile.copy()
            )

        if not usable_iterations:
            raise RuntimeError(
                f"{method_name}: 未找到同时具有 path、gradient、M "
                f"的有效 iteration。"
            )

        usable_iterations = np.asarray(
            usable_iterations,
            dtype=float,
        )

        if (
            USE_MD_CALL_COUNT
            and "count_seq" in loaded.files
        ):
            stored_count = np.asarray(
                loaded["count_seq"],
                dtype=float,
            )

            stored_index = np.asarray(
                loaded["index_seq"],
                dtype=int,
            )

            count_map = {
                int(k): float(v)
                for k, v in zip(
                    stored_index,
                    stored_count,
                )
            }

            x_values = np.asarray(
                [
                    count_map.get(
                        int(k),
                        (
                            int(k) + 1
                        )
                        * N_IMAGES
                        * n_sub,
                    )
                    for k in usable_iterations
                ],
                dtype=float,
            )

        elif USE_MD_CALL_COUNT:
            # Every outer iteration evaluates N_IMAGES reference
            # points. Each additional inner gradient substep also
            # evaluates N_IMAGES points.
            x_values = (
                usable_iterations + 1.0
            ) * float(
                N_IMAGES * n_sub
            )

        else:
            x_values = usable_iterations.copy()

        # Final path after the last outer update/reparameterization,
        # if available. This is used only in the contour panel.
        path_iter_numbers = extract_iteration_numbers(
            loaded,
            "path_iter_",
        )

        if path_iter_numbers:
            final_path_iteration = (
                path_iter_numbers[-1]
            )

            final_path_internal = np.asarray(
                loaded[
                    f"path_iter_{final_path_iteration}"
                ],
                dtype=float,
            )
        else:
            final_path_iteration = int(
                usable_iterations[-1]
            )

            final_path_internal = paths[-1].copy()

        initial_path_internal = paths[0].copy()

    return {
        "name": method_name,
        "npz": npz_path,
        "iteration": usable_iterations,
        "count": x_values,
        "I_G": np.asarray(
            I_G_values,
            dtype=float,
        ),
        "I_E": np.asarray(
            I_E_values,
            dtype=float,
        ),
        "barrier": np.asarray(
            barrier_values,
            dtype=float,
        ),
        "paths": paths,
        "profiles": profiles,
        "initial_path_internal": initial_path_internal,
        "final_path_internal": final_path_internal,
        "final_path_full": path_with_endpoints(
            final_path_internal
        ),
        "final_path_iteration": final_path_iteration,
    }


MFEP_RESULT = load_and_recompute_metrics(
    npz_path=MFEP_CONFIG["npz"],
    n_sub=MFEP_N_SUB,
    method_name="MFEP-IVM",
)

MEP_RESULT = load_and_recompute_metrics(
    npz_path=MEP_CONFIG["npz"],
    n_sub=MEP_N_SUB,
    method_name="Euclidean MEP-IVM",
)


# ============================================================
# 4. Load and crop free-energy surface
# ============================================================

if not os.path.exists(
    FES_NPZ
):
    raise FileNotFoundError(
        f"Cannot find FES file:\n"
        f"  {FES_NPZ}"
    )

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as loaded:
    gx = np.asarray(
        loaded["X"],
        dtype=float,
    )

    gy = np.asarray(
        loaded["Y"],
        dtype=float,
    )

    gz = np.asarray(
        loaded["Z"],
        dtype=float,
    )

if (
    gx.ndim == 1
    and gy.ndim == 1
):
    gx, gy = np.meshgrid(
        gx,
        gy,
        indexing="xy",
    )

if (
    gx.shape != gy.shape
    or gx.shape != gz.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{gx.shape}, "
        f"Y{gy.shape}, "
        f"Z{gz.shape}"
    )

# Stored FES coordinates are assumed to be in degrees.
gx = np.deg2rad(
    gx
)

gy = np.deg2rad(
    gy
)

gz = np.ma.masked_invalid(
    gz
)

inside = (
    (gx >= XLIM[0])
    & (gx <= XLIM[1])
    & (gy >= YLIM[0])
    & (gy <= YLIM[1])
)

if not np.any(
    inside
):
    raise RuntimeError(
        f"No FES data inside "
        f"XLIM={XLIM}, YLIM={YLIM}."
    )

rows = np.where(
    np.any(
        inside,
        axis=1,
    )
)[0]

cols = np.where(
    np.any(
        inside,
        axis=0,
    )
)[0]

r0 = rows[0]
r1 = rows[-1] + 1
c0 = cols[0]
c1 = cols[-1] + 1

Xc = gx[
    r0:r1,
    c0:c1,
]

Yc = gy[
    r0:r1,
    c0:c1,
]

Zc = gz[
    r0:r1,
    c0:c1,
]

# Set the minimum in the displayed window to zero.
Zc = (
    Zc
    - np.ma.min(Zc)
)


# ============================================================
# 5. Tuned black-to-white colormap
# ============================================================

_u = np.linspace(
    0.0,
    1.0,
    256,
)

# Same quadratic mapping as the tuned contour code.
_t = (
    _u
    + CURV
    * _u
    * (1.0 - _u)
)

_t = np.clip(
    _t,
    0.0,
    1.0,
)

_gray = (
    BG_LO
    + (1.0 - BG_LO)
    * _t
)

BG_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad",
    list(
        zip(
            _gray,
            _gray,
            _gray,
        )
    ),
)

BG_CMAP.set_over(
    "white"
)

levels = np.linspace(
    0.0,
    ENERGY_WHITE_CAP,
    CONTOUR_LEVELS,
)


# ============================================================
# 6. Nature / Science style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 7. Plot helpers
# ============================================================

tiny = np.finfo(
    float
).tiny


def safe_positive(values):
    """
    Make an array safe for logarithmic plotting.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    finite_positive = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if finite_positive.size > 0:
        floor = max(
            tiny,
            np.min(
                finite_positive
            )
            * 1e-8,
        )
    else:
        floor = tiny

    return np.where(
        np.isfinite(values)
        & (values > 0.0),
        values,
        floor,
    )


def style_log_axis(ax):
    ax.set_yscale(
        "log"
    )

    ax.yaxis.set_major_locator(
        LogLocator(
            base=10.0,
            numticks=12,
        )
    )

    ax.yaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(
                2,
                10,
            )
            * 0.1,
            numticks=100,
        )
    )

    ax.yaxis.set_minor_formatter(
        NullFormatter()
    )


def style_metric_axis(
    ax,
    log_scale,
):
    if log_scale:
        style_log_axis(
            ax
        )
    else:
        ax.minorticks_on()

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.tick_params(
        which="both",
        top=False,
        right=False,
    )

    ax.grid(
        True,
        which="major",
        axis="y",
        linestyle="-",
        linewidth=0.6,
        alpha=0.18,
        color="#666666",
    )


def plot_metric_curve(
    ax,
    result,
    key,
    color,
    marker,
    label,
    log_scale,
    zorder,
):
    x = np.asarray(
        result["count"],
        dtype=float,
    )

    y = np.asarray(
        result[key],
        dtype=float,
    )

    n_common = min(
        len(x),
        len(y),
    )

    x = x[:n_common]
    y = y[:n_common]

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if x.size == 0:
        return

    order = np.argsort(
        x
    )

    x = x[order]
    y = y[order]

    if log_scale:
        y_plot = safe_positive(
            y
        )
    else:
        y_plot = y

    markevery = max(
        1,
        len(x) // 10,
    )

    ax.plot(
        x,
        y_plot,
        color=color,
        linestyle="-",
        linewidth=2.4,
        marker=marker,
        markevery=markevery,
        markersize=6.2,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.5,
        solid_capstyle="round",
        label=label,
        zorder=zorder,
        clip_on=False,
    )

    ax.scatter(
        x[-1],
        y_plot[-1],
        s=70,
        marker=marker,
        facecolor=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=zorder + 5,
        clip_on=False,
    )


# ============================================================
# 8. Plot: three metrics + one contour
# ============================================================

fig, axes = plt.subplots(
    1,
    4,
    figsize=(19.2, 4.8),
)

ax_IG = axes[0]
ax_IE = axes[1]
ax_barrier = axes[2]
ax_contour = axes[3]

panel_tags = [
    "a",
    "b",
    "c",
    "d",
]


# ------------------------------------------------------------
# Panel (a): MFEP residual I_perp^G
# ------------------------------------------------------------

plot_metric_curve(
    ax=ax_IG,
    result=MFEP_RESULT,
    key="I_G",
    color=COLOR_MFEP,
    marker=MARKER_MFEP,
    label=r"MFEP-IVM ($G=M^{-1}$)",
    log_scale=True,
    zorder=6,
)

plot_metric_curve(
    ax=ax_IG,
    result=MEP_RESULT,
    key="I_G",
    color=COLOR_MEP,
    marker=MARKER_MEP,
    label=r"MEP-IVM ($G=I$)",
    log_scale=True,
    zorder=5,
)

ax_IG.set_xlabel(
    (
        "Restrained-MD / gradient count"
        if USE_MD_CALL_COUNT
        else "Outer iteration"
    ),
    labelpad=6,
)

ax_IG.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,G}$",
    labelpad=7,
)

ax_IG.set_title(
    "MFEP normal residual",
    fontsize=13,
    pad=12,
    fontweight="medium",
)

ax_IG.set_xlim(
    left=0,
)

style_metric_axis(
    ax_IG,
    log_scale=True,
)


# ------------------------------------------------------------
# Panel (b): Euclidean MEP residual I_perp^E
# ------------------------------------------------------------

plot_metric_curve(
    ax=ax_IE,
    result=MFEP_RESULT,
    key="I_E",
    color=COLOR_MFEP,
    marker=MARKER_MFEP,
    label=r"MFEP-IVM ($G=M^{-1}$)",
    log_scale=True,
    zorder=6,
)

plot_metric_curve(
    ax=ax_IE,
    result=MEP_RESULT,
    key="I_E",
    color=COLOR_MEP,
    marker=MARKER_MEP,
    label=r"MEP-IVM ($G=I$)",
    log_scale=True,
    zorder=5,
)

ax_IE.set_xlabel(
    (
        "Restrained-MD / gradient count"
        if USE_MD_CALL_COUNT
        else "Outer iteration"
    ),
    labelpad=6,
)

ax_IE.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,\mathrm{E}}$",
    labelpad=7,
)

ax_IE.set_title(
    "Euclidean MEP residual",
    fontsize=13,
    pad=12,
    fontweight="medium",
)

ax_IE.set_xlim(
    left=0,
)

style_metric_axis(
    ax_IE,
    log_scale=True,
)


# ------------------------------------------------------------
# Panel (c): free-energy barrier
# ------------------------------------------------------------

plot_metric_curve(
    ax=ax_barrier,
    result=MFEP_RESULT,
    key="barrier",
    color=COLOR_MFEP,
    marker=MARKER_MFEP,
    label=r"MFEP-IVM ($G=M^{-1}$)",
    log_scale=False,
    zorder=6,
)

plot_metric_curve(
    ax=ax_barrier,
    result=MEP_RESULT,
    key="barrier",
    color=COLOR_MEP,
    marker=MARKER_MEP,
    label=r"MEP-IVM ($G=I$)",
    log_scale=False,
    zorder=5,
)

ax_barrier.set_xlabel(
    (
        "Restrained-MD / gradient count"
        if USE_MD_CALL_COUNT
        else "Outer iteration"
    ),
    labelpad=6,
)

ax_barrier.set_ylabel(
    r"$\Delta F^{\dagger}$ (kcal mol$^{-1}$)",
    labelpad=7,
)

ax_barrier.set_title(
    "Free-energy barrier",
    fontsize=13,
    pad=12,
    fontweight="medium",
)

ax_barrier.set_xlim(
    left=0,
)

style_metric_axis(
    ax_barrier,
    log_scale=False,
)


# ------------------------------------------------------------
# Panel (d): one tuned black-to-white contour
# ------------------------------------------------------------

contour = ax_contour.contourf(
    Xc,
    Yc,
    Zc,
    levels=levels,
    cmap=BG_CMAP,
    extend="max",
)

# Metastable-state labels.
for (
    xx,
    yy,
    label,
    dx,
    dy,
) in SPECIAL_POINTS:
    ax_contour.plot(
        xx,
        yy,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="#111111",
        markeredgewidth=1.4,
        linestyle="None",
        zorder=12,
    )

    ax_contour.text(
        xx + dx,
        yy + dy,
        label,
        color="#111111",
        fontsize=12,
        fontweight="bold",
        zorder=12,
        ha="right",
        va="bottom",
        bbox={
            "boxstyle": "round,pad=0.12",
            "fc": "white",
            "ec": "none",
            "alpha": 0.75,
        },
    )

# Final MFEP.
P_MFEP = MFEP_RESULT[
    "final_path_full"
]

ax_contour.plot(
    P_MFEP[:, 0],
    P_MFEP[:, 1],
    color=COLOR_MFEP,
    linestyle="-",
    linewidth=2.4,
    marker=MARKER_MFEP,
    markevery=1,
    markersize=5.2,
    markerfacecolor="white",
    markeredgecolor=COLOR_MFEP,
    markeredgewidth=1.4,
    solid_capstyle="round",
    label=r"MFEP-IVM ($G=M^{-1}$)",
    zorder=8,
)

# Final Euclidean MEP.
P_MEP = MEP_RESULT[
    "final_path_full"
]

ax_contour.plot(
    P_MEP[:, 0],
    P_MEP[:, 1],
    color=COLOR_MEP,
    linestyle="-",
    linewidth=2.4,
    marker=MARKER_MEP,
    markevery=1,
    markersize=5.2,
    markerfacecolor="white",
    markeredgecolor=COLOR_MEP,
    markeredgewidth=1.4,
    solid_capstyle="round",
    label=r"MEP-IVM ($G=I$)",
    zorder=7,
)

ax_contour.set_xlabel(
    r"$\phi$ (rad)",
    labelpad=5,
)

ax_contour.set_ylabel(
    r"$\psi$ (rad)",
    labelpad=5,
)

ax_contour.set_title(
    "MFEP versus Euclidean MEP",
    fontsize=13,
    pad=12,
    fontweight="medium",
)

ax_contour.set_xlim(
    XLIM
)

ax_contour.set_ylim(
    YLIM
)

ax_contour.set_aspect(
    "equal",
    adjustable="box",
)

ax_contour.tick_params(
    which="both",
    top=False,
    right=False,
)


# ------------------------------------------------------------
# Panel labels
# ------------------------------------------------------------

for ax, tag in zip(
    axes,
    panel_tags,
):
    ax.text(
        -0.15 if ax is not ax_contour else -0.10,
        1.07,
        tag,
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left",
    )


# ============================================================
# 9. Shared legend and colorbar
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=COLOR_MFEP,
        linewidth=2.4,
        marker=MARKER_MFEP,
        markerfacecolor="white",
        markeredgecolor=COLOR_MFEP,
        markeredgewidth=1.4,
        label=r"MFEP-IVM ($G=M^{-1}$)",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_MEP,
        linewidth=2.4,
        marker=MARKER_MEP,
        markerfacecolor="white",
        markeredgecolor=COLOR_MEP,
        markeredgewidth=1.4,
        label=r"MEP-IVM ($G=I$)",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.035,
    ),
    ncol=2,
    fontsize=13,
    handlelength=2.3,
    columnspacing=2.6,
    borderaxespad=0.0,
)

# Reserve room for colorbar.
fig.subplots_adjust(
    left=0.055,
    right=0.925,
    top=0.82,
    bottom=0.15,
    wspace=0.30,
)

# Dedicated colorbar axis, aligned with the contour panel.
contour_position = ax_contour.get_position()

cax = fig.add_axes(
    [
        contour_position.x1 + 0.012,
        contour_position.y0,
        0.012,
        contour_position.height,
    ]
)

colorbar = fig.colorbar(
    contour,
    cax=cax,
    extend="max",
)

colorbar.set_label(
    r"Free energy (kcal mol$^{-1}$)",
    labelpad=8,
)


# ============================================================
# 10. Save
# ============================================================

fig.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 11. Final-value summary
# ============================================================

print()
print("=" * 100)
print("Final comparison: MFEP-IVM vs Euclidean MEP-IVM")
print("=" * 100)

for result in [
    MFEP_RESULT,
    MEP_RESULT,
]:
    print(
        f"{result['name']:>22} | "
        f"count={result['count'][-1]:>8.1f} | "
        f"I_G={result['I_G'][-1]:.6e} | "
        f"I_E={result['I_E'][-1]:.6e} | "
        f"DeltaF={result['barrier'][-1]:.6f} kcal/mol"
    )

print("-" * 100)

final_path_difference = np.linalg.norm(
    P_MFEP
    - P_MEP,
    axis=1,
)

print(
    f"Mean pointwise MFEP-MEP distance : "
    f"{np.mean(final_path_difference):.6e} rad"
)

print(
    f"Maximum MFEP-MEP distance        : "
    f"{np.max(final_path_difference):.6e} rad"
)

print("-" * 100)
print(f"MFEP result: {MFEP_CONFIG['npz']}")
print(f"MEP result : {MEP_CONFIG['npz']}")
print("=" * 100)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")

In [16]:
# -*- coding: utf-8 -*-
# ============================================================
# MFEP-IVM vs CV Euclidean-MEP-IVM vs projected 66D all-atom MEP
#
# Panels:
#   (a) MFEP residual I_perp^G
#   (b) Euclidean MEP residual I_perp^E
#   (c) Free-energy barrier Delta F^dagger
#   (d) Final three paths on one tuned black-to-white FES contour
#
# The projected 66D MEP is re-evaluated by restrained MD at each
# projected internal image to obtain:
#
#     grad F(phi, psi), M(phi, psi).
#
# A persistent cache is saved after each completed all-atom path
# state, so the restrained-MD postprocessing can resume.
# ============================================================

import os
import sys
import glob
import re
import tempfile
import numpy as np

import torch

import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, NullFormatter
from matplotlib.colors import LinearSegmentedColormap

from openmm.app import PDBFile


# ============================================================
# 0. User configuration
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# Correct 66D all-atom MEP result supplied by the user.
ALL_ATOM_MEP_NPZ = ("/Users/wuzhiyou/Downloads/Ala2_N20_lr0.04_alphaIVP100_Nsub5_CVSpacingTangentODE_results.npz")
#     "/Users/wuzhiyou/Downloads/"
#     "Ala2_N20_lr0.04_alphaIVP100_Nsub5_results.npz"
# )

# ------------------------------------------------------------
# IVM_shell.py
#
# The module must define:
#
#     M_gradient(q, discard_frac, tmin, tmax)
#
# and should end with:
#
#     if __name__ == "__main__":
#         main()
# ------------------------------------------------------------

IVM_SHELL_DIR = BASE_PATH

if IVM_SHELL_DIR not in sys.path:
    sys.path.insert(
        0,
        IVM_SHELL_DIR,
    )

import IVM_shell as ala_ivm

M_gradient = ala_ivm.M_gradient

if hasattr(
    ala_ivm,
    "GROMACS_QUIET",
):
    ala_ivm.GROMACS_QUIET = True


# ------------------------------------------------------------
# Alanine-dipeptide PDB
#
# The atom ordering in this PDB must be identical to the ordering
# used in the 66D Cartesian images.
# ------------------------------------------------------------

ALA2_PDB_CANDIDATES = [
    os.path.join(
        BASE_PATH,
        "ala2.pdb",
    ),
    os.path.abspath(
        "./ala2.pdb"
    ),
]

ALA2_PDB = None

for candidate in ALA2_PDB_CANDIDATES:
    if os.path.exists(candidate):
        ALA2_PDB = candidate
        break

if ALA2_PDB is None:
    raise FileNotFoundError(
        "Cannot find ala2.pdb. Checked:\n"
        + "\n".join(
            f"  {path}"
            for path in ALA2_PDB_CANDIDATES
        )
    )


# ------------------------------------------------------------
# CV experiment parameters
# ------------------------------------------------------------

N_IMAGES = 20
HISTORY_SIZE = 7

# MFEP-IVM: G = M^{-1}
MFEP_N_SUB = 1
MFEP_ALPHA = 1e-4
MFEP_MAXSTEP = 0.1

# CV Euclidean MEP-IVM: G = I
MEP_N_SUB = 1
MEP_ALPHA = 0.03
MEP_MAXSTEP = 0.1


# ------------------------------------------------------------
# Restrained-MD settings for projected 66D MEP
# ------------------------------------------------------------

AA_DISCARD_FRAC = 0.05
AA_TMIN = None
AA_TMAX = None

# Evaluate every saved all-atom path.
#
# Set to 2, 5, 10, ... to evaluate fewer saved states.
AA_STATE_STRIDE = 1

# Optionally limit the total number of evaluated all-atom states.
#
# None: evaluate all selected states.
AA_MAX_STATES = None

# The supplied all-atom code stores:
#
#     count_seq.append(i * N_sub)
#
# which is a per-image count. Therefore multiply by 20 to obtain
# total Cartesian force calls.
AA_COUNT_IS_PER_IMAGE = True

# Persistent cache for projected paths and restrained-MD results.
AA_CACHE_NPZ = os.path.join(
    BASE_PATH,
    "Ala2_66D_projected_MEP_restrained_MD_cache.npz",
)

# Force recomputation of the all-atom restrained-MD cache.
AA_FORCE_RECOMPUTE = False


# ------------------------------------------------------------
# Plot x axis
#
# Each method has a different oracle:
#
# CV methods:
#     restrained-MD calls.
#
# 66D method:
#     Cartesian potential-energy force calls.
#
# Hence the common axis is called method-specific oracle calls.
# ------------------------------------------------------------

USE_ORACLE_CALL_COUNT = True


# ------------------------------------------------------------
# CV endpoints and plot window
# ------------------------------------------------------------

C1 = np.asarray(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.asarray(
    [-1.33047160, 0.99118795],
    dtype=float,
)

XLIM = (
    -np.pi,
    0.0,
)

YLIM = (
    0.0,
    np.pi,
)

SPECIAL_POINTS = [
    (
        -2.62593029,
        2.77365591,
        r"$C5$",
        -0.05,
        0.16,
    ),
    (
        -1.33433281,
        0.99268968,
        r"$C7_{\mathrm{eq}}$",
        0.10,
        -0.40,
    ),
]


# ------------------------------------------------------------
# Colors and markers
# ------------------------------------------------------------

COLOR_MFEP = "#0072B2"
COLOR_MEP = "#D50087"
COLOR_AA_MEP = "#009E73"

MARKER_MFEP = "o"
MARKER_MEP = "s"
MARKER_AA_MEP = "^"


# ------------------------------------------------------------
# Tuned black-to-white contour
# ------------------------------------------------------------

ENERGY_WHITE_CAP = 60.0
BG_LO = 0.05
CURV = 2.0
CONTOUR_LEVELS = 33


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        "MFEP_vs_CV_MEP_vs_projected_66D_MEP"
        "_metrics_and_contour"
    ),
)


# ============================================================
# 1. Locate CV-space MFEP/MEP result files
# ============================================================

def parse_float_after_key(
    text,
    key,
):
    pattern = (
        rf"{re.escape(key)}"
        rf"([-+]?(?:\d+(?:\.\d*)?|\.\d+)"
        rf"(?:[eE][-+]?\d+)?)"
    )

    match = re.search(
        pattern,
        text,
    )

    if match is None:
        return None

    try:
        return float(
            match.group(1)
        )
    except ValueError:
        return None


def parse_int_after_key(
    text,
    key,
):
    pattern = (
        rf"{re.escape(key)}"
        rf"(\d+)"
    )

    match = re.search(
        pattern,
        text,
    )

    if match is None:
        return None

    try:
        return int(
            match.group(1)
        )
    except ValueError:
        return None


def discover_ivm_configs(
    prefix,
):
    pattern = os.path.join(
        BASE_PATH,
        prefix + "*",
    )

    configs = []

    for folder in sorted(
        glob.glob(pattern)
    ):
        if not os.path.isdir(folder):
            continue

        name = os.path.basename(
            folder
        )

        npz_candidates = sorted(
            glob.glob(
                os.path.join(
                    folder,
                    "alanine_*.npz",
                )
            )
        )

        if not npz_candidates:
            continue

        configs.append(
            {
                "folder": folder,
                "name": name,
                "npz": npz_candidates[-1],
                "N": parse_int_after_key(
                    name,
                    "_N",
                ),
                "N_sub": parse_int_after_key(
                    name,
                    "_Nsub",
                ),
                "alpha": parse_float_after_key(
                    name,
                    "_alpha",
                ),
                "maxstep": parse_float_after_key(
                    name,
                    "_maxstep",
                ),
                "history": parse_int_after_key(
                    name,
                    "_hist",
                ),
            }
        )

    return configs


def find_target_config(
    prefix,
    n_images,
    n_sub,
    alpha,
    maxstep,
    history,
    label,
):
    configs = discover_ivm_configs(
        prefix
    )

    matches = []

    for cfg in configs:
        if cfg["N"] != n_images:
            continue

        if cfg["N_sub"] != n_sub:
            continue

        if cfg["history"] != history:
            continue

        if (
            cfg["alpha"] is None
            or not np.isclose(
                cfg["alpha"],
                alpha,
                rtol=1e-8,
                atol=1e-14,
            )
        ):
            continue

        if (
            cfg["maxstep"] is None
            or not np.isclose(
                cfg["maxstep"],
                maxstep,
                rtol=1e-8,
                atol=1e-14,
            )
        ):
            continue

        matches.append(
            cfg
        )

    if not matches:
        available = "\n".join(
            f"  - {cfg['name']}"
            for cfg in configs
        )

        raise RuntimeError(
            f"Cannot find {label} result:\n"
            f"  prefix={prefix}\n"
            f"  N={n_images}\n"
            f"  N_sub={n_sub}\n"
            f"  alpha={alpha}\n"
            f"  maxstep={maxstep}\n"
            f"  history={history}\n\n"
            f"Available directories:\n"
            f"{available if available else '  (none)'}"
        )

    if len(matches) > 1:
        print(
            f"[WARNING] Found {len(matches)} matching {label} "
            f"directories. Using:\n"
            f"  {matches[-1]['folder']}"
        )

    return matches[-1]


MFEP_CONFIG = find_target_config(
    prefix="MFEP_image_LBFGS_B_Recycle_N",
    n_images=N_IMAGES,
    n_sub=MFEP_N_SUB,
    alpha=MFEP_ALPHA,
    maxstep=MFEP_MAXSTEP,
    history=HISTORY_SIZE,
    label="MFEP-IVM",
)

MEP_CONFIG = find_target_config(
    prefix="MEP_image_LBFGS_B_Recycle_N",
    n_images=N_IMAGES,
    n_sub=MEP_N_SUB,
    alpha=MEP_ALPHA,
    maxstep=MEP_MAXSTEP,
    history=HISTORY_SIZE,
    label="CV Euclidean MEP-IVM",
)


# ============================================================
# 2. Dihedral projection of the 66D Cartesian path
# ============================================================

def get_atom_name(
    atom,
):
    return str(
        atom.name
    ).strip().upper()


def get_residue_name(
    residue,
):
    return str(
        residue.name
    ).strip().upper()


def identify_alanine_phi_psi_indices(
    pdb_path,
):
    """
    Identify the standard alanine-dipeptide torsions:

        phi:
            C(previous) - N(ALA) - CA(ALA) - C(ALA)

        psi:
            N(ALA) - CA(ALA) - C(ALA) - N(next)

    Returns zero-based atom indices.
    """
    pdb = PDBFile(
        pdb_path
    )

    topology = pdb.topology

    residues = list(
        topology.residues()
    )

    ala_candidates = [
        i
        for i, residue in enumerate(residues)
        if get_residue_name(residue) == "ALA"
    ]

    if not ala_candidates:
        raise RuntimeError(
            "No ALA residue was found in the PDB."
        )

    # Alanine dipeptide should contain one central ALA.
    ala_residue_position = ala_candidates[0]

    if (
        ala_residue_position == 0
        or ala_residue_position
        == len(residues) - 1
    ):
        raise RuntimeError(
            "The ALA residue has no previous or next residue."
        )

    previous_residue = residues[
        ala_residue_position - 1
    ]

    ala_residue = residues[
        ala_residue_position
    ]

    next_residue = residues[
        ala_residue_position + 1
    ]

    def residue_atom_map(
        residue,
    ):
        return {
            get_atom_name(atom): int(atom.index)
            for atom in residue.atoms()
        }

    previous_atoms = residue_atom_map(
        previous_residue
    )

    ala_atoms = residue_atom_map(
        ala_residue
    )

    next_atoms = residue_atom_map(
        next_residue
    )

    required_ala = [
        "N",
        "CA",
        "C",
    ]

    for atom_name in required_ala:
        if atom_name not in ala_atoms:
            raise RuntimeError(
                f"Cannot find ALA atom {atom_name}."
            )

    if "C" not in previous_atoms:
        raise RuntimeError(
            "Cannot find the previous-residue C atom."
        )

    if "N" not in next_atoms:
        raise RuntimeError(
            "Cannot find the next-residue N atom."
        )

    phi_indices = (
        previous_atoms["C"],
        ala_atoms["N"],
        ala_atoms["CA"],
        ala_atoms["C"],
    )

    psi_indices = (
        ala_atoms["N"],
        ala_atoms["CA"],
        ala_atoms["C"],
        next_atoms["N"],
    )

    n_atoms = topology.getNumAtoms()

    print("=" * 82)
    print("Alanine dihedral definition")
    print("=" * 82)
    print(f"PDB             : {pdb_path}")
    print(f"Number of atoms : {n_atoms}")
    print(
        f"Previous residue: "
        f"{get_residue_name(previous_residue)}"
    )
    print(
        f"ALA residue     : "
        f"{get_residue_name(ala_residue)}"
    )
    print(
        f"Next residue    : "
        f"{get_residue_name(next_residue)}"
    )
    print(f"phi indices     : {phi_indices}")
    print(f"psi indices     : {psi_indices}")
    print("=" * 82)

    return (
        phi_indices,
        psi_indices,
        n_atoms,
    )


PHI_INDICES, PSI_INDICES, PDB_N_ATOMS = (
    identify_alanine_phi_psi_indices(
        ALA2_PDB
    )
)


def torsion_angle(
    positions,
    indices,
):
    """
    Compute a torsion angle in radians in [-pi, pi].
    """
    positions = np.asarray(
        positions,
        dtype=float,
    )

    i0, i1, i2, i3 = indices

    p0 = positions[i0]
    p1 = positions[i1]
    p2 = positions[i2]
    p3 = positions[i3]

    b0 = (
        p0
        - p1
    )

    b1 = (
        p2
        - p1
    )

    b2 = (
        p3
        - p2
    )

    b1_norm = np.linalg.norm(
        b1
    )

    if b1_norm <= 1e-16:
        return np.nan

    b1_unit = (
        b1
        / b1_norm
    )

    v = (
        b0
        - np.dot(
            b0,
            b1_unit,
        )
        * b1_unit
    )

    w = (
        b2
        - np.dot(
            b2,
            b1_unit,
        )
        * b1_unit
    )

    x = np.dot(
        v,
        w,
    )

    y = np.dot(
        np.cross(
            b1_unit,
            v,
        ),
        w,
    )

    return float(
        np.arctan2(
            y,
            x,
        )
    )


def wrap_angles(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        (values + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )


def torus_distance(
    a,
    b,
):
    delta = wrap_angles(
        np.asarray(a)
        - np.asarray(b)
    )

    return float(
        np.linalg.norm(delta)
    )


def project_cartesian_path_to_cv(
    cartesian_path,
):
    """
    Project one Cartesian path to (phi, psi).

    Input accepted:
        (n_images, 3*N_atoms)
        (n_images, N_atoms, 3)
    """
    cartesian_path = np.asarray(
        cartesian_path,
        dtype=float,
    )

    if cartesian_path.ndim == 2:
        if cartesian_path.shape[1] % 3 != 0:
            raise ValueError(
                f"Cartesian dimension "
                f"{cartesian_path.shape[1]} is not divisible by 3."
            )

        n_atoms = (
            cartesian_path.shape[1]
            // 3
        )

        xyz = cartesian_path.reshape(
            cartesian_path.shape[0],
            n_atoms,
            3,
        )

    elif (
        cartesian_path.ndim == 3
        and cartesian_path.shape[2] == 3
    ):
        xyz = cartesian_path

        n_atoms = xyz.shape[1]

    else:
        raise ValueError(
            f"Unexpected Cartesian path shape: "
            f"{cartesian_path.shape}"
        )

    if n_atoms != PDB_N_ATOMS:
        raise ValueError(
            "Atom-count mismatch between Cartesian result and PDB:\n"
            f"  Cartesian result: {n_atoms}\n"
            f"  PDB             : {PDB_N_ATOMS}\n"
            "The PDB atom order must match the all-atom MEP data."
        )

    cv_path = np.zeros(
        (
            xyz.shape[0],
            2,
        ),
        dtype=float,
    )

    for image_index in range(
        xyz.shape[0]
    ):
        cv_path[
            image_index,
            0,
        ] = torsion_angle(
            xyz[image_index],
            PHI_INDICES,
        )

        cv_path[
            image_index,
            1,
        ] = torsion_angle(
            xyz[image_index],
            PSI_INDICES,
        )

    if not np.all(
        np.isfinite(cv_path)
    ):
        raise FloatingPointError(
            "Non-finite projected dihedral values were obtained."
        )

    return cv_path


def orient_and_unwrap_cv_path(
    cv_path,
):
    """
    Orient the projected path from C1 to C2 and unwrap it continuously.
    """
    cv_path = wrap_angles(
        cv_path
    )

    forward_cost = (
        torus_distance(
            cv_path[0],
            C1,
        )
        +
        torus_distance(
            cv_path[-1],
            C2,
        )
    )

    reverse_cost = (
        torus_distance(
            cv_path[-1],
            C1,
        )
        +
        torus_distance(
            cv_path[0],
            C2,
        )
    )

    if reverse_cost < forward_cost:
        cv_path = cv_path[::-1].copy()

    # Unwrap along the reaction coordinate.
    cv_unwrapped = np.unwrap(
        cv_path,
        axis=0,
    )

    # Shift each coordinate by a common multiple of 2*pi so the
    # first endpoint is closest to C1.
    for dim in range(2):
        shift_count = np.round(
            (
                C1[dim]
                - cv_unwrapped[0, dim]
            )
            / (
                2.0 * np.pi
            )
        )

        cv_unwrapped[:, dim] += (
            shift_count
            * 2.0
            * np.pi
        )

    return cv_unwrapped


def shift_path_to_plot_window(
    path,
):
    """
    Shift an unwrapped path by common 2*pi multiples for display.
    """
    path = np.asarray(
        path,
        dtype=float,
    ).copy()

    centers = np.asarray(
        [
            0.5 * (
                XLIM[0]
                + XLIM[1]
            ),
            0.5 * (
                YLIM[0]
                + YLIM[1]
            ),
        ]
    )

    for dim in range(2):
        mean_value = float(
            np.mean(
                path[:, dim]
            )
        )

        shift_count = np.round(
            (
                centers[dim]
                - mean_value
            )
            / (
                2.0 * np.pi
            )
        )

        path[:, dim] += (
            shift_count
            * 2.0
            * np.pi
        )

    return path


# ============================================================
# 3. Common path-metric definitions
# ============================================================

def euclidean_normal_projector(
    tangent,
    eps=1e-16,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    norm = np.linalg.norm(
        tangent
    )

    if norm <= eps:
        return np.eye(
            tangent.shape[0],
            dtype=float,
        )

    tau = (
        tangent
        / norm
    )

    return (
        np.eye(
            tangent.shape[0],
            dtype=float,
        )
        - np.outer(
            tau,
            tau,
        )
    )


def G_normal_projector(
    tangent,
    G,
    eps=1e-16,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    G = np.asarray(
        G,
        dtype=float,
    )

    denominator = float(
        tangent
        @ G
        @ tangent
    )

    if denominator <= eps:
        return np.eye(
            tangent.shape[0],
            dtype=float,
        )

    return (
        np.eye(
            tangent.shape[0],
            dtype=float,
        )
        - np.outer(
            tangent,
            tangent @ G,
        )
        / denominator
    )


def central_tangents_from_full_path(
    full_path,
):
    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_euclidean_node_weights(
    full_path,
):
    internal = full_path[
        1:-1
    ]

    segment_minus = (
        internal
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal
    )

    return 0.5 * (
        np.linalg.norm(
            segment_minus,
            axis=1,
        )
        +
        np.linalg.norm(
            segment_plus,
            axis=1,
        )
    )


def compute_G_node_weights(
    full_path,
    G_all,
):
    internal = full_path[
        1:-1
    ]

    segment_minus = (
        internal
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal
    )

    minus_sq = np.einsum(
        "ni,nij,nj->n",
        segment_minus,
        G_all,
        segment_minus,
    )

    plus_sq = np.einsum(
        "ni,nij,nj->n",
        segment_plus,
        G_all,
        segment_plus,
    )

    return 0.5 * (
        np.sqrt(
            np.maximum(
                minus_sq,
                0.0,
            )
        )
        +
        np.sqrt(
            np.maximum(
                plus_sq,
                0.0,
            )
        )
    )


def compute_MEP_residual(
    full_path,
    grad_F,
):
    """
    Euclidean free-energy MEP residual:

        I_perp^E
        =
        sum_i |P_E grad F|^2 w_i^E.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual = np.zeros_like(
        grad_F
    )

    for image_index in range(
        n_internal
    ):
        projector = euclidean_normal_projector(
            tangents[
                image_index
            ]
        )

        residual[
            image_index
        ] = (
            projector
            @ grad_F[
                image_index
            ]
        )

    residual_sq = np.sum(
        residual ** 2,
        axis=1,
    )

    weights = compute_euclidean_node_weights(
        full_path
    )

    return {
        "value": float(
            np.sum(
                residual_sq
                * weights
            )
        ),
        "vectors": residual,
        "pointwise_sq": residual_sq,
        "weights": weights,
    }


def compute_MFEP_residual(
    full_path,
    grad_F,
    M_all,
):
    """
    MFEP residual with G=M^{-1}:

        R_G = P_G M grad F,

        I_perp^G
        =
        sum_i R_G^T G R_G w_i^G.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"M shape {M_all.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    G_all = np.linalg.inv(
        M_all
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        grad_F,
    )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual = np.zeros_like(
        Mg_all
    )

    for image_index in range(
        n_internal
    ):
        projector = G_normal_projector(
            tangents[
                image_index
            ],
            G_all[
                image_index
            ],
        )

        residual[
            image_index
        ] = (
            projector
            @ Mg_all[
                image_index
            ]
        )

    residual_G_sq = np.einsum(
        "ni,nij,nj->n",
        residual,
        G_all,
        residual,
    )

    residual_G_sq = np.maximum(
        residual_G_sq,
        0.0,
    )

    weights = compute_G_node_weights(
        full_path,
        G_all,
    )

    return {
        "value": float(
            np.sum(
                residual_G_sq
                * weights
            )
        ),
        "vectors": residual,
        "pointwise_G_sq": residual_G_sq,
        "weights": weights,
        "G": G_all,
    }


def recover_free_energy_profile(
    full_path,
    grad_F,
):
    """
    Integrate grad F along the complete path.

    Since gradients are available only at internal images, the
    first and last internal gradients are used on the endpoint
    segments.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (
            n_internal + 1,
            2,
        ),
        dtype=float,
    )

    segment_gradients[0] = (
        grad_F[0]
    )

    if n_internal > 1:
        segment_gradients[
            1:n_internal
        ] = (
            0.5
            * (
                grad_F[:-1]
                + grad_F[1:]
            )
        )

    segment_gradients[
        n_internal
    ] = grad_F[-1]

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= np.min(
        profile
    )

    barrier = float(
        np.max(profile)
        - np.min(profile)
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "increments": increments,
    }


def compute_all_metrics(
    full_path,
    grad_F,
    M_all,
):
    mep = compute_MEP_residual(
        full_path,
        grad_F,
    )

    mfep = compute_MFEP_residual(
        full_path,
        grad_F,
        M_all,
    )

    free_energy = recover_free_energy_profile(
        full_path,
        grad_F,
    )

    return {
        "I_E": mep["value"],
        "I_G": mfep["value"],
        "barrier": free_energy[
            "barrier"
        ],
        "free_energy_profile": free_energy[
            "profile"
        ],
        "MEP_vectors": mep[
            "vectors"
        ],
        "MFEP_vectors": mfep[
            "vectors"
        ],
    }


# ============================================================
# 4. Load CV MFEP/MEP and recompute common metrics
# ============================================================

def extract_iteration_numbers(
    loaded,
    prefix,
):
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    values = []

    for key in loaded.files:
        match = pattern.match(
            key
        )

        if match is not None:
            values.append(
                int(
                    match.group(1)
                )
            )

    return sorted(
        values
    )


def path_with_fixed_endpoints(
    internal_path,
):
    return np.vstack(
        [
            C1,
            internal_path,
            C2,
        ]
    )


def load_cv_result(
    npz_path,
    n_sub,
    method_name,
):
    with np.load(
        npz_path,
        allow_pickle=True,
    ) as loaded:

        iterations = extract_iteration_numbers(
            loaded,
            "phi_ref_iter_",
        )

        if not iterations:
            raise RuntimeError(
                f"{method_name}: no phi_ref_iter_* keys in:\n"
                f"  {npz_path}"
            )

        used_iterations = []
        counts = []
        I_E_values = []
        I_G_values = []
        barrier_values = []
        reference_paths = []

        if (
            "count_seq" in loaded.files
            and "index_seq" in loaded.files
        ):
            stored_count = np.asarray(
                loaded["count_seq"],
                dtype=float,
            )

            stored_index = np.asarray(
                loaded["index_seq"],
                dtype=int,
            )

            count_map = {
                int(index): float(count)
                for index, count in zip(
                    stored_index,
                    stored_count,
                )
            }

        else:
            count_map = {}

        for iteration in iterations:
            path_key = (
                f"phi_ref_iter_{iteration}"
            )

            metric_key = (
                f"M_ref_iter_{iteration}"
            )

            gradient_key = None

            for candidate in [
                f"g_ref_iter_{iteration}",
                f"F_ref_iter_{iteration}",
            ]:
                if candidate in loaded.files:
                    gradient_key = candidate
                    break

            if (
                metric_key not in loaded.files
                or gradient_key is None
            ):
                continue

            internal_path = np.asarray(
                loaded[path_key],
                dtype=float,
            )

            grad_F = np.asarray(
                loaded[gradient_key],
                dtype=float,
            )

            M_all = np.asarray(
                loaded[metric_key],
                dtype=float,
            )

            if grad_F.shape != internal_path.shape:
                continue

            if M_all.shape != (
                internal_path.shape[0],
                2,
                2,
            ):
                continue

            full_path = path_with_fixed_endpoints(
                internal_path
            )

            try:
                metrics = compute_all_metrics(
                    full_path,
                    grad_F,
                    M_all,
                )
            except np.linalg.LinAlgError:
                print(
                    f"[WARNING] {method_name}: singular M "
                    f"at iteration {iteration}; skipping."
                )
                continue

            used_iterations.append(
                iteration
            )

            reference_paths.append(
                full_path.copy()
            )

            I_E_values.append(
                metrics["I_E"]
            )

            I_G_values.append(
                metrics["I_G"]
            )

            barrier_values.append(
                metrics["barrier"]
            )

            if USE_ORACLE_CALL_COUNT:
                counts.append(
                    count_map.get(
                        iteration,
                        (
                            iteration + 1
                        )
                        * N_IMAGES
                        * n_sub,
                    )
                )
            else:
                counts.append(
                    iteration
                )

        if not used_iterations:
            raise RuntimeError(
                f"No usable states in {npz_path}"
            )

        path_iterations = extract_iteration_numbers(
            loaded,
            "path_iter_",
        )

        if path_iterations:
            final_path_internal = np.asarray(
                loaded[
                    f"path_iter_{path_iterations[-1]}"
                ],
                dtype=float,
            )

            final_path_full = path_with_fixed_endpoints(
                final_path_internal
            )
        else:
            final_path_full = reference_paths[-1].copy()

    return {
        "name": method_name,
        "npz": npz_path,
        "iteration": np.asarray(
            used_iterations,
            dtype=float,
        ),
        "count": np.asarray(
            counts,
            dtype=float,
        ),
        "I_E": np.asarray(
            I_E_values,
            dtype=float,
        ),
        "I_G": np.asarray(
            I_G_values,
            dtype=float,
        ),
        "barrier": np.asarray(
            barrier_values,
            dtype=float,
        ),
        "paths": reference_paths,
        "final_path_full": final_path_full,
    }


MFEP_RESULT = load_cv_result(
    MFEP_CONFIG["npz"],
    MFEP_N_SUB,
    "MFEP-IVM",
)

MEP_RESULT = load_cv_result(
    MEP_CONFIG["npz"],
    MEP_N_SUB,
    "CV Euclidean MEP-IVM",
)


# ============================================================
# 5. Load 66D Cartesian path sequence
# ============================================================

def load_all_atom_path_sequence(
    result_npz,
):
    if not os.path.exists(
        result_npz
    ):
        raise FileNotFoundError(
            f"Cannot find all-atom result:\n"
            f"  {result_npz}"
        )

    with np.load(
        result_npz,
        allow_pickle=True,
    ) as loaded:
        print("=" * 82)
        print("All-atom result keys")
        print("=" * 82)

        for key in loaded.files:
            array = np.asarray(
                loaded[key]
            )

            print(
                f"{key:28s} "
                f"shape={array.shape} "
                f"dtype={array.dtype}"
            )

        print("=" * 82)

        if "images_seq" not in loaded.files:
            raise KeyError(
                "The all-atom NPZ has no images_seq key."
            )

        images_seq = np.asarray(
            loaded["images_seq"],
            dtype=float,
        )

        if images_seq.ndim not in (
            3,
            4,
        ):
            raise ValueError(
                f"Unexpected images_seq shape: "
                f"{images_seq.shape}"
            )

        n_states = images_seq.shape[0]

        if "count_seq" in loaded.files:
            count_seq = np.asarray(
                loaded["count_seq"],
                dtype=float,
            ).reshape(-1)
        else:
            count_seq = np.arange(
                n_states,
                dtype=float,
            )

    n_common = min(
        n_states,
        count_seq.size,
    )

    images_seq = images_seq[
        :n_common
    ]

    count_seq = count_seq[
        :n_common
    ]

    # Select states.
    selected_indices = np.arange(
        0,
        n_common,
        max(
            1,
            int(AA_STATE_STRIDE),
        ),
        dtype=int,
    )

    if (
        selected_indices.size > 0
        and selected_indices[-1]
        != n_common - 1
    ):
        selected_indices = np.append(
            selected_indices,
            n_common - 1,
        )

    if (
        AA_MAX_STATES is not None
        and selected_indices.size
        > AA_MAX_STATES
    ):
        # Evenly select states while retaining the final state.
        choose = np.linspace(
            0,
            selected_indices.size - 1,
            AA_MAX_STATES,
        )

        choose = np.unique(
            np.round(
                choose
            ).astype(int)
        )

        selected_indices = selected_indices[
            choose
        ]

    selected_images = images_seq[
        selected_indices
    ]

    selected_counts = count_seq[
        selected_indices
    ]

    if AA_COUNT_IS_PER_IMAGE:
        selected_counts = (
            selected_counts
            * N_IMAGES
        )

    return {
        "images": selected_images,
        "source_indices": selected_indices,
        "count": selected_counts,
        "n_total_states": n_common,
    }


AA_RAW = load_all_atom_path_sequence(
    ALL_ATOM_MEP_NPZ
)


# ============================================================
# 6. Restrained-MD evaluation of projected 66D paths
# ============================================================

def atomic_savez(
    path,
    **arrays,
):
    directory = os.path.dirname(
        path
    )

    os.makedirs(
        directory,
        exist_ok=True,
    )

    file_descriptor, temporary_path = tempfile.mkstemp(
        prefix="tmp_aa_cache_",
        suffix=".npz",
        dir=directory,
    )

    os.close(
        file_descriptor
    )

    try:
        np.savez(
            temporary_path,
            **arrays,
        )

        os.replace(
            temporary_path,
            path,
        )

    finally:
        if os.path.exists(
            temporary_path
        ):
            os.remove(
                temporary_path
            )


def load_aa_cache(
    path,
):
    if (
        AA_FORCE_RECOMPUTE
        or not os.path.exists(path)
    ):
        return {
            "source_indices": [],
            "count": [],
            "paths": [],
            "grad_F": [],
            "M": [],
            "I_E": [],
            "I_G": [],
            "barrier": [],
            "profiles": [],
        }

    with np.load(
        path,
        allow_pickle=True,
    ) as loaded:
        required = [
            "source_indices",
            "count",
            "paths",
            "grad_F",
            "M",
            "I_E",
            "I_G",
            "barrier",
            "profiles",
        ]

        for key in required:
            if key not in loaded.files:
                print(
                    f"[WARNING] Cache lacks {key}; "
                    "starting from an empty cache."
                )

                return {
                    "source_indices": [],
                    "count": [],
                    "paths": [],
                    "grad_F": [],
                    "M": [],
                    "I_E": [],
                    "I_G": [],
                    "barrier": [],
                    "profiles": [],
                }

        return {
            "source_indices": list(
                np.asarray(
                    loaded["source_indices"],
                    dtype=int,
                )
            ),
            "count": list(
                np.asarray(
                    loaded["count"],
                    dtype=float,
                )
            ),
            "paths": list(
                np.asarray(
                    loaded["paths"],
                    dtype=float,
                )
            ),
            "grad_F": list(
                np.asarray(
                    loaded["grad_F"],
                    dtype=float,
                )
            ),
            "M": list(
                np.asarray(
                    loaded["M"],
                    dtype=float,
                )
            ),
            "I_E": list(
                np.asarray(
                    loaded["I_E"],
                    dtype=float,
                )
            ),
            "I_G": list(
                np.asarray(
                    loaded["I_G"],
                    dtype=float,
                )
            ),
            "barrier": list(
                np.asarray(
                    loaded["barrier"],
                    dtype=float,
                )
            ),
            "profiles": list(
                np.asarray(
                    loaded["profiles"],
                    dtype=float,
                )
            ),
        }


def save_aa_cache(
    cache,
):
    atomic_savez(
        AA_CACHE_NPZ,
        source_indices=np.asarray(
            cache["source_indices"],
            dtype=int,
        ),
        count=np.asarray(
            cache["count"],
            dtype=float,
        ),
        paths=np.asarray(
            cache["paths"],
            dtype=float,
        ),
        grad_F=np.asarray(
            cache["grad_F"],
            dtype=float,
        ),
        M=np.asarray(
            cache["M"],
            dtype=float,
        ),
        I_E=np.asarray(
            cache["I_E"],
            dtype=float,
        ),
        I_G=np.asarray(
            cache["I_G"],
            dtype=float,
        ),
        barrier=np.asarray(
            cache["barrier"],
            dtype=float,
        ),
        profiles=np.asarray(
            cache["profiles"],
            dtype=float,
        ),
        source_npz=np.asarray(
            ALL_ATOM_MEP_NPZ,
        ),
        pdb_path=np.asarray(
            ALA2_PDB,
        ),
        discard_frac=np.asarray(
            AA_DISCARD_FRAC,
        ),
    )


def evaluate_restrained_md_on_cv_path(
    full_cv_path,
):
    """
    Run restrained MD on all internal projected images.

    The path used for geometry is continuous/unwrapped, while each
    restrained-MD target is wrapped to [-pi, pi).
    """
    full_cv_path = np.asarray(
        full_cv_path,
        dtype=float,
    )

    internal_path = full_cv_path[
        1:-1
    ]

    n_internal = internal_path.shape[0]

    grad_F = np.zeros(
        (
            n_internal,
            2,
        ),
        dtype=float,
    )

    M_all = np.zeros(
        (
            n_internal,
            2,
            2,
        ),
        dtype=float,
    )

    for image_index in range(
        n_internal
    ):
        target = wrap_angles(
            internal_path[
                image_index
            ]
        )

        q_tensor = torch.tensor(
            target.reshape(
                1,
                2,
            ),
            dtype=torch.float64,
            device=torch.device(
                "cpu"
            ),
        )

        print(
            f"    restrained MD image "
            f"{image_index + 1:02d}/{n_internal:02d}: "
            f"phi={target[0]: .8f}, "
            f"psi={target[1]: .8f}"
        )

        M_i, F_original_i = M_gradient(
            q_tensor,
            discard_frac=AA_DISCARD_FRAC,
            tmin=AA_TMIN,
            tmax=AA_TMAX,
        )

        M_i = np.asarray(
            M_i,
            dtype=float,
        ).reshape(
            2,
            2,
        )

        F_original_i = np.asarray(
            F_original_i,
            dtype=float,
        ).reshape(
            2,
        )

        # kJ/mol -> kcal/mol.
        F_i = (
            F_original_i
            / 4.184
        )

        if not np.all(
            np.isfinite(M_i)
        ):
            raise FloatingPointError(
                f"Non-finite M at image {image_index}:\n"
                f"{M_i}"
            )

        if not np.all(
            np.isfinite(F_i)
        ):
            raise FloatingPointError(
                f"Non-finite grad F at image {image_index}:\n"
                f"{F_i}"
            )

        # Numerical symmetry safeguard.
        M_i = 0.5 * (
            M_i
            + M_i.T
        )

        eigenvalues = np.linalg.eigvalsh(
            M_i
        )

        if np.min(
            eigenvalues
        ) <= 0.0:
            raise FloatingPointError(
                f"M is not positive definite at image {image_index}:\n"
                f"M =\n{M_i}\n"
                f"eigenvalues = {eigenvalues}"
            )

        M_all[
            image_index
        ] = M_i

        grad_F[
            image_index
        ] = F_i

    return (
        grad_F,
        M_all,
    )


def process_all_atom_paths():
    cache = load_aa_cache(
        AA_CACHE_NPZ
    )

    cached_indices = {
        int(index)
        for index in cache[
            "source_indices"
        ]
    }

    images = AA_RAW[
        "images"
    ]

    source_indices = AA_RAW[
        "source_indices"
    ]

    counts = AA_RAW[
        "count"
    ]

    print("=" * 88)
    print("Projected 66D MEP restrained-MD postprocessing")
    print("=" * 88)
    print(f"Source result : {ALL_ATOM_MEP_NPZ}")
    print(f"Cache         : {AA_CACHE_NPZ}")
    print(f"Selected states: {len(source_indices)}")
    print(f"Cached states  : {len(cached_indices)}")
    print("=" * 88)

    for local_index, (
        source_index,
        cartesian_path,
        count,
    ) in enumerate(
        zip(
            source_indices,
            images,
            counts,
        )
    ):
        source_index = int(
            source_index
        )

        if source_index in cached_indices:
            print(
                f"[CACHE] all-atom state "
                f"{source_index} already processed."
            )

            continue

        print()
        print("=" * 88)
        print(
            f"Processing all-atom state "
            f"{source_index} "
            f"({local_index + 1}/{len(source_indices)})"
        )
        print(f"Original oracle count: {count}")
        print("=" * 88)

        projected_raw = project_cartesian_path_to_cv(
            cartesian_path
        )

        projected_full = orient_and_unwrap_cv_path(
            projected_raw
        )

        if projected_full.shape[0] == N_IMAGES:
            raise ValueError(
                "The all-atom images_seq contains only 20 internal "
                "images and no endpoints. The supplied data must contain "
                "22 images, or the Cartesian endpoints must be supplied "
                "separately."
            )

        if projected_full.shape[0] != (
            N_IMAGES + 2
        ):
            raise ValueError(
                f"Expected {N_IMAGES + 2} all-atom images, "
                f"got {projected_full.shape[0]}."
            )

        grad_F, M_all = (
            evaluate_restrained_md_on_cv_path(
                projected_full
            )
        )

        metrics = compute_all_metrics(
            projected_full,
            grad_F,
            M_all,
        )

        cache[
            "source_indices"
        ].append(
            source_index
        )

        cache[
            "count"
        ].append(
            float(count)
        )

        cache[
            "paths"
        ].append(
            projected_full.copy()
        )

        cache[
            "grad_F"
        ].append(
            grad_F.copy()
        )

        cache[
            "M"
        ].append(
            M_all.copy()
        )

        cache[
            "I_E"
        ].append(
            metrics["I_E"]
        )

        cache[
            "I_G"
        ].append(
            metrics["I_G"]
        )

        cache[
            "barrier"
        ].append(
            metrics["barrier"]
        )

        cache[
            "profiles"
        ].append(
            metrics[
                "free_energy_profile"
            ].copy()
        )

        save_aa_cache(
            cache
        )

        cached_indices.add(
            source_index
        )

        print(
            f"[SAVED CACHE] state={source_index} | "
            f"I_G={metrics['I_G']:.6e} | "
            f"I_E={metrics['I_E']:.6e} | "
            f"DeltaF={metrics['barrier']:.6f}"
        )

    # Reload cache and sort in source-state order.
    cache = load_aa_cache(
        AA_CACHE_NPZ
    )

    order = np.argsort(
        np.asarray(
            cache["source_indices"],
            dtype=int,
        )
    )

    source_indices = np.asarray(
        cache["source_indices"],
        dtype=int,
    )[order]

    counts = np.asarray(
        cache["count"],
        dtype=float,
    )[order]

    paths = np.asarray(
        cache["paths"],
        dtype=float,
    )[order]

    grad_F = np.asarray(
        cache["grad_F"],
        dtype=float,
    )[order]

    M_all = np.asarray(
        cache["M"],
        dtype=float,
    )[order]

    I_E = np.asarray(
        cache["I_E"],
        dtype=float,
    )[order]

    I_G = np.asarray(
        cache["I_G"],
        dtype=float,
    )[order]

    barrier = np.asarray(
        cache["barrier"],
        dtype=float,
    )[order]

    profiles = np.asarray(
        cache["profiles"],
        dtype=float,
    )[order]

    # Retain only states selected for the current analysis.
    selected_set = {
        int(value)
        for value in AA_RAW[
            "source_indices"
        ]
    }

    keep = np.asarray(
        [
            int(value) in selected_set
            for value in source_indices
        ],
        dtype=bool,
    )

    source_indices = source_indices[
        keep
    ]

    counts = counts[
        keep
    ]

    paths = paths[
        keep
    ]

    grad_F = grad_F[
        keep
    ]

    M_all = M_all[
        keep
    ]

    I_E = I_E[
        keep
    ]

    I_G = I_G[
        keep
    ]

    barrier = barrier[
        keep
    ]

    profiles = profiles[
        keep
    ]

    if source_indices.size == 0:
        raise RuntimeError(
            "No all-atom states are available after cache processing."
        )

    return {
        "name": "Projected 66D MEP",
        "npz": ALL_ATOM_MEP_NPZ,
        "cache": AA_CACHE_NPZ,
        "iteration": source_indices.astype(
            float
        ),
        "count": (
            counts
            if USE_ORACLE_CALL_COUNT
            else source_indices.astype(float)
        ),
        "I_E": I_E,
        "I_G": I_G,
        "barrier": barrier,
        "paths": paths,
        "grad_F": grad_F,
        "M": M_all,
        "profiles": profiles,
        "final_path_full": paths[-1],
    }


AA_MEP_RESULT = process_all_atom_paths()


# ============================================================
# 7. Load and crop FES
# ============================================================

if not os.path.exists(
    FES_NPZ
):
    raise FileNotFoundError(
        f"Cannot find FES file:\n"
        f"  {FES_NPZ}"
    )

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as loaded:
    gx = np.asarray(
        loaded["X"],
        dtype=float,
    )

    gy = np.asarray(
        loaded["Y"],
        dtype=float,
    )

    gz = np.asarray(
        loaded["Z"],
        dtype=float,
    )

if (
    gx.ndim == 1
    and gy.ndim == 1
):
    gx, gy = np.meshgrid(
        gx,
        gy,
        indexing="xy",
    )

if (
    gx.shape != gy.shape
    or gx.shape != gz.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{gx.shape}, "
        f"Y{gy.shape}, "
        f"Z{gz.shape}"
    )

gx = np.deg2rad(
    gx
)

gy = np.deg2rad(
    gy
)

gz = np.ma.masked_invalid(
    gz
)

inside = (
    (gx >= XLIM[0])
    & (gx <= XLIM[1])
    & (gy >= YLIM[0])
    & (gy <= YLIM[1])
)

if not np.any(
    inside
):
    raise RuntimeError(
        f"No FES data inside XLIM={XLIM}, YLIM={YLIM}."
    )

rows = np.where(
    np.any(
        inside,
        axis=1,
    )
)[0]

cols = np.where(
    np.any(
        inside,
        axis=0,
    )
)[0]

r0 = rows[0]
r1 = rows[-1] + 1
c0 = cols[0]
c1 = cols[-1] + 1

Xc = gx[
    r0:r1,
    c0:c1,
]

Yc = gy[
    r0:r1,
    c0:c1,
]

Zc = gz[
    r0:r1,
    c0:c1,
]

Zc = (
    Zc
    - np.ma.min(Zc)
)


# ============================================================
# 8. Tuned black-to-white colormap
# ============================================================

_u = np.linspace(
    0.0,
    1.0,
    256,
)

_t = (
    _u
    + CURV
    * _u
    * (
        1.0
        - _u
    )
)

_t = np.clip(
    _t,
    0.0,
    1.0,
)

_gray = (
    BG_LO
    + (
        1.0
        - BG_LO
    )
    * _t
)

BG_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad",
    list(
        zip(
            _gray,
            _gray,
            _gray,
        )
    ),
)

BG_CMAP.set_over(
    "white"
)

levels = np.linspace(
    0.0,
    ENERGY_WHITE_CAP,
    CONTOUR_LEVELS,
)


# ============================================================
# 9. Plot style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 10. Plot helpers
# ============================================================

TINY = np.finfo(
    float
).tiny


def safe_positive(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    positive = values[
        np.isfinite(values)
        & (
            values > 0.0
        )
    ]

    if positive.size > 0:
        floor = max(
            TINY,
            np.min(
                positive
            )
            * 1e-8,
        )
    else:
        floor = TINY

    return np.where(
        np.isfinite(values)
        & (
            values > 0.0
        ),
        values,
        floor,
    )


def style_log_axis(
    axis,
):
    axis.set_yscale(
        "log"
    )

    axis.yaxis.set_major_locator(
        LogLocator(
            base=10.0,
            numticks=12,
        )
    )

    axis.yaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(
                2,
                10,
            )
            * 0.1,
            numticks=100,
        )
    )

    axis.yaxis.set_minor_formatter(
        NullFormatter()
    )


def style_metric_axis(
    axis,
    log_scale,
):
    if log_scale:
        style_log_axis(
            axis
        )
    else:
        axis.minorticks_on()

    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        which="both",
        top=False,
        right=False,
    )

    axis.grid(
        True,
        which="major",
        axis="y",
        linestyle="-",
        linewidth=0.6,
        alpha=0.18,
        color="#666666",
    )


def plot_metric_curve(
    axis,
    result,
    key,
    color,
    marker,
    label,
    log_scale,
    zorder,
):
    x_values = np.asarray(
        result["count"],
        dtype=float,
    )

    y_values = np.asarray(
        result[key],
        dtype=float,
    )

    n_common = min(
        x_values.size,
        y_values.size,
    )

    x_values = x_values[
        :n_common
    ]

    y_values = y_values[
        :n_common
    ]

    valid = (
        np.isfinite(
            x_values
        )
        & np.isfinite(
            y_values
        )
    )

    x_values = x_values[
        valid
    ]

    y_values = y_values[
        valid
    ]

    if x_values.size == 0:
        return

    order = np.argsort(
        x_values
    )

    x_values = x_values[
        order
    ]

    y_values = y_values[
        order
    ]

    if log_scale:
        plotted_y = safe_positive(
            y_values
        )
    else:
        plotted_y = y_values

    markevery = max(
        1,
        x_values.size // 10,
    )

    axis.plot(
        x_values,
        plotted_y,
        color=color,
        linestyle="-",
        linewidth=2.4,
        marker=marker,
        markevery=markevery,
        markersize=6.2,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.5,
        solid_capstyle="round",
        label=label,
        zorder=zorder,
        clip_on=False,
    )

    axis.scatter(
        x_values[-1],
        plotted_y[-1],
        s=72,
        marker=marker,
        facecolor=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=zorder + 5,
        clip_on=False,
    )


# ============================================================
# 11. Draw four-panel figure
# ============================================================

fig, axes = plt.subplots(
    1,
    4,
    figsize=(
        19.5,
        4.8,
    ),
)

ax_IG = axes[0]
ax_IE = axes[1]
ax_barrier = axes[2]
ax_contour = axes[3]

RESULT_SERIES = [
    {
        "result": MFEP_RESULT,
        "color": COLOR_MFEP,
        "marker": MARKER_MFEP,
        "label": r"MFEP-IVM ($G=M^{-1}$)",
        "zorder": 7,
    },
    {
        "result": MEP_RESULT,
        "color": COLOR_MEP,
        "marker": MARKER_MEP,
        "label": r"CV MEP-IVM ($G=I$)",
        "zorder": 6,
    },
    {
        "result": AA_MEP_RESULT,
        "color": COLOR_AA_MEP,
        "marker": MARKER_AA_MEP,
        "label": r"Projected 66D MEP",
        "zorder": 5,
    },
]


# ------------------------------------------------------------
# Panel a: MFEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=ax_IG,
        result=series["result"],
        key="I_G",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        log_scale=True,
        zorder=series["zorder"],
    )

ax_IG.set_xlabel(
    (
        "Method-specific oracle calls"
        if USE_ORACLE_CALL_COUNT
        else "Saved iteration"
    ),
    labelpad=6,
)

ax_IG.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,G}$",
    labelpad=7,
)

ax_IG.set_title(
    "MFEP normal residual",
    pad=12,
    fontweight="medium",
)

ax_IG.set_xlim(
    left=0.0
)

style_metric_axis(
    ax_IG,
    log_scale=True,
)


# ------------------------------------------------------------
# Panel b: Euclidean MEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=ax_IE,
        result=series["result"],
        key="I_E",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        log_scale=True,
        zorder=series["zorder"],
    )

ax_IE.set_xlabel(
    (
        "Method-specific oracle calls"
        if USE_ORACLE_CALL_COUNT
        else "Saved iteration"
    ),
    labelpad=6,
)

ax_IE.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,\mathrm{E}}$",
    labelpad=7,
)

ax_IE.set_title(
    "Euclidean MEP residual",
    pad=12,
    fontweight="medium",
)

ax_IE.set_xlim(
    left=0.0
)

style_metric_axis(
    ax_IE,
    log_scale=True,
)


# ------------------------------------------------------------
# Panel c: free-energy barrier
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=ax_barrier,
        result=series["result"],
        key="barrier",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        log_scale=False,
        zorder=series["zorder"],
    )

ax_barrier.set_xlabel(
    (
        "Method-specific oracle calls"
        if USE_ORACLE_CALL_COUNT
        else "Saved iteration"
    ),
    labelpad=6,
)

ax_barrier.set_ylabel(
    r"$\Delta F^{\dagger}$ (kcal mol$^{-1}$)",
    labelpad=7,
)

ax_barrier.set_title(
    "Free-energy barrier",
    pad=12,
    fontweight="medium",
)

ax_barrier.set_xlim(
    left=0.0
)

style_metric_axis(
    ax_barrier,
    log_scale=False,
)


# ------------------------------------------------------------
# Panel d: one common black-to-white contour
# ------------------------------------------------------------

contour = ax_contour.contourf(
    Xc,
    Yc,
    Zc,
    levels=levels,
    cmap=BG_CMAP,
    extend="max",
)

for (
    x_value,
    y_value,
    label,
    dx,
    dy,
) in SPECIAL_POINTS:
    ax_contour.plot(
        x_value,
        y_value,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="#111111",
        markeredgewidth=1.4,
        linestyle="None",
        zorder=15,
    )

    ax_contour.text(
        x_value + dx,
        y_value + dy,
        label,
        color="#111111",
        fontsize=12,
        fontweight="bold",
        zorder=15,
        ha="right",
        va="bottom",
        bbox={
            "boxstyle": "round,pad=0.12",
            "fc": "white",
            "ec": "none",
            "alpha": 0.75,
        },
    )

for series in RESULT_SERIES:
    plot_path = shift_path_to_plot_window(
        series["result"][
            "final_path_full"
        ]
    )

    ax_contour.plot(
        plot_path[:, 0],
        plot_path[:, 1],
        color=series["color"],
        linestyle="-",
        linewidth=2.35,
        marker=series["marker"],
        markevery=1,
        markersize=5.1,
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.35,
        solid_capstyle="round",
        label=series["label"],
        zorder=series["zorder"] + 2,
    )

ax_contour.set_xlabel(
    r"$\phi$ (rad)",
    labelpad=5,
)

ax_contour.set_ylabel(
    r"$\psi$ (rad)",
    labelpad=5,
)

ax_contour.set_title(
    "Three transition paths",
    pad=12,
    fontweight="medium",
)

ax_contour.set_xlim(
    XLIM
)

ax_contour.set_ylim(
    YLIM
)

ax_contour.set_aspect(
    "equal",
    adjustable="box",
)

ax_contour.tick_params(
    which="both",
    top=False,
    right=False,
)


# ------------------------------------------------------------
# Panel tags
# ------------------------------------------------------------

for axis, tag in zip(
    axes,
    [
        "a",
        "b",
        "c",
        "d",
    ],
):
    axis.text(
        (
            -0.15
            if axis is not ax_contour
            else -0.10
        ),
        1.07,
        tag,
        transform=axis.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left",
    )


# ============================================================
# 12. Legend, colorbar, layout
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=series["color"],
        linewidth=2.35,
        marker=series["marker"],
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.35,
        label=series["label"],
    )
    for series in RESULT_SERIES
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.04,
    ),
    ncol=3,
    fontsize=12.5,
    handlelength=2.2,
    columnspacing=2.2,
    borderaxespad=0.0,
)

fig.subplots_adjust(
    left=0.055,
    right=0.925,
    top=0.81,
    bottom=0.15,
    wspace=0.30,
)

contour_position = ax_contour.get_position()

colorbar_axis = fig.add_axes(
    [
        contour_position.x1 + 0.012,
        contour_position.y0,
        0.012,
        contour_position.height,
    ]
)

colorbar = fig.colorbar(
    contour,
    cax=colorbar_axis,
    extend="max",
)

colorbar.set_label(
    r"Free energy (kcal mol$^{-1}$)",
    labelpad=8,
)


# ============================================================
# 13. Save figure
# ============================================================

fig.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 14. Final summary
# ============================================================

print()
print("=" * 112)
print(
    "Final comparison: "
    "MFEP-IVM vs CV Euclidean MEP-IVM vs projected 66D MEP"
)
print("=" * 112)

for series in RESULT_SERIES:
    result = series[
        "result"
    ]

    print(
        f"{result['name']:>25} | "
        f"count={result['count'][-1]:>10.1f} | "
        f"I_G={result['I_G'][-1]:.8e} | "
        f"I_E={result['I_E'][-1]:.8e} | "
        f"DeltaF={result['barrier'][-1]:.6f} kcal/mol"
    )

print("-" * 112)

final_paths = {
    "MFEP": MFEP_RESULT[
        "final_path_full"
    ],
    "CV-MEP": MEP_RESULT[
        "final_path_full"
    ],
    "66D-MEP": AA_MEP_RESULT[
        "final_path_full"
    ],
}

for name_a, name_b in [
    (
        "MFEP",
        "CV-MEP",
    ),
    (
        "MFEP",
        "66D-MEP",
    ),
    (
        "CV-MEP",
        "66D-MEP",
    ),
]:
    path_a = final_paths[
        name_a
    ]

    path_b = final_paths[
        name_b
    ]

    if path_a.shape == path_b.shape:
        difference = wrap_angles(
            path_a
            - path_b
        )

        pointwise_distance = np.linalg.norm(
            difference,
            axis=1,
        )

        print(
            f"{name_a:>7} vs {name_b:<7} | "
            f"mean distance="
            f"{np.mean(pointwise_distance):.6e} rad | "
            f"max distance="
            f"{np.max(pointwise_distance):.6e} rad"
        )
    else:
        print(
            f"{name_a:>7} vs {name_b:<7} | "
            "different image counts; pointwise distance skipped."
        )

print("-" * 112)
print(f"MFEP result       : {MFEP_CONFIG['npz']}")
print(f"CV MEP result     : {MEP_CONFIG['npz']}")
print(f"66D MEP result    : {ALL_ATOM_MEP_NPZ}")
print(f"66D MD cache      : {AA_CACHE_NPZ}")
print("=" * 112)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")

phi_-129.38355934985685_psi_146.7159947301419/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-129.38355934985685_psi_146.7159947301419/PROJ: discarding 125/2501 frames (5.0%)
    restrained MD image 05/20: phi=-2.11874481, psi= 2.54780700
phi_-121.39513520886548_psi_145.97858801908836/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-121.39513520886548_psi_145.97858801908836/PROJ: discarding 125/2501 frames (5.0%)
    restrained MD image 06/20: phi=-1.97504897, psi= 2.52949958
phi_-113.16197011080166_psi_144.9296504096621/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-113.16197011080166_psi_144.9296504096621/PROJ: discarding 125/2501 frames (5.0%)
    restrained MD image 07/20: phi=-1.82655904, psi= 2.53475706
phi_-104.65412422878813_psi_145.23088140548595/COLVAR_restrained: discarding 125/2501 frames (5.0%)
phi_-104.65412422878813_psi_145.23088140548595/PROJ: discarding 125/2501 frames (5.0%)
    restrained MD image 08/20: phi=-1.67991504, psi= 2.52403985


/var/folders/2s/djr7zscd64n5yrwn2mghzt2m0000gn/T/ipykernel_69214/3424207024.py:3417: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()



Final comparison: MFEP-IVM vs CV Euclidean MEP-IVM vs projected 66D MEP
                 MFEP-IVM | count=    4020.0 | I_G=4.91226905e-01 | I_E=4.41820873e-01 | DeltaF=1.790164 kcal/mol
     CV Euclidean MEP-IVM | count=    4020.0 | I_G=1.17644293e+01 | I_E=4.26226493e-02 | DeltaF=1.801081 kcal/mol
        Projected 66D MEP | count=   11700.0 | I_G=4.39898682e+00 | I_E=6.21241950e-01 | DeltaF=1.949711 kcal/mol
----------------------------------------------------------------------------------------------------------------
   MFEP vs CV-MEP  | mean distance=1.640498e-02 rad | max distance=4.393923e-02 rad
   MFEP vs 66D-MEP | mean distance=6.395816e-02 rad | max distance=1.509676e-01 rad
 CV-MEP vs 66D-MEP | mean distance=6.973363e-02 rad | max distance=1.701029e-01 rad
----------------------------------------------------------------------------------------------------------------
MFEP result       : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MFEP_image_LBFGS_B_Recycle_N20_Nsub1_

In [17]:
# -*- coding: utf-8 -*-
# ============================================================
# MFEP-IVM vs CV Euclidean-MEP-IVM vs projected 66D MEP
#
# Plot only: no restrained MD is run in this script.
#
# 一行四张：
#   (a) MFEP residual I_perp^G
#   (b) Euclidean MEP residual I_perp^E
#   (c) Free-energy barrier Delta F^dagger
#   (d) Final three paths on one black-to-white FES contour
#
# 三个指标图的横坐标统一为：
#
#   Normalized optimization progress in [0, 1]
#
# 每种方法分别按照自己的 count / iteration 归一化：
#
#   progress = (count-count_initial)/(count_final-count_initial)
#
# 本脚本直接读取已经保存好的：
#
#   1. MFEP-IVM NPZ
#   2. CV Euclidean MEP-IVM NPZ
#   3. Projected 66D MEP restrained-MD cache
#
# 不会重新执行 restrained MD。
# ============================================================

import os
import re
import numpy as np

import matplotlib

# 不弹窗；只保存图片。
DISPLAY_FIGURE = False

if not DISPLAY_FIGURE:
    matplotlib.use("Agg")

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, NullFormatter
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# 0. File paths
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# ------------------------------------------------------------
# MFEP-IVM result
# ------------------------------------------------------------
MFEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# CV Euclidean MEP-IVM result
# ------------------------------------------------------------
MEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# Correct all-atom 66D MEP source data
#
# This file is included only for metadata/output reporting.
# The metric curves are loaded from AA_CACHE_NPZ.
# ------------------------------------------------------------
ALL_ATOM_MEP_NPZ = (
    "/Users/wuzhiyou/Downloads/"
    "Ala2_N20_lr0.04_alphaIVP100_Nsub5_results.npz"
)

# ------------------------------------------------------------
# Previously generated projected-66D restrained-MD cache
# ------------------------------------------------------------
AA_CACHE_NPZ = os.path.join(
    BASE_PATH,
    "Ala2_66D_projected_MEP_restrained_MD_cache.npz",
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        "MFEP_vs_CV_MEP_vs_projected_66D_MEP"
        "_normalized_progress"
        "_metrics_and_contour"
    ),
)


# ============================================================
# 1. Basic settings
# ============================================================

N_IMAGES = 20

# Fixed CV endpoints.
C1 = np.asarray(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.asarray(
    [-1.33047160, 0.99118795],
    dtype=float,
)

XLIM = (
    -np.pi,
    0.0,
)

YLIM = (
    0.0,
    np.pi,
)

SPECIAL_POINTS = [
    (
        -2.62593029,
        2.77365591,
        r"$C5$",
        -0.05,
        0.16,
    ),
    (
        -1.33433281,
        0.99268968,
        r"$C7_{\mathrm{eq}}$",
        0.10,
        -0.40,
    ),
]


# ============================================================
# 2. Colors and markers
# ============================================================

# Color-blind-friendly colors.
COLOR_MFEP = "#0072B2"       # blue
COLOR_CV_MEP = "#D50087"     # magenta
COLOR_66D_MEP = "#009E73"    # green

MARKER_MFEP = "o"
MARKER_CV_MEP = "s"
MARKER_66D_MEP = "^"


# ============================================================
# 3. Tuned black-to-white contour settings
# ============================================================

# Free energy at and above this value is white.
ENERGY_WHITE_CAP = 60.0

# Minimum-energy grayscale value.
#
# 0.0 = pure black
# 0.05 = nearly black
BG_LO = 0.05

# Quadratic grayscale mapping.
CURV = 2.0

CONTOUR_LEVELS = 33


# ============================================================
# 4. Check input files
# ============================================================

for label, path in [
    ("FES", FES_NPZ),
    ("MFEP-IVM", MFEP_NPZ),
    ("CV MEP-IVM", MEP_NPZ),
    ("Projected 66D MEP cache", AA_CACHE_NPZ),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find {label} file:\n"
            f"  {path}"
        )

print("=" * 100)
print("Input files")
print("=" * 100)
print(f"MFEP-IVM result        : {MFEP_NPZ}")
print(f"CV MEP-IVM result      : {MEP_NPZ}")
print(f"66D projected MD cache : {AA_CACHE_NPZ}")
print(f"66D original result    : {ALL_ATOM_MEP_NPZ}")
print(f"FES                     : {FES_NPZ}")
print("=" * 100)


# ============================================================
# 5. General helpers
# ============================================================

def extract_iteration_numbers(
    loaded,
    prefix,
):
    """
    Extract iteration numbers from keys such as:

        phi_ref_iter_0
        phi_ref_iter_1
        ...
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    iterations = []

    for key in loaded.files:
        match = pattern.match(key)

        if match is not None:
            iterations.append(
                int(match.group(1))
            )

    return sorted(iterations)


def normalized_progress(
    raw_coordinate,
):
    """
    Normalize any monotone optimization coordinate to [0, 1].

    The first finite point becomes 0 and the final finite point
    becomes 1.

    If only one point exists, it is placed at progress=1.
    """
    raw_coordinate = np.asarray(
        raw_coordinate,
        dtype=float,
    ).reshape(-1)

    if raw_coordinate.size == 0:
        return np.zeros(
            0,
            dtype=float,
        )

    if raw_coordinate.size == 1:
        return np.ones(
            1,
            dtype=float,
        )

    first_value = float(
        raw_coordinate[0]
    )

    final_value = float(
        raw_coordinate[-1]
    )

    denominator = (
        final_value
        - first_value
    )

    if (
        not np.isfinite(denominator)
        or abs(denominator) <= 1e-16
    ):
        return np.linspace(
            0.0,
            1.0,
            raw_coordinate.size,
        )

    progress = (
        raw_coordinate
        - first_value
    ) / denominator

    return np.clip(
        progress,
        0.0,
        1.0,
    )


def wrap_angles(
    values,
):
    """
    Wrap angular differences to [-pi, pi).
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        (values + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )


def path_with_fixed_endpoints(
    internal_path,
):
    """
    Add the two fixed CV endpoints to an internal-image path.
    """
    internal_path = np.asarray(
        internal_path,
        dtype=float,
    )

    return np.vstack(
        [
            C1,
            internal_path,
            C2,
        ]
    )


def shift_path_to_plot_window(
    path,
):
    """
    Shift an unwrapped angular path by common multiples of 2*pi
    so that it lies near the requested plotting window.

    The same shift is applied to all images in each coordinate,
    preserving path continuity.
    """
    path = np.asarray(
        path,
        dtype=float,
    ).copy()

    target_center = np.asarray(
        [
            0.5 * (
                XLIM[0]
                + XLIM[1]
            ),
            0.5 * (
                YLIM[0]
                + YLIM[1]
            ),
        ],
        dtype=float,
    )

    for coordinate in range(2):
        path_center = float(
            np.mean(
                path[:, coordinate]
            )
        )

        shift_count = np.round(
            (
                target_center[coordinate]
                - path_center
            )
            / (
                2.0 * np.pi
            )
        )

        path[:, coordinate] += (
            shift_count
            * 2.0
            * np.pi
        )

    return path


# ============================================================
# 6. Common geometry and residual definitions
# ============================================================

def euclidean_normal_projector(
    tangent,
    eps=1e-16,
):
    """
    Euclidean normal projector:

        P_E = I - tau_hat tau_hat^T.
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    tangent_norm = float(
        np.linalg.norm(
            tangent
        )
    )

    if tangent_norm <= eps:
        return np.eye(
            tangent.size,
            dtype=float,
        )

    unit_tangent = (
        tangent
        / tangent_norm
    )

    return (
        np.eye(
            tangent.size,
            dtype=float,
        )
        - np.outer(
            unit_tangent,
            unit_tangent,
        )
    )


def G_normal_projector(
    tangent,
    G,
    eps=1e-16,
):
    """
    G-orthogonal normal projector:

        P_G
        =
        I - tangent (tangent^T G)/(tangent^T G tangent).
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    G = np.asarray(
        G,
        dtype=float,
    )

    denominator = float(
        tangent
        @ G
        @ tangent
    )

    if denominator <= eps:
        return np.eye(
            tangent.size,
            dtype=float,
        )

    return (
        np.eye(
            tangent.size,
            dtype=float,
        )
        - np.outer(
            tangent,
            tangent @ G,
        )
        / denominator
    )


def central_tangents_from_full_path(
    full_path,
):
    """
    Central-difference tangent at each internal image.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_euclidean_node_weights(
    full_path,
):
    """
    Euclidean arclength quadrature weights.
    """
    internal_path = full_path[
        1:-1
    ]

    segment_minus = (
        internal_path
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal_path
    )

    return 0.5 * (
        np.linalg.norm(
            segment_minus,
            axis=1,
        )
        +
        np.linalg.norm(
            segment_plus,
            axis=1,
        )
    )


def compute_G_node_weights(
    full_path,
    G_all,
):
    """
    G-arclength quadrature weights.
    """
    internal_path = full_path[
        1:-1
    ]

    segment_minus = (
        internal_path
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal_path
    )

    segment_minus_G_sq = np.einsum(
        "ni,nij,nj->n",
        segment_minus,
        G_all,
        segment_minus,
    )

    segment_plus_G_sq = np.einsum(
        "ni,nij,nj->n",
        segment_plus,
        G_all,
        segment_plus,
    )

    return 0.5 * (
        np.sqrt(
            np.maximum(
                segment_minus_G_sq,
                0.0,
            )
        )
        +
        np.sqrt(
            np.maximum(
                segment_plus_G_sq,
                0.0,
            )
        )
    )


def compute_MEP_residual(
    full_path,
    grad_F,
):
    """
    Euclidean free-energy MEP residual:

        I_perp^E
        =
        sum_i |P_E grad F|^2 w_i^E.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual_vectors = np.zeros_like(
        grad_F
    )

    for image_index in range(
        number_internal
    ):
        projector = euclidean_normal_projector(
            tangents[image_index]
        )

        residual_vectors[image_index] = (
            projector
            @ grad_F[image_index]
        )

    pointwise_squared_norm = np.sum(
        residual_vectors ** 2,
        axis=1,
    )

    node_weights = compute_euclidean_node_weights(
        full_path
    )

    residual_value = float(
        np.sum(
            pointwise_squared_norm
            * node_weights
        )
    )

    return {
        "value": residual_value,
        "vectors": residual_vectors,
        "pointwise_sq": pointwise_squared_norm,
        "weights": node_weights,
    }


def compute_MFEP_residual(
    full_path,
    grad_F,
    M_all,
):
    """
    MFEP residual under G=M^{-1}:

        R_G = P_G M grad F,

        I_perp^G
        =
        sum_i R_G^T G R_G w_i^G.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    if M_all.shape != (
        number_internal,
        2,
        2,
    ):
        raise ValueError(
            f"M shape {M_all.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    G_all = np.linalg.inv(
        M_all
    )

    M_grad_F = np.einsum(
        "nij,nj->ni",
        M_all,
        grad_F,
    )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual_vectors = np.zeros_like(
        M_grad_F
    )

    for image_index in range(
        number_internal
    ):
        projector = G_normal_projector(
            tangents[image_index],
            G_all[image_index],
        )

        residual_vectors[image_index] = (
            projector
            @ M_grad_F[image_index]
        )

    pointwise_G_squared_norm = np.einsum(
        "ni,nij,nj->n",
        residual_vectors,
        G_all,
        residual_vectors,
    )

    pointwise_G_squared_norm = np.maximum(
        pointwise_G_squared_norm,
        0.0,
    )

    node_weights = compute_G_node_weights(
        full_path,
        G_all,
    )

    residual_value = float(
        np.sum(
            pointwise_G_squared_norm
            * node_weights
        )
    )

    return {
        "value": residual_value,
        "vectors": residual_vectors,
        "pointwise_G_sq": pointwise_G_squared_norm,
        "weights": node_weights,
        "G": G_all,
    }


def recover_free_energy_profile(
    full_path,
    grad_F,
):
    """
    Integrate grad F along the complete path.

    grad F is available at the internal images. The first and last
    internal gradients are used on the endpoint segments.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    path_segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (
            number_internal + 1,
            2,
        ),
        dtype=float,
    )

    segment_gradients[0] = (
        grad_F[0]
    )

    if number_internal > 1:
        segment_gradients[
            1:number_internal
        ] = 0.5 * (
            grad_F[:-1]
            + grad_F[1:]
        )

    segment_gradients[
        number_internal
    ] = grad_F[-1]

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        path_segments,
    )

    profile = np.zeros(
        number_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= np.min(
        profile
    )

    barrier = float(
        np.max(profile)
        - np.min(profile)
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "increments": increments,
    }


def compute_all_metrics(
    full_path,
    grad_F,
    M_all,
):
    """
    Compute the three common path metrics.
    """
    mep_result = compute_MEP_residual(
        full_path,
        grad_F,
    )

    mfep_result = compute_MFEP_residual(
        full_path,
        grad_F,
        M_all,
    )

    free_energy_result = recover_free_energy_profile(
        full_path,
        grad_F,
    )

    return {
        "I_E": mep_result["value"],
        "I_G": mfep_result["value"],
        "barrier": free_energy_result["barrier"],
        "free_energy_profile": free_energy_result[
            "profile"
        ],
    }


# ============================================================
# 7. Load the two CV-space results
# ============================================================

def load_cv_result(
    npz_path,
    method_name,
):
    """
    Load a CV-space IVM run and recompute all three metrics using
    the common definitions above.
    """
    with np.load(
        npz_path,
        allow_pickle=True,
    ) as loaded:

        iterations = extract_iteration_numbers(
            loaded,
            "phi_ref_iter_",
        )

        if not iterations:
            raise RuntimeError(
                f"{method_name}: no phi_ref_iter_* keys in:\n"
                f"  {npz_path}"
            )

        # Optional stored cost coordinate.
        count_map = {}

        if (
            "count_seq" in loaded.files
            and "index_seq" in loaded.files
        ):
            stored_counts = np.asarray(
                loaded["count_seq"],
                dtype=float,
            ).reshape(-1)

            stored_indices = np.asarray(
                loaded["index_seq"],
                dtype=int,
            ).reshape(-1)

            for index_value, count_value in zip(
                stored_indices,
                stored_counts,
            ):
                count_map[
                    int(index_value)
                ] = float(count_value)

        used_iterations = []
        raw_coordinate = []

        I_E_values = []
        I_G_values = []
        barrier_values = []

        reference_paths = []
        profiles = []

        for iteration in iterations:
            path_key = (
                f"phi_ref_iter_{iteration}"
            )

            metric_key = (
                f"M_ref_iter_{iteration}"
            )

            gradient_key = None

            for candidate in [
                f"g_ref_iter_{iteration}",
                f"F_ref_iter_{iteration}",
            ]:
                if candidate in loaded.files:
                    gradient_key = candidate
                    break

            if gradient_key is None:
                continue

            if metric_key not in loaded.files:
                continue

            internal_path = np.asarray(
                loaded[path_key],
                dtype=float,
            )

            grad_F = np.asarray(
                loaded[gradient_key],
                dtype=float,
            )

            M_all = np.asarray(
                loaded[metric_key],
                dtype=float,
            )

            if internal_path.ndim != 2:
                continue

            if internal_path.shape[1] != 2:
                continue

            if grad_F.shape != internal_path.shape:
                continue

            if M_all.shape != (
                internal_path.shape[0],
                2,
                2,
            ):
                continue

            full_path = path_with_fixed_endpoints(
                internal_path
            )

            try:
                metrics = compute_all_metrics(
                    full_path,
                    grad_F,
                    M_all,
                )

            except np.linalg.LinAlgError:
                print(
                    f"[WARNING] {method_name}: "
                    f"singular M at iteration {iteration}; skipping."
                )
                continue

            used_iterations.append(
                iteration
            )

            raw_coordinate.append(
                count_map.get(
                    iteration,
                    float(iteration),
                )
            )

            I_E_values.append(
                metrics["I_E"]
            )

            I_G_values.append(
                metrics["I_G"]
            )

            barrier_values.append(
                metrics["barrier"]
            )

            reference_paths.append(
                full_path.copy()
            )

            profiles.append(
                metrics[
                    "free_energy_profile"
                ].copy()
            )

        if not used_iterations:
            raise RuntimeError(
                f"No usable states were found in:\n"
                f"  {npz_path}"
            )

        # Final post-update/reparameterized path for the contour.
        path_iterations = extract_iteration_numbers(
            loaded,
            "path_iter_",
        )

        if path_iterations:
            final_internal_path = np.asarray(
                loaded[
                    f"path_iter_{path_iterations[-1]}"
                ],
                dtype=float,
            )

            final_full_path = path_with_fixed_endpoints(
                final_internal_path
            )
        else:
            final_full_path = (
                reference_paths[-1].copy()
            )

    raw_coordinate = np.asarray(
        raw_coordinate,
        dtype=float,
    )

    order = np.argsort(
        raw_coordinate
    )

    raw_coordinate = raw_coordinate[
        order
    ]

    used_iterations = np.asarray(
        used_iterations,
        dtype=float,
    )[order]

    I_E_values = np.asarray(
        I_E_values,
        dtype=float,
    )[order]

    I_G_values = np.asarray(
        I_G_values,
        dtype=float,
    )[order]

    barrier_values = np.asarray(
        barrier_values,
        dtype=float,
    )[order]

    reference_paths = [
        reference_paths[index]
        for index in order
    ]

    profiles = [
        profiles[index]
        for index in order
    ]

    return {
        "name": method_name,
        "npz": npz_path,
        "iteration": used_iterations,
        "raw_coordinate": raw_coordinate,
        "progress": normalized_progress(
            raw_coordinate
        ),
        "I_E": I_E_values,
        "I_G": I_G_values,
        "barrier": barrier_values,
        "paths": reference_paths,
        "profiles": profiles,
        "final_path_full": final_full_path,
    }


MFEP_RESULT = load_cv_result(
    npz_path=MFEP_NPZ,
    method_name="MFEP-IVM",
)

CV_MEP_RESULT = load_cv_result(
    npz_path=MEP_NPZ,
    method_name="CV Euclidean MEP-IVM",
)


# ============================================================
# 8. Load the already computed 66D projected-MEP cache
# ============================================================

def load_projected_66d_cache(
    cache_path,
):
    """
    Load previously completed restrained-MD postprocessing of the
    projected 66D MEP.

    No restrained MD is run here.
    """
    with np.load(
        cache_path,
        allow_pickle=True,
    ) as loaded:

        required_keys = [
            "source_indices",
            "count",
            "paths",
            "grad_F",
            "M",
            "I_E",
            "I_G",
            "barrier",
            "profiles",
        ]

        missing_keys = [
            key
            for key in required_keys
            if key not in loaded.files
        ]

        if missing_keys:
            raise KeyError(
                "The projected-66D cache is incomplete. Missing keys:\n"
                + "\n".join(
                    f"  {key}"
                    for key in missing_keys
                )
            )

        source_indices = np.asarray(
            loaded["source_indices"],
            dtype=int,
        ).reshape(-1)

        counts = np.asarray(
            loaded["count"],
            dtype=float,
        ).reshape(-1)

        paths = np.asarray(
            loaded["paths"],
            dtype=float,
        )

        grad_F = np.asarray(
            loaded["grad_F"],
            dtype=float,
        )

        M_all = np.asarray(
            loaded["M"],
            dtype=float,
        )

        stored_I_E = np.asarray(
            loaded["I_E"],
            dtype=float,
        ).reshape(-1)

        stored_I_G = np.asarray(
            loaded["I_G"],
            dtype=float,
        ).reshape(-1)

        stored_barrier = np.asarray(
            loaded["barrier"],
            dtype=float,
        ).reshape(-1)

        profiles = np.asarray(
            loaded["profiles"],
            dtype=float,
        )

    number_states = min(
        source_indices.size,
        counts.size,
        paths.shape[0],
        grad_F.shape[0],
        M_all.shape[0],
        stored_I_E.size,
        stored_I_G.size,
        stored_barrier.size,
        profiles.shape[0],
    )

    source_indices = source_indices[
        :number_states
    ]

    counts = counts[
        :number_states
    ]

    paths = paths[
        :number_states
    ]

    grad_F = grad_F[
        :number_states
    ]

    M_all = M_all[
        :number_states
    ]

    stored_I_E = stored_I_E[
        :number_states
    ]

    stored_I_G = stored_I_G[
        :number_states
    ]

    stored_barrier = stored_barrier[
        :number_states
    ]

    profiles = profiles[
        :number_states
    ]

    # Sort by original all-atom source state/count.
    order = np.lexsort(
        (
            source_indices,
            counts,
        )
    )

    source_indices = source_indices[
        order
    ]

    counts = counts[
        order
    ]

    paths = paths[
        order
    ]

    grad_F = grad_F[
        order
    ]

    M_all = M_all[
        order
    ]

    stored_I_E = stored_I_E[
        order
    ]

    stored_I_G = stored_I_G[
        order
    ]

    stored_barrier = stored_barrier[
        order
    ]

    profiles = profiles[
        order
    ]

    # Remove duplicate source states, retaining the last occurrence.
    unique_source_indices = []
    keep_indices = []

    for array_index, source_index in enumerate(
        source_indices
    ):
        source_index = int(
            source_index
        )

        if (
            unique_source_indices
            and source_index
            == unique_source_indices[-1]
        ):
            keep_indices[-1] = array_index
        else:
            unique_source_indices.append(
                source_index
            )

            keep_indices.append(
                array_index
            )

    keep_indices = np.asarray(
        keep_indices,
        dtype=int,
    )

    source_indices = source_indices[
        keep_indices
    ]

    counts = counts[
        keep_indices
    ]

    paths = paths[
        keep_indices
    ]

    grad_F = grad_F[
        keep_indices
    ]

    M_all = M_all[
        keep_indices
    ]

    stored_I_E = stored_I_E[
        keep_indices
    ]

    stored_I_G = stored_I_G[
        keep_indices
    ]

    stored_barrier = stored_barrier[
        keep_indices
    ]

    profiles = profiles[
        keep_indices
    ]

    # Recompute the metrics from the cached paths/gradients/M to
    # guarantee the same definitions as the two CV methods.
    recomputed_I_E = []
    recomputed_I_G = []
    recomputed_barrier = []
    recomputed_profiles = []

    valid_state_indices = []

    for state_index in range(
        source_indices.size
    ):
        full_path = np.asarray(
            paths[state_index],
            dtype=float,
        )

        state_grad_F = np.asarray(
            grad_F[state_index],
            dtype=float,
        )

        state_M = np.asarray(
            M_all[state_index],
            dtype=float,
        )

        if full_path.ndim != 2:
            continue

        if full_path.shape[1] != 2:
            continue

        if state_grad_F.shape != (
            full_path.shape[0] - 2,
            2,
        ):
            continue

        if state_M.shape != (
            full_path.shape[0] - 2,
            2,
            2,
        ):
            continue

        try:
            metrics = compute_all_metrics(
                full_path,
                state_grad_F,
                state_M,
            )

        except np.linalg.LinAlgError:
            print(
                f"[WARNING] Projected 66D MEP: singular M "
                f"at source state {source_indices[state_index]}; "
                "skipping."
            )
            continue

        valid_state_indices.append(
            state_index
        )

        recomputed_I_E.append(
            metrics["I_E"]
        )

        recomputed_I_G.append(
            metrics["I_G"]
        )

        recomputed_barrier.append(
            metrics["barrier"]
        )

        recomputed_profiles.append(
            metrics[
                "free_energy_profile"
            ]
        )

    if not valid_state_indices:
        raise RuntimeError(
            "No valid projected-66D states were found in the cache."
        )

    valid_state_indices = np.asarray(
        valid_state_indices,
        dtype=int,
    )

    source_indices = source_indices[
        valid_state_indices
    ]

    counts = counts[
        valid_state_indices
    ]

    paths = paths[
        valid_state_indices
    ]

    grad_F = grad_F[
        valid_state_indices
    ]

    M_all = M_all[
        valid_state_indices
    ]

    return {
        "name": "Projected 66D MEP",
        "npz": ALL_ATOM_MEP_NPZ,
        "cache": cache_path,
        "iteration": source_indices.astype(
            float
        ),
        "raw_coordinate": counts,
        "progress": normalized_progress(
            counts
        ),
        "I_E": np.asarray(
            recomputed_I_E,
            dtype=float,
        ),
        "I_G": np.asarray(
            recomputed_I_G,
            dtype=float,
        ),
        "barrier": np.asarray(
            recomputed_barrier,
            dtype=float,
        ),
        "paths": paths,
        "grad_F": grad_F,
        "M": M_all,
        "profiles": np.asarray(
            recomputed_profiles,
            dtype=float,
        ),
        "final_path_full": paths[-1],
    }


AA_MEP_RESULT = load_projected_66d_cache(
    AA_CACHE_NPZ
)


# ============================================================
# 9. Print loaded sequence information
# ============================================================

print()
print("=" * 100)
print("Loaded optimization sequences")
print("=" * 100)

for result in [
    MFEP_RESULT,
    CV_MEP_RESULT,
    AA_MEP_RESULT,
]:
    print(
        f"{result['name']:>25} | "
        f"states={len(result['progress']):>4d} | "
        f"raw start={result['raw_coordinate'][0]:>10.2f} | "
        f"raw final={result['raw_coordinate'][-1]:>10.2f} | "
        f"progress=[{result['progress'][0]:.3f}, "
        f"{result['progress'][-1]:.3f}]"
    )

print("=" * 100)


# ============================================================
# 10. Load and crop FES
# ============================================================

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as loaded:
    grid_x = np.asarray(
        loaded["X"],
        dtype=float,
    )

    grid_y = np.asarray(
        loaded["Y"],
        dtype=float,
    )

    grid_z = np.asarray(
        loaded["Z"],
        dtype=float,
    )

if (
    grid_x.ndim == 1
    and grid_y.ndim == 1
):
    grid_x, grid_y = np.meshgrid(
        grid_x,
        grid_y,
        indexing="xy",
    )

if (
    grid_x.shape != grid_y.shape
    or grid_x.shape != grid_z.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{grid_x.shape}, "
        f"Y{grid_y.shape}, "
        f"Z{grid_z.shape}"
    )

# FES coordinates are stored in degrees.
grid_x = np.deg2rad(
    grid_x
)

grid_y = np.deg2rad(
    grid_y
)

grid_z = np.ma.masked_invalid(
    grid_z
)

inside_window = (
    (grid_x >= XLIM[0])
    & (grid_x <= XLIM[1])
    & (grid_y >= YLIM[0])
    & (grid_y <= YLIM[1])
)

if not np.any(
    inside_window
):
    raise RuntimeError(
        f"No FES data inside XLIM={XLIM}, YLIM={YLIM}."
    )

rows = np.where(
    np.any(
        inside_window,
        axis=1,
    )
)[0]

columns = np.where(
    np.any(
        inside_window,
        axis=0,
    )
)[0]

row_start = rows[0]
row_stop = rows[-1] + 1
column_start = columns[0]
column_stop = columns[-1] + 1

X_contour = grid_x[
    row_start:row_stop,
    column_start:column_stop,
]

Y_contour = grid_y[
    row_start:row_stop,
    column_start:column_stop,
]

Z_contour = grid_z[
    row_start:row_stop,
    column_start:column_stop,
]

# Set minimum energy in the displayed window to zero.
Z_contour = (
    Z_contour
    - np.ma.min(
        Z_contour
    )
)


# ============================================================
# 11. Tuned black-to-white colormap
# ============================================================

unit_values = np.linspace(
    0.0,
    1.0,
    256,
)

mapped_values = (
    unit_values
    + CURV
    * unit_values
    * (
        1.0
        - unit_values
    )
)

mapped_values = np.clip(
    mapped_values,
    0.0,
    1.0,
)

gray_values = (
    BG_LO
    + (
        1.0
        - BG_LO
    )
    * mapped_values
)

BACKGROUND_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad",
    list(
        zip(
            gray_values,
            gray_values,
            gray_values,
        )
    ),
)

BACKGROUND_CMAP.set_over(
    "white"
)

contour_levels = np.linspace(
    0.0,
    ENERGY_WHITE_CAP,
    CONTOUR_LEVELS,
)


# ============================================================
# 12. Nature / Science plotting style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 13. Plot helpers
# ============================================================

FLOAT_TINY = np.finfo(
    float
).tiny


def safe_positive(
    values,
):
    """
    Make values safe for log-scale plotting.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if positive_values.size > 0:
        floor_value = max(
            FLOAT_TINY,
            float(
                np.min(
                    positive_values
                )
            )
            * 1e-8,
        )
    else:
        floor_value = FLOAT_TINY

    return np.where(
        np.isfinite(values)
        & (values > 0.0),
        values,
        floor_value,
    )


def style_log_axis(
    axis,
):
    axis.set_yscale(
        "log"
    )

    axis.yaxis.set_major_locator(
        LogLocator(
            base=10.0,
            numticks=12,
        )
    )

    axis.yaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(
                2,
                10,
            )
            * 0.1,
            numticks=100,
        )
    )

    axis.yaxis.set_minor_formatter(
        NullFormatter()
    )


def style_metric_axis(
    axis,
    use_log_scale,
):
    if use_log_scale:
        style_log_axis(
            axis
        )
    else:
        axis.minorticks_on()

    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        which="both",
        top=False,
        right=False,
    )

    axis.grid(
        True,
        which="major",
        axis="y",
        linestyle="-",
        linewidth=0.6,
        alpha=0.18,
        color="#666666",
    )

    # All metric panels use exactly the same x range.
    axis.set_xlim(
        0.0,
        1.0,
    )

    axis.set_xticks(
        np.linspace(
            0.0,
            1.0,
            6,
        )
    )


def plot_metric_curve(
    axis,
    result,
    metric_key,
    color,
    marker,
    label,
    use_log_scale,
    zorder,
):
    """
    Plot one metric against normalized progress.
    """
    progress = np.asarray(
        result["progress"],
        dtype=float,
    )

    metric_values = np.asarray(
        result[metric_key],
        dtype=float,
    )

    number_common = min(
        progress.size,
        metric_values.size,
    )

    progress = progress[
        :number_common
    ]

    metric_values = metric_values[
        :number_common
    ]

    valid = (
        np.isfinite(progress)
        & np.isfinite(metric_values)
    )

    progress = progress[
        valid
    ]

    metric_values = metric_values[
        valid
    ]

    if progress.size == 0:
        return

    order = np.argsort(
        progress
    )

    progress = progress[
        order
    ]

    metric_values = metric_values[
        order
    ]

    if use_log_scale:
        plotted_values = safe_positive(
            metric_values
        )
    else:
        plotted_values = metric_values

    mark_every = max(
        1,
        progress.size // 10,
    )

    axis.plot(
        progress,
        plotted_values,
        color=color,
        linestyle="-",
        linewidth=2.4,
        marker=marker,
        markevery=mark_every,
        markersize=6.2,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.5,
        solid_capstyle="round",
        label=label,
        zorder=zorder,
        clip_on=False,
    )

    # Highlight final point.
    axis.scatter(
        progress[-1],
        plotted_values[-1],
        s=72,
        marker=marker,
        facecolor=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=zorder + 5,
        clip_on=False,
    )


# ============================================================
# 14. Result-series definitions
# ============================================================

RESULT_SERIES = [
    {
        "result": MFEP_RESULT,
        "color": COLOR_MFEP,
        "marker": MARKER_MFEP,
        "label": r"MFEP-IVM ($G=M^{-1}$)",
        "zorder": 7,
    },
    {
        "result": CV_MEP_RESULT,
        "color": COLOR_CV_MEP,
        "marker": MARKER_CV_MEP,
        "label": r"CV MEP-IVM ($G=I$)",
        "zorder": 6,
    },
    {
        "result": AA_MEP_RESULT,
        "color": COLOR_66D_MEP,
        "marker": MARKER_66D_MEP,
        "label": r"Projected 66D MEP",
        "zorder": 5,
    },
]


# ============================================================
# 15. Draw the four-panel figure
# ============================================================

figure, axes = plt.subplots(
    1,
    4,
    figsize=(
        19.5,
        4.8,
    ),
)

axis_I_G = axes[0]
axis_I_E = axes[1]
axis_barrier = axes[2]
axis_contour = axes[3]


# ------------------------------------------------------------
# Panel (a): MFEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_I_G,
        result=series["result"],
        metric_key="I_G",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=True,
        zorder=series["zorder"],
    )

axis_I_G.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_I_G.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,G}$",
    labelpad=7,
)

axis_I_G.set_title(
    "MFEP normal residual",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_I_G,
    use_log_scale=True,
)


# ------------------------------------------------------------
# Panel (b): Euclidean MEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_I_E,
        result=series["result"],
        metric_key="I_E",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=True,
        zorder=series["zorder"],
    )

axis_I_E.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_I_E.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,\mathrm{E}}$",
    labelpad=7,
)

axis_I_E.set_title(
    "Euclidean MEP residual",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_I_E,
    use_log_scale=True,
)


# ------------------------------------------------------------
# Panel (c): free-energy barrier
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_barrier,
        result=series["result"],
        metric_key="barrier",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=False,
        zorder=series["zorder"],
    )

axis_barrier.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_barrier.set_ylabel(
    r"$\Delta F^{\dagger}$ (kcal mol$^{-1}$)",
    labelpad=7,
)

axis_barrier.set_title(
    "Free-energy barrier",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_barrier,
    use_log_scale=False,
)


# ------------------------------------------------------------
# Panel (d): one common black-to-white contour
# ------------------------------------------------------------

contour = axis_contour.contourf(
    X_contour,
    Y_contour,
    Z_contour,
    levels=contour_levels,
    cmap=BACKGROUND_CMAP,
    extend="max",
)

# Metastable-state labels.
for (
    x_value,
    y_value,
    state_label,
    x_offset,
    y_offset,
) in SPECIAL_POINTS:
    axis_contour.plot(
        x_value,
        y_value,
        marker="o",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="#111111",
        markeredgewidth=1.4,
        linestyle="None",
        zorder=15,
    )

    axis_contour.text(
        x_value + x_offset,
        y_value + y_offset,
        state_label,
        color="#111111",
        fontsize=12,
        fontweight="bold",
        zorder=15,
        ha="right",
        va="bottom",
        bbox={
            "boxstyle": "round,pad=0.12",
            "fc": "white",
            "ec": "none",
            "alpha": 0.75,
        },
    )

# Plot the final path from each method.
for series in RESULT_SERIES:
    final_path = shift_path_to_plot_window(
        series["result"][
            "final_path_full"
        ]
    )

    axis_contour.plot(
        final_path[:, 0],
        final_path[:, 1],
        color=series["color"],
        linestyle="-",
        linewidth=2.35,
        marker=series["marker"],
        markevery=1,
        markersize=5.1,
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.35,
        solid_capstyle="round",
        label=series["label"],
        zorder=series["zorder"] + 2,
    )

axis_contour.set_xlabel(
    r"$\phi$ (rad)",
    labelpad=5,
)

axis_contour.set_ylabel(
    r"$\psi$ (rad)",
    labelpad=5,
)

axis_contour.set_title(
    "Three transition paths",
    pad=12,
    fontweight="medium",
)

axis_contour.set_xlim(
    XLIM
)

axis_contour.set_ylim(
    YLIM
)

axis_contour.set_aspect(
    "equal",
    adjustable="box",
)

axis_contour.tick_params(
    which="both",
    top=False,
    right=False,
)


# ------------------------------------------------------------
# Panel tags
# ------------------------------------------------------------

for axis, panel_tag in zip(
    axes,
    [
        "a",
        "b",
        "c",
        "d",
    ],
):
    axis.text(
        (
            -0.15
            if axis is not axis_contour
            else -0.10
        ),
        1.07,
        panel_tag,
        transform=axis.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left",
    )


# ============================================================
# 16. Shared legend and colorbar
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=series["color"],
        linewidth=2.35,
        marker=series["marker"],
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.35,
        label=series["label"],
    )
    for series in RESULT_SERIES
]

figure.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.04,
    ),
    ncol=3,
    fontsize=12.5,
    handlelength=2.2,
    columnspacing=2.2,
    borderaxespad=0.0,
)

figure.subplots_adjust(
    left=0.055,
    right=0.925,
    top=0.81,
    bottom=0.15,
    wspace=0.30,
)

# Dedicated colorbar aligned with the contour panel.
contour_position = axis_contour.get_position()

colorbar_axis = figure.add_axes(
    [
        contour_position.x1 + 0.012,
        contour_position.y0,
        0.012,
        contour_position.height,
    ]
)

colorbar = figure.colorbar(
    contour,
    cax=colorbar_axis,
    extend="max",
)

colorbar.set_label(
    r"Free energy (kcal mol$^{-1}$)",
    labelpad=8,
)


# ============================================================
# 17. Save figure
# ============================================================

figure.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

if DISPLAY_FIGURE:
    plt.show()

plt.close(
    figure
)


# ============================================================
# 18. Final-value summary
# ============================================================

print()
print("=" * 118)
print(
    "Final comparison with normalized optimization progress"
)
print("=" * 118)

for series in RESULT_SERIES:
    result = series[
        "result"
    ]

    print(
        f"{result['name']:>25} | "
        f"states={len(result['progress']):>4d} | "
        f"progress={result['progress'][-1]:.3f} | "
        f"I_G={result['I_G'][-1]:.8e} | "
        f"I_E={result['I_E'][-1]:.8e} | "
        f"DeltaF={result['barrier'][-1]:.6f} kcal/mol"
    )

print("-" * 118)


# ============================================================
# 19. Final-path distances
# ============================================================

final_paths = {
    "MFEP": MFEP_RESULT[
        "final_path_full"
    ],
    "CV-MEP": CV_MEP_RESULT[
        "final_path_full"
    ],
    "66D-MEP": AA_MEP_RESULT[
        "final_path_full"
    ],
}

for first_name, second_name in [
    (
        "MFEP",
        "CV-MEP",
    ),
    (
        "MFEP",
        "66D-MEP",
    ),
    (
        "CV-MEP",
        "66D-MEP",
    ),
]:
    first_path = np.asarray(
        final_paths[first_name],
        dtype=float,
    )

    second_path = np.asarray(
        final_paths[second_name],
        dtype=float,
    )

    if first_path.shape == second_path.shape:
        angular_difference = wrap_angles(
            first_path
            - second_path
        )

        pointwise_distance = np.linalg.norm(
            angular_difference,
            axis=1,
        )

        print(
            f"{first_name:>7} vs {second_name:<7} | "
            f"mean distance="
            f"{np.mean(pointwise_distance):.6e} rad | "
            f"max distance="
            f"{np.max(pointwise_distance):.6e} rad"
        )

    else:
        print(
            f"{first_name:>7} vs {second_name:<7} | "
            f"different path shapes: "
            f"{first_path.shape} vs {second_path.shape}; "
            "pointwise distance skipped."
        )

print("-" * 118)
print(f"MFEP result       : {MFEP_NPZ}")
print(f"CV MEP result     : {MEP_NPZ}")
print(f"66D MEP result    : {ALL_ATOM_MEP_NPZ}")
print(f"66D MD cache      : {AA_CACHE_NPZ}")
print("=" * 118)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")

Input files
MFEP-IVM result        : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam/alanine_MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam.npz
CV MEP-IVM result      : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam/alanine_MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam.npz
66D projected MD cache : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/Ala2_66D_projected_MEP_restrained_MD_cache.npz
66D original result    : /Users/wuzhiyou/Downloads/Ala2_N20_lr0.04_alphaIVP100_Nsub5_results.npz
FES                     : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/wtmeta_phi_psi/fes_XYZ.npz

Loaded optimization sequences
                 MFEP-IVM | states= 201 | raw start=      0.00 | raw final=    200.00 | progress=[0.000, 1.000]
     CV Euclidean MEP-IVM | states= 201 | raw s

In [21]:
# -*- coding: utf-8 -*-
# ============================================================
# Final-path comparison:
#
# 一行两张：
#   (a) Full FES contour:
#           [-pi, 0] x [0, pi]
#
#   (b) Zoomed contour:
#           [-3, -1.5] x [1.5, 3]
#
# Paths:
#   1. MFEP-IVM, G = M^{-1}
#   2. CV Euclidean MEP-IVM, G = I
#   3. Projected 66D all-atom MEP
#
# 只读取已有结果，不重新运行 restrained MD。
# ============================================================

import os
import re
import numpy as np

import matplotlib

# False: 只保存图片，不弹窗。
# True : 保存并显示图片。
DISPLAY_FIGURE = False

if not DISPLAY_FIGURE:
    matplotlib.use("Agg")

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# 0. File paths
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# ------------------------------------------------------------
# MFEP-IVM result
# ------------------------------------------------------------

MFEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# CV Euclidean MEP-IVM result
# ------------------------------------------------------------

MEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# Existing projected-66D restrained-MD cache
# ------------------------------------------------------------

AA_CACHE_NPZ = os.path.join(
    BASE_PATH,
    "Ala2_66D_projected_MEP_restrained_MD_cache.npz",
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        "MFEP_CVMEP_66DMEP"
        "_full_and_zoom_contour"
    ),
)


# ============================================================
# 1. Plotting windows
# ============================================================

FULL_XLIM = (
    -np.pi,
    0.0,
)

FULL_YLIM = (
    0.0,
    np.pi,
)

ZOOM_XLIM = (
    -2.8,
    -1.5,
)

ZOOM_YLIM = (
    1.8,
    3.1,
)


# ============================================================
# 2. Fixed endpoints and metastable states
# ============================================================

C1 = np.asarray(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.asarray(
    [-1.33047160, 0.99118795],
    dtype=float,
)

SPECIAL_POINTS = [
    {
        "x": -2.62593029,
        "y": 2.77365591,
        "label": r"$C5$",
        "dx": -0.05,
        "dy": 0.16,
    },
    {
        "x": -1.33433281,
        "y": 0.99268968,
        "label": r"$C7_{\mathrm{eq}}$",
        "dx": 0.10,
        "dy": -0.40,
    },
]


# ============================================================
# 3. Path colors and markers
# ============================================================

COLOR_MFEP = "#0072B2"      # blue
COLOR_CV_MEP = "#D50087"    # magenta
COLOR_66D_MEP = "#009E73"   # green

MARKER_MFEP = "o"
MARKER_CV_MEP = "s"
MARKER_66D_MEP = "^"


# ============================================================
# 4. Tuned black-to-white contour
# ============================================================

# Free energy above this value is white.
ENERGY_WHITE_CAP = 60.0

# Minimum-energy gray value.
#
# 0.0 = pure black
# 0.05 = almost black
BG_LO = 0.05

# Quadratic grayscale mapping.
CURV = 2.0

CONTOUR_LEVELS = 33


# ============================================================
# 5. Check files
# ============================================================

for label, path in [
    ("FES", FES_NPZ),
    ("MFEP-IVM", MFEP_NPZ),
    ("CV MEP-IVM", MEP_NPZ),
    ("Projected 66D MEP cache", AA_CACHE_NPZ),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find {label} file:\n"
            f"  {path}"
        )

print("=" * 100)
print("Input files")
print("=" * 100)
print(f"FES            : {FES_NPZ}")
print(f"MFEP-IVM       : {MFEP_NPZ}")
print(f"CV MEP-IVM     : {MEP_NPZ}")
print(f"Projected 66D  : {AA_CACHE_NPZ}")
print("=" * 100)


# ============================================================
# 6. Helpers
# ============================================================

def extract_iteration_numbers(
    loaded,
    prefix,
):
    """
    Extract iteration numbers from keys such as:

        path_iter_0
        path_iter_1
        ...
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    iteration_numbers = []

    for key in loaded.files:
        match = pattern.match(key)

        if match is not None:
            iteration_numbers.append(
                int(match.group(1))
            )

    return sorted(
        iteration_numbers
    )


def path_with_fixed_endpoints(
    internal_path,
):
    """
    Add fixed endpoints to a CV path containing only internal images.
    """
    internal_path = np.asarray(
        internal_path,
        dtype=float,
    )

    if (
        internal_path.ndim != 2
        or internal_path.shape[1] != 2
    ):
        raise ValueError(
            f"Expected internal path shape (N,2), "
            f"got {internal_path.shape}."
        )

    return np.vstack(
        [
            C1,
            internal_path,
            C2,
        ]
    )


def load_final_cv_path(
    npz_path,
    method_name,
):
    """
    Load the final post-update/reparameterized CV path.

    Priority:
        path_iter_<largest iteration>

    Fallback:
        phi_ref_iter_<largest iteration>
    """
    with np.load(
        npz_path,
        allow_pickle=True,
    ) as loaded:

        path_iterations = extract_iteration_numbers(
            loaded,
            "path_iter_",
        )

        if path_iterations:
            final_iteration = path_iterations[-1]

            internal_path = np.asarray(
                loaded[
                    f"path_iter_{final_iteration}"
                ],
                dtype=float,
            )

            source_key = (
                f"path_iter_{final_iteration}"
            )

        else:
            reference_iterations = extract_iteration_numbers(
                loaded,
                "phi_ref_iter_",
            )

            if not reference_iterations:
                raise RuntimeError(
                    f"{method_name}: no path_iter_* or "
                    f"phi_ref_iter_* keys in:\n"
                    f"  {npz_path}"
                )

            final_iteration = reference_iterations[-1]

            internal_path = np.asarray(
                loaded[
                    f"phi_ref_iter_{final_iteration}"
                ],
                dtype=float,
            )

            source_key = (
                f"phi_ref_iter_{final_iteration}"
            )

    full_path = path_with_fixed_endpoints(
        internal_path
    )

    print(
        f"{method_name:>24}: "
        f"loaded {source_key}, "
        f"shape={full_path.shape}"
    )

    return full_path


def load_final_66d_projected_path(
    cache_path,
):
    """
    Load the final projected 66D MEP path from the existing cache.

    The cache paths already include both endpoints.
    """
    with np.load(
        cache_path,
        allow_pickle=True,
    ) as loaded:

        if "paths" not in loaded.files:
            raise KeyError(
                f"Cache has no 'paths' key:\n"
                f"  {cache_path}"
            )

        paths = np.asarray(
            loaded["paths"],
            dtype=float,
        )

        if paths.ndim != 3:
            raise ValueError(
                f"Expected cached paths shape "
                f"(n_states,n_images,2), got {paths.shape}."
            )

        if paths.shape[2] != 2:
            raise ValueError(
                f"Expected final coordinate dimension 2, "
                f"got paths shape {paths.shape}."
            )

        # Use source_indices or count to identify the final state.
        if (
            "source_indices" in loaded.files
            and len(loaded["source_indices"]) == paths.shape[0]
        ):
            source_indices = np.asarray(
                loaded["source_indices"],
                dtype=int,
            )

            final_array_index = int(
                np.argmax(
                    source_indices
                )
            )

            final_identifier = int(
                source_indices[
                    final_array_index
                ]
            )

            identifier_name = "source state"

        elif (
            "count" in loaded.files
            and len(loaded["count"]) == paths.shape[0]
        ):
            counts = np.asarray(
                loaded["count"],
                dtype=float,
            )

            final_array_index = int(
                np.argmax(
                    counts
                )
            )

            final_identifier = float(
                counts[
                    final_array_index
                ]
            )

            identifier_name = "count"

        else:
            final_array_index = (
                paths.shape[0]
                - 1
            )

            final_identifier = final_array_index
            identifier_name = "array index"

        final_path = paths[
            final_array_index
        ].copy()

    if (
        final_path.ndim != 2
        or final_path.shape[1] != 2
    ):
        raise ValueError(
            f"Unexpected projected 66D path shape: "
            f"{final_path.shape}"
        )

    print(
        f"{'Projected 66D MEP':>24}: "
        f"loaded {identifier_name}={final_identifier}, "
        f"shape={final_path.shape}"
    )

    return final_path


def shift_path_to_window(
    path,
    xlim,
    ylim,
):
    """
    Shift a continuous angular path by common multiples of 2*pi,
    separately in phi and psi, so its center lies near the selected
    plotting window.

    The same shift is applied to all path images in one coordinate.
    """
    path = np.asarray(
        path,
        dtype=float,
    ).copy()

    window_center = np.asarray(
        [
            0.5 * (
                xlim[0]
                + xlim[1]
            ),
            0.5 * (
                ylim[0]
                + ylim[1]
            ),
        ],
        dtype=float,
    )

    for coordinate in range(2):
        path_center = float(
            np.mean(
                path[:, coordinate]
            )
        )

        shift_count = np.round(
            (
                window_center[coordinate]
                - path_center
            )
            / (
                2.0 * np.pi
            )
        )

        path[:, coordinate] += (
            shift_count
            * 2.0
            * np.pi
        )

    return path


def crop_regular_grid(
    grid_x,
    grid_y,
    grid_z,
    xlim,
    ylim,
):
    """
    Crop a regular meshgrid to the requested window.
    """
    inside = (
        (grid_x >= xlim[0])
        & (grid_x <= xlim[1])
        & (grid_y >= ylim[0])
        & (grid_y <= ylim[1])
    )

    if not np.any(inside):
        raise RuntimeError(
            f"No FES grid points inside "
            f"xlim={xlim}, ylim={ylim}."
        )

    rows = np.where(
        np.any(
            inside,
            axis=1,
        )
    )[0]

    columns = np.where(
        np.any(
            inside,
            axis=0,
        )
    )[0]

    row_start = rows[0]
    row_stop = rows[-1] + 1

    column_start = columns[0]
    column_stop = columns[-1] + 1

    return (
        grid_x[
            row_start:row_stop,
            column_start:column_stop,
        ],
        grid_y[
            row_start:row_stop,
            column_start:column_stop,
        ],
        grid_z[
            row_start:row_stop,
            column_start:column_stop,
        ],
    )


def point_inside_window(
    x,
    y,
    xlim,
    ylim,
):
    return (
        xlim[0] <= x <= xlim[1]
        and ylim[0] <= y <= ylim[1]
    )


# ============================================================
# 7. Load final paths
# ============================================================

PATH_MFEP = load_final_cv_path(
    MFEP_NPZ,
    "MFEP-IVM",
)

PATH_CV_MEP = load_final_cv_path(
    MEP_NPZ,
    "CV MEP-IVM",
)

PATH_66D_MEP = load_final_66d_projected_path(
    AA_CACHE_NPZ
)


# ============================================================
# 8. Load FES
# ============================================================

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as loaded:

    grid_x = np.asarray(
        loaded["X"],
        dtype=float,
    )

    grid_y = np.asarray(
        loaded["Y"],
        dtype=float,
    )

    grid_z = np.asarray(
        loaded["Z"],
        dtype=float,
    )

if (
    grid_x.ndim == 1
    and grid_y.ndim == 1
):
    grid_x, grid_y = np.meshgrid(
        grid_x,
        grid_y,
        indexing="xy",
    )

if (
    grid_x.shape != grid_y.shape
    or grid_x.shape != grid_z.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{grid_x.shape}, "
        f"Y{grid_y.shape}, "
        f"Z{grid_z.shape}."
    )

# FES coordinates are stored in degrees.
grid_x = np.deg2rad(
    grid_x
)

grid_y = np.deg2rad(
    grid_y
)

grid_z = np.ma.masked_invalid(
    grid_z
)

# Use the minimum over the full displayed region as the common zero.
full_inside = (
    (grid_x >= FULL_XLIM[0])
    & (grid_x <= FULL_XLIM[1])
    & (grid_y >= FULL_YLIM[0])
    & (grid_y <= FULL_YLIM[1])
)

if not np.any(full_inside):
    raise RuntimeError(
        "No FES points inside the full plotting window."
    )

common_energy_minimum = float(
    np.ma.min(
        grid_z[
            full_inside
        ]
    )
)

grid_z_relative = (
    grid_z
    - common_energy_minimum
)

# Full-region FES.
X_FULL, Y_FULL, Z_FULL = crop_regular_grid(
    grid_x,
    grid_y,
    grid_z_relative,
    FULL_XLIM,
    FULL_YLIM,
)

# Zoom-region FES.
X_ZOOM, Y_ZOOM, Z_ZOOM = crop_regular_grid(
    grid_x,
    grid_y,
    grid_z_relative,
    ZOOM_XLIM,
    ZOOM_YLIM,
)


# ============================================================
# 9. Tuned black-to-white colormap
# ============================================================

unit_values = np.linspace(
    0.0,
    1.0,
    256,
)

mapped_values = (
    unit_values
    + CURV
    * unit_values
    * (
        1.0
        - unit_values
    )
)

mapped_values = np.clip(
    mapped_values,
    0.0,
    1.0,
)

gray_values = (
    BG_LO
    + (
        1.0
        - BG_LO
    )
    * mapped_values
)

BACKGROUND_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad",
    list(
        zip(
            gray_values,
            gray_values,
            gray_values,
        )
    ),
)

BACKGROUND_CMAP.set_over(
    "white"
)

contour_levels = np.linspace(
    0.0,
    ENERGY_WHITE_CAP,
    CONTOUR_LEVELS,
)


# ============================================================
# 10. Plot style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 11. Path-series definitions
# ============================================================

PATH_SERIES = [
    {
        "path": PATH_MFEP,
        "color": COLOR_MFEP,
        "marker": MARKER_MFEP,
        "label": r"MFEP-IVM ($G=M^{-1}$)",
        "zorder": 9,
    },
    {
        "path": PATH_CV_MEP,
        "color": COLOR_CV_MEP,
        "marker": MARKER_CV_MEP,
        "label": r"CV MEP-IVM ($G=I$)",
        "zorder": 8,
    },
    {
        "path": PATH_66D_MEP,
        "color": COLOR_66D_MEP,
        "marker": MARKER_66D_MEP,
        "label": r"Projected 66D MEP",
        "zorder": 7,
    },
]


# ============================================================
# 12. Panel drawing helper
# ============================================================

def draw_contour_panel(
    axis,
    contour_x,
    contour_y,
    contour_z,
    xlim,
    ylim,
    title,
    panel_tag,
):
    """
    Draw one contour panel with all three paths.
    """
    contour = axis.contourf(
        contour_x,
        contour_y,
        contour_z,
        levels=contour_levels,
        cmap=BACKGROUND_CMAP,
        extend="max",
    )

    # --------------------------------------------------------
    # Metastable states
    # --------------------------------------------------------
    for state in SPECIAL_POINTS:
        state_x = state["x"]
        state_y = state["y"]

        if not point_inside_window(
            state_x,
            state_y,
            xlim,
            ylim,
        ):
            continue

        axis.plot(
            state_x,
            state_y,
            marker="o",
            markersize=6.5,
            markerfacecolor="white",
            markeredgecolor="#111111",
            markeredgewidth=1.4,
            linestyle="None",
            zorder=15,
        )

        # Keep labels inside the zoomed panel if needed.
        text_x = (
            state_x
            + state["dx"]
        )

        text_y = (
            state_y
            + state["dy"]
        )

        if text_x < xlim[0]:
            text_x = (
                state_x
                + 0.18
            )

        if text_x > xlim[1]:
            text_x = (
                state_x
                - 0.18
            )

        if text_y < ylim[0]:
            text_y = (
                state_y
                + 0.15
            )

        if text_y > ylim[1]:
            text_y = (
                state_y
                - 0.15
            )

        axis.text(
            text_x,
            text_y,
            state["label"],
            color="#111111",
            fontsize=12,
            fontweight="bold",
            zorder=15,
            ha="right",
            va="bottom",
            bbox={
                "boxstyle": "round,pad=0.12",
                "fc": "white",
                "ec": "none",
                "alpha": 0.75,
            },
        )

    # --------------------------------------------------------
    # Three paths
    # --------------------------------------------------------
    for series in PATH_SERIES:
        plot_path = shift_path_to_window(
            series["path"],
            xlim,
            ylim,
        )

        axis.plot(
            plot_path[:, 0],
            plot_path[:, 1],
            color=series["color"],
            linestyle="-",
            linewidth=2.45,
            marker=series["marker"],
            markevery=1,
            markersize=5.4,
            markerfacecolor="white",
            markeredgecolor=series["color"],
            markeredgewidth=1.4,
            solid_capstyle="round",
            label=series["label"],
            zorder=series["zorder"],
            clip_on=True,
        )

    axis.set_xlabel(
        r"$\phi$ (rad)",
        labelpad=5,
    )

    axis.set_ylabel(
        r"$\psi$ (rad)",
        labelpad=5,
    )

    axis.set_title(
        title,
        pad=12,
        fontweight="medium",
    )

    axis.set_xlim(
        xlim
    )

    axis.set_ylim(
        ylim
    )

    axis.set_aspect(
        "equal",
        adjustable="box",
    )

    axis.tick_params(
        which="both",
        top=False,
        right=False,
    )

    axis.text(
        -0.08,
        1.06,
        panel_tag,
        transform=axis.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left",
    )

    return contour


# ============================================================
# 13. Draw one-row, two-panel figure
# ============================================================

figure, axes = plt.subplots(
    1,
    2,
    figsize=(
        11.8,
        5.1,
    ),
)

axis_full = axes[0]
axis_zoom = axes[1]

contour_full = draw_contour_panel(
    axis=axis_full,
    contour_x=X_FULL,
    contour_y=Y_FULL,
    contour_z=Z_FULL,
    xlim=FULL_XLIM,
    ylim=FULL_YLIM,
    title="Full free-energy landscape",
    panel_tag="a",
)

contour_zoom = draw_contour_panel(
    axis=axis_zoom,
    contour_x=X_ZOOM,
    contour_y=Y_ZOOM,
    contour_z=Z_ZOOM,
    xlim=ZOOM_XLIM,
    ylim=ZOOM_YLIM,
    title="Zoomed transition region",
    panel_tag="b",
)


# ============================================================
# 14. Shared legend
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=series["color"],
        linewidth=2.45,
        marker=series["marker"],
        markersize=6,
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.4,
        label=series["label"],
    )
    for series in PATH_SERIES
]

figure.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.03,
    ),
    ncol=3,
    fontsize=12.2,
    handlelength=2.2,
    columnspacing=1.8,
    borderaxespad=0.0,
)


# ============================================================
# 15. Layout and shared colorbar
# ============================================================

figure.subplots_adjust(
    left=0.075,
    right=0.90,
    top=0.82,
    bottom=0.14,
    wspace=0.20,
)

# Colorbar aligned with the right panel.
zoom_position = axis_zoom.get_position()

colorbar_axis = figure.add_axes(
    [
        zoom_position.x1 + 0.018,
        zoom_position.y0,
        0.018,
        zoom_position.height,
    ]
)

colorbar = figure.colorbar(
    contour_zoom,
    cax=colorbar_axis,
    extend="max",
)

colorbar.set_label(
    r"Free energy (kcal mol$^{-1}$)",
    labelpad=8,
)


# ============================================================
# 16. Save
# ============================================================

figure.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

if DISPLAY_FIGURE:
    plt.show()

plt.close(
    figure
)


# ============================================================
# 17. Summary
# ============================================================

print()
print("=" * 100)
print("Final path comparison figure")
print("=" * 100)
print(
    f"Full window : "
    f"{FULL_XLIM} x {FULL_YLIM}"
)
print(
    f"Zoom window : "
    f"{ZOOM_XLIM} x {ZOOM_YLIM}"
)
print("-" * 100)

for series in PATH_SERIES:
    print(
        f"{series['label']:>32} | "
        f"path shape={series['path'].shape}"
    )

print("-" * 100)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")
print("=" * 100)

Input files
FES            : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/wtmeta_phi_psi/fes_XYZ.npz
MFEP-IVM       : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam/alanine_MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam.npz
CV MEP-IVM     : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam/alanine_MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam.npz
Projected 66D  : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/Ala2_66D_projected_MEP_restrained_MD_cache.npz
                MFEP-IVM: loaded path_iter_200, shape=(22, 2)
              CV MEP-IVM: loaded path_iter_200, shape=(22, 2)
       Projected 66D MEP: loaded source state=117, shape=(22, 2)

Final path comparison figure
Full window : (-3.141592653589793, 0.0) x (0.0, 3.141592653589793)
Zoom window : (-2.8, -1.5) x (1.8,

In [29]:
# -*- coding: utf-8 -*-
# ============================================================
# MFEP-IVM vs CV Euclidean-MEP-IVM vs projected 66D MEP
#
# Plot only: no restrained MD is run in this script.
#
# 一行三张：
#   (a) MFEP residual I_perp^G
#   (b) Euclidean MEP residual I_perp^E
#   (c) Free-energy barrier Delta F^dagger
#
# 三个指标图的横坐标统一为：
#
#   Normalized optimization progress in [0, 1]
#
# 每种方法分别按照自己的 count / iteration 归一化：
#
#   progress = (count-count_initial)/(count_final-count_initial)
#
# 本脚本直接读取已经保存好的：
#
#   1. MFEP-IVM NPZ
#   2. CV Euclidean MEP-IVM NPZ
#   3. Projected 66D MEP restrained-MD cache
#
# 不会重新执行 restrained MD。
#
# *** 修改说明 ***
# MFEP-IVM 和 CV MEP-IVM 的 progress=0 数据点，
# 已强制替换为公共初始路径的评估值 (MFEP_INITIAL_PATH_common.npz)，
# 以使所有方法的初始状态基准一致。
# ============================================================

import os
import re
import numpy as np

import matplotlib

# 不弹窗；只保存图片。
DISPLAY_FIGURE = False

if not DISPLAY_FIGURE:
    matplotlib.use("Agg")

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, NullFormatter
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# 0. File paths
# ============================================================

BASE_PATH = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

FES_NPZ = os.path.join(
    BASE_PATH,
    "wtmeta_phi_psi",
    "fes_XYZ.npz",
)

# ------------------------------------------------------------
# MFEP-IVM result
# ------------------------------------------------------------
MFEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MFEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.0001"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# CV Euclidean MEP-IVM result
# ------------------------------------------------------------
MEP_NPZ = os.path.join(
    BASE_PATH,
    (
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam"
    ),
    (
        "alanine_"
        "MEP_image_LBFGS_B_Recycle_N20"
        "_Nsub1"
        "_alpha0.03"
        "_maxstep0.1"
        "_hist7"
        "_NEBreparam.npz"
    ),
)

# ------------------------------------------------------------
# Correct all-atom 66D MEP source data
#
# This file is included only for metadata/output reporting.
# The metric curves are loaded from AA_CACHE_NPZ.
# ------------------------------------------------------------
ALL_ATOM_MEP_NPZ = (
    "/Users/wuzhiyou/Downloads/"
    "Ala2_N20_lr0.04_alphaIVP100_Nsub5_results.npz"
)

# ------------------------------------------------------------
# Previously generated projected-66D restrained-MD cache
# ------------------------------------------------------------
AA_CACHE_NPZ = os.path.join(
    BASE_PATH,
    "Ala2_66D_projected_MEP_restrained_MD_cache.npz",
)

# ------------------------------------------------------------
# Public initial path (common starting point)
# ------------------------------------------------------------
INIT_NPZ = os.path.join(
    BASE_PATH,
    "MFEP_INITIAL_PATH_common.npz",
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
OUTPUT_STEM = os.path.join(
    BASE_PATH,
    (
        "MFEP_vs_CV_MEP_vs_projected_66D_MEP"
        "_normalized_progress"
        "_metrics_and_contour"
    ),
)


# ============================================================
# 1. Basic settings
# ============================================================

N_IMAGES = 20

# Fixed CV endpoints.
C1 = np.asarray(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.asarray(
    [-1.33047160, 0.99118795],
    dtype=float,
)

XLIM = (
    -np.pi,
    0.0,
)

YLIM = (
    0.0,
    np.pi,
)

SPECIAL_POINTS = [
    (
        -2.62593029,
        2.77365591,
        r"$C5$",
        -0.05,
        0.16,
    ),
    (
        -1.33433281,
        0.99268968,
        r"$C7_{\mathrm{eq}}$",
        0.10,
        -0.40,
    ),
]


# ============================================================
# 2. Colors and markers
# ============================================================

# Color-blind-friendly colors.
COLOR_MFEP = "#0072B2"       # blue
COLOR_CV_MEP = "#D50087"     # magenta
COLOR_66D_MEP = "#009E73"    # green

MARKER_MFEP = "o"
MARKER_CV_MEP = "s"
MARKER_66D_MEP = "^"


# ============================================================
# 3. Tuned black-to-white contour settings
# ============================================================

# Free energy at and above this value is white.
ENERGY_WHITE_CAP = 60.0

# Minimum-energy grayscale value.
#
# 0.0 = pure black
# 0.05 = nearly black
BG_LO = 0.05

# Quadratic grayscale mapping.
CURV = 2.0

CONTOUR_LEVELS = 33


# ============================================================
# 4. Check input files
# ============================================================

for label, path in [
    ("FES", FES_NPZ),
    ("MFEP-IVM", MFEP_NPZ),
    ("CV MEP-IVM", MEP_NPZ),
    ("Projected 66D MEP cache", AA_CACHE_NPZ),
    ("Initial common path", INIT_NPZ),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find {label} file:\n"
            f"  {path}"
        )

print("=" * 100)
print("Input files")
print("=" * 100)
print(f"MFEP-IVM result        : {MFEP_NPZ}")
print(f"CV MEP-IVM result      : {MEP_NPZ}")
print(f"66D projected MD cache : {AA_CACHE_NPZ}")
print(f"66D original result    : {ALL_ATOM_MEP_NPZ}")
print(f"FES                     : {FES_NPZ}")
print(f"Initial common path     : {INIT_NPZ}")
print("=" * 100)


# ============================================================
# 5. General helpers
# ============================================================

def extract_iteration_numbers(
    loaded,
    prefix,
):
    """
    Extract iteration numbers from keys such as:

        phi_ref_iter_0
        phi_ref_iter_1
        ...
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}(\d+)$"
    )

    iterations = []

    for key in loaded.files:
        match = pattern.match(key)

        if match is not None:
            iterations.append(
                int(match.group(1))
            )

    return sorted(iterations)


def normalized_progress(
    raw_coordinate,
):
    """
    Normalize any monotone optimization coordinate to [0, 1].

    The first finite point becomes 0 and the final finite point
    becomes 1.

    If only one point exists, it is placed at progress=1.
    """
    raw_coordinate = np.asarray(
        raw_coordinate,
        dtype=float,
    ).reshape(-1)

    if raw_coordinate.size == 0:
        return np.zeros(
            0,
            dtype=float,
        )

    if raw_coordinate.size == 1:
        return np.ones(
            1,
            dtype=float,
        )

    first_value = float(
        raw_coordinate[0]
    )

    final_value = float(
        raw_coordinate[-1]
    )

    denominator = (
        final_value
        - first_value
    )

    if (
        not np.isfinite(denominator)
        or abs(denominator) <= 1e-16
    ):
        return np.linspace(
            0.0,
            1.0,
            raw_coordinate.size,
        )

    progress = (
        raw_coordinate
        - first_value
    ) / denominator

    return np.clip(
        progress,
        0.0,
        1.0,
    )


def wrap_angles(
    values,
):
    """
    Wrap angular differences to [-pi, pi).
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        (values + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )


def path_with_fixed_endpoints(
    internal_path,
):
    """
    Add the two fixed CV endpoints to an internal-image path.
    """
    internal_path = np.asarray(
        internal_path,
        dtype=float,
    )

    return np.vstack(
        [
            C1,
            internal_path,
            C2,
        ]
    )


def shift_path_to_plot_window(
    path,
):
    """
    Shift an unwrapped angular path by common multiples of 2*pi
    so that it lies near the requested plotting window.

    The same shift is applied to all images in each coordinate,
    preserving path continuity.
    """
    path = np.asarray(
        path,
        dtype=float,
    ).copy()

    target_center = np.asarray(
        [
            0.5 * (
                XLIM[0]
                + XLIM[1]
            ),
            0.5 * (
                YLIM[0]
                + YLIM[1]
            ),
        ],
        dtype=float,
    )

    for coordinate in range(2):
        path_center = float(
            np.mean(
                path[:, coordinate]
            )
        )

        shift_count = np.round(
            (
                target_center[coordinate]
                - path_center
            )
            / (
                2.0 * np.pi
            )
        )

        path[:, coordinate] += (
            shift_count
            * 2.0
            * np.pi
        )

    return path


# ============================================================
# 6. Common geometry and residual definitions
# ============================================================

def euclidean_normal_projector(
    tangent,
    eps=1e-16,
):
    """
    Euclidean normal projector:

        P_E = I - tau_hat tau_hat^T.
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    tangent_norm = float(
        np.linalg.norm(
            tangent
        )
    )

    if tangent_norm <= eps:
        return np.eye(
            tangent.size,
            dtype=float,
        )

    unit_tangent = (
        tangent
        / tangent_norm
    )

    return (
        np.eye(
            tangent.size,
            dtype=float,
        )
        - np.outer(
            unit_tangent,
            unit_tangent,
        )
    )


def G_normal_projector(
    tangent,
    G,
    eps=1e-16,
):
    """
    G-orthogonal normal projector:

        P_G
        =
        I - tangent (tangent^T G)/(tangent^T G tangent).
    """
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    G = np.asarray(
        G,
        dtype=float,
    )

    denominator = float(
        tangent
        @ G
        @ tangent
    )

    if denominator <= eps:
        return np.eye(
            tangent.size,
            dtype=float,
        )

    return (
        np.eye(
            tangent.size,
            dtype=float,
        )
        - np.outer(
            tangent,
            tangent @ G,
        )
        / denominator
    )


def central_tangents_from_full_path(
    full_path,
):
    """
    Central-difference tangent at each internal image.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


def compute_euclidean_node_weights(
    full_path,
):
    """
    Euclidean arclength quadrature weights.
    """
    internal_path = full_path[
        1:-1
    ]

    segment_minus = (
        internal_path
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal_path
    )

    return 0.5 * (
        np.linalg.norm(
            segment_minus,
            axis=1,
        )
        +
        np.linalg.norm(
            segment_plus,
            axis=1,
        )
    )


def compute_G_node_weights(
    full_path,
    G_all,
):
    """
    G-arclength quadrature weights.
    """
    internal_path = full_path[
        1:-1
    ]

    segment_minus = (
        internal_path
        - full_path[:-2]
    )

    segment_plus = (
        full_path[2:]
        - internal_path
    )

    segment_minus_G_sq = np.einsum(
        "ni,nij,nj->n",
        segment_minus,
        G_all,
        segment_minus,
    )

    segment_plus_G_sq = np.einsum(
        "ni,nij,nj->n",
        segment_plus,
        G_all,
        segment_plus,
    )

    return 0.5 * (
        np.sqrt(
            np.maximum(
                segment_minus_G_sq,
                0.0,
            )
        )
        +
        np.sqrt(
            np.maximum(
                segment_plus_G_sq,
                0.0,
            )
        )
    )


def compute_MEP_residual(
    full_path,
    grad_F,
):
    """
    Euclidean free-energy MEP residual:

        I_perp^E
        =
        sum_i |P_E grad F|^2 w_i^E.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual_vectors = np.zeros_like(
        grad_F
    )

    for image_index in range(
        number_internal
    ):
        projector = euclidean_normal_projector(
            tangents[image_index]
        )

        residual_vectors[image_index] = (
            projector
            @ grad_F[image_index]
        )

    pointwise_squared_norm = np.sum(
        residual_vectors ** 2,
        axis=1,
    )

    node_weights = compute_euclidean_node_weights(
        full_path
    )

    residual_value = float(
        np.sum(
            pointwise_squared_norm
            * node_weights
        )
    )

    return {
        "value": residual_value,
        "vectors": residual_vectors,
        "pointwise_sq": pointwise_squared_norm,
        "weights": node_weights,
    }


def compute_MFEP_residual(
    full_path,
    grad_F,
    M_all,
):
    """
    MFEP residual under G=M^{-1}:

        R_G = P_G M grad F,

        I_perp^G
        =
        sum_i R_G^T G R_G w_i^G.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    if M_all.shape != (
        number_internal,
        2,
        2,
    ):
        raise ValueError(
            f"M shape {M_all.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    G_all = np.linalg.inv(
        M_all
    )

    M_grad_F = np.einsum(
        "nij,nj->ni",
        M_all,
        grad_F,
    )

    tangents = central_tangents_from_full_path(
        full_path
    )

    residual_vectors = np.zeros_like(
        M_grad_F
    )

    for image_index in range(
        number_internal
    ):
        projector = G_normal_projector(
            tangents[image_index],
            G_all[image_index],
        )

        residual_vectors[image_index] = (
            projector
            @ M_grad_F[image_index]
        )

    pointwise_G_squared_norm = np.einsum(
        "ni,nij,nj->n",
        residual_vectors,
        G_all,
        residual_vectors,
    )

    pointwise_G_squared_norm = np.maximum(
        pointwise_G_squared_norm,
        0.0,
    )

    node_weights = compute_G_node_weights(
        full_path,
        G_all,
    )

    residual_value = float(
        np.sum(
            pointwise_G_squared_norm
            * node_weights
        )
    )

    return {
        "value": residual_value,
        "vectors": residual_vectors,
        "pointwise_G_sq": pointwise_G_squared_norm,
        "weights": node_weights,
        "G": G_all,
    }


def recover_free_energy_profile(
    full_path,
    grad_F,
):
    """
    Integrate grad F along the complete path.

    grad F is available at the internal images. The first and last
    internal gradients are used on the endpoint segments.
    """
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    grad_F = np.asarray(
        grad_F,
        dtype=float,
    )

    number_internal = (
        full_path.shape[0]
        - 2
    )

    if grad_F.shape != (
        number_internal,
        2,
    ):
        raise ValueError(
            f"grad_F shape {grad_F.shape} is incompatible "
            f"with path shape {full_path.shape}."
        )

    path_segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (
            number_internal + 1,
            2,
        ),
        dtype=float,
    )

    segment_gradients[0] = (
        grad_F[0]
    )

    if number_internal > 1:
        segment_gradients[
            1:number_internal
        ] = 0.5 * (
            grad_F[:-1]
            + grad_F[1:]
        )

    segment_gradients[
        number_internal
    ] = grad_F[-1]

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        path_segments,
    )

    profile = np.zeros(
        number_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= np.min(
        profile
    )

    barrier = float(
        np.max(profile)
        - np.min(profile)
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "increments": increments,
    }


def compute_all_metrics(
    full_path,
    grad_F,
    M_all,
):
    """
    Compute the three common path metrics.
    """
    mep_result = compute_MEP_residual(
        full_path,
        grad_F,
    )

    mfep_result = compute_MFEP_residual(
        full_path,
        grad_F,
        M_all,
    )

    free_energy_result = recover_free_energy_profile(
        full_path,
        grad_F,
    )

    return {
        "I_E": mep_result["value"],
        "I_G": mfep_result["value"],
        "barrier": free_energy_result["barrier"],
        "free_energy_profile": free_energy_result[
            "profile"
        ],
    }


# ============================================================
# 7. Load the two CV-space results
# ============================================================

def load_cv_result(
    npz_path,
    method_name,
):
    """
    Load a CV-space IVM run and recompute all three metrics using
    the common definitions above.
    """
    with np.load(
        npz_path,
        allow_pickle=True,
    ) as loaded:

        iterations = extract_iteration_numbers(
            loaded,
            "phi_ref_iter_",
        )

        if not iterations:
            raise RuntimeError(
                f"{method_name}: no phi_ref_iter_* keys in:\n"
                f"  {npz_path}"
            )

        # Optional stored cost coordinate.
        count_map = {}

        if (
            "count_seq" in loaded.files
            and "index_seq" in loaded.files
        ):
            stored_counts = np.asarray(
                loaded["count_seq"],
                dtype=float,
            ).reshape(-1)

            stored_indices = np.asarray(
                loaded["index_seq"],
                dtype=int,
            ).reshape(-1)

            for index_value, count_value in zip(
                stored_indices,
                stored_counts,
            ):
                count_map[
                    int(index_value)
                ] = float(count_value)

        used_iterations = []
        raw_coordinate = []

        I_E_values = []
        I_G_values = []
        barrier_values = []

        reference_paths = []
        profiles = []

        for iteration in iterations:
            path_key = (
                f"phi_ref_iter_{iteration}"
            )

            metric_key = (
                f"M_ref_iter_{iteration}"
            )

            gradient_key = None

            for candidate in [
                f"g_ref_iter_{iteration}",
                f"F_ref_iter_{iteration}",
            ]:
                if candidate in loaded.files:
                    gradient_key = candidate
                    break

            if gradient_key is None:
                continue

            if metric_key not in loaded.files:
                continue

            internal_path = np.asarray(
                loaded[path_key],
                dtype=float,
            )

            grad_F = np.asarray(
                loaded[gradient_key],
                dtype=float,
            )

            M_all = np.asarray(
                loaded[metric_key],
                dtype=float,
            )

            if internal_path.ndim != 2:
                continue

            if internal_path.shape[1] != 2:
                continue

            if grad_F.shape != internal_path.shape:
                continue

            if M_all.shape != (
                internal_path.shape[0],
                2,
                2,
            ):
                continue

            full_path = path_with_fixed_endpoints(
                internal_path
            )

            try:
                metrics = compute_all_metrics(
                    full_path,
                    grad_F,
                    M_all,
                )

            except np.linalg.LinAlgError:
                print(
                    f"[WARNING] {method_name}: "
                    f"singular M at iteration {iteration}; skipping."
                )
                continue

            used_iterations.append(
                iteration
            )

            raw_coordinate.append(
                count_map.get(
                    iteration,
                    float(iteration),
                )
            )

            I_E_values.append(
                metrics["I_E"]
            )

            I_G_values.append(
                metrics["I_G"]
            )

            barrier_values.append(
                metrics["barrier"]
            )

            reference_paths.append(
                full_path.copy()
            )

            profiles.append(
                metrics[
                    "free_energy_profile"
                ].copy()
            )

        if not used_iterations:
            raise RuntimeError(
                f"No usable states were found in:\n"
                f"  {npz_path}"
            )

        # Final post-update/reparameterized path for the contour.
        path_iterations = extract_iteration_numbers(
            loaded,
            "path_iter_",
        )

        if path_iterations:
            final_internal_path = np.asarray(
                loaded[
                    f"path_iter_{path_iterations[-1]}"
                ],
                dtype=float,
            )

            final_full_path = path_with_fixed_endpoints(
                final_internal_path
            )
        else:
            final_full_path = (
                reference_paths[-1].copy()
            )

    raw_coordinate = np.asarray(
        raw_coordinate,
        dtype=float,
    )

    order = np.argsort(
        raw_coordinate
    )

    raw_coordinate = raw_coordinate[
        order
    ]

    used_iterations = np.asarray(
        used_iterations,
        dtype=float,
    )[order]

    I_E_values = np.asarray(
        I_E_values,
        dtype=float,
    )[order]

    I_G_values = np.asarray(
        I_G_values,
        dtype=float,
    )[order]

    barrier_values = np.asarray(
        barrier_values,
        dtype=float,
    )[order]

    reference_paths = [
        reference_paths[index]
        for index in order
    ]

    profiles = [
        profiles[index]
        for index in order
    ]

    return {
        "name": method_name,
        "npz": npz_path,
        "iteration": used_iterations,
        "raw_coordinate": raw_coordinate,
        "progress": normalized_progress(
            raw_coordinate
        ),
        "I_E": I_E_values,
        "I_G": I_G_values,
        "barrier": barrier_values,
        "paths": reference_paths,
        "profiles": profiles,
        "final_path_full": final_full_path,
    }


MFEP_RESULT = load_cv_result(
    npz_path=MFEP_NPZ,
    method_name="MFEP-IVM",
)

CV_MEP_RESULT = load_cv_result(
    npz_path=MEP_NPZ,
    method_name="CV Euclidean MEP-IVM",
)


# ============================================================
# 8. Load the already computed 66D projected-MEP cache
# ============================================================

def load_projected_66d_cache(
    cache_path,
):
    """
    Load previously completed restrained-MD postprocessing of the
    projected 66D MEP.

    No restrained MD is run here.
    """
    with np.load(
        cache_path,
        allow_pickle=True,
    ) as loaded:

        required_keys = [
            "source_indices",
            "count",
            "paths",
            "grad_F",
            "M",
            "I_E",
            "I_G",
            "barrier",
            "profiles",
        ]

        missing_keys = [
            key
            for key in required_keys
            if key not in loaded.files
        ]

        if missing_keys:
            raise KeyError(
                "The projected-66D cache is incomplete. Missing keys:\n"
                + "\n".join(
                    f"  {key}"
                    for key in missing_keys
                )
            )

        source_indices = np.asarray(
            loaded["source_indices"],
            dtype=int,
        ).reshape(-1)

        counts = np.asarray(
            loaded["count"],
            dtype=float,
        ).reshape(-1)

        paths = np.asarray(
            loaded["paths"],
            dtype=float,
        )

        grad_F = np.asarray(
            loaded["grad_F"],
            dtype=float,
        )

        M_all = np.asarray(
            loaded["M"],
            dtype=float,
        )

        stored_I_E = np.asarray(
            loaded["I_E"],
            dtype=float,
        ).reshape(-1)

        stored_I_G = np.asarray(
            loaded["I_G"],
            dtype=float,
        ).reshape(-1)

        stored_barrier = np.asarray(
            loaded["barrier"],
            dtype=float,
        ).reshape(-1)

        profiles = np.asarray(
            loaded["profiles"],
            dtype=float,
        )

    number_states = min(
        source_indices.size,
        counts.size,
        paths.shape[0],
        grad_F.shape[0],
        M_all.shape[0],
        stored_I_E.size,
        stored_I_G.size,
        stored_barrier.size,
        profiles.shape[0],
    )

    source_indices = source_indices[
        :number_states
    ]

    counts = counts[
        :number_states
    ]

    paths = paths[
        :number_states
    ]

    grad_F = grad_F[
        :number_states
    ]

    M_all = M_all[
        :number_states
    ]

    stored_I_E = stored_I_E[
        :number_states
    ]

    stored_I_G = stored_I_G[
        :number_states
    ]

    stored_barrier = stored_barrier[
        :number_states
    ]

    profiles = profiles[
        :number_states
    ]

    # Sort by original all-atom source state/count.
    order = np.lexsort(
        (
            source_indices,
            counts,
        )
    )

    source_indices = source_indices[
        order
    ]

    counts = counts[
        order
    ]

    paths = paths[
        order
    ]

    grad_F = grad_F[
        order
    ]

    M_all = M_all[
        order
    ]

    stored_I_E = stored_I_E[
        order
    ]

    stored_I_G = stored_I_G[
        order
    ]

    stored_barrier = stored_barrier[
        order
    ]

    profiles = profiles[
        order
    ]

    # Remove duplicate source states, retaining the last occurrence.
    unique_source_indices = []
    keep_indices = []

    for array_index, source_index in enumerate(
        source_indices
    ):
        source_index = int(
            source_index
        )

        if (
            unique_source_indices
            and source_index
            == unique_source_indices[-1]
        ):
            keep_indices[-1] = array_index
        else:
            unique_source_indices.append(
                source_index
            )

            keep_indices.append(
                array_index
            )

    keep_indices = np.asarray(
        keep_indices,
        dtype=int,
    )

    source_indices = source_indices[
        keep_indices
    ]

    counts = counts[
        keep_indices
    ]

    paths = paths[
        keep_indices
    ]

    grad_F = grad_F[
        keep_indices
    ]

    M_all = M_all[
        keep_indices
    ]

    stored_I_E = stored_I_E[
        keep_indices
    ]

    stored_I_G = stored_I_G[
        keep_indices
    ]

    stored_barrier = stored_barrier[
        keep_indices
    ]

    profiles = profiles[
        keep_indices
    ]

    # Recompute the metrics from the cached paths/gradients/M to
    # guarantee the same definitions as the two CV methods.
    recomputed_I_E = []
    recomputed_I_G = []
    recomputed_barrier = []
    recomputed_profiles = []

    valid_state_indices = []

    for state_index in range(
        source_indices.size
    ):
        full_path = np.asarray(
            paths[state_index],
            dtype=float,
        )

        state_grad_F = np.asarray(
            grad_F[state_index],
            dtype=float,
        )

        state_M = np.asarray(
            M_all[state_index],
            dtype=float,
        )

        if full_path.ndim != 2:
            continue

        if full_path.shape[1] != 2:
            continue

        if state_grad_F.shape != (
            full_path.shape[0] - 2,
            2,
        ):
            continue

        if state_M.shape != (
            full_path.shape[0] - 2,
            2,
            2,
        ):
            continue

        try:
            metrics = compute_all_metrics(
                full_path,
                state_grad_F,
                state_M,
            )

        except np.linalg.LinAlgError:
            print(
                f"[WARNING] Projected 66D MEP: singular M "
                f"at source state {source_indices[state_index]}; "
                "skipping."
            )
            continue

        valid_state_indices.append(
            state_index
        )

        recomputed_I_E.append(
            metrics["I_E"]
        )

        recomputed_I_G.append(
            metrics["I_G"]
        )

        recomputed_barrier.append(
            metrics["barrier"]
        )

        recomputed_profiles.append(
            metrics[
                "free_energy_profile"
            ]
        )

    if not valid_state_indices:
        raise RuntimeError(
            "No valid projected-66D states were found in the cache."
        )

    valid_state_indices = np.asarray(
        valid_state_indices,
        dtype=int,
    )

    source_indices = source_indices[
        valid_state_indices
    ]

    counts = counts[
        valid_state_indices
    ]

    paths = paths[
        valid_state_indices
    ]

    grad_F = grad_F[
        valid_state_indices
    ]

    M_all = M_all[
        valid_state_indices
    ]

    return {
        "name": "Projected 66D MEP",
        "npz": ALL_ATOM_MEP_NPZ,
        "cache": cache_path,
        "iteration": source_indices.astype(
            float
        ),
        "raw_coordinate": counts,
        "progress": normalized_progress(
            counts
        ),
        "I_E": np.asarray(
            recomputed_I_E,
            dtype=float,
        ),
        "I_G": np.asarray(
            recomputed_I_G,
            dtype=float,
        ),
        "barrier": np.asarray(
            recomputed_barrier,
            dtype=float,
        ),
        "paths": paths,
        "grad_F": grad_F,
        "M": M_all,
        "profiles": np.asarray(
            recomputed_profiles,
            dtype=float,
        ),
        "final_path_full": paths[-1],
    }


AA_MEP_RESULT = load_projected_66d_cache(
    AA_CACHE_NPZ
)


# ============================================================
# 8b. Override initial points with common initial path
# ============================================================

def load_initial_metrics_from_npz(npz_path):
    """
    Load the three metrics (I_G, I_E, barrier) from the common
    initial path NPZ file.
    """
    with np.load(npz_path, allow_pickle=True) as data:
        # Try multiple possible key names
        for key in ['I_perp_G', 'I_G', 'I_perp_g']:
            if key in data.files:
                I_G_init = float(data[key])
                break
        else:
            raise KeyError(f"Could not find I_perp_G in {npz_path}")

        for key in ['I_perp_E', 'I_E', 'I_perp_e']:
            if key in data.files:
                I_E_init = float(data[key])
                break
        else:
            raise KeyError(f"Could not find I_perp_E in {npz_path}")

        for key in ['Delta_F_barrier', 'barrier', 'DeltaF_barrier']:
            if key in data.files:
                barrier_init = float(data[key])
                break
        else:
            raise KeyError(f"Could not find barrier in {npz_path}")

    return {
        'I_G': I_G_init,
        'I_E': I_E_init,
        'barrier': barrier_init,
    }


init_metrics = load_initial_metrics_from_npz(INIT_NPZ)
print("\nCommon initial metrics loaded:")
print(f"  I_perp_G = {init_metrics['I_G']:.6e}")
print(f"  I_perp_E = {init_metrics['I_E']:.6e}")
print(f"  barrier  = {init_metrics['barrier']:.6f} kcal/mol\n")

# Override the first point for MFEP and CV MEP results
for result in [MFEP_RESULT, CV_MEP_RESULT]:
    # progress[0] is 0.0, so we replace the corresponding metric values
    result['I_G'][0] = init_metrics['I_G']
    result['I_E'][0] = init_metrics['I_E']
    result['barrier'][0] = init_metrics['barrier']
    print(f"Initial point of {result['name']} set to common initial path values.")


# ============================================================
# 9. Print loaded sequence information
# ============================================================

print()
print("=" * 100)
print("Loaded optimization sequences")
print("=" * 100)

for result in [
    MFEP_RESULT,
    CV_MEP_RESULT,
    AA_MEP_RESULT,
]:
    print(
        f"{result['name']:>25} | "
        f"states={len(result['progress']):>4d} | "
        f"raw start={result['raw_coordinate'][0]:>10.2f} | "
        f"raw final={result['raw_coordinate'][-1]:>10.2f} | "
        f"progress=[{result['progress'][0]:.3f}, "
        f"{result['progress'][-1]:.3f}]"
    )

print("=" * 100)


# ============================================================
# 10. Load and crop FES
# ============================================================

with np.load(
    FES_NPZ,
    allow_pickle=True,
) as loaded:
    grid_x = np.asarray(
        loaded["X"],
        dtype=float,
    )

    grid_y = np.asarray(
        loaded["Y"],
        dtype=float,
    )

    grid_z = np.asarray(
        loaded["Z"],
        dtype=float,
    )

if (
    grid_x.ndim == 1
    and grid_y.ndim == 1
):
    grid_x, grid_y = np.meshgrid(
        grid_x,
        grid_y,
        indexing="xy",
    )

if (
    grid_x.shape != grid_y.shape
    or grid_x.shape != grid_z.shape
):
    raise ValueError(
        f"FES shape mismatch: "
        f"X{grid_x.shape}, "
        f"Y{grid_y.shape}, "
        f"Z{grid_z.shape}"
    )

# FES coordinates are stored in degrees.
grid_x = np.deg2rad(
    grid_x
)

grid_y = np.deg2rad(
    grid_y
)

grid_z = np.ma.masked_invalid(
    grid_z
)

inside_window = (
    (grid_x >= XLIM[0])
    & (grid_x <= XLIM[1])
    & (grid_y >= YLIM[0])
    & (grid_y <= YLIM[1])
)

if not np.any(
    inside_window
):
    raise RuntimeError(
        f"No FES data inside XLIM={XLIM}, YLIM={YLIM}."
    )

rows = np.where(
    np.any(
        inside_window,
        axis=1,
    )
)[0]

columns = np.where(
    np.any(
        inside_window,
        axis=0,
    )
)[0]

row_start = rows[0]
row_stop = rows[-1] + 1
column_start = columns[0]
column_stop = columns[-1] + 1

X_contour = grid_x[
    row_start:row_stop,
    column_start:column_stop,
]

Y_contour = grid_y[
    row_start:row_stop,
    column_start:column_stop,
]

Z_contour = grid_z[
    row_start:row_stop,
    column_start:column_stop,
]

# Set minimum energy in the displayed window to zero.
Z_contour = (
    Z_contour
    - np.ma.min(
        Z_contour
    )
)


# ============================================================
# 11. Tuned black-to-white colormap
# ============================================================

unit_values = np.linspace(
    0.0,
    1.0,
    256,
)

mapped_values = (
    unit_values
    + CURV
    * unit_values
    * (
        1.0
        - unit_values
    )
)

mapped_values = np.clip(
    mapped_values,
    0.0,
    1.0,
)

gray_values = (
    BG_LO
    + (
        1.0
        - BG_LO
    )
    * mapped_values
)

BACKGROUND_CMAP = LinearSegmentedColormap.from_list(
    "dark_to_white_quad",
    list(
        zip(
            gray_values,
            gray_values,
            gray_values,
        )
    ),
)

BACKGROUND_CMAP.set_over(
    "white"
)

contour_levels = np.linspace(
    0.0,
    ENERGY_WHITE_CAP,
    CONTOUR_LEVELS,
)


# ============================================================
# 12. Nature / Science plotting style
# ============================================================

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "mathtext.fontset": "dejavusans",
        "axes.linewidth": 1.1,
        "axes.edgecolor": "#222222",
        "axes.labelcolor": "#111111",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "text.color": "#111111",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.size": 2.6,
        "ytick.minor.size": 2.6,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 13,
        "legend.frameon": False,
        "savefig.dpi": 400,
        "figure.dpi": 120,
    }
)


# ============================================================
# 13. Plot helpers
# ============================================================

FLOAT_TINY = np.finfo(
    float
).tiny


def safe_positive(
    values,
):
    """
    Make values safe for log-scale plotting.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if positive_values.size > 0:
        floor_value = max(
            FLOAT_TINY,
            float(
                np.min(
                    positive_values
                )
            )
            * 1e-8,
        )
    else:
        floor_value = FLOAT_TINY

    return np.where(
        np.isfinite(values)
        & (values > 0.0),
        values,
        floor_value,
    )


def style_log_axis(
    axis,
):
    axis.set_yscale(
        "log"
    )

    axis.yaxis.set_major_locator(
        LogLocator(
            base=10.0,
            numticks=12,
        )
    )

    axis.yaxis.set_minor_locator(
        LogLocator(
            base=10.0,
            subs=np.arange(
                2,
                10,
            )
            * 0.1,
            numticks=100,
        )
    )

    axis.yaxis.set_minor_formatter(
        NullFormatter()
    )


def style_metric_axis(
    axis,
    use_log_scale,
):
    if use_log_scale:
        style_log_axis(
            axis
        )
    else:
        axis.minorticks_on()

    axis.spines["top"].set_visible(
        False
    )

    axis.spines["right"].set_visible(
        False
    )

    axis.tick_params(
        which="both",
        top=False,
        right=False,
    )

    axis.grid(
        True,
        which="major",
        axis="y",
        linestyle="-",
        linewidth=0.6,
        alpha=0.18,
        color="#666666",
    )

    # All metric panels use exactly the same x range.
    axis.set_xlim(
        0.0,
        1.0,
    )

    axis.set_xticks(
        np.linspace(
            0.0,
            1.0,
            6,
        )
    )


def plot_metric_curve(
    axis,
    result,
    metric_key,
    color,
    marker,
    label,
    use_log_scale,
    zorder,
):
    """
    Plot one metric against normalized progress.
    """
    progress = np.asarray(
        result["progress"],
        dtype=float,
    )

    metric_values = np.asarray(
        result[metric_key],
        dtype=float,
    )

    number_common = min(
        progress.size,
        metric_values.size,
    )

    progress = progress[
        :number_common
    ]

    metric_values = metric_values[
        :number_common
    ]

    valid = (
        np.isfinite(progress)
        & np.isfinite(metric_values)
    )

    progress = progress[
        valid
    ]

    metric_values = metric_values[
        valid
    ]

    if progress.size == 0:
        return

    order = np.argsort(
        progress
    )

    progress = progress[
        order
    ]

    metric_values = metric_values[
        order
    ]

    if use_log_scale:
        plotted_values = safe_positive(
            metric_values
        )
    else:
        plotted_values = metric_values

    mark_every = max(
        1,
        progress.size // 10,
    )

    axis.plot(
        progress,
        plotted_values,
        color=color,
        linestyle="-",
        linewidth=2.4,
        marker=marker,
        markevery=mark_every,
        markersize=6.2,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.5,
        solid_capstyle="round",
        label=label,
        zorder=zorder,
        clip_on=False,
    )

    # Highlight final point.
    axis.scatter(
        progress[-1],
        plotted_values[-1],
        s=72,
        marker=marker,
        facecolor=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=zorder + 5,
        clip_on=False,
    )


# ============================================================
# 14. Result-series definitions
# ============================================================

RESULT_SERIES = [
    {
        "result": MFEP_RESULT,
        "color": COLOR_MFEP,
        "marker": MARKER_MFEP,
        "label": r"MFEP-IVM ($G=M^{-1}$)",
        "zorder": 7,
    },
    {
        "result": CV_MEP_RESULT,
        "color": COLOR_CV_MEP,
        "marker": MARKER_CV_MEP,
        "label": r"CV MEP-IVM ($G=I$)",
        "zorder": 6,
    },
    {
        "result": AA_MEP_RESULT,
        "color": COLOR_66D_MEP,
        "marker": MARKER_66D_MEP,
        "label": r"Projected 66D MEP",
        "zorder": 5,
    },
]


# ============================================================
# 15. Draw the three-panel figure
# ============================================================

figure, axes = plt.subplots(
    1,
    3,
    figsize=(
        15.5,
        4.8,
    ),
)

axis_I_G = axes[0]
axis_I_E = axes[1]
axis_barrier = axes[2]


# ------------------------------------------------------------
# Panel (a): MFEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_I_G,
        result=series["result"],
        metric_key="I_G",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=True,
        zorder=series["zorder"],
    )

axis_I_G.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_I_G.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,G}$",
    labelpad=7,
)

axis_I_G.set_title(
    "MFEP residual",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_I_G,
    use_log_scale=True,
)


# ------------------------------------------------------------
# Panel (b): Euclidean MEP residual
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_I_E,
        result=series["result"],
        metric_key="I_E",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=True,
        zorder=series["zorder"],
    )

axis_I_E.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_I_E.set_ylabel(
    r"$\widehat{I}_{\perp}^{\,\mathrm{E}}$",
    labelpad=7,
)

axis_I_E.set_title(
    "MEP residual",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_I_E,
    use_log_scale=True,
)


# ------------------------------------------------------------
# Panel (c): free-energy barrier
# ------------------------------------------------------------

for series in RESULT_SERIES:
    plot_metric_curve(
        axis=axis_barrier,
        result=series["result"],
        metric_key="barrier",
        color=series["color"],
        marker=series["marker"],
        label=series["label"],
        use_log_scale=False,
        zorder=series["zorder"],
    )

axis_barrier.set_xlabel(
    "Normalized optimization progress",
    labelpad=6,
)

axis_barrier.set_ylabel(
    r"$\Delta F^{\dagger}$ (kcal mol$^{-1}$)",
    labelpad=7,
)

axis_barrier.set_title(
    "Free-energy barrier",
    pad=12,
    fontweight="medium",
)

style_metric_axis(
    axis_barrier,
    use_log_scale=False,
)


# ------------------------------------------------------------
# Panel tags
# ------------------------------------------------------------

for axis, panel_tag in zip(
    axes,
    [
        "a",
        "b",
        "c",
    ],
):
    axis.text(
        -0.15,
        1.07,
        panel_tag,
        transform=axis.transAxes,
        fontsize=17,
        fontweight="bold",
        va="top",
        ha="left",
    )


# ============================================================
# 16. Shared legend
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=series["color"],
        linewidth=2.35,
        marker=series["marker"],
        markerfacecolor="white",
        markeredgecolor=series["color"],
        markeredgewidth=1.35,
        label=series["label"],
    )
    for series in RESULT_SERIES
]

figure.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.04,
    ),
    ncol=3,
    fontsize=12.5,
    handlelength=2.2,
    columnspacing=2.2,
    borderaxespad=0.0,
)

figure.subplots_adjust(
    left=0.07,
    right=0.97,
    top=0.81,
    bottom=0.15,
    wspace=0.35,
)


# ============================================================
# 17. Save figure
# ============================================================

figure.savefig(
    OUTPUT_STEM + ".pdf",
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".png",
    dpi=400,
    bbox_inches="tight",
)

figure.savefig(
    OUTPUT_STEM + ".svg",
    bbox_inches="tight",
)

if DISPLAY_FIGURE:
    plt.show()

plt.close(
    figure
)


# ============================================================
# 18. Final-value summary
# ============================================================

print()
print("=" * 118)
print(
    "Final comparison with normalized optimization progress"
)
print("=" * 118)

for series in RESULT_SERIES:
    result = series[
        "result"
    ]

    print(
        f"{result['name']:>25} | "
        f"states={len(result['progress']):>4d} | "
        f"progress={result['progress'][-1]:.3f} | "
        f"I_G={result['I_G'][-1]:.8e} | "
        f"I_E={result['I_E'][-1]:.8e} | "
        f"DeltaF={result['barrier'][-1]:.6f} kcal/mol"
    )

print("-" * 118)


# ============================================================
# 19. Final-path distances
# ============================================================

final_paths = {
    "MFEP": MFEP_RESULT[
        "final_path_full"
    ],
    "CV-MEP": CV_MEP_RESULT[
        "final_path_full"
    ],
    "66D-MEP": AA_MEP_RESULT[
        "final_path_full"
    ],
}

for first_name, second_name in [
    (
        "MFEP",
        "CV-MEP",
    ),
    (
        "MFEP",
        "66D-MEP",
    ),
    (
        "CV-MEP",
        "66D-MEP",
    ),
]:
    first_path = np.asarray(
        final_paths[first_name],
        dtype=float,
    )

    second_path = np.asarray(
        final_paths[second_name],
        dtype=float,
    )

    if first_path.shape == second_path.shape:
        angular_difference = wrap_angles(
            first_path
            - second_path
        )

        pointwise_distance = np.linalg.norm(
            angular_difference,
            axis=1,
        )

        print(
            f"{first_name:>7} vs {second_name:<7} | "
            f"mean distance="
            f"{np.mean(pointwise_distance):.6e} rad | "
            f"max distance="
            f"{np.max(pointwise_distance):.6e} rad"
        )

    else:
        print(
            f"{first_name:>7} vs {second_name:<7} | "
            f"different path shapes: "
            f"{first_path.shape} vs {second_path.shape}; "
            "pointwise distance skipped."
        )

print("-" * 118)
print(f"MFEP result       : {MFEP_NPZ}")
print(f"CV MEP result     : {MEP_NPZ}")
print(f"66D MEP result    : {ALL_ATOM_MEP_NPZ}")
print(f"66D MD cache      : {AA_CACHE_NPZ}")
print(f"Initial path NPZ  : {INIT_NPZ}")
print("=" * 118)
print(f"[SAVED] {OUTPUT_STEM}.pdf")
print(f"[SAVED] {OUTPUT_STEM}.png")
print(f"[SAVED] {OUTPUT_STEM}.svg")

Input files
MFEP-IVM result        : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam/alanine_MFEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.0001_maxstep0.1_hist7_NEBreparam.npz
CV MEP-IVM result      : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam/alanine_MEP_image_LBFGS_B_Recycle_N20_Nsub1_alpha0.03_maxstep0.1_hist7_NEBreparam.npz
66D projected MD cache : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/Ala2_66D_projected_MEP_restrained_MD_cache.npz
66D original result    : /Users/wuzhiyou/Downloads/Ala2_N20_lr0.04_alphaIVP100_Nsub5_results.npz
FES                     : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/wtmeta_phi_psi/fes_XYZ.npz
Initial common path     : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/MFEP_INITIAL_PATH_common.npz

Common initial metrics loaded:
  I_perp_G = 1.902623e+02
  I_perp_E = 1.006851e+01
 